# Predicting Delivery Lateness in an Integrated GRP Pipe Production–Distribution System

**Reproducible research notebook — publication workflow**

This notebook reconstructs the order-level simulation, feature-selection procedure, final 14-predictor dataset, Linear Regression and XGBoost models, process-informed benchmarks, SHAP analysis, and six publication figures.

## Data provenance and interpretation

- The industrial reference configuration is based on company-authorized information for three pipe diameters: **900, 1,000, and 1,200 mm**, with reference quantities of 50, 100, and 40 pipes, respectively.
- The analytical dataset contains **2,000 order-level observations**: one factory-informed reference order and 1,999 simulated orders. Simulated observations must not be described as historical company records.
- The four production stages are represented sequentially. The 2,000 records are order-level simulated scenarios, not a full discrete-event simulation of shared-resource interactions. Warehouse and transportation uncertainty, the 300-km illustrative route, and the synthetic delivery-commitment policy are simulation assumptions rather than verified historical operating records.
- The delivery promise is generated by a transparent synthetic planning rule, not by actual contractual deadlines. Results therefore evaluate prediction under the stated data-generating process; they do not constitute external validation against historical delivery outcomes.
- The prediction point is immediately before dispatch. `pre_dispatch_remaining_slack_h` is mathematically linked to the lateness target; SHAP values are predictive attributions, not causal effects.

## Reproducibility

Run the notebook from top to bottom in a clean Python environment. The random seed is fixed at 2026. The notebook writes publication tables and figures to `Final_Publication_Results/`. Required package versions are reported in the environment cell below and pinned in the accompanying `requirements.txt`. The saved outputs from the source notebook were cleared because they did not consistently correspond to the current source code; the final notebook should be executed to regenerate all results.

**Important configuration checks before public release:** confirm the time unit and exact interpretation of the factory-provided operation start times, confirm the manual job sequence, and replace the destination placeholder if disclosure is permitted. The route distance remains illustrative unless company records establish otherwise.

## 0. Software environment

In [15]:
import sys, importlib.metadata as metadata

print("Python:", sys.version)
for package in ["numpy", "pandas", "scipy", "scikit-learn", "xgboost", "shap", "matplotlib", "openpyxl"]:
    try:
        print(f"{package}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

Python: 3.13.3 (tags/v3.13.3:6280bb5, Apr  8 2025, 14:47:33) [MSC v.1943 64 bit (AMD64)]
numpy: 2.2.6
pandas: 2.3.0
scipy: 1.15.3
scikit-learn: 1.7.0
xgboost: 3.2.0
shap: 0.51.0
matplotlib: 3.10.3
openpyxl: 3.1.5


## 1. Industrial configuration and reference production schedule

In [16]:

# ==============================================================
# CELL 1 — INDUSTRIAL SYSTEM CONFIGURATION
# PIP PIPE MANUFACTURING: FOUR-STAGE FLOW-SHOP SYSTEM
# ==============================================================

import numpy as np
import pandas as pd


# ==============================================================
# 1. REPRODUCIBILITY
# ==============================================================

RANDOM_SEED = 2026

rng = np.random.default_rng(RANDOM_SEED)


# ==============================================================
# 2. REAL PIP MANUFACTURING STRUCTURE
# ==============================================================

# Product specifications supplied for the PIP manufacturing case.

PRODUCT_TYPES = [900, 1000, 1200]

PRODUCT_DIAMETER_UNIT = "mm"

# Four sequential machine stages.
# One machine at each stage.

MACHINE_STAGES = ["M1", "M2", "M3", "M4"]

N_MACHINES_PER_STAGE = {
    "M1": 1,
    "M2": 1,
    "M3": 1,
    "M4": 1
}

N_MACHINE_STAGES = len(MACHINE_STAGES)

FLOW_SHOP_ROUTE = (
    "M1",
    "M2",
    "M3",
    "M4"
)

# Production scheduling characteristics.

SEQUENCING_POLICY = "MANUAL"

TRANSFER_POLICY = "FULL_BATCH"

ALLOW_PREEMPTION = False


# ==============================================================
# 3. CUSTOMER ORDER AND PRODUCTION JOB INFORMATION
# ==============================================================

# The three production jobs are consolidated into one
# customer order after production for delivery to a
# specified border destination.
#
# Customer order ID and destination code are placeholders
# until their actual identifiers are provided.

CUSTOMER_ORDER_ID = "CO-001"

DESTINATION_CODE = "BORDER_DESTINATION_TO_CONFIRM"

# Job order as supplied in the production table.
# The actual manual production sequence must be confirmed
# against the factory's scheduling records.

MANUAL_JOB_SEQUENCE = [1, 2, 3]

jobs_df = pd.DataFrame({

    "customer_order_id": [
        CUSTOMER_ORDER_ID,
        CUSTOMER_ORDER_ID,
        CUSTOMER_ORDER_ID
    ],

    "job_id": [
        1,
        2,
        3
    ],

    "product_diameter_mm": [
        900,
        1000,
        1200
    ],

    "order_quantity": [
        50,
        100,
        40
    ],

    "manual_sequence": [
        1,
        2,
        3
    ],

    "destination_code": [
        DESTINATION_CODE,
        DESTINATION_CODE,
        DESTINATION_CODE
    ]

})


# Consolidated order quantity.

TOTAL_ORDER_QUANTITY = int(
    jobs_df["order_quantity"].sum()
)


# ==============================================================
# 4. ACTUAL JOB–MACHINE OPERATION DATA
# ==============================================================

# Data supplied for the industrial case.
#
# Time unit is assumed to be hours and must be confirmed.
#
# Processing times are recorded at the JOB–MACHINE level.
# They are NOT processing times per individual pipe.
#
# Start times are retained exactly as supplied.
# Their time reference and relationship with setup time
# must be confirmed before scheduling calculations.

TIME_UNIT = "hours (provisional; confirm with factory)"

operation_records = [

    # ----------------------------------------------------------
    # JOB 1 — 50 PIPES, Ø900 mm
    # ----------------------------------------------------------

    {
        "job_id": 1,
        "machine_id": "M1",
        "setup_time": 1.5,
        "start_time": 1.5,
        "processing_time": 12.5
    },

    {
        "job_id": 1,
        "machine_id": "M2",
        "setup_time": 1.0,
        "start_time": 17.43,
        "processing_time": 8.33
    },

    {
        "job_id": 1,
        "machine_id": "M3",
        "setup_time": 0.8,
        "start_time": 28.44,
        "processing_time": 4.17
    },

    {
        "job_id": 1,
        "machine_id": "M4",
        "setup_time": 0.5,
        "start_time": 34.57,
        "processing_time": 1.0
    },


    # ----------------------------------------------------------
    # JOB 2 — 100 PIPES, Ø1000 mm
    # ----------------------------------------------------------

    {
        "job_id": 2,
        "machine_id": "M1",
        "setup_time": 2.0,
        "start_time": 2.0,
        "processing_time": 25.0
    },

    {
        "job_id": 2,
        "machine_id": "M2",
        "setup_time": 1.5,
        "start_time": 30.71,
        "processing_time": 16.4
    },

    {
        "job_id": 2,
        "machine_id": "M3",
        "setup_time": 1.0,
        "start_time": 49.69,
        "processing_time": 8.33
    },

    {
        "job_id": 2,
        "machine_id": "M4",
        "setup_time": 0.8,
        "start_time": 60.04,
        "processing_time": 2.0
    },


    # ----------------------------------------------------------
    # JOB 3 — 40 PIPES, Ø1200 mm
    # ----------------------------------------------------------

    {
        "job_id": 3,
        "machine_id": "M1",
        "setup_time": 2.5,
        "start_time": 2.5,
        "processing_time": 50.0
    },

    {
        "job_id": 3,
        "machine_id": "M2",
        "setup_time": 2.0,
        "start_time": 57.57,
        "processing_time": 33.4
    },

    {
        "job_id": 3,
        "machine_id": "M3",
        "setup_time": 1.5,
        "start_time": 94.4,
        "processing_time": 16.68
    },

    {
        "job_id": 3,
        "machine_id": "M4",
        "setup_time": 1.0,
        "start_time": 113.9,
        "processing_time": 4.0
    }

]


job_operation_df = pd.DataFrame(
    operation_records
)

# Attach product and order information to each operation.

job_operation_df = job_operation_df.merge(
    jobs_df[
        [
            "customer_order_id",
            "job_id",
            "product_diameter_mm",
            "order_quantity",
            "manual_sequence"
        ]
    ],
    on="job_id",
    how="left",
    validate="many_to_one"
)

job_operation_df = job_operation_df[
    [
        "customer_order_id",
        "job_id",
        "product_diameter_mm",
        "order_quantity",
        "manual_sequence",
        "machine_id",
        "setup_time",
        "start_time",
        "processing_time"
    ]
]


# ==============================================================
# 5. CONFIGURATION OF REMAINING OPERATIONAL PARAMETERS
# ==============================================================

# These parameters are retained from the original DGP.
# They are illustrative assumptions until calibrated using
# additional industrial observations.

# Production operating conditions.

UTIL_MIN = 55.0
UTIL_MAX = 98.0

AVAIL_MIN = 80.0
AVAIL_MAX = 100.0

QUEUE_MAX = 15

QUEUE_THRESHOLD = 8

UTIL_THRESHOLD = 85.0


# --------------------------------------------------------------
# Warehouse conditions
# --------------------------------------------------------------

INVENTORY_MIN = 25.0
INVENTORY_MAX = 150.0

WAREHOUSE_PRESSURE_MIN = 0.30
WAREHOUSE_PRESSURE_MAX = 1.50

WAREHOUSE_PRESSURE_THRESHOLD = 1.00

STORAGE_MIN = 1.5
STORAGE_MAX = 16.0

LOADING_MIN = 0.6
LOADING_MAX = 5.0


# --------------------------------------------------------------
# Transportation conditions
# --------------------------------------------------------------

DISTANCE_MIN = 30.0
DISTANCE_MAX = 750.0

TRAFFIC_LEVELS = np.array([1, 2, 3, 4, 5])

TRAFFIC_PROBABILITIES = np.array([
    0.15,
    0.25,
    0.30,
    0.20,
    0.10
])

WEATHER_LEVELS = np.array([0, 1, 2, 3])

WEATHER_PROBABILITIES = np.array([
    0.70,
    0.20,
    0.08,
    0.02
])

ROAD_LEVELS = np.array([1, 2, 3, 4, 5])

ROAD_PROBABILITIES = np.array([
    0.15,
    0.25,
    0.30,
    0.20,
    0.10
])


# --------------------------------------------------------------
# Stochastic disruption probabilities
# --------------------------------------------------------------

# These are preliminary event probabilities.
# The observation interval must be specified and calibrated.

PRODUCTION_DISRUPTION_PROB = 0.08

WAREHOUSE_DISRUPTION_PROB = 0.06

TRANSPORT_DISRUPTION_PROB = 0.07


# --------------------------------------------------------------
# Stochastic variability
# --------------------------------------------------------------

# All noise standard deviations are provisionally expressed
# in hours where they apply to time-related outcomes.

PRODUCTION_NOISE_SD = 1.5

WAREHOUSE_NOISE_SD = 1.2

TRANSPORT_NOISE_SD = 1.5

TOTAL_NOISE_SD = 1.5


# ==============================================================
# 6. PARAMETER PROVENANCE
# ==============================================================

PARAMETER_PROVENANCE = {

    "Product diameters and quantities":
        "User-supplied company information",

    "Four machine stages":
        "User-supplied company information",

    "One machine per stage":
        "Confirmed system configuration",

    "Manual production sequencing":
        "Confirmed system configuration; exact sequence needs verification",

    "Full-batch transfer":
        "Confirmed system configuration",

    "Job-machine setup times":
        "User-supplied operational information",

    "Job-machine start times":
        "User-supplied operational information; time reference needs verification",

    "Job-machine processing times":
        "User-supplied operational information",

    "Warehouse operating parameters":
        "Illustrative simulation assumptions",

    "Transportation distributions":
        "Illustrative simulation assumptions",

    "Disruption probabilities":
        "Illustrative simulation assumptions",

    "Noise parameters":
        "Illustrative simulation assumptions"

}


# ==============================================================
# 7. CONFIGURATION VALIDATION
# ==============================================================

def validate_industrial_configuration():

    # Product configuration.

    assert PRODUCT_TYPES == [900, 1000, 1200]

    assert len(MACHINE_STAGES) == 4

    assert all(
        N_MACHINES_PER_STAGE[m] == 1
        for m in MACHINE_STAGES
    )

    assert FLOW_SHOP_ROUTE == (
        "M1", "M2", "M3", "M4"
    )

    # Job information.

    assert len(jobs_df) == 3

    assert jobs_df["job_id"].is_unique

    assert (
        jobs_df["product_diameter_mm"]
        .isin(PRODUCT_TYPES)
        .all()
    )

    assert (
        jobs_df["order_quantity"] > 0
    ).all()

    assert jobs_df["manual_sequence"].is_unique

    assert (
        jobs_df["order_quantity"].sum()
        == TOTAL_ORDER_QUANTITY
    )

    # Operation records.

    assert len(job_operation_df) == (
        len(jobs_df) * len(MACHINE_STAGES)
    )

    assert not job_operation_df.duplicated(
        ["job_id", "machine_id"]
    ).any()

    assert set(
        job_operation_df["machine_id"]
    ) == set(MACHINE_STAGES)

    assert set(
        job_operation_df["job_id"]
    ) == set(jobs_df["job_id"])

    assert (
        job_operation_df["setup_time"] >= 0
    ).all()

    assert (
        job_operation_df["start_time"] >= 0
    ).all()

    assert (
        job_operation_df["processing_time"] > 0
    ).all()

    # Warehouse and production ranges.

    assert 0 <= UTIL_MIN <= UTIL_MAX <= 100

    assert 0 <= AVAIL_MIN <= AVAIL_MAX <= 100

    assert 0 < STORAGE_MIN <= STORAGE_MAX

    assert 0 < LOADING_MIN <= LOADING_MAX

    assert (
        WAREHOUSE_PRESSURE_MIN
        <= WAREHOUSE_PRESSURE_THRESHOLD
        <= WAREHOUSE_PRESSURE_MAX
    )

    # Transportation probabilities.

    probability_definitions = [
        (TRAFFIC_LEVELS, TRAFFIC_PROBABILITIES),
        (WEATHER_LEVELS, WEATHER_PROBABILITIES),
        (ROAD_LEVELS, ROAD_PROBABILITIES)
    ]

    for levels, probabilities in probability_definitions:

        assert len(levels) == len(probabilities)

        assert np.all(probabilities >= 0)

        assert np.isclose(
            probabilities.sum(),
            1.0
        )

    # Disruption probabilities.

    assert all(
        0 <= p <= 1
        for p in [
            PRODUCTION_DISRUPTION_PROB,
            WAREHOUSE_DISRUPTION_PROB,
            TRANSPORT_DISRUPTION_PROB
        ]
    )

    return True



# ==============================================================
# ADDITION TO CELL 1 — OPERATION TIME DEFINITIONS
# ==============================================================

TIME_UNIT = "hours"

SETUP_TIME_DEFINITION = (
    "Duration required to prepare the machine before processing."
)

START_TIME_DEFINITION = (
    "Actual processing start time, after machine setup."
)

PROCESSING_TIME_DEFINITION = (
    "Total processing duration for the complete production job "
    "at the specified machine stage."
)

SETUP_START_DEFINITION = (
    "Processing start time minus setup duration."
)

COMPLETION_TIME_DEFINITION = (
    "Processing start time plus processing duration."
)

SETUP_INCLUDED_IN_PROCESSING_TIME = False

START_TIME_REFERENCE = (
    "Common production schedule time origin; requires confirmation."
)


# ==============================================================
# 8. DISPLAY INDUSTRIAL CONFIGURATION
# ==============================================================

validate_industrial_configuration()

print("=" * 80)
print("PIP MANUFACTURING — INDUSTRIAL SYSTEM CONFIGURATION")
print("=" * 80)

print("\nCUSTOMER ORDER:")
print("Customer order ID:", CUSTOMER_ORDER_ID)
print("Destination:", DESTINATION_CODE)

print("\nPRODUCTION JOBS:")

print(
    jobs_df.to_string(index=False)
)

print(
    "\nTotal consolidated quantity:",
    TOTAL_ORDER_QUANTITY,
    "pipes"
)

print("\nPRODUCTION ROUTE:")
print(" -> ".join(FLOW_SHOP_ROUTE))

print("\nMACHINE CONFIGURATION:")
print(N_MACHINES_PER_STAGE)

print("\nSEQUENCING POLICY:")
print(SEQUENCING_POLICY)

print("\nTRANSFER POLICY:")
print(TRANSFER_POLICY)

print("\nJOB-MACHINE OPERATION DATA:")

print(
    job_operation_df[
        [
            "job_id",
            "product_diameter_mm",
            "order_quantity",
            "machine_id",
            "setup_time",
            "start_time",
            "processing_time"
        ]
    ].to_string(index=False)
)

print("\nPARAMETER PROVENANCE:")

for parameter, source in PARAMETER_PROVENANCE.items():

    print(f"{parameter}: {source}")

print("\n" + "=" * 80)
print("INDUSTRIAL CONFIGURATION VALIDATION: PASSED")
print("=" * 80)


PIP MANUFACTURING — INDUSTRIAL SYSTEM CONFIGURATION

CUSTOMER ORDER:
Customer order ID: CO-001
Destination: BORDER_DESTINATION_TO_CONFIRM

PRODUCTION JOBS:
customer_order_id  job_id  product_diameter_mm  order_quantity  manual_sequence              destination_code
           CO-001       1                  900              50                1 BORDER_DESTINATION_TO_CONFIRM
           CO-001       2                 1000             100                2 BORDER_DESTINATION_TO_CONFIRM
           CO-001       3                 1200              40                3 BORDER_DESTINATION_TO_CONFIRM

Total consolidated quantity: 190 pipes

PRODUCTION ROUTE:
M1 -> M2 -> M3 -> M4

MACHINE CONFIGURATION:
{'M1': 1, 'M2': 1, 'M3': 1, 'M4': 1}

SEQUENCING POLICY:
MANUAL

TRANSFER POLICY:
FULL_BATCH

JOB-MACHINE OPERATION DATA:
 job_id  product_diameter_mm  order_quantity machine_id  setup_time  start_time  processing_time
      1                  900              50         M1         1.5        1.50  

In [17]:

# ==============================================================
# CELL 2 — CUSTOMER ORDER AND PRODUCTION OPERATION DATA
# PIP MANUFACTURING: THREE JOBS, FOUR MACHINE STAGES
# ==============================================================


# ==============================================================
# 1. CONSOLIDATED CUSTOMER ORDER
# ==============================================================

customer_order_df = pd.DataFrame({

    "customer_order_id": [CUSTOMER_ORDER_ID],

    "destination_code": [DESTINATION_CODE],

    "number_of_jobs": [len(jobs_df)],

    "total_quantity_pipes": [
        jobs_df["order_quantity"].sum()
    ],

    "number_of_machine_stages": [
        N_MACHINE_STAGES
    ],

    "sequencing_policy": [SEQUENCING_POLICY],

    "transfer_policy": [TRANSFER_POLICY]

})


# ==============================================================
# 2. PRODUCTION JOBS
# ==============================================================

production_jobs_df = (
    jobs_df
    .sort_values("manual_sequence")
    .reset_index(drop=True)
    .copy()
)

assert production_jobs_df["job_id"].is_unique

assert production_jobs_df["manual_sequence"].is_unique

assert production_jobs_df["product_diameter_mm"].isin(
    PRODUCT_TYPES
).all()

assert production_jobs_df["order_quantity"].sum() == 190

assert production_jobs_df["customer_order_id"].nunique() == 1


# ==============================================================
# 3. MACHINE ROUTING
# ==============================================================

machine_route_df = pd.DataFrame({

    "machine_id": MACHINE_STAGES,

    "stage_sequence": range(1, N_MACHINE_STAGES + 1),

    "machines_available": [
        N_MACHINES_PER_STAGE[m]
        for m in MACHINE_STAGES
    ]

})


# ==============================================================
# 4. JOB–MACHINE OPERATION DATA
# ==============================================================

production_operations_df = (
    job_operation_df.copy()
)

machine_stage_mapping = {

    machine: sequence

    for sequence, machine in enumerate(
        MACHINE_STAGES,
        start=1
    )

}

production_operations_df["stage_sequence"] = (
    production_operations_df["machine_id"].map(
        machine_stage_mapping
    )
)

production_operations_df = (
    production_operations_df
    .sort_values(
        ["manual_sequence", "stage_sequence"]
    )
    .reset_index(drop=True)
)


# ==============================================================
# 5. CALCULATE SETUP START AND PROCESSING COMPLETION
# ==============================================================

# Start_time means actual PROCESSING START.
#
# Therefore:
#
# Setup start = Processing start - Setup duration
#
# Processing completion = Processing start + Processing duration

production_operations_df["setup_start_time_h"] = (

    production_operations_df["start_time"]

    - production_operations_df["setup_time"]

)

production_operations_df["processing_start_time_h"] = (
    production_operations_df["start_time"]
)

production_operations_df["processing_completion_time_h"] = (

    production_operations_df["start_time"]

    + production_operations_df["processing_time"]

)

# Machine occupation includes both setup and processing.

production_operations_df["machine_occupied_start_h"] = (
    production_operations_df["setup_start_time_h"]
)

production_operations_df["machine_occupied_end_h"] = (
    production_operations_df["processing_completion_time_h"]
)


# ==============================================================
# 6. VERIFY JOB–MACHINE OPERATION COMPLETENESS
# ==============================================================

operations_per_job = (

    production_operations_df

    .groupby("job_id")

    .agg(

        operation_count=("machine_id", "count"),

        machine_count=("machine_id", "nunique")

    )

)

assert (
    operations_per_job["operation_count"]
    == N_MACHINE_STAGES
).all()

assert (
    operations_per_job["machine_count"]
    == N_MACHINE_STAGES
).all()

assert len(production_operations_df) == 12


# ==============================================================
# 7. CHECK WITHIN-JOB PRODUCTION PRECEDENCE
# ==============================================================

# Each job must complete its previous machine operation
# before processing can begin at the next machine.
#
# Setup is allowed to be prepared before the job arrives
# only if the factory's setup policy permits it.
#
# This initial check evaluates processing precedence.

precedence_records = []

for job_id in production_jobs_df["job_id"]:

    job_operations = (

        production_operations_df.loc[
            production_operations_df["job_id"] == job_id
        ]

        .sort_values("stage_sequence")

        .reset_index(drop=True)

    )

    for j in range(1, len(job_operations)):

        previous_operation = job_operations.iloc[j - 1]

        current_operation = job_operations.iloc[j]

        previous_completion = (
            previous_operation[
                "processing_completion_time_h"
            ]
        )

        current_processing_start = (
            current_operation[
                "processing_start_time_h"
            ]
        )

        precedence_satisfied = (
            current_processing_start >= previous_completion
        )

        precedence_records.append({

            "job_id": job_id,

            "previous_machine": previous_operation["machine_id"],

            "current_machine": current_operation["machine_id"],

            "previous_completion_h": previous_completion,

            "current_processing_start_h": current_processing_start,

            "precedence_satisfied": precedence_satisfied

        })

precedence_df = pd.DataFrame(precedence_records)


# ==============================================================
# 8. CHECK MACHINE RESOURCE CONFLICTS
# ==============================================================

# With one machine per stage, setup and processing intervals
# belonging to different jobs must not overlap.
#
# Occupation interval:
# [setup_start_time, processing_completion_time]

overlap_records = []

for machine in MACHINE_STAGES:

    machine_operations = (

        production_operations_df.loc[
            production_operations_df["machine_id"] == machine
        ]

        .sort_values("machine_occupied_start_h")

        .reset_index(drop=True)

    )

    for i in range(len(machine_operations)):

        for j in range(i + 1, len(machine_operations)):

            first = machine_operations.iloc[i]

            second = machine_operations.iloc[j]

            overlap_start = max(
                first["machine_occupied_start_h"],
                second["machine_occupied_start_h"]
            )

            overlap_end = min(
                first["machine_occupied_end_h"],
                second["machine_occupied_end_h"]
            )

            if overlap_start < overlap_end:

                overlap_records.append({

                    "machine_id": machine,

                    "job_1": first["job_id"],

                    "job_2": second["job_id"],

                    "job_1_occupied_start_h":
                        first["machine_occupied_start_h"],

                    "job_1_occupied_end_h":
                        first["machine_occupied_end_h"],

                    "job_2_occupied_start_h":
                        second["machine_occupied_start_h"],

                    "job_2_occupied_end_h":
                        second["machine_occupied_end_h"],

                    "overlap_duration_h":
                        overlap_end - overlap_start

                })

machine_conflicts_df = pd.DataFrame(
    overlap_records
)


# ==============================================================
# 9. CANDIDATE JOB COMPLETION TIMES
# ==============================================================

job_completion_df = (

    production_operations_df.loc[
        production_operations_df["machine_id"] == "M4"
    ]

    [[
        "job_id",
        "product_diameter_mm",
        "order_quantity",
        "start_time",
        "processing_time",
        "processing_completion_time_h"
    ]]

    .copy()

)

job_completion_df = job_completion_df.rename({

    "start_time": "M4_processing_start_h",

    "processing_time": "M4_processing_duration_h"

}, axis=1)


# ==============================================================
# 10. CANDIDATE CONSOLIDATED ORDER READINESS
# ==============================================================

# All three jobs must complete M4 before the customer order
# can be considered ready for warehouse dispatch.

candidate_order_ready_time_h = (

    job_completion_df[
        "processing_completion_time_h"
    ].max()

)

order_readiness_df = pd.DataFrame({

    "customer_order_id": [CUSTOMER_ORDER_ID],

    "total_quantity_pipes": [190],

    "number_of_completed_jobs_required": [3],

    "candidate_production_ready_time_h": [
        candidate_order_ready_time_h
    ]

})


# ==============================================================
# 11. OVERALL RESOURCE FEASIBILITY
# ==============================================================

precedence_feasible = (
    precedence_df["precedence_satisfied"].all()
)

machine_feasible = machine_conflicts_df.empty

observed_schedule_feasible = (
    precedence_feasible and machine_feasible
)


# ==============================================================
# 12. DISPLAY RESULTS
# ==============================================================

print("=" * 80)

print("PIP MANUFACTURING — CUSTOMER ORDER AND OPERATION DATA")

print("=" * 80)

print("\nCONSOLIDATED CUSTOMER ORDER:")

print(
    customer_order_df.to_string(index=False)
)

print("\nPRODUCTION JOBS:")

print(
    production_jobs_df[
        [
            "job_id",
            "product_diameter_mm",
            "order_quantity",
            "manual_sequence"
        ]
    ].to_string(index=False)
)

print("\nJOB–MACHINE OPERATION TIMES:")

print(
    production_operations_df[
        [
            "job_id",
            "product_diameter_mm",
            "machine_id",
            "setup_time",
            "setup_start_time_h",
            "processing_start_time_h",
            "processing_time",
            "processing_completion_time_h"
        ]
    ].round(3).to_string(index=False)
)


print("\nWITHIN-JOB PRECEDENCE CHECK:")

print(
    precedence_df.round(3).to_string(index=False)
)


print("\nMACHINE RESOURCE CONFLICTS:")

if machine_conflicts_df.empty:

    print("No machine overlaps detected.")

else:

    print(
        machine_conflicts_df.round(3).to_string(index=False)
    )


print("\nCANDIDATE CUSTOMER ORDER READINESS:")

print(
    order_readiness_df.to_string(index=False)
)


print("\nOVERALL OBSERVED-SCHEDULE STATUS:")

if observed_schedule_feasible:

    print("FEASIBLE under the defined machine constraints.")

else:

    print(
        "WARNING: The supplied timestamps do not define "
        "a feasible schedule for the specified one-machine "
        "per-stage configuration."
    )


# ==============================================================
# 13. DATA VALIDATION
# ==============================================================

assert len(customer_order_df) == 1

assert len(production_jobs_df) == 3

assert len(production_operations_df) == 12

assert production_jobs_df["order_quantity"].sum() == 190

assert production_operations_df[
    "processing_time"
].gt(0).all()

assert production_operations_df[
    "setup_time"
].ge(0).all()

assert np.isfinite(
    production_operations_df[
        [
            "setup_time",
            "start_time",
            "processing_time",
            "setup_start_time_h",
            "processing_completion_time_h"
        ]
    ].to_numpy()
).all()

assert precedence_feasible


print("\n✓ Customer order structure validated.")
print("✓ Three production jobs validated.")
print("✓ Twelve machine operations validated.")
print("✓ Setup and processing definitions applied.")
print("✓ Within-job precedence checked.")
print("✓ Single-machine resource conflicts checked.")

print("\nCell 2 — operation data verification completed.")


PIP MANUFACTURING — CUSTOMER ORDER AND OPERATION DATA

CONSOLIDATED CUSTOMER ORDER:
customer_order_id              destination_code  number_of_jobs  total_quantity_pipes  number_of_machine_stages sequencing_policy transfer_policy
           CO-001 BORDER_DESTINATION_TO_CONFIRM               3                   190                         4            MANUAL      FULL_BATCH

PRODUCTION JOBS:
 job_id  product_diameter_mm  order_quantity  manual_sequence
      1                  900              50                1
      2                 1000             100                2
      3                 1200              40                3

JOB–MACHINE OPERATION TIMES:
 job_id  product_diameter_mm machine_id  setup_time  setup_start_time_h  processing_start_time_h  processing_time  processing_completion_time_h
      1                  900         M1         1.5                0.00                     1.50            12.50                         14.00
      1                  900         M2 

In [18]:

# ==============================================================
# CELL 3 — SERIAL FOUR-STAGE FLOW-SHOP PRODUCTION SCHEDULING
# PIP MANUFACTURING: ONE JOB COMPLETES BEFORE THE NEXT STARTS
# ==============================================================


# ==============================================================
# 1. SCHEDULING CONFIGURATION
# ==============================================================

ORDER_RELEASE_TIME_H = 0.0

SETUP_INCLUDED_IN_PROCESSING_TIME = False

# Recorded processing start times are measured relative to
# the beginning of EACH individual job.

START_TIME_REFERENCE = "JOB_RELATIVE"

# Jobs are processed sequentially through all four machines.

NO_INTER_JOB_OVERLAP = True


# ==============================================================
# 2. SERIAL PRODUCTION SCHEDULING FUNCTION
# ==============================================================

def simulate_serial_flowshop(
    jobs,
    operations,
    machine_stages,
    order_release_time=0.0,
    setup_included_in_processing=False
):
    """
    Serial four-stage flow-shop simulation.

    Industrial operating policy:
    - One machine per stage.
    - One consolidated customer order.
    - Three product-specific production jobs.
    - Manual sequence.
    - Full-batch transfer.
    - Each job completes ALL four stages before
      the next job enters the production line.
    - Recorded start times are job-relative processing starts.
    - Setup and processing durations are separate.
    - No inter-job overlap is permitted.

    The supplied within-job timing pattern is preserved.
    Each subsequent job receives a global time offset equal
    to the previous job's global completion time.
    """

    # ----------------------------------------------------------
    # A. Prepare and validate inputs
    # ----------------------------------------------------------

    jobs = (
        jobs
        .sort_values("manual_sequence")
        .reset_index(drop=True)
        .copy()
    )

    operations = operations.copy()

    required_job_columns = [
        "job_id",
        "customer_order_id",
        "product_diameter_mm",
        "order_quantity",
        "manual_sequence"
    ]

    required_operation_columns = [
        "job_id",
        "machine_id",
        "setup_time",
        "processing_time",
        "start_time"
    ]

    for column in required_job_columns:

        if column not in jobs.columns:

            raise ValueError(
                f"Missing job column: {column}"
            )

    for column in required_operation_columns:

        if column not in operations.columns:

            raise ValueError(
                f"Missing operation column: {column}"
            )

    if jobs["job_id"].duplicated().any():

        raise ValueError(
            "Job IDs must be unique."
        )

    if jobs["manual_sequence"].duplicated().any():

        raise ValueError(
            "Manual sequence values must be unique."
        )

    if operations.duplicated(
        ["job_id", "machine_id"]
    ).any():

        raise ValueError(
            "Duplicate job-machine operations detected."
        )

    for column in [
        "setup_time",
        "processing_time",
        "start_time"
    ]:

        operations[column] = pd.to_numeric(
            operations[column],
            errors="raise"
        )

    if (operations["setup_time"] < 0).any():

        raise ValueError(
            "Setup durations cannot be negative."
        )

    if (operations["processing_time"] <= 0).any():

        raise ValueError(
            "Processing durations must be positive."
        )

    if (operations["start_time"] < 0).any():

        raise ValueError(
            "Job-relative processing start times cannot be negative."
        )

    # ----------------------------------------------------------
    # B. Confirm all jobs follow the four-stage route
    # ----------------------------------------------------------

    for job_id in jobs["job_id"]:

        actual_route = operations.loc[
            operations["job_id"] == job_id,
            "machine_id"
        ].tolist()

        if (
            len(actual_route) != len(machine_stages)
            or set(actual_route) != set(machine_stages)
        ):

            raise ValueError(
                f"Job {job_id} does not have all four "
                "machine operations."
            )

    # ----------------------------------------------------------
    # C. Prepare operation lookup
    # ----------------------------------------------------------

    operation_lookup = {

        (row["job_id"], row["machine_id"]): row

        for _, row in operations.iterrows()

    }

    schedule_records = []

    job_summary_records = []

    machine_busy_time = {
        machine: 0.0
        for machine in machine_stages
    }

    # First job starts at the consolidated order release time.

    next_job_global_release = float(
        order_release_time
    )

    previous_job_global_completion = None

    # ==========================================================
    # 3. PROCESS JOBS SEQUENTIALLY
    # ==============================================================

    for _, job in jobs.iterrows():

        job_id = job["job_id"]

        product = job["product_diameter_mm"]

        quantity = job["order_quantity"]

        manual_sequence = job["manual_sequence"]

        # ------------------------------------------------------
        # Global release time of the current job
        # ------------------------------------------------------

        job_global_release = next_job_global_release

        local_completion_by_machine = {}

        local_setup_start_by_machine = {}

        job_operation_records = []

        # ------------------------------------------------------
        # Execute the four stages for the current job
        # ------------------------------------------------------

        for stage_index, machine in enumerate(machine_stages):

            operation = operation_lookup[
                (job_id, machine)
            ]

            setup_duration = float(
                operation["setup_time"]
            )

            processing_duration = float(
                operation["processing_time"]
            )

            # Actual processing-start time recorded by factory,
            # expressed relative to this job's own release.

            local_processing_start = float(
                operation["start_time"]
            )

            # Setup is completed before processing begins.

            applied_setup = (
                0.0
                if setup_included_in_processing
                else setup_duration
            )

            local_setup_start = (
                local_processing_start
                - applied_setup
            )

            local_processing_completion = (
                local_processing_start
                + processing_duration
            )

            # --------------------------------------------------
            # Within-job stage precedence
            # --------------------------------------------------

            if stage_index == 0:

                previous_stage_completion = 0.0

            else:

                previous_machine = (
                    machine_stages[stage_index - 1]
                )

                previous_stage_completion = (
                    local_completion_by_machine[
                        previous_machine
                    ]
                )

            if (
                local_setup_start
                < previous_stage_completion - 1e-9
            ):

                raise ValueError(

                    f"Within-job precedence violation: "
                    f"Job {job_id}, Machine {machine}. "
                    f"Setup starts before the previous "
                    f"stage finishes."
                )

            # Time between upstream completion and setup start.
            # This is an interstage gap, not automatically a
            # measured queue delay.

            interstage_gap = max(
                0.0,
                local_setup_start
                - previous_stage_completion
            )

            # --------------------------------------------------
            # Convert job-relative times into global times
            # --------------------------------------------------

            global_setup_start = (
                job_global_release
                + local_setup_start
            )

            global_processing_start = (
                job_global_release
                + local_processing_start
            )

            global_completion = (
                job_global_release
                + local_processing_completion
            )

            global_upstream_ready = (
                job_global_release
                + previous_stage_completion
            )

            global_interstage_gap = (
                interstage_gap
            )

            # --------------------------------------------------
            # Save machine and job state
            # --------------------------------------------------

            local_completion_by_machine[
                machine
            ] = local_processing_completion

            local_setup_start_by_machine[
                machine
            ] = local_setup_start

            machine_busy_duration = (
                applied_setup + processing_duration
            )

            machine_busy_time[machine] += (
                machine_busy_duration
            )

            job_operation_records.append({

                "customer_order_id":
                    job["customer_order_id"],

                "job_id":
                    job_id,

                "product_diameter_mm":
                    product,

                "order_quantity":
                    quantity,

                "manual_sequence":
                    manual_sequence,

                "machine_id":
                    machine,

                "stage_sequence":
                    stage_index + 1,

                "job_global_release_time_h":
                    job_global_release,

                "reported_processing_start_local_h":
                    local_processing_start,

                "setup_start_local_h":
                    local_setup_start,

                "processing_completion_local_h":
                    local_processing_completion,

                "upstream_ready_global_h":
                    global_upstream_ready,

                "setup_start_global_h":
                    global_setup_start,

                "processing_start_global_h":
                    global_processing_start,

                "processing_completion_global_h":
                    global_completion,

                "setup_time_h":
                    applied_setup,

                "processing_time_h":
                    processing_duration,

                "interstage_gap_h":
                    global_interstage_gap,

                "machine_busy_time_h":
                    machine_busy_duration

            })

        # ------------------------------------------------------
        # Full-batch completion of the current job
        # ------------------------------------------------------

        job_local_completion = (
            local_completion_by_machine[
                machine_stages[-1]
            ]
        )

        job_global_completion = (
            job_global_release
            + job_local_completion
        )

        total_processing = sum(
            record["processing_time_h"]
            for record in job_operation_records
        )

        total_setup = sum(
            record["setup_time_h"]
            for record in job_operation_records
        )

        total_interstage_gap = sum(
            record["interstage_gap_h"]
            for record in job_operation_records
        )

        # Save operation records.

        schedule_records.extend(
            job_operation_records
        )

        # ------------------------------------------------------
        # Job-level summary
        # ------------------------------------------------------

        job_summary_records.append({

            "customer_order_id":
                job["customer_order_id"],

            "job_id":
                job_id,

            "product_diameter_mm":
                product,

            "order_quantity":
                quantity,

            "manual_sequence":
                manual_sequence,

            "global_job_release_h":
                job_global_release,

            "total_setup_time_h":
                total_setup,

            "total_processing_time_h":
                total_processing,

            "total_interstage_gap_h":
                total_interstage_gap,

            "job_production_duration_h":
                job_local_completion,

            "global_job_completion_h":
                job_global_completion

        })

        # ------------------------------------------------------
        # Release next job only after M4 completion
        # ------------------------------------------------------

        previous_job_global_completion = (
            job_global_completion
        )

        next_job_global_release = (
            job_global_completion
        )

    # ==========================================================
    # 4. OPERATION-LEVEL SCHEDULE
    # ==============================================================

    production_schedule_df = pd.DataFrame(
        schedule_records
    )

    production_schedule_df = (
        production_schedule_df
        .sort_values(
            ["manual_sequence", "stage_sequence"]
        )
        .reset_index(drop=True)
    )

    # ==========================================================
    # 5. JOB-LEVEL SUMMARY
    # ==============================================================

    job_schedule_summary_df = pd.DataFrame(
        job_summary_records
    )

    # ==========================================================
    # 6. CONSOLIDATED CUSTOMER ORDER READINESS
    # ==============================================================

    production_ready_time = (
        job_schedule_summary_df[
            "global_job_completion_h"
        ].max()
    )

    production_horizon = (
        production_ready_time - order_release_time
    )

    production_order_summary_df = pd.DataFrame([{

        "customer_order_id":
            jobs["customer_order_id"].iloc[0],

        "destination_code":
            jobs["destination_code"].iloc[0],

        "number_of_jobs":
            len(jobs),

        "total_order_quantity":
            jobs["order_quantity"].sum(),

        "order_release_time_h":
            order_release_time,

        "production_ready_time_h":
            production_ready_time,

        "production_flow_time_h":
            production_horizon,

        "total_processing_workload_h":
            job_schedule_summary_df[
                "total_processing_time_h"
            ].sum(),

        "total_setup_workload_h":
            job_schedule_summary_df[
                "total_setup_time_h"
            ].sum(),

        "total_interstage_gap_h":
            job_schedule_summary_df[
                "total_interstage_gap_h"
            ].sum()

    }])

    # ==========================================================
    # 7. MACHINE UTILIZATION
    # ==============================================================

    machine_summary_records = []

    for machine in machine_stages:

        busy_time = machine_busy_time[machine]

        utilization = (
            100.0 * busy_time / production_horizon
            if production_horizon > 0
            else 0.0
        )

        machine_summary_records.append({

            "machine_id":
                machine,

            "total_busy_time_h":
                busy_time,

            "production_horizon_h":
                production_horizon,

            "machine_utilization_pct":
                utilization

        })

    machine_schedule_summary_df = pd.DataFrame(
        machine_summary_records
    )

    return (
        production_schedule_df,
        job_schedule_summary_df,
        machine_schedule_summary_df,
        production_order_summary_df
    )


# ==============================================================
# 8. EXECUTE SERIAL PRODUCTION SCHEDULING
# ==============================================================

(
    production_schedule_df,
    job_schedule_summary_df,
    machine_schedule_summary_df,
    production_order_summary_df
) = simulate_serial_flowshop(

    jobs=production_jobs_df,

    operations=production_operations_df,

    machine_stages=MACHINE_STAGES,

    order_release_time=ORDER_RELEASE_TIME_H,

    setup_included_in_processing=(
        SETUP_INCLUDED_IN_PROCESSING_TIME
    )

)


# ==============================================================
# 9. VERIFY NO MACHINE OVERLAPS
# ==============================================================

machine_conflicts = []

for machine in MACHINE_STAGES:

    machine_ops = (

        production_schedule_df.loc[
            production_schedule_df["machine_id"] == machine
        ]

        .sort_values("setup_start_global_h")

        .reset_index(drop=True)

    )

    for i in range(1, len(machine_ops)):

        previous_end = machine_ops.loc[
            i - 1,
            "processing_completion_global_h"
        ]

        current_start = machine_ops.loc[
            i,
            "setup_start_global_h"
        ]

        if current_start < previous_end - 1e-9:

            machine_conflicts.append({

                "machine_id": machine,

                "previous_job": machine_ops.loc[
                    i - 1, "job_id"
                ],

                "current_job": machine_ops.loc[
                    i, "job_id"
                ]

            })


# ==============================================================
# 10. VERIFY SEQUENTIAL JOB RELEASE
# ==============================================================

job_sequence_valid = True

for i in range(1, len(job_schedule_summary_df)):

    previous_completion = (
        job_schedule_summary_df.loc[
            i - 1,
            "global_job_completion_h"
        ]
    )

    current_release = (
        job_schedule_summary_df.loc[
            i,
            "global_job_release_h"
        ]
    )

    if current_release < previous_completion - 1e-9:

        job_sequence_valid = False


# ==============================================================
# 11. DISPLAY PRODUCTION SCHEDULE
# ==============================================================

print("=" * 80)

print("PIP SERIAL FOUR-STAGE PRODUCTION SCHEDULE")

print("=" * 80)

print("\nGLOBAL OPERATION-LEVEL SCHEDULE:")

print(
    production_schedule_df[
        [
            "job_id",
            "product_diameter_mm",
            "machine_id",
            "job_global_release_time_h",
            "setup_start_global_h",
            "processing_start_global_h",
            "processing_completion_global_h",
            "interstage_gap_h"
        ]
    ].round(3).to_string(index=False)
)


print("\nJOB COMPLETION SUMMARY:")

print(
    job_schedule_summary_df.round(3).to_string(
        index=False
    )
)


print("\nMACHINE UTILIZATION:")

print(
    machine_schedule_summary_df.round(3).to_string(
        index=False
    )
)


print("\nCONSOLIDATED CUSTOMER ORDER:")

print(
    production_order_summary_df.round(3).to_string(
        index=False
    )
)


# ==============================================================
# 12. FINAL VALIDATION
# ==============================================================

assert len(production_schedule_df) == 12

assert len(job_schedule_summary_df) == 3

assert not machine_conflicts, (
    "Machine overlap detected in the reconstructed timeline."
)

assert job_sequence_valid, (
    "The sequential job-release policy was violated."
)

assert (
    production_schedule_df[
        "processing_completion_global_h"
    ]
    >=
    production_schedule_df[
        "processing_start_global_h"
    ]
).all()

assert (
    production_order_summary_df[
        "total_order_quantity"
    ].iloc[0] == 190
)

assert np.isfinite(
    production_schedule_df[
        [
            "setup_start_global_h",
            "processing_start_global_h",
            "processing_completion_global_h"
        ]
    ].to_numpy()
).all()


# ==============================================================
# 13. FINAL RESULTS
# ==============================================================

print("\n" + "=" * 80)

print("CELL 3 — SERIAL FLOW-SHOP SCHEDULING PASSED")

print("=" * 80)

print("✓ Job 1 completes all four stages before Job 2 starts.")

print("✓ Job 2 completes all four stages before Job 3 starts.")

print("✓ Job-relative recorded processing times preserved.")

print("✓ Global timestamps reconstructed.")

print("✓ Setup and processing durations distinguished.")

print("✓ No inter-job production overlap.")

print("✓ No machine resource conflicts.")

print("✓ Consolidated order readiness calculated.")

print(
    "\nTotal production completion time:",
    round(
        production_order_summary_df[
            "production_ready_time_h"
        ].iloc[0],
        3
    ),
    "hours"
)

print("\nReady for Cell 4 — production-state generation.")


PIP SERIAL FOUR-STAGE PRODUCTION SCHEDULE

GLOBAL OPERATION-LEVEL SCHEDULE:
 job_id  product_diameter_mm machine_id  job_global_release_time_h  setup_start_global_h  processing_start_global_h  processing_completion_global_h  interstage_gap_h
      1                  900         M1                       0.00                  0.00                       1.50                           14.00              0.00
      1                  900         M2                       0.00                 16.43                      17.43                           25.76              2.43
      1                  900         M3                       0.00                 27.64                      28.44                           32.61              1.88
      1                  900         M4                       0.00                 34.07                      34.57                           35.57              1.46
      2                 1000         M1                      35.57                 35.57      

In [19]:

# ==============================================================
# CELL 4 — PRODUCTION STATE AND INDUSTRIAL WORKLOAD INDICATORS
# PIP FOUR-STAGE SERIAL FLOW-SHOP MANUFACTURING SYSTEM
# ==============================================================


# ==============================================================
# 1. EXTRACT PRODUCTION SCHEDULE INFORMATION
# ==============================================================

total_order_quantity = (
    production_jobs_df["order_quantity"].sum()
)

number_of_jobs = len(production_jobs_df)

production_horizon = float(
    production_order_summary_df[
        "production_flow_time_h"
    ].iloc[0]
)

production_processing_workload = float(
    production_order_summary_df[
        "total_processing_workload_h"
    ].iloc[0]
)

production_setup_workload = float(
    production_order_summary_df[
        "total_setup_workload_h"
    ].iloc[0]
)

production_interstage_gap = float(
    production_schedule_df[
        "interstage_gap_h"
    ].sum()
)


# ==============================================================
# 2. JOB-LEVEL PRODUCTION STATE
# ==============================================================

production_job_state_df = (

    production_jobs_df.merge(

        job_schedule_summary_df,

        on=[
            "customer_order_id",
            "job_id",
            "product_diameter_mm",
            "order_quantity",
            "manual_sequence"
        ],

        how="left",

        validate="one_to_one"

    )

)

# Number of jobs waiting outside the production line
# when each job begins its processing sequence.

production_job_state_df[
    "pending_jobs_before_line_entry"
] = (

    number_of_jobs
    -
    production_job_state_df["manual_sequence"]

)

# Elapsed duration for each job, measured from its
# own global production release to its M4 completion.

production_job_state_df[
    "job_production_duration_h"
] = (

    production_job_state_df[
        "global_job_completion_h"
    ]

    -

    production_job_state_df[
        "global_job_release_h"
    ]

)


# ==============================================================
# 3. MACHINE-LEVEL PRODUCTION STATE
# ==============================================================

production_machine_state_df = (
    machine_schedule_summary_df.copy()
)

# Explicitly define utilization relative to the complete
# consolidated production horizon.
#
# This is scheduled-horizon utilization, NOT OEE.

production_machine_state_df[
    "scheduled_horizon_utilization_pct"
] = (

    production_machine_state_df[
        "machine_utilization_pct"
    ]

)

# Machine idle time during the complete production horizon.

production_machine_state_df[
    "scheduled_idle_time_h"
] = (

    production_horizon

    -

    production_machine_state_df[
        "total_busy_time_h"
    ]

)

# Validate machine workload.

assert (
    production_machine_state_df[
        "total_busy_time_h"
    ]
    <= production_horizon + 1e-9
).all()


# ==============================================================
# 4. PRODUCTION SEQUENCE OCCUPATION
# ==============================================================

# In the confirmed serial production policy, only one job
# occupies the production sequence at a time.
#
# The occupation ratio measures the proportion of the
# production horizon spent on setup and processing.

total_machine_busy_time = (

    production_processing_workload

    +

    production_setup_workload

)

production_sequence_occupation_ratio = (

    total_machine_busy_time

    /

    production_horizon

    if production_horizon > 0
    else 0.0

)

production_sequence_occupation_pct = (
    production_sequence_occupation_ratio * 100.0
)


# ==============================================================
# 5. INTERSTAGE WAITING AND TRANSFER GAPS
# ==============================================================

# These gaps are calculated from the recorded job-relative
# operation timings.
#
# They represent elapsed time between completion at one
# machine and setup at the next machine.
#
# They are NOT automatically assumed to be pure queue delays.

interstage_gap_mean = float(
    production_schedule_df[
        "interstage_gap_h"
    ].mean()
)

interstage_gap_max = float(
    production_schedule_df[
        "interstage_gap_h"
    ].max()
)

interstage_gap_share_pct = (

    production_interstage_gap
    /
    production_horizon
    * 100.0

    if production_horizon > 0
    else 0.0

)


# ==============================================================
# 6. PRODUCTION BACKLOG
# ==============================================================

# The production backlog represents jobs waiting outside
# the line according to the manual sequence.
#
# It is NOT a machine-specific queue.

production_backlog_max = int(
    production_job_state_df[
        "pending_jobs_before_line_entry"
    ].max()
)

production_backlog_mean = float(
    production_job_state_df[
        "pending_jobs_before_line_entry"
    ].mean()
)


# ==============================================================
# 7. NONLINEAR PRODUCTION PRESSURE
# ==============================================================

# The sequence occupation ratio captures the workload of the
# serial production system.
#
# A nonlinear pressure indicator is retained for the DGP,
# but its threshold is an illustrative model assumption.

PRODUCTION_OCCUPATION_THRESHOLD = 85.0

production_occupation_excess = max(

    0.0,

    production_sequence_occupation_pct
    - PRODUCTION_OCCUPATION_THRESHOLD

)

high_production_occupation_pressure = (

    production_occupation_excess
    /
    (100.0 - PRODUCTION_OCCUPATION_THRESHOLD)

) ** 2


# ==============================================================
# 8. BUILD CONSOLIDATED PRODUCTION-STATE DATAFRAME
# ==============================================================

production_record = (

    production_order_summary_df.iloc[0].to_dict()

)

# Product-specific quantities.

for diameter in PRODUCT_TYPES:

    product_quantity = (

        production_jobs_df.loc[

            production_jobs_df[
                "product_diameter_mm"
            ] == diameter,

            "order_quantity"

        ].sum()

    )

    production_record[
        f"quantity_d{diameter}_pipes"
    ] = int(product_quantity)


# Aggregate production indicators.

production_record.update({

    "total_processing_workload_h":
        production_processing_workload,

    "total_setup_workload_h":
        production_setup_workload,

    "total_machine_busy_time_h":
        total_machine_busy_time,

    "total_interstage_gap_h":
        production_interstage_gap,

    "mean_interstage_gap_h":
        interstage_gap_mean,

    "maximum_interstage_gap_h":
        interstage_gap_max,

    "interstage_gap_share_pct":
        interstage_gap_share_pct,

    "production_sequence_occupation_ratio":
        production_sequence_occupation_ratio,

    "production_sequence_occupation_pct":
        production_sequence_occupation_pct,

    "production_backlog_max_jobs":
        production_backlog_max,

    "production_backlog_mean_jobs":
        production_backlog_mean,

    "high_production_occupation_pressure":
        high_production_occupation_pressure,

    "number_of_jobs":
        number_of_jobs,

    "order_quantity":
        total_order_quantity,

    "production_time":
        production_horizon

})


# Add individual machine workload and utilization.

for _, row in production_machine_state_df.iterrows():

    machine = row["machine_id"]

    production_record[
        f"{machine}_busy_time_h"
    ] = row["total_busy_time_h"]

    production_record[
        f"{machine}_utilization_pct"
    ] = row["scheduled_horizon_utilization_pct"]


# Average utilization across the four physical machines.

production_record["machine_utilization"] = (
    production_machine_state_df[
        "scheduled_horizon_utilization_pct"
    ].mean()
)

production_record["maximum_machine_utilization_pct"] = (
    production_machine_state_df[
        "scheduled_horizon_utilization_pct"
    ].max()
)


# One row represents one consolidated customer order.

production_state_df = pd.DataFrame([
    production_record
])


# ==============================================================
# 9. PRODUCTION-STATE STATISTICS
# ==============================================================

print("=" * 80)

print("PIP MANUFACTURING — PRODUCTION STATE")

print("=" * 80)

print("\nConsolidated order information:")

print(
    production_state_df[
        [
            "customer_order_id",
            "total_order_quantity",
            "number_of_jobs",
            "production_ready_time_h",
            "production_flow_time_h"
        ]
    ].round(3).to_string(index=False)
)


print("\nProduction workload:")

print(
    production_state_df[
        [
            "total_processing_workload_h",
            "total_setup_workload_h",
            "total_machine_busy_time_h",
            "total_interstage_gap_h"
        ]
    ].round(3).to_string(index=False)
)


print("\nMachine utilization:")

print(
    production_machine_state_df[
        [
            "machine_id",
            "total_busy_time_h",
            "scheduled_idle_time_h",
            "scheduled_horizon_utilization_pct"
        ]
    ].round(3).to_string(index=False)
)


print("\nProduction sequence occupation:")

print(
    f"{production_sequence_occupation_pct:.2f}%"
)


print("\nProduction backlog:")

print(
    "Maximum pending jobs:",
    production_backlog_max
)

print(
    "Mean pending jobs:",
    round(production_backlog_mean, 3)
)


print("\nNonlinear production occupation pressure:")

print(
    round(
        high_production_occupation_pressure,
        4
    )
)


# ==============================================================
# 10. DATA VALIDATION
# ==============================================================

assert len(production_state_df) == 1

assert (
    production_state_df[
        "total_order_quantity"
    ].iloc[0] == 190
)

assert (
    production_processing_workload > 0
)

assert (
    production_setup_workload >= 0
)

assert (
    production_interstage_gap >= 0
)

assert (
    0 <= production_sequence_occupation_pct <= 100
)

assert (
    production_backlog_max == 2
)

assert (
    production_backlog_mean == 1.0
)

assert (
    production_machine_state_df[
        "scheduled_horizon_utilization_pct"
    ].between(0, 100).all()
)

assert np.isfinite(
    production_state_df.select_dtypes(
        include=np.number
    ).to_numpy()
).all()


# ==============================================================
# 11. FINAL MESSAGE
# ==============================================================

print("\n" + "=" * 80)

print("CELL 4 — INDUSTRIAL PRODUCTION STATE COMPLETED")

print("=" * 80)

print("✓ Actual job-machine schedule incorporated.")

print("✓ Product-specific order quantities retained.")

print("✓ Machine-level workload calculated.")

print("✓ Setup and processing workload separated.")

print("✓ Machine utilization calculated from the schedule.")

print("✓ Interstage gaps quantified.")

print("✓ Production backlog represented.")

print("✓ Nonlinear production occupation pressure calculated.")

print("✓ Consolidated customer-order production state created.")

print("\nReady for Cell 5 — production-delay generation.")


PIP MANUFACTURING — PRODUCTION STATE

Consolidated order information:
customer_order_id  total_order_quantity  number_of_jobs  production_ready_time_h  production_flow_time_h
           CO-001                   190               3                   215.51                  215.51

Production workload:
 total_processing_workload_h  total_setup_workload_h  total_machine_busy_time_h  total_interstage_gap_h
                      181.81                    16.1                     197.91                    17.6

Machine utilization:
machine_id  total_busy_time_h  scheduled_idle_time_h  scheduled_horizon_utilization_pct
        M1              93.50                 122.01                             43.385
        M2              62.63                 152.88                             29.061
        M3              32.48                 183.03                             15.071
        M4               9.30                 206.21                              4.315

Production sequence occupat

In [20]:

# ==============================================================
# CELL 5 — PRODUCTION TIME AND INTERSTAGE OPERATIONAL ACTIVITIES
# PIP FOUR-STAGE SERIAL FLOW-SHOP SYSTEM
# ==============================================================


# ==============================================================
# 1. VALIDATE PRODUCTION SCHEDULE VARIABLES
# ==============================================================

required_columns = [

    "job_id",

    "machine_id",

    "stage_sequence",

    "setup_time_h",

    "processing_time_h",

    "setup_start_global_h",

    "processing_completion_global_h",

    "interstage_gap_h",

    "job_global_release_time_h"

]

missing_columns = [

    col for col in required_columns

    if col not in production_schedule_df.columns

]

if missing_columns:

    raise ValueError(
        f"Missing scheduling variables: {missing_columns}"
    )


# ==============================================================
# 2. AGGREGATE JOB-LEVEL PROCESSING AND SETUP INFORMATION
# ==============================================================

job_workload_df = (

    production_schedule_df

    .groupby("job_id")

    .agg(

        total_processing_time_h=(
            "processing_time_h",
            "sum"
        ),

        total_setup_time_h=(
            "setup_time_h",
            "sum"
        ),

        interstage_nonprocessing_time_h=(
            "interstage_gap_h",
            "sum"
        ),

        job_global_release_time_h=(
            "job_global_release_time_h",
            "first"
        )

    )

    .reset_index()

)


# ==============================================================
# 3. RETRIEVE FINAL JOB COMPLETION AT MACHINE M4
# ==============================================================

job_final_completion_df = (

    production_schedule_df.loc[

        production_schedule_df["machine_id"] == "M4",

        [
            "job_id",
            "processing_completion_global_h"
        ]

    ]

    .rename({

        "processing_completion_global_h":
            "job_global_completion_time_h"

    }, axis=1)

)


# ==============================================================
# 4. MERGE JOB INFORMATION
# ==============================================================

job_workload_df = job_workload_df.merge(

    production_jobs_df[

        [
            "job_id",
            "product_diameter_mm",
            "order_quantity",
            "manual_sequence"
        ]

    ],

    on="job_id",

    how="left",

    validate="one_to_one"

)

job_workload_df = job_workload_df.merge(

    job_final_completion_df,

    on="job_id",

    how="left",

    validate="one_to_one"

)

job_workload_df = (

    job_workload_df

    .sort_values("manual_sequence")

    .reset_index(drop=True)

)


# ==============================================================
# 5. CALCULATE JOB-LEVEL ELAPSED PRODUCTION TIME
# ==============================================================

job_workload_df["job_production_elapsed_time_h"] = (

    job_workload_df[
        "job_global_completion_time_h"
    ]

    -

    job_workload_df[
        "job_global_release_time_h"
    ]

)


# ==============================================================
# 6. CALCULATE NON-PROCESSING TIME
# ==============================================================

# Non-processing time is the elapsed time not occupied
# by setup or actual processing.

job_workload_df["nonprocessing_time_h"] = (

    job_workload_df[
        "job_production_elapsed_time_h"
    ]

    -

    job_workload_df[
        "total_setup_time_h"
    ]

    -

    job_workload_df[
        "total_processing_time_h"
    ]

)

# Compare the independently calculated non-processing time
# with the interstage gaps derived from Cell 3.

job_workload_df["time_balance_error_h"] = (

    job_workload_df[
        "nonprocessing_time_h"
    ]

    -

    job_workload_df[
        "interstage_nonprocessing_time_h"
    ]

)


# ==============================================================
# 7. CREATE INTERSTAGE GAP DETAIL TABLE
# ==============================================================

# Only transitions between successive production stages
# are included in this table.

interstage_gap_detail_df = (

    production_schedule_df.loc[

        production_schedule_df["stage_sequence"] > 1,

        [

            "job_id",

            "product_diameter_mm",

            "order_quantity",

            "manual_sequence",

            "machine_id",

            "stage_sequence",

            "upstream_ready_global_h",

            "setup_start_global_h",

            "interstage_gap_h"

        ]

    ]

    .copy()

)

# Identify the previous machine stage.

interstage_gap_detail_df["previous_machine"] = (

    interstage_gap_detail_df[
        "stage_sequence"
    ]

    .map({

        2: "M1",

        3: "M2",

        4: "M3"

    })

)

interstage_gap_detail_df = (
    interstage_gap_detail_df.rename({

        "machine_id": "next_machine",

        "interstage_gap_h":
            "interstage_nonprocessing_time_h"

    }, axis=1)
)

# Do not assign invented causes.
# The following fields are intended for later calibration
# with the factory manager's operational records.

interstage_gap_detail_df["gap_cause"] = (
    "TO_BE_CONFIRMED"
)

interstage_gap_detail_df["gap_cause_status"] = (
    "Not separately measured"
)


# Potential operational classifications.

GAP_CAUSE_CATEGORIES = [

    "Cooling",

    "Pipe transfer",

    "Inspection / quality control",

    "Material handling",

    "Operator unavailability",

    "Production queue waiting",

    "Other",

    "Multiple activities"

]


# ==============================================================
# 8. PRODUCT-SPECIFIC PRODUCTION WORKLOAD
# ==============================================================

product_workload_df = (

    job_workload_df

    .groupby("product_diameter_mm")

    .agg(

        total_quantity=(
            "order_quantity",
            "sum"
        ),

        processing_workload_h=(
            "total_processing_time_h",
            "sum"
        ),

        setup_workload_h=(
            "total_setup_time_h",
            "sum"
        ),

        nonprocessing_time_h=(
            "interstage_nonprocessing_time_h",
            "sum"
        ),

        total_elapsed_time_h=(
            "job_production_elapsed_time_h",
            "sum"
        )

    )

    .reset_index()

)


# ==============================================================
# 9. CONSOLIDATED PRODUCTION PERFORMANCE
# ==============================================================

production_processing_workload_h = float(

    job_workload_df[
        "total_processing_time_h"
    ].sum()

)

production_setup_workload_h = float(

    job_workload_df[
        "total_setup_time_h"
    ].sum()

)

production_nonprocessing_time_h = float(

    job_workload_df[
        "interstage_nonprocessing_time_h"
    ].sum()

)

production_elapsed_time_h = float(

    production_order_summary_df[
        "production_flow_time_h"
    ].iloc[0]

)

production_total_busy_time_h = (

    production_processing_workload_h

    +

    production_setup_workload_h

)


# ==============================================================
# 10. CALCULATE TIME DISTRIBUTION
# ==============================================================

production_processing_share_pct = (

    100.0
    * production_processing_workload_h
    / production_elapsed_time_h

    if production_elapsed_time_h > 0

    else 0.0

)

production_setup_share_pct = (

    100.0
    * production_setup_workload_h
    / production_elapsed_time_h

    if production_elapsed_time_h > 0

    else 0.0

)

production_nonprocessing_share_pct = (

    100.0
    * production_nonprocessing_time_h
    / production_elapsed_time_h

    if production_elapsed_time_h > 0

    else 0.0

)


# ==============================================================
# 11. MACHINE-SPECIFIC UTILIZATION
# ==============================================================

# Utilization is based on setup + processing time relative
# to the complete serial production horizon.
#
# It is NOT OEE or machine availability.

machine_utilization_summary_df = (

    machine_schedule_summary_df.copy()

)

machine_utilization_summary_df[
    "machine_utilization_pct"
] = (

    100.0

    * machine_utilization_summary_df[
        "total_busy_time_h"
    ]

    /

    production_elapsed_time_h

)


# ==============================================================
# 12. PRODUCTION SEQUENCE OCCUPATION
# ==============================================================

production_sequence_occupation_pct = (

    100.0

    * production_total_busy_time_h

    /

    production_elapsed_time_h

)


# ==============================================================
# 13. UPDATE PRODUCTION-STATE DATAFRAME
# ==============================================================

production_state_df["production_processing_workload_h"] = (
    production_processing_workload_h
)

production_state_df["production_setup_workload_h"] = (
    production_setup_workload_h
)

production_state_df["production_total_busy_time_h"] = (
    production_total_busy_time_h
)

production_state_df["production_nonprocessing_time_h"] = (
    production_nonprocessing_time_h
)

production_state_df["production_elapsed_time_h"] = (
    production_elapsed_time_h
)

production_state_df["production_processing_share_pct"] = (
    production_processing_share_pct
)

production_state_df["production_setup_share_pct"] = (
    production_setup_share_pct
)

production_state_df["production_nonprocessing_share_pct"] = (
    production_nonprocessing_share_pct
)

production_state_df["production_sequence_occupation_pct"] = (
    production_sequence_occupation_pct
)

production_state_df["production_time"] = (
    production_elapsed_time_h
)

production_state_df["production_interstage_nonprocessing_h"] = (
    production_nonprocessing_time_h
)


# Add machine-specific features.

for _, row in machine_utilization_summary_df.iterrows():

    machine = row["machine_id"]

    production_state_df[
        f"{machine}_utilization_pct"
    ] = row["machine_utilization_pct"]

    production_state_df[
        f"{machine}_busy_time_h"
    ] = row["total_busy_time_h"]


# ==============================================================
# 14. DISPLAY JOB-LEVEL RESULTS
# ==============================================================

print("=" * 80)

print("PRODUCTION PERFORMANCE BY JOB")

print("=" * 80)

print(

    job_workload_df[

        [

            "job_id",

            "product_diameter_mm",

            "order_quantity",

            "total_processing_time_h",

            "total_setup_time_h",

            "interstage_nonprocessing_time_h",

            "job_production_elapsed_time_h"

        ]

    ].round(3).to_string(index=False)

)


# ==============================================================
# 15. DISPLAY GAP CAUSE INFORMATION
# ==============================================================

print("\n" + "=" * 80)

print("INTERSTAGE OPERATIONAL GAP ANALYSIS")

print("=" * 80)

print(

    interstage_gap_detail_df[

        [

            "job_id",

            "previous_machine",

            "next_machine",

            "interstage_nonprocessing_time_h",

            "gap_cause",

            "gap_cause_status"

        ]

    ].round(3).to_string(index=False)

)

print("\nPossible gap classifications:")

print(GAP_CAUSE_CATEGORIES)


# ==============================================================
# 16. DISPLAY PRODUCT-SPECIFIC WORKLOAD
# ==============================================================

print("\n" + "=" * 80)

print("PRODUCTION WORKLOAD BY PIPE DIAMETER")

print("=" * 80)

print(

    product_workload_df.round(3).to_string(
        index=False
    )

)


# ==============================================================
# 17. DISPLAY CONSOLIDATED TIME DISTRIBUTION
# ==============================================================

production_summary_df = pd.DataFrame({

    "Indicator": [

        "Total pipes",

        "Production elapsed time",

        "Processing workload",

        "Setup workload",

        "Interstage non-processing time",

        "Processing time share (%)",

        "Setup time share (%)",

        "Non-processing time share (%)",

        "Production sequence occupation (%)"

    ],

    "Value": [

        total_order_quantity,

        production_elapsed_time_h,

        production_processing_workload_h,

        production_setup_workload_h,

        production_nonprocessing_time_h,

        production_processing_share_pct,

        production_setup_share_pct,

        production_nonprocessing_share_pct,

        production_sequence_occupation_pct

    ]

})

print("\n" + "=" * 80)

print("CONSOLIDATED PRODUCTION TIME DISTRIBUTION")

print("=" * 80)

print(
    production_summary_df.round(3).to_string(
        index=False
    )
)


# ==============================================================
# 18. VALIDATION
# ==============================================================

assert len(job_workload_df) == 3

assert len(interstage_gap_detail_df) == 9

assert production_processing_workload_h > 0

assert production_setup_workload_h >= 0

assert production_nonprocessing_time_h >= 0

assert np.allclose(

    job_workload_df[
        "time_balance_error_h"
    ],

    0.0,

    atol=1e-8

)

assert np.isclose(

    production_processing_workload_h
    + production_setup_workload_h
    + production_nonprocessing_time_h,

    production_elapsed_time_h,

    atol=1e-8

)

assert (
    machine_utilization_summary_df[
        "machine_utilization_pct"
    ].between(0, 100).all()
)

assert np.isfinite(

    production_state_df.select_dtypes(
        include=np.number
    ).to_numpy()

).all()


# ==============================================================
# 19. FINAL MESSAGE
# ==============================================================

print("\n" + "=" * 80)

print("CELL 5 — PRODUCTION PERFORMANCE VALIDATION PASSED")

print("=" * 80)

print("✓ Actual processing durations retained.")

print("✓ Actual setup durations retained.")

print("✓ Job-specific production workloads calculated.")

print("✓ Interstage non-processing time calculated.")

print("✓ Potential operational gap causes documented.")

print("✓ No unsupported cause allocation introduced.")

print("✓ Production time balance verified.")

print("✓ Consolidated production indicators updated.")

print("\nReady for Cell 6 — stochastic operational disruptions.")


PRODUCTION PERFORMANCE BY JOB
 job_id  product_diameter_mm  order_quantity  total_processing_time_h  total_setup_time_h  interstage_nonprocessing_time_h  job_production_elapsed_time_h
      1                  900              50                    26.00                 3.8                             5.77                          35.57
      2                 1000             100                    51.73                 5.3                             5.01                          62.04
      3                 1200              40                   104.08                 7.0                             6.82                         117.90

INTERSTAGE OPERATIONAL GAP ANALYSIS
 job_id previous_machine next_machine  interstage_nonprocessing_time_h       gap_cause        gap_cause_status
      1               M1           M2                             2.43 TO_BE_CONFIRMED Not separately measured
      1               M2           M3                             1.88 TO_BE_CONFIRMED Not sepa

## 2. Stochastic production, warehouse, and transportation data generation

In [21]:
# ============================================================
# CELL 6 — PRODUCTION-STAGE SCENARIO GENERATION
# ============================================================
# Purpose:
# 1. Use the validated serial production schedule as the baseline.
# 2. Generate stochastic machine-availability scenarios.
# 3. Estimate additional production downtime.
# 4. Calculate scenario-level production completion times.
#
# Production configuration:
# - Four sequential machines: M1, M2, M3, M4
# - One machine at each stage
# - Fixed manual job sequence
# - Full-batch transfer
# - Each job completes all four stages before the next job starts
#
# Important:
# This is an availability-adjusted production model.
# It is not an event-level discrete-event failure simulator.
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Validate required data frames and columns
# ------------------------------------------------------------

required_order_columns = [
    "customer_order_id",
    "total_order_quantity",
    "number_of_jobs",
    "production_flow_time_h"
]

required_machine_columns = [
    "machine_id"
]

required_schedule_columns = [
    "customer_order_id",
    "job_id",
    "machine_id",
    "stage_sequence",
    "setup_time_h",
    "processing_time_h",
    "interstage_gap_h"
]

missing_order_columns = [
    col for col in required_order_columns
    if col not in production_order_summary_df.columns
]

missing_machine_columns = [
    col for col in required_machine_columns
    if col not in machine_schedule_summary_df.columns
]

missing_schedule_columns = [
    col for col in required_schedule_columns
    if col not in production_schedule_df.columns
]

if missing_order_columns:
    raise ValueError(
        f"Missing columns in production_order_summary_df: "
        f"{missing_order_columns}"
    )

if missing_machine_columns:
    raise ValueError(
        f"Missing columns in machine_schedule_summary_df: "
        f"{missing_machine_columns}"
    )

if missing_schedule_columns:
    raise ValueError(
        f"Missing columns in production_schedule_df: "
        f"{missing_schedule_columns}"
    )

if "rng" not in globals():
    raise RuntimeError(
        "Random generator 'rng' is missing. "
        "Run Cell 1 before running Cell 6."
    )

if "MACHINE_STAGES" not in globals():
    raise RuntimeError(
        "MACHINE_STAGES is not defined. Check Cell 1."
    )

if not isinstance(production_schedule_df, pd.DataFrame):
    raise TypeError("production_schedule_df must be a pandas DataFrame.")

if not isinstance(machine_schedule_summary_df, pd.DataFrame):
    raise TypeError(
        "machine_schedule_summary_df must be a pandas DataFrame."
    )

if not isinstance(production_order_summary_df, pd.DataFrame):
    raise TypeError(
        "production_order_summary_df must be a pandas DataFrame."
    )

if len(production_order_summary_df) != 1:
    raise ValueError(
        "Cell 6 currently expects one reference customer order."
    )


# ------------------------------------------------------------
# 2. Retrieve reference customer-order information
# ------------------------------------------------------------

order_info = production_order_summary_df.iloc[0].copy()

customer_order_id = order_info["customer_order_id"]

order_quantity = float(
    order_info["total_order_quantity"]
)

number_of_jobs = int(
    order_info["number_of_jobs"]
)

# Use zero as the reference release time if it is not available.
order_release_time_h = order_info.get(
    "order_release_time_h", 0.0
)

if pd.isna(order_release_time_h):
    order_release_time_h = 0.0

order_release_time_h = float(order_release_time_h)

if order_quantity <= 0:
    raise ValueError("Total customer-order quantity must be positive.")

if number_of_jobs <= 0:
    raise ValueError("Number of production jobs must be positive.")


# ------------------------------------------------------------
# 3. Validate the four production stages
# ------------------------------------------------------------

stage_ids = list(MACHINE_STAGES)

if len(stage_ids) != 4:
    raise ValueError(
        f"Expected exactly four production stages; "
        f"found {len(stage_ids)}."
    )

if len(set(stage_ids)) != len(stage_ids):
    raise ValueError("Duplicate machine IDs found in MACHINE_STAGES.")

if machine_schedule_summary_df["machine_id"].duplicated().any():
    raise ValueError(
        "Duplicate machine IDs detected in machine_schedule_summary_df."
    )

summary_machine_ids = set(
    machine_schedule_summary_df["machine_id"].astype(str)
)

missing_stages = [
    machine_id
    for machine_id in stage_ids
    if str(machine_id) not in summary_machine_ids
]

if missing_stages:
    raise ValueError(
        f"The following production stages are missing from "
        f"machine_schedule_summary_df: {missing_stages}"
    )


# ------------------------------------------------------------
# 4. Validate and prepare the operation-level schedule
# ------------------------------------------------------------

production_schedule_working = production_schedule_df.copy()

production_schedule_working["stage_sequence"] = pd.to_numeric(
    production_schedule_working["stage_sequence"],
    errors="coerce"
)

production_schedule_working["setup_time_h"] = pd.to_numeric(
    production_schedule_working["setup_time_h"],
    errors="coerce"
)

production_schedule_working["processing_time_h"] = pd.to_numeric(
    production_schedule_working["processing_time_h"],
    errors="coerce"
)

production_schedule_working["interstage_gap_h"] = pd.to_numeric(
    production_schedule_working["interstage_gap_h"],
    errors="coerce"
)

numeric_schedule_columns = [
    "stage_sequence",
    "setup_time_h",
    "processing_time_h",
    "interstage_gap_h"
]

if production_schedule_working[numeric_schedule_columns].isna().any().any():
    raise ValueError(
        "Missing or invalid numerical values were found "
        "in the production schedule."
    )

if (
    production_schedule_working[
        ["setup_time_h", "processing_time_h", "interstage_gap_h"]
    ] < 0
).any().any():
    raise ValueError(
        "Negative setup, processing, or interstage times were found."
    )

# Verify that the schedule contains all four machine stages.
schedule_machine_ids = set(
    production_schedule_working["machine_id"].astype(str)
)

if not all(str(machine_id) in schedule_machine_ids
           for machine_id in stage_ids):
    raise ValueError(
        "The operation-level schedule does not contain all four machines."
    )


# ------------------------------------------------------------
# 5. Derive machine workloads from the operation schedule
# ------------------------------------------------------------

# The operation-level production schedule is the source of truth.
# We do not depend on optional setup_workload_h or
# processing_workload_h columns in the machine summary.

machine_workload_df = (
    production_schedule_working
    .groupby("machine_id")
    .agg(
        setup_workload_h=("setup_time_h", "sum"),
        processing_workload_h=("processing_time_h", "sum")
    )
    .reindex(stage_ids)
)

if machine_workload_df.isna().any().any():
    raise ValueError(
        "Machine setup or processing workload information is incomplete."
    )

machine_workload_df["total_busy_time_h"] = (
    machine_workload_df["setup_workload_h"]
    + machine_workload_df["processing_workload_h"]
)

machine_busy_hours = (
    machine_workload_df["total_busy_time_h"]
    .astype(float)
    .to_numpy()
)

total_setup_time_h = float(
    machine_workload_df["setup_workload_h"].sum()
)

total_processing_time_h = float(
    machine_workload_df["processing_workload_h"].sum()
)

total_machine_busy_h = float(
    machine_workload_df["total_busy_time_h"].sum()
)

if total_machine_busy_h <= 0:
    raise ValueError(
        "Total machine busy time must be greater than zero."
    )


# ------------------------------------------------------------
# 6. Derive interstage non-processing time
# ------------------------------------------------------------

# Stage 1 has no preceding production stage.
# Therefore, only stages 2, 3, and 4 are included.
#
# These intervals may include cooling, pipe transfer,
# quality inspection, material handling, operator unavailability,
# or simple waiting.
#
# Their individual causes have not yet been quantified.
# Accordingly, they are NOT classified as pure waiting time.

interstage_mask = (
    production_schedule_working["stage_sequence"] > 1
)

interstage_nonprocessing_time_h = float(
    production_schedule_working.loc[
        interstage_mask,
        "interstage_gap_h"
    ].sum()
)


# ------------------------------------------------------------
# 7. Derive the maximum pending-job count
# ------------------------------------------------------------

# This indicator represents the number of jobs that have not
# yet entered production under the fixed manual sequence.
#
# It is NOT a measured machine-level congestion queue.

if (
    "production_jobs_df" in globals()
    and "manual_sequence" in production_jobs_df.columns
):

    manual_sequence = pd.to_numeric(
        production_jobs_df["manual_sequence"],
        errors="coerce"
    )

    if manual_sequence.isna().any():
        raise ValueError(
            "Invalid manual sequence values found in production_jobs_df."
        )

    pending_jobs_before_entry = (
        number_of_jobs - manual_sequence
    ).clip(lower=0)

    maximum_queue_length = int(
        pending_jobs_before_entry.max()
    )

else:
    maximum_queue_length = max(number_of_jobs - 1, 0)


# ------------------------------------------------------------
# 8. Verify baseline production-time balance
# ------------------------------------------------------------

# Baseline production flow time =
# Total setup time
# + Total processing time
# + Interstage non-processing time

baseline_production_flow_time_h = float(
    order_info["production_flow_time_h"]
)

calculated_baseline_flow_h = (
    total_machine_busy_h
    + interstage_nonprocessing_time_h
)

if not np.isclose(
    baseline_production_flow_time_h,
    calculated_baseline_flow_h,
    atol=1e-3
):
    raise ValueError(
        "Production-time balance does not hold.\n"
        f"Production flow time in order summary: "
        f"{baseline_production_flow_time_h:.3f} h\n"
        f"Machine busy time: {total_machine_busy_h:.3f} h\n"
        f"Interstage non-processing time: "
        f"{interstage_nonprocessing_time_h:.3f} h\n"
        f"Calculated total: {calculated_baseline_flow_h:.3f} h\n"
        "Check Cells 3–5 before continuing."
    )


# ------------------------------------------------------------
# 9. Define provisional machine-availability assumptions
# ------------------------------------------------------------

N_PRODUCTION_SCENARIOS = 2000

AVAILABILITY_BETA_ALPHA = 18.0
AVAILABILITY_BETA_BETA = 2.0

MIN_MACHINE_AVAILABILITY = 0.80
MAX_MACHINE_AVAILABILITY = 1.00

AVAILABILITY_PARAMETER_STATUS = (
    "Provisional simulation assumption. "
    "Calibration using factory reliability records "
    "or justified external evidence is required."
)


# ------------------------------------------------------------
# 10. Generate machine-availability scenarios
# ------------------------------------------------------------

# Each scenario represents one simulated realization of the
# reference customer order.
#
# Each row corresponds to one scenario.
# Each column corresponds to one production machine.
#
# The machine availability values are independently sampled
# across machines and held constant throughout each scenario.

availability_matrix = rng.beta(
    AVAILABILITY_BETA_ALPHA,
    AVAILABILITY_BETA_BETA,
    size=(
        N_PRODUCTION_SCENARIOS,
        len(stage_ids)
    )
)

availability_matrix = np.clip(
    availability_matrix,
    MIN_MACHINE_AVAILABILITY,
    MAX_MACHINE_AVAILABILITY
)


# ------------------------------------------------------------
# 11. Calculate adjusted machine occupation time
# ------------------------------------------------------------

# Availability definition:
#
# A = operating time / (operating time + downtime)
#
# Thus:
#
# Adjusted occupation time = Scheduled busy time / A
#
# Additional downtime = Adjusted occupation time - Scheduled busy time
#
# This approximation applies the sampled availability to both
# setup and processing occupation time.

adjusted_machine_occupation_h = (
    machine_busy_hours[np.newaxis, :]
    / availability_matrix
)

machine_downtime_h = (
    adjusted_machine_occupation_h
    - machine_busy_hours[np.newaxis, :]
)

total_production_downtime_h = (
    machine_downtime_h.sum(axis=1)
)


# ------------------------------------------------------------
# 12. Calculate scenario-level production completion times
# ------------------------------------------------------------

# The production jobs are processed sequentially.
# Therefore, all machine occupation and interstage intervals
# contribute to the total production flow time.

production_flow_time_scenario_h = (
    interstage_nonprocessing_time_h
    + adjusted_machine_occupation_h.sum(axis=1)
)

production_completion_time_h = (
    order_release_time_h
    + production_flow_time_scenario_h
)

production_time_increase_h = (
    production_flow_time_scenario_h
    - baseline_production_flow_time_h
)


# ------------------------------------------------------------
# 13. Calculate production-system indicators
# ------------------------------------------------------------

# Busy-time-weighted availability across the four machines.

weighted_machine_availability = (
    availability_matrix @ machine_busy_hours
) / total_machine_busy_h

# Overall machine occupation ratio relative to the complete
# scenario production-flow horizon.

system_machine_utilization = (
    total_machine_busy_h
    / production_flow_time_scenario_h
)

# Machine-specific occupation ratios over the complete
# production-flow horizon.

machine_utilization_matrix = (
    machine_busy_hours[np.newaxis, :]
    / production_flow_time_scenario_h[:, np.newaxis]
)


# ------------------------------------------------------------
# 14. Construct the production-scenario data frame
# ------------------------------------------------------------

production_scenarios_df = pd.DataFrame({

    "scenario_id": np.arange(
        1,
        N_PRODUCTION_SCENARIOS + 1
    ),

    "customer_order_id": customer_order_id,

    "order_quantity": order_quantity,

    "number_of_jobs": number_of_jobs,

    "production_baseline_flow_time_h":
        baseline_production_flow_time_h,

    "production_processing_time_h":
        total_processing_time_h,

    "production_setup_time_h":
        total_setup_time_h,

    "interstage_nonprocessing_time_h":
        interstage_nonprocessing_time_h,

    "production_downtime_h":
        total_production_downtime_h,

    "production_time_increase_h":
        production_time_increase_h,

    "production_time_h":
        production_flow_time_scenario_h,

    "production_completion_time_h":
        production_completion_time_h,

    "machine_availability":
        weighted_machine_availability,

    "machine_utilization":
        system_machine_utilization,

    "queue_length":
        maximum_queue_length,

    "pending_job_count":
        maximum_queue_length
})


# ------------------------------------------------------------
# 15. Add machine-specific indicators
# ------------------------------------------------------------

for machine_index, machine_id in enumerate(stage_ids):

    production_scenarios_df[
        f"machine_availability_{machine_id}"
    ] = availability_matrix[:, machine_index]

    production_scenarios_df[
        f"machine_downtime_{machine_id}_h"
    ] = machine_downtime_h[:, machine_index]

    production_scenarios_df[
        f"machine_utilization_{machine_id}_pct"
    ] = (
        machine_utilization_matrix[:, machine_index] * 100
    )


# Overall availability and utilization as percentages.

production_scenarios_df[
    "machine_availability_pct"
] = (
    production_scenarios_df["machine_availability"] * 100
)

production_scenarios_df[
    "machine_utilization_pct"
] = (
    production_scenarios_df["machine_utilization"] * 100
)


# ------------------------------------------------------------
# 16. Quality-control checks
# ------------------------------------------------------------

if not (
    production_scenarios_df["production_downtime_h"] >= 0
).all():
    raise ValueError(
        "Negative production downtime was generated."
    )

if not (
    production_scenarios_df["production_time_h"]
    >= baseline_production_flow_time_h
).all():
    raise ValueError(
        "A simulated production time is shorter than the baseline."
    )

if not production_scenarios_df["machine_availability"].between(
    MIN_MACHINE_AVAILABILITY,
    MAX_MACHINE_AVAILABILITY
).all():
    raise ValueError(
        "Machine availability is outside the specified bounds."
    )

if not np.allclose(
    production_scenarios_df["production_time_increase_h"],
    production_scenarios_df["production_downtime_h"],
    atol=1e-8
):
    raise ValueError(
        "Production-time increase does not match additional downtime."
    )

if production_scenarios_df["scenario_id"].duplicated().any():
    raise ValueError(
        "Duplicate scenario IDs were generated."
    )

if not np.isfinite(
    production_scenarios_df[
        [
            "production_time_h",
            "production_downtime_h",
            "machine_availability",
            "machine_utilization"
        ]
    ].to_numpy()
).all():
    raise ValueError(
        "Non-finite values were generated in production scenarios."
    )


# ------------------------------------------------------------
# 17. Generate descriptive statistics
# ------------------------------------------------------------

production_scenario_summary_df = (
    production_scenarios_df[
        [
            "machine_availability_pct",
            "machine_utilization_pct",
            "production_downtime_h",
            "production_time_h",
            "production_time_increase_h"
        ]
    ]
    .agg([
        "mean",
        "std",
        "min",
        "median",
        "max"
    ])
    .T
    .reset_index()
    .rename(
        columns={"index": "metric"}
    )
)


# ------------------------------------------------------------
# 18. Display results
# ------------------------------------------------------------

print("CELL 6 COMPLETED SUCCESSFULLY")
print("=" * 55)

print(f"Reference customer order: {customer_order_id}")
print(f"Total order quantity: {order_quantity:.0f} pipes")
print(f"Number of production jobs: {number_of_jobs}")
print(f"Number of machines: {len(stage_ids)}")
print(f"Maximum pending-job count: {maximum_queue_length}")
print(f"Number of production scenarios: {N_PRODUCTION_SCENARIOS}")

print("-" * 55)

print(
    f"Baseline processing workload: "
    f"{total_processing_time_h:.2f} h"
)

print(
    f"Baseline setup workload: "
    f"{total_setup_time_h:.2f} h"
)

print(
    f"Total machine busy time: "
    f"{total_machine_busy_h:.2f} h"
)

print(
    f"Interstage non-processing time: "
    f"{interstage_nonprocessing_time_h:.2f} h"
)

print(
    f"Baseline production flow time: "
    f"{baseline_production_flow_time_h:.2f} h"
)

print("-" * 55)

display(production_scenario_summary_df)

display(
    production_scenarios_df.head()
)

CELL 6 COMPLETED SUCCESSFULLY
Reference customer order: CO-001
Total order quantity: 190 pipes
Number of production jobs: 3
Number of machines: 4
Maximum pending-job count: 2
Number of production scenarios: 2000
-------------------------------------------------------
Baseline processing workload: 181.81 h
Baseline setup workload: 16.10 h
Total machine busy time: 197.91 h
Interstage non-processing time: 17.60 h
Baseline production flow time: 215.51 h
-------------------------------------------------------


,metric,mean,std,min,median,max
0,machine_availability_pct,90.389190,3.296107,80.333514,90.660566,98.285308
1,machine_utilization_pct,83.460317,2.873496,74.975797,83.662947,90.379132
2,production_downtime_h,21.905565,8.286835,3.467540,21.046333,48.455182
3,production_time_h,237.415565,8.286835,218.977540,236.556333,263.965182
4,production_time_increase_h,21.905565,8.286835,3.467540,21.046333,48.455182


,scenario_id,customer_order_id,order_quantity,number_of_jobs,production_baseline_flow_time_h,production_processing_time_h,production_setup_time_h,interstage_nonprocessing_time_h,production_downtime_h,production_time_increase_h,...,machine_downtime_M2_h,machine_utilization_M2_pct,machine_availability_M3,machine_downtime_M3_h,machine_utilization_M3_pct,machine_availability_M4,machine_downtime_M4_h,machine_utilization_M4_pct,machine_availability_pct,machine_utilization_pct
0,1,CO-001,190.0,3,215.51,181.81,16.1,17.6,11.842324,11.842324,...,3.960326,27.547552,0.856304,5.450433,14.286197,0.902334,1.006603,4.090567,94.591538,87.049913
1,2,CO-001,190.0,3,215.51,181.81,16.1,17.6,16.125065,16.125065,...,5.510197,27.038221,0.957724,1.433720,14.022057,0.958171,0.405997,4.014936,92.497103,85.440432
2,3,CO-001,190.0,3,215.51,181.81,16.1,17.6,27.528919,27.528919,...,4.774049,25.769535,0.910902,3.176948,13.364115,0.849364,1.649368,3.826548,87.987157,81.431402
3,4,CO-001,190.0,3,215.51,181.81,16.1,17.6,39.028477,39.028477,...,12.738342,24.605317,0.931209,2.399394,12.760350,0.947458,0.515741,3.653672,83.826811,77.752488
4,5,CO-001,190.0,3,215.51,181.81,16.1,17.6,11.212367,11.212367,...,5.345751,27.624094,0.952476,1.620597,14.325891,0.868251,1.411191,4.101933,94.722043,87.291785


In [22]:
# ============================================================
# CELL 6A — CORRECTION OF MACHINE UTILIZATION INDICATORS
# ============================================================

# Calculate the utilization of each machine over the complete
# customer-order production horizon.

machine_utilization_columns = [
    f"machine_utilization_{machine_id}_pct"
    for machine_id in stage_ids
]

# Average utilization across the four machines.
production_scenarios_df["machine_utilization"] = (
    production_scenarios_df[machine_utilization_columns]
    .mean(axis=1) / 100
)

production_scenarios_df["machine_utilization_pct"] = (
    production_scenarios_df["machine_utilization"] * 100
)

# Retain the former aggregate indicator under a more accurate name.
production_scenarios_df[
    "aggregate_machine_occupation_ratio"
] = (
    total_machine_busy_h
    / production_scenarios_df["production_time_h"]
)

# Recalculate descriptive statistics.
production_scenario_summary_df = (
    production_scenarios_df[
        [
            "machine_availability_pct",
            "machine_utilization_pct",
            "production_downtime_h",
            "production_time_h",
            "production_time_increase_h",
            "aggregate_machine_occupation_ratio"
        ]
    ]
    .agg(["mean", "std", "min", "median", "max"])
    .T
    .reset_index()
    .rename(columns={"index": "metric"})
)

print("MACHINE UTILIZATION CORRECTION COMPLETED")
print("----------------------------------------")

display(production_scenario_summary_df)

MACHINE UTILIZATION CORRECTION COMPLETED
----------------------------------------


,metric,mean,std,min,median,max
0,machine_availability_pct,90.389190,3.296107,80.333514,90.660566,98.285308
1,machine_utilization_pct,20.865079,0.718374,18.743949,20.915737,22.594783
2,production_downtime_h,21.905565,8.286835,3.467540,21.046333,48.455182
3,production_time_h,237.415565,8.286835,218.977540,236.556333,263.965182
4,production_time_increase_h,21.905565,8.286835,3.467540,21.046333,48.455182
5,aggregate_machine_occupation_ratio,0.834603,0.028735,0.749758,0.836629,0.903791


In [23]:

# ============================================================
# CELL 7 — WAREHOUSE OPERATIONS MODELING
# ============================================================
# Research framework:
# Integrated production–warehouse–transportation system
#
# Purpose:
# 1. Receive completed production orders from Cell 6.
# 2. Model pre-existing warehouse workload and queue waiting.
# 3. Model inbound handling and storage/staging.
# 4. Model loading operations.
# 5. Calculate warehouse completion time for each scenario.
#
# Industrial configuration:
# - One consolidated customer order.
# - Three product jobs within the order.
# - Total reference quantity: 190 pipes.
# - Warehouse operations start after production completion.
# - Loading is completed before transportation begins.
#
# IMPORTANT:
# Warehouse parameters are provisional assumptions.
# They must be calibrated using company records or justified
# external evidence before being reported as factory measurements.
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Validate production scenarios from Cell 6
# ------------------------------------------------------------

required_production_columns = [
    "scenario_id",
    "customer_order_id",
    "order_quantity",
    "production_completion_time_h",
    "production_time_h",
    "machine_availability",
    "machine_utilization"
]

missing_production_columns = [
    col for col in required_production_columns
    if col not in production_scenarios_df.columns
]

if missing_production_columns:
    raise ValueError(
        "Missing columns in production_scenarios_df: "
        f"{missing_production_columns}. "
        "Run Cells 1–6 before running Cell 7."
    )

if "rng" not in globals():
    raise RuntimeError(
        "Random generator 'rng' is missing. "
        "Run Cell 1 before running Cell 7."
    )

if not isinstance(production_scenarios_df, pd.DataFrame):
    raise TypeError(
        "production_scenarios_df must be a pandas DataFrame."
    )

if production_scenarios_df.empty:
    raise ValueError(
        "production_scenarios_df is empty."
    )

if production_scenarios_df["scenario_id"].duplicated().any():
    raise ValueError(
        "Duplicate scenario IDs detected in production_scenarios_df."
    )

if production_scenarios_df["customer_order_id"].nunique() != 1:
    raise ValueError(
        "This version of Cell 7 expects scenarios for one reference "
        "customer order."
    )

if (
    production_scenarios_df["production_completion_time_h"] < 0
).any():
    raise ValueError(
        "Negative production completion times were detected."
    )

if (
    production_scenarios_df["order_quantity"] <= 0
).any():
    raise ValueError(
        "Customer-order quantities must be positive."
    )


# ------------------------------------------------------------
# 2. Preserve the production scenarios
# ------------------------------------------------------------

# Each warehouse scenario remains paired with the corresponding
# production scenario.
#
# This preserves the scenario-level connection between the
# production and warehouse stages.

warehouse_scenarios_df = (
    production_scenarios_df.copy()
    .reset_index(drop=True)
)

N_WAREHOUSE_SCENARIOS = len(warehouse_scenarios_df)

customer_order_id = (
    warehouse_scenarios_df["customer_order_id"].iloc[0]
)

order_quantity = (
    warehouse_scenarios_df["order_quantity"]
    .astype(float)
    .to_numpy()
)

# Warehouse operations begin immediately after production
# completion in the current reference model.

warehouse_start_time_h = (
    warehouse_scenarios_df[
        "production_completion_time_h"
    ].astype(float).to_numpy()
)


# ------------------------------------------------------------
# 3. Define provisional warehouse parameters
# ------------------------------------------------------------

# The following parameters are illustrative placeholders.
# They are NOT verified Farasan warehouse measurements.
#
# They should eventually be replaced or calibrated using:
# - Historical warehouse processing records.
# - Handling and loading productivity records.
# - Warehouse arrival and dispatch timestamps.
# - Warehouse queue and staging records.
# - Management input where historical records are unavailable.

# A. Inbound handling capacity:
# Number of pipes handled per hour.

WAREHOUSE_INBOUND_RATE_MIN = 12.0
WAREHOUSE_INBOUND_RATE_MODE = 18.0
WAREHOUSE_INBOUND_RATE_MAX = 26.0

# B. Loading capacity:
# Number of pipes loaded per hour.

WAREHOUSE_LOADING_RATE_MIN = 15.0
WAREHOUSE_LOADING_RATE_MODE = 22.0
WAREHOUSE_LOADING_RATE_MAX = 30.0

# C. Previously accumulated warehouse workload:
# Expressed in pipe-equivalent handling units.
#
# Gamma distribution parameters are provisional.
# Mean background workload = shape * scale = 80 units.

WAREHOUSE_BACKLOG_GAMMA_SHAPE = 4.0
WAREHOUSE_BACKLOG_GAMMA_SCALE = 20.0

# D. Storage/staging duration:
# Time between completion of inbound handling and beginning
# of loading.
#
# This is treated as an operational staging interval, not
# automatically as avoidable delay.

WAREHOUSE_STORAGE_MIN_H = 0.5
WAREHOUSE_STORAGE_MODE_H = 1.5
WAREHOUSE_STORAGE_MAX_H = 4.0

# Parameter provenance.

WAREHOUSE_PARAMETER_PROVENANCE = {
    "inbound_handling_rate":
        "Illustrative assumption; requires factory calibration.",

    "loading_rate":
        "Illustrative assumption; requires factory calibration.",

    "background_queue_workload":
        "Synthetic scenario assumption; requires warehouse records.",

    "storage_staging_time":
        "Illustrative assumption; requires dispatch and staging records."
}


# ------------------------------------------------------------
# 4. Generate warehouse operating conditions
# ------------------------------------------------------------

# Inbound handling capacity.

warehouse_inbound_rate = rng.triangular(
    WAREHOUSE_INBOUND_RATE_MIN,
    WAREHOUSE_INBOUND_RATE_MODE,
    WAREHOUSE_INBOUND_RATE_MAX,
    size=N_WAREHOUSE_SCENARIOS
)

# Loading capacity.

warehouse_loading_rate = rng.triangular(
    WAREHOUSE_LOADING_RATE_MIN,
    WAREHOUSE_LOADING_RATE_MODE,
    WAREHOUSE_LOADING_RATE_MAX,
    size=N_WAREHOUSE_SCENARIOS
)

# Background workload waiting ahead of the incoming customer order.

warehouse_background_queue_workload = rng.gamma(
    shape=WAREHOUSE_BACKLOG_GAMMA_SHAPE,
    scale=WAREHOUSE_BACKLOG_GAMMA_SCALE,
    size=N_WAREHOUSE_SCENARIOS
)

# Storage/staging duration.

warehouse_storage_dwell_time_h = rng.triangular(
    WAREHOUSE_STORAGE_MIN_H,
    WAREHOUSE_STORAGE_MODE_H,
    WAREHOUSE_STORAGE_MAX_H,
    size=N_WAREHOUSE_SCENARIOS
)


# ------------------------------------------------------------
# 5. Calculate warehouse queue waiting time
# ------------------------------------------------------------

# Assumption:
# The previously accumulated warehouse workload is processed
# before the incoming customer order, following a FIFO policy.
#
# A single effective inbound-handling resource is assumed.
#
# Queue waiting time =
# Background workload / Inbound handling rate

warehouse_queue_wait_time_h = (
    warehouse_background_queue_workload
    / warehouse_inbound_rate
)


# ------------------------------------------------------------
# 6. Calculate inbound handling time
# ------------------------------------------------------------

# The complete consolidated customer order is handled as a batch.
#
# Inbound handling time =
# Total order quantity / Inbound handling rate

warehouse_inbound_handling_time_h = (
    order_quantity / warehouse_inbound_rate
)


# ------------------------------------------------------------
# 7. Calculate storage and staging time
# ------------------------------------------------------------

# This is the assumed period during which the completed order
# remains in the warehouse/staging area before loading.
#
# It is kept separate from queue waiting and handling time.

warehouse_storage_dwell_time_h = (
    warehouse_storage_dwell_time_h
)


# ------------------------------------------------------------
# 8. Calculate loading time
# ------------------------------------------------------------

# Loading time =
# Total order quantity / Loading rate

warehouse_loading_time_h = (
    order_quantity / warehouse_loading_rate
)


# ------------------------------------------------------------
# 9. Calculate warehouse workload and pressure indicators
# ------------------------------------------------------------

# Total inbound workload expressed in handling hours.

warehouse_background_processing_load_h = (
    warehouse_background_queue_workload
    / warehouse_inbound_rate
)

warehouse_order_handling_load_h = (
    order_quantity / warehouse_inbound_rate
)

# Total handling workload ahead of and including the new order.

warehouse_total_inbound_workload_h = (
    warehouse_background_processing_load_h
    + warehouse_order_handling_load_h
)

# Ratio between pre-existing workload and incoming order quantity.
# It is a relative backlog indicator, not a physical storage
# occupancy percentage.

warehouse_backlog_ratio = (
    warehouse_background_queue_workload
    / order_quantity
)


# ------------------------------------------------------------
# 10. Calculate total warehouse elapsed time
# ------------------------------------------------------------

# Warehouse elapsed time consists of:
#
# 1. Queue waiting.
# 2. Inbound handling.
# 3. Storage/staging.
# 4. Loading.
#
# Loading is included in warehouse completion because the order
# is not ready for transportation until it has been loaded.

warehouse_processing_time_h = (
    warehouse_inbound_handling_time_h
    + warehouse_loading_time_h
)

warehouse_total_waiting_time_h = (
    warehouse_queue_wait_time_h
    + warehouse_storage_dwell_time_h
)

warehouse_total_elapsed_time_h = (
    warehouse_queue_wait_time_h
    + warehouse_inbound_handling_time_h
    + warehouse_storage_dwell_time_h
    + warehouse_loading_time_h
)


# ------------------------------------------------------------
# 11. Calculate warehouse completion time
# ------------------------------------------------------------

# The warehouse stage starts only after production completion.

warehouse_completion_time_h = (
    warehouse_start_time_h
    + warehouse_total_elapsed_time_h
)


# ------------------------------------------------------------
# 12. Add warehouse information to the integrated scenarios
# ------------------------------------------------------------

warehouse_scenarios_df[
    "warehouse_start_time_h"
] = warehouse_start_time_h

warehouse_scenarios_df[
    "warehouse_background_queue_workload_pipes"
] = warehouse_background_queue_workload

warehouse_scenarios_df[
    "warehouse_inbound_rate_pipes_per_h"
] = warehouse_inbound_rate

warehouse_scenarios_df[
    "warehouse_loading_rate_pipes_per_h"
] = warehouse_loading_rate

warehouse_scenarios_df[
    "warehouse_queue_wait_time_h"
] = warehouse_queue_wait_time_h

warehouse_scenarios_df[
    "warehouse_inbound_handling_time_h"
] = warehouse_inbound_handling_time_h

warehouse_scenarios_df[
    "warehouse_storage_dwell_time_h"
] = warehouse_storage_dwell_time_h

warehouse_scenarios_df[
    "warehouse_loading_time_h"
] = warehouse_loading_time_h

warehouse_scenarios_df[
    "warehouse_processing_time_h"
] = warehouse_processing_time_h

warehouse_scenarios_df[
    "warehouse_total_waiting_time_h"
] = warehouse_total_waiting_time_h

warehouse_scenarios_df[
    "warehouse_total_elapsed_time_h"
] = warehouse_total_elapsed_time_h

warehouse_scenarios_df[
    "warehouse_total_inbound_workload_h"
] = warehouse_total_inbound_workload_h

warehouse_scenarios_df[
    "warehouse_backlog_ratio"
] = warehouse_backlog_ratio

warehouse_scenarios_df[
    "warehouse_completion_time_h"
] = warehouse_completion_time_h


# ------------------------------------------------------------
# 13. Quality-control checks
# ------------------------------------------------------------

# Check that all generated numerical values are finite.

warehouse_numeric_columns = [
    "warehouse_start_time_h",
    "warehouse_inbound_rate_pipes_per_h",
    "warehouse_loading_rate_pipes_per_h",
    "warehouse_queue_wait_time_h",
    "warehouse_inbound_handling_time_h",
    "warehouse_storage_dwell_time_h",
    "warehouse_loading_time_h",
    "warehouse_total_elapsed_time_h",
    "warehouse_completion_time_h"
]

if not np.isfinite(
    warehouse_scenarios_df[
        warehouse_numeric_columns
    ].to_numpy()
).all():
    raise ValueError(
        "Non-finite values were generated in the warehouse scenarios."
    )

# Queue waiting, handling, staging, and loading times
# must be non-negative.

warehouse_duration_columns = [
    "warehouse_queue_wait_time_h",
    "warehouse_inbound_handling_time_h",
    "warehouse_storage_dwell_time_h",
    "warehouse_loading_time_h",
    "warehouse_total_elapsed_time_h"
]

if (
    warehouse_scenarios_df[warehouse_duration_columns] < 0
).any().any():
    raise ValueError(
        "Negative warehouse durations were generated."
    )

# The warehouse must start after production completion.

if not np.allclose(
    warehouse_scenarios_df["warehouse_start_time_h"],
    warehouse_scenarios_df["production_completion_time_h"],
    atol=1e-8
):
    raise ValueError(
        "Warehouse start times do not match production completion times."
    )

# Warehouse completion must not precede warehouse start.

if not (
    warehouse_scenarios_df["warehouse_completion_time_h"]
    >= warehouse_scenarios_df["warehouse_start_time_h"]
).all():
    raise ValueError(
        "Warehouse completion precedes warehouse start."
    )

# Check the warehouse time balance.

calculated_warehouse_elapsed_h = (
    warehouse_scenarios_df["warehouse_queue_wait_time_h"]
    + warehouse_scenarios_df["warehouse_inbound_handling_time_h"]
    + warehouse_scenarios_df["warehouse_storage_dwell_time_h"]
    + warehouse_scenarios_df["warehouse_loading_time_h"]
)

if not np.allclose(
    warehouse_scenarios_df["warehouse_total_elapsed_time_h"],
    calculated_warehouse_elapsed_h,
    atol=1e-8
):
    raise ValueError(
        "Warehouse elapsed-time balance does not hold."
    )

# Verify the sequential production-to-warehouse relationship.

if not np.allclose(
    warehouse_scenarios_df["warehouse_completion_time_h"],
    (
        warehouse_scenarios_df["production_completion_time_h"]
        + warehouse_scenarios_df["warehouse_total_elapsed_time_h"]
    ),
    atol=1e-8
):
    raise ValueError(
        "Production-to-warehouse time integration is inconsistent."
    )


# ------------------------------------------------------------
# 14. Generate warehouse descriptive statistics
# ------------------------------------------------------------

warehouse_summary_columns = [
    "warehouse_background_queue_workload_pipes",
    "warehouse_inbound_rate_pipes_per_h",
    "warehouse_loading_rate_pipes_per_h",
    "warehouse_queue_wait_time_h",
    "warehouse_inbound_handling_time_h",
    "warehouse_storage_dwell_time_h",
    "warehouse_loading_time_h",
    "warehouse_processing_time_h",
    "warehouse_total_waiting_time_h",
    "warehouse_total_elapsed_time_h",
    "warehouse_completion_time_h",
    "warehouse_backlog_ratio"
]

warehouse_scenario_summary_df = (
    warehouse_scenarios_df[warehouse_summary_columns]
    .agg([
        "mean",
        "std",
        "min",
        "median",
        "max"
    ])
    .T
    .reset_index()
    .rename(columns={"index": "metric"})
)


# ------------------------------------------------------------
# 15. Display results
# ------------------------------------------------------------

print("CELL 7 COMPLETED SUCCESSFULLY")
print("=" * 60)

print(f"Reference customer order: {customer_order_id}")
print(f"Number of warehouse scenarios: {N_WAREHOUSE_SCENARIOS}")
print(
    f"Total order quantity: "
    f"{order_quantity[0]:.0f} pipes"
)

print("-" * 60)

print(
    f"Mean production completion time: "
    f"{warehouse_scenarios_df['production_completion_time_h'].mean():.2f} h"
)

print(
    f"Mean warehouse queue waiting time: "
    f"{warehouse_scenarios_df['warehouse_queue_wait_time_h'].mean():.2f} h"
)

print(
    f"Mean inbound handling time: "
    f"{warehouse_scenarios_df['warehouse_inbound_handling_time_h'].mean():.2f} h"
)

print(
    f"Mean storage/staging time: "
    f"{warehouse_scenarios_df['warehouse_storage_dwell_time_h'].mean():.2f} h"
)

print(
    f"Mean loading time: "
    f"{warehouse_scenarios_df['warehouse_loading_time_h'].mean():.2f} h"
)

print(
    f"Mean total warehouse elapsed time: "
    f"{warehouse_scenarios_df['warehouse_total_elapsed_time_h'].mean():.2f} h"
)

print(
    f"Mean warehouse completion time: "
    f"{warehouse_scenarios_df['warehouse_completion_time_h'].mean():.2f} h"
)

print("-" * 60)

display(warehouse_scenario_summary_df)

display(warehouse_scenarios_df.head())


CELL 7 COMPLETED SUCCESSFULLY
Reference customer order: CO-001
Number of warehouse scenarios: 2000
Total order quantity: 190 pipes
------------------------------------------------------------
Mean production completion time: 237.42 h
Mean warehouse queue waiting time: 4.34 h
Mean inbound handling time: 10.38 h
Mean storage/staging time: 1.99 h
Mean loading time: 8.70 h
Mean total warehouse elapsed time: 25.40 h
Mean warehouse completion time: 262.82 h
------------------------------------------------------------


,metric,mean,std,min,median,max
0,warehouse_background_queue_workload_pipes,79.182904,38.714666,10.490339,72.732018,272.910794
1,warehouse_inbound_rate_pipes_per_h,18.736641,2.810049,12.210410,18.599394,25.909102
2,warehouse_loading_rate_pipes_per_h,22.263109,3.065913,15.272672,22.185862,29.689649
3,warehouse_queue_wait_time_h,4.335445,2.277884,0.501603,3.871511,15.824562
4,warehouse_inbound_handling_time_h,10.377422,1.605388,7.333330,10.215387,15.560493
5,warehouse_storage_dwell_time_h,1.986517,0.743441,0.542190,1.889239,3.985670
6,warehouse_loading_time_h,8.702856,1.241921,6.399537,8.564012,12.440521
7,warehouse_processing_time_h,19.080278,2.002520,14.409511,18.951202,26.845722
8,warehouse_total_waiting_time_h,6.321962,2.384850,1.475161,5.906233,18.182982
9,warehouse_total_elapsed_time_h,25.402240,3.462032,16.565277,25.027698,40.178175


,scenario_id,customer_order_id,order_quantity,number_of_jobs,production_baseline_flow_time_h,production_processing_time_h,production_setup_time_h,interstage_nonprocessing_time_h,production_downtime_h,production_time_increase_h,...,warehouse_queue_wait_time_h,warehouse_inbound_handling_time_h,warehouse_storage_dwell_time_h,warehouse_loading_time_h,warehouse_processing_time_h,warehouse_total_waiting_time_h,warehouse_total_elapsed_time_h,warehouse_total_inbound_workload_h,warehouse_backlog_ratio,warehouse_completion_time_h
0,1,CO-001,190.0,3,215.51,181.81,16.1,17.6,11.842324,11.842324,...,2.897136,10.634416,2.392722,6.967930,17.602347,5.289858,22.892205,13.531553,0.272430,250.244528
1,2,CO-001,190.0,3,215.51,181.81,16.1,17.6,16.125065,16.125065,...,2.776766,12.605710,2.230981,8.292749,20.898459,5.007747,25.906206,15.382475,0.220278,257.541271
2,3,CO-001,190.0,3,215.51,181.81,16.1,17.6,27.528919,27.528919,...,4.991484,12.553421,1.747203,7.886557,20.439978,6.738687,27.178666,17.544905,0.397619,270.217584
3,4,CO-001,190.0,3,215.51,181.81,16.1,17.6,39.028477,39.028477,...,5.407261,10.456086,1.224337,10.261091,20.717177,6.631599,27.348776,15.863347,0.517140,281.887253
4,5,CO-001,190.0,3,215.51,181.81,16.1,17.6,11.212367,11.212367,...,6.571688,11.678904,2.030495,7.392035,19.070939,8.602183,27.673122,18.250592,0.562697,254.395489


In [24]:

# ============================================================
# CELL 8 — TRANSPORTATION OPERATIONS MODELING
# ============================================================
# Research framework:
# Integrated production–warehouse–transportation system
#
# Purpose:
# 1. Receive completed and loaded customer orders from Cell 7.
# 2. Model vehicle availability and dispatch waiting.
# 3. Model normal transportation time.
# 4. Incorporate traffic, weather, and road conditions.
# 5. Calculate additional transportation delay.
# 6. Estimate customer-order delivery completion time.
#
# Important:
# - The reference order has one specified destination.
# - Route distance is constant across its scenarios.
# - Transportation conditions vary stochastically.
# - Numerical parameters are provisional assumptions.
# - No actual company transportation records are claimed.
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Validate inputs from Cell 7
# ------------------------------------------------------------

required_warehouse_columns = [
    "scenario_id",
    "customer_order_id",
    "order_quantity",
    "warehouse_completion_time_h",
    "warehouse_loading_time_h",
    "warehouse_total_elapsed_time_h"
]

missing_warehouse_columns = [
    col for col in required_warehouse_columns
    if col not in warehouse_scenarios_df.columns
]

if missing_warehouse_columns:
    raise ValueError(
        "Missing columns in warehouse_scenarios_df: "
        f"{missing_warehouse_columns}. "
        "Run Cells 1–7 before running Cell 8."
    )

if "rng" not in globals():
    raise RuntimeError(
        "Random generator 'rng' is missing. "
        "Run Cell 1 before running Cell 8."
    )

if warehouse_scenarios_df.empty:
    raise ValueError(
        "warehouse_scenarios_df is empty."
    )

if warehouse_scenarios_df["scenario_id"].duplicated().any():
    raise ValueError(
        "Duplicate scenario IDs detected in warehouse_scenarios_df."
    )

if warehouse_scenarios_df["customer_order_id"].nunique() != 1:
    raise ValueError(
        "This version of Cell 8 expects one reference customer order."
    )


# ------------------------------------------------------------
# 2. Preserve the integrated production-warehouse scenarios
# ------------------------------------------------------------

transportation_scenarios_df = (
    warehouse_scenarios_df.copy()
    .reset_index(drop=True)
)

N_TRANSPORT_SCENARIOS = len(transportation_scenarios_df)

customer_order_id = (
    transportation_scenarios_df["customer_order_id"].iloc[0]
)

order_quantity = (
    transportation_scenarios_df["order_quantity"]
    .astype(float)
    .to_numpy()
)

# The order becomes available for transportation only after
# warehouse operations, including loading, are completed.

transportation_start_time_h = (
    transportation_scenarios_df[
        "warehouse_completion_time_h"
    ].astype(float).to_numpy()
)


# ------------------------------------------------------------
# 3. Define destination and route distance
# ------------------------------------------------------------

# The exact border destination and actual road distance must
# be confirmed using company dispatch records or a verified route.
#
# The following distance is a temporary illustrative placeholder.
# Replace it with the actual one-way road distance in kilometers.

REFERENCE_DESTINATION_CODE = (
    "BORDER_DESTINATION_TO_CONFIRM"
)

REFERENCE_ROUTE_DISTANCE_KM = 300.0

ROUTE_DISTANCE_STATUS = (
    "Illustrative placeholder only; replace with verified "
    "one-way road distance to the actual customer destination."
)

# Determine destination from the available case-study information.

if (
    "destination_code" in production_order_summary_df.columns
    and pd.notna(
        production_order_summary_df["destination_code"].iloc[0]
    )
):
    destination_code = (
        production_order_summary_df["destination_code"].iloc[0]
    )
elif "DESTINATION_CODE" in globals():
    destination_code = DESTINATION_CODE
else:
    destination_code = REFERENCE_DESTINATION_CODE

# Use the placeholder distance for all scenarios of the same order.
# Do not sample a different physical route distance for each replication.

delivery_distance_km = np.full(
    N_TRANSPORT_SCENARIOS,
    REFERENCE_ROUTE_DISTANCE_KM,
    dtype=float
)


# ------------------------------------------------------------
# 4. Define provisional transportation parameters
# ------------------------------------------------------------

# A. Uncongested heavy-vehicle average speed (km/h).
#
# This is a scenario assumption, not a recorded truck speed.

BASE_SPEED_MIN_KMH = 45.0
BASE_SPEED_MODE_KMH = 60.0
BASE_SPEED_MAX_KMH = 75.0

# B. Vehicle availability.
#
# This represents the effective proportion of the delivery fleet
# available for dispatch, not the availability of a specific truck.

VEHICLE_AVAILABILITY_BETA_ALPHA = 16.0
VEHICLE_AVAILABILITY_BETA_BETA = 4.0

MIN_VEHICLE_AVAILABILITY = 0.60
MAX_VEHICLE_AVAILABILITY = 1.00

# C. Dispatch waiting calibration parameter.
#
# This is a provisional reference-time scale used to convert
# vehicle unavailability into dispatch waiting time.

VEHICLE_DISPATCH_REFERENCE_WAIT_H = 8.0

# D. Traffic categories:
# 1 = Low
# 2 = Moderate
# 3 = High
# 4 = Severe

TRAFFIC_LEVELS = [1, 2, 3, 4]

TRAFFIC_PROBABILITIES = [
    0.25,
    0.40,
    0.25,
    0.10
]

TRAFFIC_TIME_MULTIPLIERS = {
    1: 1.00,
    2: 1.15,
    3: 1.35,
    4: 1.60
}

# E. Weather categories:
# 1 = Favorable
# 2 = Adverse
# 3 = Severe

WEATHER_LEVELS = [1, 2, 3]

WEATHER_PROBABILITIES = [
    0.65,
    0.25,
    0.10
]

WEATHER_TIME_MULTIPLIERS = {
    1: 1.00,
    2: 1.10,
    3: 1.25
}

# F. Road complexity categories:
# 1 = Good
# 2 = Moderate
# 3 = Difficult

ROAD_COMPLEXITY_LEVELS = [1, 2, 3]

ROAD_COMPLEXITY_PROBABILITIES = [
    0.40,
    0.40,
    0.20
]

ROAD_TIME_MULTIPLIERS = {
    1: 1.00,
    2: 1.08,
    3: 1.20
}


# ------------------------------------------------------------
# 5. Generate transportation operating conditions
# ------------------------------------------------------------

# Base driving speed.

base_travel_speed_kmh = rng.triangular(
    BASE_SPEED_MIN_KMH,
    BASE_SPEED_MODE_KMH,
    BASE_SPEED_MAX_KMH,
    size=N_TRANSPORT_SCENARIOS
)

# Vehicle availability.

vehicle_availability = rng.beta(
    VEHICLE_AVAILABILITY_BETA_ALPHA,
    VEHICLE_AVAILABILITY_BETA_BETA,
    size=N_TRANSPORT_SCENARIOS
)

vehicle_availability = np.clip(
    vehicle_availability,
    MIN_VEHICLE_AVAILABILITY,
    MAX_VEHICLE_AVAILABILITY
)

# Traffic conditions.

traffic_level = rng.choice(
    TRAFFIC_LEVELS,
    size=N_TRANSPORT_SCENARIOS,
    p=TRAFFIC_PROBABILITIES
)

# Weather conditions.

weather_condition = rng.choice(
    WEATHER_LEVELS,
    size=N_TRANSPORT_SCENARIOS,
    p=WEATHER_PROBABILITIES
)

# Road complexity.

road_complexity = rng.choice(
    ROAD_COMPLEXITY_LEVELS,
    size=N_TRANSPORT_SCENARIOS,
    p=ROAD_COMPLEXITY_PROBABILITIES
)


# ------------------------------------------------------------
# 6. Convert operational conditions into time multipliers
# ------------------------------------------------------------

traffic_time_multiplier = np.array([
    TRAFFIC_TIME_MULTIPLIERS[int(level)]
    for level in traffic_level
])

weather_time_multiplier = np.array([
    WEATHER_TIME_MULTIPLIERS[int(level)]
    for level in weather_condition
])

road_time_multiplier = np.array([
    ROAD_TIME_MULTIPLIERS[int(level)]
    for level in road_complexity
])

# Combined multiplier represents the cumulative influence
# of traffic, weather, and road conditions on driving time.

combined_travel_time_multiplier = (
    traffic_time_multiplier
    * weather_time_multiplier
    * road_time_multiplier
)


# ------------------------------------------------------------
# 7. Calculate normal transportation time
# ------------------------------------------------------------

# Normal travel time is calculated using:
#
# Distance / baseline driving speed
#
# It represents the reference travel duration without
# the additional effects of adverse traffic, weather,
# or road conditions.

transportation_base_travel_time_h = (
    delivery_distance_km
    / base_travel_speed_kmh
)


# ------------------------------------------------------------
# 8. Calculate realized transportation time
# ------------------------------------------------------------

# Actual driving time accounts for the operating conditions.

transportation_realized_travel_time_h = (
    transportation_base_travel_time_h
    * combined_travel_time_multiplier
)


# ------------------------------------------------------------
# 9. Calculate additional travel-time delay
# ------------------------------------------------------------

transportation_travel_time_delay_h = (
    transportation_realized_travel_time_h
    - transportation_base_travel_time_h
)

# Numerical protection against negligible floating-point values.

transportation_travel_time_delay_h = np.maximum(
    transportation_travel_time_delay_h,
    0.0
)


# ------------------------------------------------------------
# 10. Calculate vehicle dispatch waiting time
# ------------------------------------------------------------

# A simple availability-adjusted approximation:
#
# Dispatch waiting =
# Reference waiting scale * (1 - availability) / availability
#
# This is a provisional representation of the delay associated
# with limited vehicle availability.
#
# It does not represent an explicit fleet-size or vehicle-routing
# optimization model.

vehicle_dispatch_wait_time_h = (
    VEHICLE_DISPATCH_REFERENCE_WAIT_H
    * (1.0 - vehicle_availability)
    / vehicle_availability
)


# ------------------------------------------------------------
# 11. Calculate cross-condition interaction indicators
# ------------------------------------------------------------

# These indicators describe compound operating conditions.
# They are not causal effects.

weather_road_interaction = (
    (weather_time_multiplier - 1.0)
    * (road_time_multiplier - 1.0)
)

traffic_weather_interaction = (
    (traffic_time_multiplier - 1.0)
    * (weather_time_multiplier - 1.0)
)

traffic_road_interaction = (
    (traffic_time_multiplier - 1.0)
    * (road_time_multiplier - 1.0)
)


# ------------------------------------------------------------
# 12. Calculate total transportation elapsed time
# ------------------------------------------------------------

# Transportation elapsed time comprises:
#
# 1. Vehicle dispatch waiting time.
# 2. Realized driving time.
#
# Normal driving time is not itself an additional delay.
# It is part of the actual delivery lead time.

transportation_total_elapsed_time_h = (
    vehicle_dispatch_wait_time_h
    + transportation_realized_travel_time_h
)

# Total transportation delay relative to normal travel:
#
# Dispatch waiting + additional driving time.

transportation_total_delay_h = (
    vehicle_dispatch_wait_time_h
    + transportation_travel_time_delay_h
)


# ------------------------------------------------------------
# 13. Calculate actual customer-order delivery completion time
# ------------------------------------------------------------

# Transportation starts only after warehouse completion.

transportation_completion_time_h = (
    transportation_start_time_h
    + transportation_total_elapsed_time_h
)

# At this stage, actual delivery completion time is the
# completion time of the transportation stage.
#
# The promised delivery time will be introduced later when
# the final customer-order delivery-lateness target is constructed.

actual_delivery_time_h = (
    transportation_completion_time_h
)


# ------------------------------------------------------------
# 14. Add transportation variables to the integrated dataset
# ------------------------------------------------------------

transportation_scenarios_df[
    "destination_code"
] = destination_code

transportation_scenarios_df[
    "delivery_distance_km"
] = delivery_distance_km

transportation_scenarios_df[
    "base_travel_speed_kmh"
] = base_travel_speed_kmh

transportation_scenarios_df[
    "vehicle_availability"
] = vehicle_availability

transportation_scenarios_df[
    "vehicle_availability_pct"
] = vehicle_availability * 100

transportation_scenarios_df[
    "traffic_level"
] = traffic_level

transportation_scenarios_df[
    "weather_condition"
] = weather_condition

transportation_scenarios_df[
    "road_complexity"
] = road_complexity

transportation_scenarios_df[
    "traffic_time_multiplier"
] = traffic_time_multiplier

transportation_scenarios_df[
    "weather_time_multiplier"
] = weather_time_multiplier

transportation_scenarios_df[
    "road_time_multiplier"
] = road_time_multiplier

transportation_scenarios_df[
    "combined_travel_time_multiplier"
] = combined_travel_time_multiplier

transportation_scenarios_df[
    "transportation_start_time_h"
] = transportation_start_time_h

transportation_scenarios_df[
    "transportation_base_travel_time_h"
] = transportation_base_travel_time_h

transportation_scenarios_df[
    "transportation_realized_travel_time_h"
] = transportation_realized_travel_time_h

transportation_scenarios_df[
    "transportation_travel_time_delay_h"
] = transportation_travel_time_delay_h

transportation_scenarios_df[
    "vehicle_dispatch_wait_time_h"
] = vehicle_dispatch_wait_time_h

transportation_scenarios_df[
    "weather_road_interaction"
] = weather_road_interaction

transportation_scenarios_df[
    "traffic_weather_interaction"
] = traffic_weather_interaction

transportation_scenarios_df[
    "traffic_road_interaction"
] = traffic_road_interaction

transportation_scenarios_df[
    "transportation_total_elapsed_time_h"
] = transportation_total_elapsed_time_h

transportation_scenarios_df[
    "transportation_total_delay_h"
] = transportation_total_delay_h

transportation_scenarios_df[
    "transportation_completion_time_h"
] = transportation_completion_time_h

transportation_scenarios_df[
    "actual_delivery_time_h"
] = actual_delivery_time_h


# ------------------------------------------------------------
# 15. Quality-control checks
# ------------------------------------------------------------

transportation_duration_columns = [
    "transportation_base_travel_time_h",
    "transportation_realized_travel_time_h",
    "transportation_travel_time_delay_h",
    "vehicle_dispatch_wait_time_h",
    "transportation_total_elapsed_time_h",
    "transportation_total_delay_h"
]

if (
    transportation_scenarios_df[
        transportation_duration_columns
    ] < 0
).any().any():
    raise ValueError(
        "Negative transportation durations were generated."
    )

if not np.isfinite(
    transportation_scenarios_df[
        transportation_duration_columns
    ].to_numpy()
).all():
    raise ValueError(
        "Non-finite transportation duration values were generated."
    )

if not transportation_scenarios_df[
    "vehicle_availability"
].between(
    MIN_VEHICLE_AVAILABILITY,
    MAX_VEHICLE_AVAILABILITY
).all():
    raise ValueError(
        "Vehicle availability is outside the specified bounds."
    )

if not np.allclose(
    transportation_scenarios_df[
        "transportation_start_time_h"
    ],
    transportation_scenarios_df[
        "warehouse_completion_time_h"
    ],
    atol=1e-8
):
    raise ValueError(
        "Transportation start time does not match warehouse completion."
    )

# Verify normal travel time calculation.

calculated_base_travel_time_h = (
    transportation_scenarios_df["delivery_distance_km"]
    / transportation_scenarios_df["base_travel_speed_kmh"]
)

if not np.allclose(
    transportation_scenarios_df[
        "transportation_base_travel_time_h"
    ],
    calculated_base_travel_time_h,
    atol=1e-8
):
    raise ValueError(
        "Base transportation time calculation is inconsistent."
    )

# Verify realized travel time.

calculated_realized_travel_time_h = (
    transportation_scenarios_df[
        "transportation_base_travel_time_h"
    ]
    * transportation_scenarios_df[
        "combined_travel_time_multiplier"
    ]
)

if not np.allclose(
    transportation_scenarios_df[
        "transportation_realized_travel_time_h"
    ],
    calculated_realized_travel_time_h,
    atol=1e-8
):
    raise ValueError(
        "Realized transportation time calculation is inconsistent."
    )

# Verify transportation elapsed time.

calculated_transport_elapsed_h = (
    transportation_scenarios_df[
        "vehicle_dispatch_wait_time_h"
    ]
    + transportation_scenarios_df[
        "transportation_realized_travel_time_h"
    ]
)

if not np.allclose(
    transportation_scenarios_df[
        "transportation_total_elapsed_time_h"
    ],
    calculated_transport_elapsed_h,
    atol=1e-8
):
    raise ValueError(
        "Transportation elapsed-time balance does not hold."
    )

# Verify total transportation delay.

calculated_transport_delay_h = (
    transportation_scenarios_df[
        "vehicle_dispatch_wait_time_h"
    ]
    + transportation_scenarios_df[
        "transportation_travel_time_delay_h"
    ]
)

if not np.allclose(
    transportation_scenarios_df[
        "transportation_total_delay_h"
    ],
    calculated_transport_delay_h,
    atol=1e-8
):
    raise ValueError(
        "Transportation delay calculation is inconsistent."
    )

# Verify actual order delivery completion time.

calculated_delivery_time_h = (
    transportation_scenarios_df[
        "warehouse_completion_time_h"
    ]
    + transportation_scenarios_df[
        "transportation_total_elapsed_time_h"
    ]
)

if not np.allclose(
    transportation_scenarios_df[
        "actual_delivery_time_h"
    ],
    calculated_delivery_time_h,
    atol=1e-8
):
    raise ValueError(
        "Integrated delivery completion time is inconsistent."
    )


# ------------------------------------------------------------
# 16. Generate transportation descriptive statistics
# ------------------------------------------------------------

transportation_summary_columns = [
    "delivery_distance_km",
    "base_travel_speed_kmh",
    "vehicle_availability_pct",
    "traffic_level",
    "weather_condition",
    "road_complexity",
    "vehicle_dispatch_wait_time_h",
    "transportation_base_travel_time_h",
    "transportation_realized_travel_time_h",
    "transportation_travel_time_delay_h",
    "transportation_total_elapsed_time_h",
    "transportation_total_delay_h",
    "actual_delivery_time_h"
]

transportation_scenario_summary_df = (
    transportation_scenarios_df[
        transportation_summary_columns
    ]
    .agg([
        "mean",
        "std",
        "min",
        "median",
        "max"
    ])
    .T
    .reset_index()
    .rename(columns={"index": "metric"})
)


# ------------------------------------------------------------
# 17. Display results
# ------------------------------------------------------------

print("CELL 8 COMPLETED SUCCESSFULLY")
print("=" * 65)

print(f"Reference customer order: {customer_order_id}")
print(f"Destination: {destination_code}")
print(f"Number of transportation scenarios: {N_TRANSPORT_SCENARIOS}")
print(
    f"Total order quantity: "
    f"{order_quantity[0]:.0f} pipes"
)

print("-" * 65)

print(
    f"Route distance (provisional): "
    f"{REFERENCE_ROUTE_DISTANCE_KM:.2f} km"
)

print(
    f"Mean vehicle availability: "
    f"{transportation_scenarios_df['vehicle_availability_pct'].mean():.2f}%"
)

print(
    f"Mean dispatch waiting time: "
    f"{transportation_scenarios_df['vehicle_dispatch_wait_time_h'].mean():.2f} h"
)

print(
    f"Mean normal travel time: "
    f"{transportation_scenarios_df['transportation_base_travel_time_h'].mean():.2f} h"
)

print(
    f"Mean realized travel time: "
    f"{transportation_scenarios_df['transportation_realized_travel_time_h'].mean():.2f} h"
)

print(
    f"Mean additional travel-time delay: "
    f"{transportation_scenarios_df['transportation_travel_time_delay_h'].mean():.2f} h"
)

print(
    f"Mean total transportation elapsed time: "
    f"{transportation_scenarios_df['transportation_total_elapsed_time_h'].mean():.2f} h"
)

print(
    f"Mean transportation completion time: "
    f"{transportation_scenarios_df['actual_delivery_time_h'].mean():.2f} h"
)

print("-" * 65)

print("ROUTE DISTANCE STATUS:")
print(ROUTE_DISTANCE_STATUS)

display(transportation_scenario_summary_df)

display(transportation_scenarios_df.head())


CELL 8 COMPLETED SUCCESSFULLY
Reference customer order: CO-001
Destination: BORDER_DESTINATION_TO_CONFIRM
Number of transportation scenarios: 2000
Total order quantity: 190 pipes
-----------------------------------------------------------------
Route distance (provisional): 300.00 km
Mean vehicle availability: 80.04%
Mean dispatch waiting time: 2.12 h
Mean normal travel time: 5.08 h
Mean realized travel time: 6.91 h
Mean additional travel-time delay: 1.83 h
Mean total transportation elapsed time: 9.02 h
Mean transportation completion time: 271.84 h
-----------------------------------------------------------------
ROUTE DISTANCE STATUS:
Illustrative placeholder only; replace with verified one-way road distance to the actual customer destination.


,metric,mean,std,min,median,max
0,delivery_distance_km,300.000000,0.000000,300.000000,300.000000,300.000000
1,base_travel_speed_kmh,59.721934,6.041716,45.136318,59.906731,74.388136
2,vehicle_availability_pct,80.038186,8.512131,60.000000,80.933158,97.760624
3,traffic_level,2.202000,0.942677,1.000000,2.000000,4.000000
4,weather_condition,1.446500,0.662840,1.000000,1.000000,3.000000
5,road_complexity,1.809500,0.752657,1.000000,2.000000,3.000000
6,vehicle_dispatch_wait_time_h,2.116995,1.155832,0.183254,1.884700,5.333333
7,transportation_base_travel_time_h,5.076042,0.525821,4.032901,5.007785,6.646532
8,transportation_realized_travel_time_h,6.906519,1.449591,4.038741,6.684421,13.381507
9,transportation_travel_time_delay_h,1.830477,1.280213,0.000000,1.624136,7.342223


,scenario_id,customer_order_id,order_quantity,number_of_jobs,production_baseline_flow_time_h,production_processing_time_h,production_setup_time_h,interstage_nonprocessing_time_h,production_downtime_h,production_time_increase_h,...,transportation_realized_travel_time_h,transportation_travel_time_delay_h,vehicle_dispatch_wait_time_h,weather_road_interaction,traffic_weather_interaction,traffic_road_interaction,transportation_total_elapsed_time_h,transportation_total_delay_h,transportation_completion_time_h,actual_delivery_time_h
0,1,CO-001,190.0,3,215.51,181.81,16.1,17.6,11.842324,11.842324,...,5.586135,0.000000,3.214148,0.00,0.0000,0.00,8.800283,3.214148,259.044811,259.044811
1,2,CO-001,190.0,3,215.51,181.81,16.1,17.6,16.125065,16.125065,...,7.688075,2.117006,0.903465,0.00,0.0000,0.03,8.591540,3.020471,266.132811,266.132811
2,3,CO-001,190.0,3,215.51,181.81,16.1,17.6,27.528919,27.528919,...,7.361438,2.240438,3.647601,0.00,0.0375,0.00,11.009039,5.888039,281.226623,281.226623
3,4,CO-001,190.0,3,215.51,181.81,16.1,17.6,39.028477,39.028477,...,8.079604,2.459010,3.961075,0.00,0.0375,0.00,12.040679,6.420085,293.927932,293.927932
4,5,CO-001,190.0,3,215.51,181.81,16.1,17.6,11.212367,11.212367,...,8.776403,2.994846,3.723981,0.02,0.0150,0.03,12.500384,6.718827,266.895873,266.895873


In [25]:

# ============================================================
# CELL 9 — INTEGRATED DATASET AND DELIVERY-TARGET CONSTRUCTION
# ============================================================
# Research framework:
# Integrated production–warehouse–transportation system
#
# Purpose:
# 1. Integrate production, warehouse, and transportation outputs.
# 2. Verify chronological continuity between all three stages.
# 3. Calculate actual customer-order delivery lead time.
# 4. Incorporate the promised delivery lead time from the order.
# 5. Calculate signed delivery deviation and non-negative lateness.
# 6. Prepare the integrated dataset for ML feature engineering.
#
# Observation unit:
# One consolidated customer order per simulation scenario.
#
# Primary target:
# delivery_lateness_h
#
# IMPORTANT:
# The promised delivery period must come from the actual customer order.
# Do not replace it with an arbitrary simulation-generated deadline.
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Validate inputs from Cells 6–8
# ------------------------------------------------------------

required_production_columns = [
    "scenario_id",
    "customer_order_id",
    "production_completion_time_h",
    "production_time_h"
]

required_warehouse_columns = [
    "scenario_id",
    "customer_order_id",
    "warehouse_start_time_h",
    "warehouse_completion_time_h",
    "warehouse_total_elapsed_time_h"
]

required_transport_columns = [
    "scenario_id",
    "customer_order_id",
    "transportation_start_time_h",
    "transportation_completion_time_h",
    "actual_delivery_time_h",
    "transportation_total_elapsed_time_h"
]

def check_required_columns(dataframe, required_columns, dataframe_name):

    missing = [
        column
        for column in required_columns
        if column not in dataframe.columns
    ]

    if missing:
        raise ValueError(
            f"Missing columns in {dataframe_name}: {missing}"
        )


check_required_columns(
    production_scenarios_df,
    required_production_columns,
    "production_scenarios_df"
)

check_required_columns(
    warehouse_scenarios_df,
    required_warehouse_columns,
    "warehouse_scenarios_df"
)

check_required_columns(
    transportation_scenarios_df,
    required_transport_columns,
    "transportation_scenarios_df"
)


# ------------------------------------------------------------
# 2. Check scenario consistency
# ------------------------------------------------------------

# Every production scenario must have exactly one corresponding
# warehouse scenario and one transportation scenario.

for dataframe, dataframe_name in [
    (production_scenarios_df, "production"),
    (warehouse_scenarios_df, "warehouse"),
    (transportation_scenarios_df, "transportation")
]:

    if dataframe["scenario_id"].duplicated().any():
        raise ValueError(
            f"Duplicate scenario IDs in {dataframe_name} data."
        )

    if dataframe.empty:
        raise ValueError(
            f"{dataframe_name} scenario data are empty."
        )

production_ids = set(
    production_scenarios_df["scenario_id"]
)

warehouse_ids = set(
    warehouse_scenarios_df["scenario_id"]
)

transportation_ids = set(
    transportation_scenarios_df["scenario_id"]
)

if not (
    production_ids == warehouse_ids == transportation_ids
):
    raise ValueError(
        "Production, warehouse, and transportation scenarios "
        "do not have matching scenario IDs."
    )

if (
    production_scenarios_df["customer_order_id"].nunique() != 1
    or warehouse_scenarios_df["customer_order_id"].nunique() != 1
    or transportation_scenarios_df["customer_order_id"].nunique() != 1
):
    raise ValueError(
        "This version of Cell 9 expects one reference customer order."
    )

order_ids = {
    production_scenarios_df["customer_order_id"].iloc[0],
    warehouse_scenarios_df["customer_order_id"].iloc[0],
    transportation_scenarios_df["customer_order_id"].iloc[0]
}

if len(order_ids) != 1:
    raise ValueError(
        "Customer-order IDs differ between the operational stages."
    )

customer_order_id = order_ids.pop()

N_INTEGRATED_SCENARIOS = len(production_scenarios_df)


# ------------------------------------------------------------
# 3. Retrieve order-release time
# ------------------------------------------------------------

if (
    "order_release_time_h" in production_order_summary_df.columns
):

    order_release_time_h = float(
        production_order_summary_df[
            "order_release_time_h"
        ].iloc[0]
    )

else:
    order_release_time_h = 0.0

if not np.isfinite(order_release_time_h):
    raise ValueError(
        "Invalid customer-order release time."
    )


# ------------------------------------------------------------
# 4. Integrate the three operational stages
# ------------------------------------------------------------

# Transportation scenarios already retain the previous
# production and warehouse variables.
#
# We use them as the main integrated dataset and independently
# verify that the corresponding stage outputs agree.

integrated_delivery_dataset_df = (
    transportation_scenarios_df.copy()
    .sort_values("scenario_id")
    .reset_index(drop=True)
)

# Sort the other stage data to ensure consistent scenario order.

production_check_df = (
    production_scenarios_df
    .sort_values("scenario_id")
    .reset_index(drop=True)
)

warehouse_check_df = (
    warehouse_scenarios_df
    .sort_values("scenario_id")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 5. Verify production-to-warehouse continuity
# ------------------------------------------------------------

if not np.allclose(
    warehouse_check_df["warehouse_start_time_h"],
    production_check_df["production_completion_time_h"],
    atol=1e-8
):
    raise ValueError(
        "Production completion and warehouse start times "
        "are not consistent."
    )


# ------------------------------------------------------------
# 6. Verify warehouse-to-transportation continuity
# ------------------------------------------------------------

if not np.allclose(
    integrated_delivery_dataset_df[
        "transportation_start_time_h"
    ],
    warehouse_check_df[
        "warehouse_completion_time_h"
    ],
    atol=1e-8
):
    raise ValueError(
        "Warehouse completion and transportation start times "
        "are not consistent."
    )


# ------------------------------------------------------------
# 7. Verify delivery completion time
# ------------------------------------------------------------

if not np.allclose(
    integrated_delivery_dataset_df["actual_delivery_time_h"],
    integrated_delivery_dataset_df[
        "transportation_completion_time_h"
    ],
    atol=1e-8
):
    raise ValueError(
        "Actual delivery time does not match transportation completion."
    )


# ------------------------------------------------------------
# 8. Calculate actual customer-order delivery lead time
# ------------------------------------------------------------

# Actual delivery lead time is measured from order release
# until the simulated order reaches its destination.
#
# This is different from delivery lateness.
#
# Delivery lead time =
# Actual delivery completion time - Order release time.

integrated_delivery_dataset_df[
    "order_release_time_h"
] = order_release_time_h

integrated_delivery_dataset_df[
    "actual_delivery_lead_time_h"
] = (
    integrated_delivery_dataset_df["actual_delivery_time_h"]
    - order_release_time_h
)

if (
    integrated_delivery_dataset_df[
        "actual_delivery_lead_time_h"
    ] < 0
).any():
    raise ValueError(
        "Actual delivery lead time cannot be negative."
    )


# ------------------------------------------------------------
# 9. Define promised delivery lead time
# ------------------------------------------------------------


PROMISED_DELIVERY_DAYS_BY_ORDER = {
    "CO-001": None
}

promised_delivery_days = (
    PROMISED_DELIVERY_DAYS_BY_ORDER.get(
        str(customer_order_id),
        None
    )
)

# If a promised delivery period is already stored in the
# customer-order summary, use it when the dictionary is empty.

if promised_delivery_days is None:

    if (
        "promised_delivery_days"
        in production_order_summary_df.columns
    ):

        promised_delivery_days = (
            production_order_summary_df[
                "promised_delivery_days"
            ].iloc[0]
        )

# Validate the contractual delivery period.

if promised_delivery_days is not None:

    promised_delivery_days = float(
        promised_delivery_days
    )

    if (
        not np.isfinite(promised_delivery_days)
        or promised_delivery_days <= 0
    ):
        raise ValueError(
            "Promised delivery days must be a positive finite number."
        )


# ------------------------------------------------------------
# 10. Calculate promised delivery time
# ------------------------------------------------------------

# Contractual promised delivery period is converted from days to hours.
#
# Promised delivery deadline =
# Order release time + Promised delivery days * 24.

if promised_delivery_days is not None:

    promised_delivery_lead_time_h = (
        promised_delivery_days * 24.0
    )

    promised_delivery_deadline_h = (
        order_release_time_h
        + promised_delivery_lead_time_h
    )

    integrated_delivery_dataset_df[
        "promised_delivery_days"
    ] = promised_delivery_days

    integrated_delivery_dataset_df[
        "promised_delivery_lead_time_h"
    ] = promised_delivery_lead_time_h

    integrated_delivery_dataset_df[
        "promised_delivery_deadline_h"
    ] = promised_delivery_deadline_h

else:

    # Keep the target columns available, but do not fabricate
    # a promised delivery period.

    integrated_delivery_dataset_df[
        "promised_delivery_days"
    ] = np.nan

    integrated_delivery_dataset_df[
        "promised_delivery_lead_time_h"
    ] = np.nan

    integrated_delivery_dataset_df[
        "promised_delivery_deadline_h"
    ] = np.nan


# ------------------------------------------------------------
# 11. Calculate signed delivery deviation and lateness
# ------------------------------------------------------------

if promised_delivery_days is not None:

    # Signed deviation:
    # Positive = late delivery.
    # Negative = early delivery.
    # Zero = exactly on the promised deadline.

    signed_delivery_deviation_h = (
        integrated_delivery_dataset_df[
            "actual_delivery_time_h"
        ]
        - integrated_delivery_dataset_df[
            "promised_delivery_deadline_h"
        ]
    )

    integrated_delivery_dataset_df[
        "signed_delivery_deviation_h"
    ] = signed_delivery_deviation_h

    # Primary research target:
    # Non-negative customer-order delivery lateness.

    integrated_delivery_dataset_df[
        "delivery_lateness_h"
    ] = np.maximum(
        signed_delivery_deviation_h,
        0.0
    )

    # Early delivery is stored separately.

    integrated_delivery_dataset_df[
        "early_delivery_h"
    ] = np.maximum(
        -signed_delivery_deviation_h,
        0.0
    )

    # On-time delivery indicator.

    integrated_delivery_dataset_df[
        "on_time_delivery"
    ] = (
        signed_delivery_deviation_h <= 0
    )

    # Delivery status.

    integrated_delivery_dataset_df[
        "delivery_status"
    ] = np.where(
        signed_delivery_deviation_h > 0,
        "Late",
        "On time"
    )

    DELIVERY_TARGET_STATUS = "CALCULATED"

else:

    # The true target cannot be calculated without the
    # contractual promised delivery period.

    integrated_delivery_dataset_df[
        "signed_delivery_deviation_h"
    ] = np.nan

    integrated_delivery_dataset_df[
        "delivery_lateness_h"
    ] = np.nan

    integrated_delivery_dataset_df[
        "early_delivery_h"
    ] = np.nan

    integrated_delivery_dataset_df[
        "on_time_delivery"
    ] = pd.Series(
        [pd.NA] * N_INTEGRATED_SCENARIOS,
        dtype="boolean"
    )

    integrated_delivery_dataset_df[
        "delivery_status"
    ] = "PENDING_PROMISED_DELIVERY_PERIOD"

    DELIVERY_TARGET_STATUS = "PENDING"


# ------------------------------------------------------------
# 12. Verify integrated order-level time balance
# ------------------------------------------------------------

# Actual delivery lead time =
# Production flow time
# + Warehouse elapsed time
# + Transportation elapsed time.

calculated_delivery_lead_time_h = (
    integrated_delivery_dataset_df["production_time_h"]
    + integrated_delivery_dataset_df[
        "warehouse_total_elapsed_time_h"
    ]
    + integrated_delivery_dataset_df[
        "transportation_total_elapsed_time_h"
    ]
)

if not np.allclose(
    integrated_delivery_dataset_df[
        "actual_delivery_lead_time_h"
    ],
    calculated_delivery_lead_time_h,
    atol=1e-6
):
    raise ValueError(
        "Integrated production–warehouse–transportation "
        "time balance does not hold."
    )


# ------------------------------------------------------------
# 13. Check the target definition
# ------------------------------------------------------------

if promised_delivery_days is not None:

    if not np.allclose(
        integrated_delivery_dataset_df[
            "delivery_lateness_h"
        ],
        np.maximum(
            integrated_delivery_dataset_df[
                "signed_delivery_deviation_h"
            ],
            0.0
        ),
        atol=1e-8
    ):
        raise ValueError(
            "Delivery lateness does not match its mathematical definition."
        )

    if (
        integrated_delivery_dataset_df[
            "delivery_lateness_h"
        ] < 0
    ).any():
        raise ValueError(
            "Negative delivery lateness was generated."
        )


# ------------------------------------------------------------
# 14. Define the primary target and prediction unit
# ------------------------------------------------------------

TARGET_VARIABLE = "delivery_lateness_h"

OBSERVATION_UNIT = "Consolidated customer order"

TARGET_DEFINITION = (
    "Non-negative difference between actual customer-order "
    "delivery completion time and the contractual promised "
    "delivery deadline, measured in hours."
)

TIME_UNIT = "hours"


# ------------------------------------------------------------
# 15. Prepare data-quality indicators
# ------------------------------------------------------------

integrated_delivery_dataset_df[
    "delivery_target_available"
] = (
    DELIVERY_TARGET_STATUS == "CALCULATED"
)

if integrated_delivery_dataset_df["scenario_id"].duplicated().any():
    raise ValueError(
        "Duplicate scenario IDs exist in the integrated dataset."
    )

if integrated_delivery_dataset_df.isna().all(axis=1).any():
    raise ValueError(
        "An entirely empty observation was found in the integrated dataset."
    )

if not np.isfinite(
    integrated_delivery_dataset_df[
        [
            "actual_delivery_time_h",
            "actual_delivery_lead_time_h",
            "production_time_h",
            "warehouse_total_elapsed_time_h",
            "transportation_total_elapsed_time_h"
        ]
    ].to_numpy()
).all():
    raise ValueError(
        "Non-finite integrated delivery-time values were generated."
    )


# ------------------------------------------------------------
# 16. Generate integrated time summary
# ------------------------------------------------------------

integrated_time_summary_df = (
    integrated_delivery_dataset_df[
        [
            "production_time_h",
            "warehouse_total_elapsed_time_h",
            "transportation_total_elapsed_time_h",
            "actual_delivery_lead_time_h"
        ]
    ]
    .agg([
        "mean",
        "std",
        "min",
        "median",
        "max"
    ])
    .T
    .reset_index()
    .rename(columns={"index": "metric"})
)


# Add target statistics only if the contractual deadline is known.

if DELIVERY_TARGET_STATUS == "CALCULATED":

    target_summary_df = (
        integrated_delivery_dataset_df[
            [
                "signed_delivery_deviation_h",
                "delivery_lateness_h",
                "early_delivery_h"
            ]
        ]
        .agg([
            "mean",
            "std",
            "min",
            "median",
            "max"
        ])
        .T
        .reset_index()
        .rename(columns={"index": "metric"})
    )

else:
    target_summary_df = pd.DataFrame({
        "metric": [
            "promised_delivery_days",
            "signed_delivery_deviation_h",
            "delivery_lateness_h"
        ],
        "status": [
            "Pending contractual order information",
            "Cannot be calculated yet",
            "Cannot be calculated yet"
        ]
    })


# ------------------------------------------------------------
# 17. Display final results
# ------------------------------------------------------------

print("CELL 9 COMPLETED SUCCESSFULLY")
print("=" * 70)

print(f"Customer order: {customer_order_id}")
print(f"Observation unit: {OBSERVATION_UNIT}")
print(f"Number of integrated scenarios: {N_INTEGRATED_SCENARIOS}")

print("-" * 70)

print(
    f"Mean production completion duration: "
    f"{integrated_delivery_dataset_df['production_time_h'].mean():.2f} h"
)

print(
    f"Mean warehouse elapsed time: "
    f"{integrated_delivery_dataset_df['warehouse_total_elapsed_time_h'].mean():.2f} h"
)

print(
    f"Mean transportation elapsed time: "
    f"{integrated_delivery_dataset_df['transportation_total_elapsed_time_h'].mean():.2f} h"
)

print(
    f"Mean actual delivery lead time: "
    f"{integrated_delivery_dataset_df['actual_delivery_lead_time_h'].mean():.2f} h"
)

print("-" * 70)

print(f"Delivery target status: {DELIVERY_TARGET_STATUS}")

if DELIVERY_TARGET_STATUS == "CALCULATED":

    print(
        f"Promised delivery period: "
        f"{promised_delivery_days:.2f} days"
    )

    print(
        f"Promised delivery lead time: "
        f"{promised_delivery_lead_time_h:.2f} h"
    )

    print(
        f"Mean delivery lateness: "
        f"{integrated_delivery_dataset_df['delivery_lateness_h'].mean():.2f} h"
    )

    print(
        f"Late-order scenario proportion: "
        f"{(~integrated_delivery_dataset_df['on_time_delivery']).mean() * 100:.2f}%"
    )

else:

    print(
        "The promised delivery period for this reference order "
        "has not yet been entered."
    )

    print(
        "Actual delivery lead time has been calculated, "
        "but delivery lateness remains pending."
    )

print("=" * 70)

display(integrated_time_summary_df)

display(target_summary_df)

display(
    integrated_delivery_dataset_df[
        [
            "scenario_id",
            "customer_order_id",
            "production_time_h",
            "warehouse_total_elapsed_time_h",
            "transportation_total_elapsed_time_h",
            "actual_delivery_time_h",
            "actual_delivery_lead_time_h",
            "promised_delivery_days",
            "signed_delivery_deviation_h",
            "delivery_lateness_h",
            "delivery_status"
        ]
    ].head()
)


CELL 9 COMPLETED SUCCESSFULLY
Customer order: CO-001
Observation unit: Consolidated customer order
Number of integrated scenarios: 2000
----------------------------------------------------------------------
Mean production completion duration: 237.42 h
Mean warehouse elapsed time: 25.40 h
Mean transportation elapsed time: 9.02 h
Mean actual delivery lead time: 271.84 h
----------------------------------------------------------------------
Delivery target status: PENDING
The promised delivery period for this reference order has not yet been entered.
Actual delivery lead time has been calculated, but delivery lateness remains pending.


,metric,mean,std,min,median,max
0,production_time_h,237.415565,8.286835,218.977540,236.556333,263.965182
1,warehouse_total_elapsed_time_h,25.402240,3.462032,16.565277,25.027698,40.178175
2,transportation_total_elapsed_time_h,9.023514,1.859560,4.951326,8.757577,16.701907
3,actual_delivery_lead_time_h,271.841319,9.164654,251.498936,270.703969,305.955896


,metric,status
0,promised_delivery_days,Pending contractual order information
1,signed_delivery_deviation_h,Cannot be calculated yet
2,delivery_lateness_h,Cannot be calculated yet


,scenario_id,customer_order_id,production_time_h,warehouse_total_elapsed_time_h,transportation_total_elapsed_time_h,actual_delivery_time_h,actual_delivery_lead_time_h,promised_delivery_days,signed_delivery_deviation_h,delivery_lateness_h,delivery_status
0,1,CO-001,227.352324,22.892205,8.800283,259.044811,259.044811,NaN,NaN,NaN,PENDING_PROMISED_DELIVERY_PERIOD
1,2,CO-001,231.635065,25.906206,8.591540,266.132811,266.132811,NaN,NaN,NaN,PENDING_PROMISED_DELIVERY_PERIOD
2,3,CO-001,243.038919,27.178666,11.009039,281.226623,281.226623,NaN,NaN,NaN,PENDING_PROMISED_DELIVERY_PERIOD
3,4,CO-001,254.538477,27.348776,12.040679,293.927932,293.927932,NaN,NaN,NaN,PENDING_PROMISED_DELIVERY_PERIOD
4,5,CO-001,226.722367,27.673122,12.500384,266.895873,266.895873,NaN,NaN,NaN,PENDING_PROMISED_DELIVERY_PERIOD


In [26]:

# ============================================================
# CELL 10 — GENERATION OF A DIVERSIFIED CUSTOMER-ORDER PORTFOLIO
# ============================================================
# Purpose:
# 1. Generate distinct simulated customer orders.
# 2. Preserve CO-001 as the factory-informed reference order.
# 3. Introduce variation in product mix and order quantities.
# 4. Reconstruct the four-stage serial production schedules.
# 5. Preserve setup times and reference interstage intervals.
# 6. Calculate order-specific nominal production workloads.
#
# Observation unit:
# One distinct consolidated customer order.
#
# IMPORTANT:
# Synthetic order characteristics are simulation assumptions.
# They must not be described as actual company orders.
#
# Processing times are scaled linearly from reference
# job-machine observations. This assumption requires validation.
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Validate the reference production schedule
# ------------------------------------------------------------

required_columns = [
    "customer_order_id",
    "job_id",
    "product_diameter_mm",
    "order_quantity",
    "machine_id",
    "stage_sequence",
    "setup_time_h",
    "processing_time_h",
    "interstage_gap_h"
]

missing_columns = [
    col for col in required_columns
    if col not in production_schedule_df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing columns in production_schedule_df: {missing_columns}"
    )

if "MACHINE_STAGES" not in globals():
    raise RuntimeError("MACHINE_STAGES is missing. Check Cell 1.")

if "PRODUCT_TYPES" not in globals():
    raise RuntimeError("PRODUCT_TYPES is missing. Check Cell 1.")

if "RANDOM_SEED" not in globals():
    raise RuntimeError("RANDOM_SEED is missing. Check Cell 1.")

if "rng" not in globals():
    raise RuntimeError("Random generator 'rng' is missing.")

if production_schedule_df.empty:
    raise ValueError("The reference production schedule is empty.")

if len(production_order_summary_df) != 1:
    raise ValueError(
        "Cell 10 expects the single reference order CO-001."
    )


# ------------------------------------------------------------
# 2. Define reproducible Cell 10 random generator
# ------------------------------------------------------------

# An independent seed makes Cell 10 reproducible when rerun.
# It does not advance the random generator used in Cells 6–8.

cell10_rng = np.random.default_rng(
    RANDOM_SEED + 10
)


# ------------------------------------------------------------
# 3. Define the customer-order portfolio
# ------------------------------------------------------------

N_DISTINCT_CUSTOMER_ORDERS = 2000

REFERENCE_ORDER_ID = "CO-001"

# The first scenario is the reference order.
# The remaining scenarios are synthetic customer orders.

MIN_PRODUCT_QUANTITY = 20
MAX_PRODUCT_QUANTITY = 120

PRODUCT_INCLUSION_PROBABILITY = 0.85

# These parameters are provisional scenario assumptions.
# The triangular distribution is centred on the observed
# reference quantity for each product diameter.

# Product sequence is fixed in ascending diameter order.
# This is an assumed manual operating policy for the
# additional synthetic orders, not an optimization result.

product_types = sorted(
    [int(x) for x in PRODUCT_TYPES]
)

stage_ids = list(MACHINE_STAGES)

if len(stage_ids) != 4:
    raise ValueError(
        "Exactly four production stages are required."
    )

if len(product_types) != 3:
    raise ValueError(
        "This version of Cell 10 expects the three reference "
        "product diameters."
    )

if product_types != [900, 1000, 1200]:
    raise ValueError(
        f"Unexpected product diameter configuration: {product_types}"
    )


# ------------------------------------------------------------
# 4. Build reference product-machine processing profiles
# ------------------------------------------------------------

reference_profile_df = production_schedule_df[
    [
        "product_diameter_mm",
        "order_quantity",
        "machine_id",
        "stage_sequence",
        "setup_time_h",
        "processing_time_h",
        "interstage_gap_h"
    ]
].copy()

numeric_profile_columns = [
    "product_diameter_mm",
    "order_quantity",
    "stage_sequence",
    "setup_time_h",
    "processing_time_h",
    "interstage_gap_h"
]

for col in numeric_profile_columns:
    reference_profile_df[col] = pd.to_numeric(
        reference_profile_df[col],
        errors="coerce"
    )

if reference_profile_df[numeric_profile_columns].isna().any().any():
    raise ValueError(
        "Invalid values detected in the reference operation profiles."
    )

# Verify one observation per product-machine combination.

if reference_profile_df.duplicated(
    subset=["product_diameter_mm", "machine_id"]
).any():
    raise ValueError(
        "Duplicate product-machine profiles found in the reference schedule."
    )

expected_profile_count = (
    len(product_types) * len(stage_ids)
)

if len(reference_profile_df) != expected_profile_count:
    raise ValueError(
        f"Expected {expected_profile_count} product-machine records, "
        f"but found {len(reference_profile_df)}."
    )

# Verify that every product has all four machine stages.

for diameter in product_types:

    product_profile = reference_profile_df[
        reference_profile_df["product_diameter_mm"] == diameter
    ]

    if set(product_profile["machine_id"]) != set(stage_ids):
        raise ValueError(
            f"Incomplete machine route for diameter {diameter} mm."
        )

    if product_profile["order_quantity"].nunique() != 1:
        raise ValueError(
            f"Inconsistent reference quantity for diameter {diameter} mm."
        )

    product_profile = product_profile.sort_values("stage_sequence")

    if product_profile["stage_sequence"].tolist() != [1, 2, 3, 4]:
        raise ValueError(
            f"Incorrect machine sequence for diameter {diameter} mm."
        )


# Reference quantity for each product.

reference_quantity_by_diameter = (
    reference_profile_df
    .groupby("product_diameter_mm")["order_quantity"]
    .first()
    .astype(float)
    .to_dict()
)

# Processing time per pipe at each product-machine combination.
#
# This rate is a derived simulation parameter, not necessarily
# a physically constant production rate.

reference_profile_df[
    "processing_time_per_pipe_h"
] = (
    reference_profile_df["processing_time_h"]
    / reference_profile_df["order_quantity"]
)

if (
    reference_profile_df["processing_time_per_pipe_h"] <= 0
).any():
    raise ValueError(
        "Processing time per pipe must be positive."
    )

reference_profile_df = (
    reference_profile_df
    .sort_values(
        ["product_diameter_mm", "stage_sequence"]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 5. Create a lookup table for each product-machine combination
# ------------------------------------------------------------

profile_lookup = {}

for _, row in reference_profile_df.iterrows():

    key = (
        int(row["product_diameter_mm"]),
        row["machine_id"]
    )

    profile_lookup[key] = {
        "reference_quantity": float(row["order_quantity"]),
        "setup_time_h": float(row["setup_time_h"]),
        "processing_time_per_pipe_h":
            float(row["processing_time_per_pipe_h"]),
        "interstage_gap_h": float(row["interstage_gap_h"]),
        "stage_sequence": int(row["stage_sequence"])
    }


# ------------------------------------------------------------
# 6. Define reference destination and route information
# ------------------------------------------------------------

reference_destination_code = (
    production_order_summary_df["destination_code"].iloc[0]
    if "destination_code" in production_order_summary_df.columns
    else "BORDER_DESTINATION_TO_CONFIRM"
)

if pd.isna(reference_destination_code):
    reference_destination_code = "BORDER_DESTINATION_TO_CONFIRM"

# The existing 300-km route is still an illustrative placeholder.
# No new destinations or distances are invented in this cell.

if "REFERENCE_ROUTE_DISTANCE_KM" in globals():
    reference_route_distance_km = float(
        REFERENCE_ROUTE_DISTANCE_KM
    )
else:
    reference_route_distance_km = 300.0

# Promised delivery days are imported only if supplied.
# Missing contractual values remain NaN.

if "PROMISED_DELIVERY_DAYS_BY_ORDER" in globals():
    promised_days_lookup = (
        PROMISED_DELIVERY_DAYS_BY_ORDER.copy()
    )
else:
    promised_days_lookup = {}


# ------------------------------------------------------------
# 7. Retrieve reference order quantities
# ------------------------------------------------------------

reference_order_quantities = {}

for diameter in product_types:

    reference_order_quantities[diameter] = int(
        round(reference_quantity_by_diameter[diameter])
    )

# Verify that the supplied reference quantities have not changed.

expected_reference_quantities = {
    900: 50,
    1000: 100,
    1200: 40
}

if reference_order_quantities != expected_reference_quantities:
    raise ValueError(
        "Reference product quantities differ from the confirmed "
        f"factory information: {reference_order_quantities}"
    )


# ------------------------------------------------------------
# 8. Generate distinct customer-order configurations
# ------------------------------------------------------------

order_records = []

for scenario_id in range(
    1,
    N_DISTINCT_CUSTOMER_ORDERS + 1
):

    # --------------------------------------------------------
    # Scenario 1: retain the reference order exactly.
    # --------------------------------------------------------

    if scenario_id == 1:

        customer_order_id = REFERENCE_ORDER_ID

        selected_products = product_types.copy()

        quantity_by_diameter = (
            reference_order_quantities.copy()
        )

        order_data_origin = (
            "Factory-informed reference configuration; "
            "operational scenario outcomes are simulated."
        )

    # --------------------------------------------------------
    # Remaining scenarios: generate synthetic customer orders.
    # --------------------------------------------------------

    else:

        customer_order_id = (
            f"SIM-{scenario_id:06d}"
        )

        # Select the product mix.
        product_presence = (
            cell10_rng.random(len(product_types))
            < PRODUCT_INCLUSION_PROBABILITY
        )

        selected_products = [
            diameter
            for diameter, included
            in zip(product_types, product_presence)
            if included
        ]

        # Every customer order must contain at least one product.
        if len(selected_products) == 0:

            selected_products = [
                int(cell10_rng.choice(product_types))
            ]

        # Generate quantities using the provisional
        # triangular distribution.

        quantity_by_diameter = {}

        for diameter in selected_products:

            reference_quantity = (
                reference_quantity_by_diameter[diameter]
            )

            generated_quantity = cell10_rng.triangular(
                MIN_PRODUCT_QUANTITY,
                reference_quantity,
                MAX_PRODUCT_QUANTITY
            )

            quantity_by_diameter[diameter] = int(
                np.clip(
                    np.rint(generated_quantity),
                    MIN_PRODUCT_QUANTITY,
                    MAX_PRODUCT_QUANTITY
                )
            )

        order_data_origin = (
            "Synthetic customer-order scenario generated "
            "from provisional product-mix and quantity assumptions."
        )

    # --------------------------------------------------------
    # Calculate order-level characteristics.
    # --------------------------------------------------------

    total_order_quantity = int(
        sum(quantity_by_diameter.values())
    )

    number_of_jobs = len(selected_products)

    product_mix = ", ".join(
        f"Ø{diameter}"
        for diameter in selected_products
    )

    product_mix_code = "-".join(
        str(diameter)
        for diameter in selected_products
    )

    quantity_900 = quantity_by_diameter.get(900, 0)
    quantity_1000 = quantity_by_diameter.get(1000, 0)
    quantity_1200 = quantity_by_diameter.get(1200, 0)

    promised_days = promised_days_lookup.get(
        customer_order_id,
        np.nan
    )

    if promised_days is None:
        promised_days = np.nan

    order_records.append({

        "scenario_id": scenario_id,

        "customer_order_id": customer_order_id,

        "destination_code": reference_destination_code,

        "delivery_distance_km": reference_route_distance_km,

        "number_of_jobs": number_of_jobs,

        "total_order_quantity": total_order_quantity,

        "product_mix": product_mix,

        "product_mix_code": product_mix_code,

        "quantity_900mm": quantity_900,

        "quantity_1000mm": quantity_1000,

        "quantity_1200mm": quantity_1200,

        "manual_sequence_policy": "ASCENDING_DIAMETER",

        "promised_delivery_days": promised_days,

        "order_data_origin": order_data_origin

    })


order_master_df = pd.DataFrame(order_records)


# ------------------------------------------------------------
# 9. Reconstruct production schedules for every generated order
# ------------------------------------------------------------

# Each order follows:
# - One job per product diameter present in the order.
# - Fixed ascending-diameter manual sequence.
# - M1 -> M2 -> M3 -> M4 for every job.
# - A complete job finishes before the next job starts.
# - Full-batch transfer between stages.
#
# Processing durations are scaled from the reference product-
# machine processing rates.
#
# Setup durations and interstage gaps remain fixed at their
# reference values for each product-machine combination.

operation_records = []
order_production_records = []

for _, order in order_master_df.iterrows():

    scenario_id = int(order["scenario_id"])

    customer_order_id = order["customer_order_id"]

    selected_products = [
        int(x)
        for x in order["product_mix_code"].split("-")
    ]

    current_job_release_global_h = 0.0

    order_processing_total_h = 0.0
    order_setup_total_h = 0.0
    order_interstage_total_h = 0.0

    machine_workload = {
        machine_id: 0.0
        for machine_id in stage_ids
    }

    job_completion_times = []

    # --------------------------------------------------------
    # Process each product job in the fixed manual sequence.
    # --------------------------------------------------------

    for manual_sequence, diameter in enumerate(
        selected_products,
        start=1
    ):

        job_id = (
            f"{customer_order_id}-J{manual_sequence}"
        )

        quantity = int(
            order[f"quantity_{diameter}mm"]
        )

        if quantity <= 0:
            raise ValueError(
                f"Invalid job quantity for {job_id}."
            )

        job_release_global_h = (
            current_job_release_global_h
        )

        upstream_completion_global_h = (
            job_release_global_h
        )

        job_setup_total_h = 0.0
        job_processing_total_h = 0.0
        job_interstage_total_h = 0.0

        # ----------------------------------------------------
        # Execute the four sequential machine stages.
        # ----------------------------------------------------

        for machine_id in stage_ids:

            profile = profile_lookup[
                (diameter, machine_id)
            ]

            stage_sequence = profile[
                "stage_sequence"
            ]

            reference_quantity = profile[
                "reference_quantity"
            ]

            setup_time_h = profile[
                "setup_time_h"
            ]

            processing_time_per_pipe_h = profile[
                "processing_time_per_pipe_h"
            ]

            # Processing time scales linearly with job quantity.

            processing_time_h = (
                processing_time_per_pipe_h * quantity
            )

            # The first stage has no interstage gap.
            # Subsequent stages retain their reference gap.

            if stage_sequence == 1:
                interstage_gap_h = 0.0
            else:
                interstage_gap_h = profile[
                    "interstage_gap_h"
                ]

            # The job starts only after the previous complete
            # job has finished its fourth machine stage.

            upstream_ready_global_h = (
                upstream_completion_global_h
            )

            setup_start_global_h = (
                upstream_ready_global_h
                + interstage_gap_h
            )

            processing_start_global_h = (
                setup_start_global_h
                + setup_time_h
            )

            processing_completion_global_h = (
                processing_start_global_h
                + processing_time_h
            )

            machine_busy_time_h = (
                setup_time_h
                + processing_time_h
            )

            # Record operation-level information.

            operation_records.append({

                "scenario_id": scenario_id,

                "customer_order_id": customer_order_id,

                "job_id": job_id,

                "product_diameter_mm": diameter,

                "order_quantity": quantity,

                "manual_sequence": manual_sequence,

                "machine_id": machine_id,

                "stage_sequence": stage_sequence,

                "job_global_release_time_h":
                    job_release_global_h,

                "upstream_ready_global_h":
                    upstream_ready_global_h,

                "setup_start_global_h":
                    setup_start_global_h,

                "processing_start_global_h":
                    processing_start_global_h,

                "processing_completion_global_h":
                    processing_completion_global_h,

                "setup_start_local_h":
                    setup_start_global_h - job_release_global_h,

                "processing_start_local_h":
                    processing_start_global_h - job_release_global_h,

                "processing_completion_local_h":
                    processing_completion_global_h - job_release_global_h,

                "setup_time_h":
                    setup_time_h,

                "processing_time_h":
                    processing_time_h,

                "interstage_gap_h":
                    interstage_gap_h,

                "machine_busy_time_h":
                    machine_busy_time_h

            })

            # Accumulate job- and machine-level workloads.

            job_setup_total_h += setup_time_h

            job_processing_total_h += processing_time_h

            job_interstage_total_h += interstage_gap_h

            machine_workload[machine_id] += (
                machine_busy_time_h
            )

            # The next machine begins after this machine
            # has completed processing.

            upstream_completion_global_h = (
                processing_completion_global_h
            )

        # ----------------------------------------------------
        # Complete the current job.
        # ----------------------------------------------------

        job_completion_global_h = (
            upstream_completion_global_h
        )

        job_completion_times.append(
            job_completion_global_h
        )

        order_processing_total_h += (
            job_processing_total_h
        )

        order_setup_total_h += (
            job_setup_total_h
        )

        order_interstage_total_h += (
            job_interstage_total_h
        )

        # The next complete job is released only after the
        # current job finishes M4.

        current_job_release_global_h = (
            job_completion_global_h
        )

    # --------------------------------------------------------
    # Calculate order-level nominal production indicators.
    # --------------------------------------------------------

    nominal_production_completion_h = (
        current_job_release_global_h
    )

    nominal_production_flow_time_h = (
        nominal_production_completion_h
    )

    total_machine_busy_h = (
        sum(machine_workload.values())
    )

    calculated_production_time_h = (
        order_processing_total_h
        + order_setup_total_h
        + order_interstage_total_h
    )

    if not np.isclose(
        nominal_production_flow_time_h,
        calculated_production_time_h,
        atol=1e-6
    ):
        raise ValueError(
            f"Production time balance failed for {customer_order_id}."
        )

    # Mean machine occupation ratio over the complete production
    # horizon. This is not OEE or machine availability.

    machine_utilization_values = [
        machine_workload[machine_id]
        / nominal_production_flow_time_h
        for machine_id in stage_ids
    ]

    average_machine_utilization = float(
        np.mean(machine_utilization_values)
    )

    production_record = {

        "scenario_id": scenario_id,

        "customer_order_id": customer_order_id,

        "nominal_production_completion_time_h":
            nominal_production_completion_h,

        "nominal_production_flow_time_h":
            nominal_production_flow_time_h,

        "production_processing_time_h":
            order_processing_total_h,

        "production_setup_time_h":
            order_setup_total_h,

        "production_interstage_nonprocessing_time_h":
            order_interstage_total_h,

        "total_machine_busy_time_h":
            total_machine_busy_h,

        "average_machine_utilization":
            average_machine_utilization,

        "maximum_pending_job_count":
            max(int(order["number_of_jobs"]) - 1, 0)

    }

    # Add machine-specific workload and utilization.

    for machine_id in stage_ids:

        production_record[
            f"busy_time_{machine_id}_h"
        ] = machine_workload[machine_id]

        production_record[
            f"utilization_{machine_id}"
        ] = (
            machine_workload[machine_id]
            / nominal_production_flow_time_h
        )

    order_production_records.append(
        production_record
    )


# ------------------------------------------------------------
# 10. Convert generated records into data frames
# ------------------------------------------------------------

simulated_order_operations_df = pd.DataFrame(
    operation_records
)

simulated_order_production_df = pd.DataFrame(
    order_production_records
)


# ------------------------------------------------------------
# 11. Integrate order master and production information
# ------------------------------------------------------------

multi_order_scenarios_df = (
    order_master_df
    .merge(
        simulated_order_production_df,
        on=[
            "scenario_id",
            "customer_order_id"
        ],
        how="inner",
        validate="one_to_one"
    )
    .sort_values("scenario_id")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 12. Verify reference order reconstruction
# ------------------------------------------------------------

reference_row = multi_order_scenarios_df[
    multi_order_scenarios_df["customer_order_id"]
    == REFERENCE_ORDER_ID
]

if len(reference_row) != 1:
    raise ValueError(
        "The reference order CO-001 was not reconstructed exactly once."
    )

reference_production_flow_h = float(
    reference_row[
        "nominal_production_flow_time_h"
    ].iloc[0]
)

original_reference_flow_h = float(
    production_order_summary_df[
        "production_flow_time_h"
    ].iloc[0]
)

if not np.isclose(
    reference_production_flow_h,
    original_reference_flow_h,
    atol=1e-6
):
    raise ValueError(
        "Reconstructed CO-001 production flow time differs from "
        "the original reference schedule. "
        f"Original = {original_reference_flow_h:.4f} h; "
        f"reconstructed = {reference_production_flow_h:.4f} h."
    )


# Verify reference order quantities.

reference_quantity_check = (
    reference_row[
        [
            "quantity_900mm",
            "quantity_1000mm",
            "quantity_1200mm"
        ]
    ]
    .iloc[0]
    .to_dict()
)

expected_quantity_check = {
    "quantity_900mm": 50,
    "quantity_1000mm": 100,
    "quantity_1200mm": 40
}

if reference_quantity_check != expected_quantity_check:
    raise ValueError(
        "Reference order product quantities changed during reconstruction."
    )


# ------------------------------------------------------------
# 13. Verify all generated production schedules
# ------------------------------------------------------------

if len(multi_order_scenarios_df) != N_DISTINCT_CUSTOMER_ORDERS:
    raise ValueError(
        "Generated customer-order count does not match the target."
    )

if multi_order_scenarios_df[
    "customer_order_id"
].nunique() != N_DISTINCT_CUSTOMER_ORDERS:
    raise ValueError(
        "Customer-order IDs are not unique."
    )

if (
    multi_order_scenarios_df[
        "total_order_quantity"
    ] <= 0
).any():
    raise ValueError(
        "Non-positive customer-order quantities were generated."
    )

if not np.allclose(
    multi_order_scenarios_df[
        "nominal_production_flow_time_h"
    ],
    (
        multi_order_scenarios_df[
            "production_processing_time_h"
        ]
        + multi_order_scenarios_df[
            "production_setup_time_h"
        ]
        + multi_order_scenarios_df[
            "production_interstage_nonprocessing_time_h"
        ]
    ),
    atol=1e-6
):
    raise ValueError(
        "One or more customer-order production-time balances failed."
    )

if not np.isfinite(
    multi_order_scenarios_df[
        [
            "nominal_production_flow_time_h",
            "production_processing_time_h",
            "production_setup_time_h",
            "production_interstage_nonprocessing_time_h"
        ]
    ].to_numpy()
).all():
    raise ValueError(
        "Non-finite production values were generated."
    )


# ------------------------------------------------------------
# 14. Create product-mix and quantity summaries
# ------------------------------------------------------------

product_mix_frequency_df = (
    multi_order_scenarios_df
    .groupby("product_mix")
    .agg(
        number_of_orders=("customer_order_id", "count"),
        mean_order_quantity=("total_order_quantity", "mean"),
        mean_production_time_h=(
            "nominal_production_flow_time_h", "mean"
        )
    )
    .reset_index()
    .sort_values(
        "number_of_orders",
        ascending=False
    )
)

order_portfolio_summary_df = (
    multi_order_scenarios_df[
        [
            "total_order_quantity",
            "number_of_jobs",
            "nominal_production_flow_time_h",
            "production_processing_time_h",
            "production_setup_time_h",
            "production_interstage_nonprocessing_time_h"
        ]
    ]
    .agg([
        "mean",
        "std",
        "min",
        "median",
        "max"
    ])
    .T
    .reset_index()
    .rename(columns={"index": "metric"})
)


# ------------------------------------------------------------
# 15. Display results
# ------------------------------------------------------------

print("CELL 10 COMPLETED SUCCESSFULLY")
print("=" * 70)

print(
    f"Number of distinct customer orders: "
    f"{len(multi_order_scenarios_df)}"
)

print(
    f"Number of reference orders: "
    f"{(multi_order_scenarios_df['order_data_origin'].str.startswith('Factory-informed')).sum()}"
)

print(
    f"Number of synthetic orders: "
    f"{(multi_order_scenarios_df['order_data_origin'].str.startswith('Synthetic')).sum()}"
)

print(
    f"Number of operation records: "
    f"{len(simulated_order_operations_df)}"
)

print("-" * 70)

print(
    "Reference CO-001 nominal production time: "
    f"{reference_production_flow_h:.2f} h"
)

print(
    "Original CO-001 production time: "
    f"{original_reference_flow_h:.2f} h"
)

print("-" * 70)

print(
    f"Mean order quantity: "
    f"{multi_order_scenarios_df['total_order_quantity'].mean():.2f} pipes"
)

print(
    f"Mean number of product jobs: "
    f"{multi_order_scenarios_df['number_of_jobs'].mean():.2f}"
)

print(
    f"Mean nominal production flow time: "
    f"{multi_order_scenarios_df['nominal_production_flow_time_h'].mean():.2f} h"
)

print("-" * 70)

print(
    "Promised delivery periods available: "
    f"{multi_order_scenarios_df['promised_delivery_days'].notna().sum()}"
)

print(
    "Destination diversity: "
    f"{multi_order_scenarios_df['destination_code'].nunique()} destination(s)"
)

print("=" * 70)

display(order_portfolio_summary_df)

display(product_mix_frequency_df)

display(
    multi_order_scenarios_df.head(10)
)

display(
    simulated_order_operations_df.head(12)
)


CELL 10 COMPLETED SUCCESSFULLY
Number of distinct customer orders: 2000
Number of reference orders: 1
Number of synthetic orders: 1999
Number of operation records: 20488
----------------------------------------------------------------------
Reference CO-001 nominal production time: 215.51 h
Original CO-001 production time: 215.51 h
----------------------------------------------------------------------
Mean order quantity: 174.12 pipes
Mean number of product jobs: 2.56
Mean nominal production flow time: 227.04 h
----------------------------------------------------------------------
Promised delivery periods available: 0
Destination diversity: 1 destination(s)


,metric,mean,std,min,median,max
0,total_order_quantity,174.122500,54.155710,28.0000,178.0000,320.0000
1,number_of_jobs,2.561000,0.602877,1.0000,3.0000,3.0000
2,nominal_production_flow_time_h,227.044575,84.337616,24.7944,231.0243,433.0341
3,production_processing_time_h,198.254945,79.666143,14.4844,199.7186,399.3341
4,production_setup_time_h,13.757850,3.333100,3.8000,16.1000,16.1000
5,production_interstage_nonprocessing_time_h,15.031780,3.565122,5.0100,17.6000,17.6000


,product_mix,number_of_orders,mean_order_quantity,mean_production_time_h
5,"Ø900, Ø1000, Ø1200",1239,204.420500,265.956666
1,"Ø1000, Ø1200",225,136.866667,217.113077
6,"Ø900, Ø1200",221,125.058824,215.903050
4,"Ø900, Ø1000",198,145.424242,95.280992
3,Ø900,43,65.813953,43.793256
0,Ø1000,41,74.902439,49.057032
2,Ø1200,33,55.757576,158.901212


,scenario_id,customer_order_id,destination_code,delivery_distance_km,number_of_jobs,total_order_quantity,product_mix,product_mix_code,quantity_900mm,quantity_1000mm,...,average_machine_utilization,maximum_pending_job_count,busy_time_M1_h,utilization_M1,busy_time_M2_h,utilization_M2,busy_time_M3_h,utilization_M3,busy_time_M4_h,utilization_M4
0,1,CO-001,BORDER_DESTINATION_TO_CONFIRM,300.0,3,190,"Ø900, Ø1000, Ø1200",900-1000-1200,50,100,...,0.229583,2,93.50,0.433855,62.6300,0.290613,32.4800,0.150712,9.30,0.043153
1,2,SIM-000002,BORDER_DESTINATION_TO_CONFIRM,300.0,2,122,"Ø1000, Ø1200",1000-1200,0,67,...,0.235352,1,90.00,0.445767,60.4130,0.299224,31.0161,0.153622,8.64,0.042794
2,3,SIM-000003,BORDER_DESTINATION_TO_CONFIRM,300.0,2,125,"Ø1000, Ø1200",1000-1200,0,73,...,0.235002,1,87.75,0.444987,58.8920,0.298646,30.2649,0.153476,8.46,0.042901
3,4,SIM-000004,BORDER_DESTINATION_TO_CONFIRM,300.0,3,239,"Ø900, Ø1000, Ø1200",900-1000-1200,76,99,...,0.234878,2,129.75,0.445937,86.8376,0.298451,44.5731,0.153193,12.20,0.041930
4,5,SIM-000005,BORDER_DESTINATION_TO_CONFIRM,300.0,2,106,"Ø1000, Ø1200",1000-1200,0,58,...,0.233480,1,79.00,0.441268,53.0920,0.296555,27.3474,0.152754,7.76,0.043345
5,6,SIM-000006,BORDER_DESTINATION_TO_CONFIRM,300.0,2,125,"Ø900, Ø1200",900-1200,36,0,...,0.238500,1,124.25,0.453984,83.3126,0.304407,42.4154,0.154977,11.12,0.040630
6,7,SIM-000007,BORDER_DESTINATION_TO_CONFIRM,300.0,2,118,"Ø1000, Ø1200",1000-1200,0,66,...,0.234722,1,86.00,0.444270,57.7440,0.298302,29.6818,0.153334,8.32,0.042981
7,8,SIM-000008,BORDER_DESTINATION_TO_CONFIRM,300.0,3,190,"Ø900, Ø1000, Ø1200",900-1000-1200,95,40,...,0.232179,2,108.50,0.439446,72.8120,0.294902,37.4900,0.151842,10.50,0.042527
8,9,SIM-000009,BORDER_DESTINATION_TO_CONFIRM,300.0,2,119,"Ø900, Ø1000",900-1000,57,62,...,0.216970,1,33.25,0.407512,22.1642,0.271645,11.7184,0.143621,3.68,0.045102
9,10,SIM-000010,BORDER_DESTINATION_TO_CONFIRM,300.0,3,209,"Ø900, Ø1000, Ø1200",900-1000-1200,84,65,...,0.233528,2,118.25,0.442677,79.2544,0.296695,40.7401,0.152514,11.28,0.042228


,scenario_id,customer_order_id,job_id,product_diameter_mm,order_quantity,manual_sequence,machine_id,stage_sequence,job_global_release_time_h,upstream_ready_global_h,setup_start_global_h,processing_start_global_h,processing_completion_global_h,setup_start_local_h,processing_start_local_h,processing_completion_local_h,setup_time_h,processing_time_h,interstage_gap_h,machine_busy_time_h
0,1,CO-001,CO-001-J1,900,50,1,M1,1,0.00,0.00,0.00,1.50,14.00,0.00,1.50,14.00,1.5,12.50,0.00,14.00
1,1,CO-001,CO-001-J1,900,50,1,M2,2,0.00,14.00,16.43,17.43,25.76,16.43,17.43,25.76,1.0,8.33,2.43,9.33
2,1,CO-001,CO-001-J1,900,50,1,M3,3,0.00,25.76,27.64,28.44,32.61,27.64,28.44,32.61,0.8,4.17,1.88,4.97
3,1,CO-001,CO-001-J1,900,50,1,M4,4,0.00,32.61,34.07,34.57,35.57,34.07,34.57,35.57,0.5,1.00,1.46,1.50
4,1,CO-001,CO-001-J2,1000,100,2,M1,1,35.57,35.57,35.57,37.57,62.57,0.00,2.00,27.00,2.0,25.00,0.00,27.00
5,1,CO-001,CO-001-J2,1000,100,2,M2,2,35.57,62.57,64.78,66.28,82.68,29.21,30.71,47.11,1.5,16.40,2.21,17.90
6,1,CO-001,CO-001-J2,1000,100,2,M3,3,35.57,82.68,84.26,85.26,93.59,48.69,49.69,58.02,1.0,8.33,1.58,9.33
7,1,CO-001,CO-001-J2,1000,100,2,M4,4,35.57,93.59,94.81,95.61,97.61,59.24,60.04,62.04,0.8,2.00,1.22,2.80
8,1,CO-001,CO-001-J3,1200,40,3,M1,1,97.61,97.61,97.61,100.11,150.11,0.00,2.50,52.50,2.5,50.00,0.00,52.50
9,1,CO-001,CO-001-J3,1200,40,3,M2,2,97.61,150.11,153.18,155.18,188.58,55.57,57.57,90.97,2.0,33.40,3.07,35.40


In [27]:

# ============================================================
# CELL 11 — STOCHASTIC PRODUCTION SCENARIOS FOR DISTINCT ORDERS
# ============================================================
# Purpose:
# 1. Generate production availability conditions for each order.
# 2. Apply machine-specific availability to its own workload.
# 3. Calculate production downtime and adjusted production time.
# 4. Calculate machine-level and system-level indicators.
# 5. Produce one integrated production scenario per distinct order.
#
# Production policy:
# - Four machines: M1, M2, M3, M4.
# - Fixed manual sequence.
# - Full-batch transfer.
# - A complete job finishes all four stages before the next job.
#
# Observation unit:
# One distinct consolidated customer order.
#
# IMPORTANT:
# Availability distributions are provisional simulation assumptions.
# This is an availability-adjusted production model, not an
# event-level breakdown-and-repair simulation.
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Validate inputs from Cell 10
# ------------------------------------------------------------

required_order_columns = [
    "scenario_id",
    "customer_order_id",
    "total_order_quantity",
    "number_of_jobs",
    "nominal_production_flow_time_h",
    "production_processing_time_h",
    "production_setup_time_h",
    "production_interstage_nonprocessing_time_h",
    "maximum_pending_job_count"
]

missing_order_columns = [
    col for col in required_order_columns
    if col not in multi_order_scenarios_df.columns
]

if missing_order_columns:
    raise ValueError(
        "Missing columns in multi_order_scenarios_df: "
        f"{missing_order_columns}"
    )

if "MACHINE_STAGES" not in globals():
    raise RuntimeError("MACHINE_STAGES is missing. Check Cell 1.")

if "RANDOM_SEED" not in globals():
    raise RuntimeError("RANDOM_SEED is missing. Check Cell 1.")

if multi_order_scenarios_df.empty:
    raise ValueError("The multi-order portfolio is empty.")

if multi_order_scenarios_df["scenario_id"].duplicated().any():
    raise ValueError("Duplicate scenario IDs detected.")

if multi_order_scenarios_df["customer_order_id"].duplicated().any():
    raise ValueError("Duplicate customer-order IDs detected.")

stage_ids = list(MACHINE_STAGES)

if len(stage_ids) != 4:
    raise ValueError(
        "Exactly four production machines are required."
    )


# ------------------------------------------------------------
# 2. Create an independent reproducible random generator
# ------------------------------------------------------------

# Using an independent seed means that rerunning this cell
# does not alter the random sequence in Cells 6–10.

cell11_rng = np.random.default_rng(
    RANDOM_SEED + 11
)


# ------------------------------------------------------------
# 3. Validate machine workload columns
# ------------------------------------------------------------

machine_busy_columns = [
    f"busy_time_{machine_id}_h"
    for machine_id in stage_ids
]

missing_busy_columns = [
    col for col in machine_busy_columns
    if col not in multi_order_scenarios_df.columns
]

if missing_busy_columns:
    raise ValueError(
        "Missing machine workload columns: "
        f"{missing_busy_columns}"
    )

if multi_order_scenarios_df[
    machine_busy_columns
].isna().any().any():
    raise ValueError(
        "Missing machine workload values were detected."
    )

if (
    multi_order_scenarios_df[machine_busy_columns] < 0
).any().any():
    raise ValueError(
        "Negative machine workload values were detected."
    )


# ------------------------------------------------------------
# 4. Retrieve baseline order-level workloads
# ------------------------------------------------------------

multi_order_production_scenarios_df = (
    multi_order_scenarios_df.copy()
    .sort_values("scenario_id")
    .reset_index(drop=True)
)

N_ORDER_SCENARIOS = len(
    multi_order_production_scenarios_df
)

order_busy_matrix = (
    multi_order_production_scenarios_df[
        machine_busy_columns
    ]
    .astype(float)
    .to_numpy()
)

total_machine_busy_h = order_busy_matrix.sum(axis=1)

baseline_production_flow_time_h = (
    multi_order_production_scenarios_df[
        "nominal_production_flow_time_h"
    ].astype(float).to_numpy()
)

total_setup_time_h = (
    multi_order_production_scenarios_df[
        "production_setup_time_h"
    ].astype(float).to_numpy()
)

total_processing_time_h = (
    multi_order_production_scenarios_df[
        "production_processing_time_h"
    ].astype(float).to_numpy()
)

interstage_nonprocessing_time_h = (
    multi_order_production_scenarios_df[
        "production_interstage_nonprocessing_time_h"
    ].astype(float).to_numpy()
)

if np.any(total_machine_busy_h <= 0):
    raise ValueError(
        "Every order must have positive total machine workload."
    )


# ------------------------------------------------------------
# 5. Verify the nominal production-time balance
# ------------------------------------------------------------

calculated_nominal_production_time_h = (
    total_machine_busy_h
    + interstage_nonprocessing_time_h
)

if not np.allclose(
    baseline_production_flow_time_h,
    calculated_nominal_production_time_h,
    atol=1e-6
):
    raise ValueError(
        "One or more nominal production-time balances are inconsistent."
    )

if not np.allclose(
    total_machine_busy_h,
    total_setup_time_h + total_processing_time_h,
    atol=1e-6
):
    raise ValueError(
        "Machine busy time does not equal setup plus processing time."
    )


# ------------------------------------------------------------
# 6. Define machine-availability assumptions
# ------------------------------------------------------------

# Retain the same provisional distribution used in Cell 6.

AVAILABILITY_BETA_ALPHA_CELL11 = 18.0
AVAILABILITY_BETA_BETA_CELL11 = 2.0

MIN_MACHINE_AVAILABILITY_CELL11 = 0.80
MAX_MACHINE_AVAILABILITY_CELL11 = 1.00

AVAILABILITY_PARAMETER_STATUS_CELL11 = (
    "Provisional Beta-distribution assumption. "
    "Requires calibration using machine-level reliability records."
)


# ------------------------------------------------------------
# 7. Generate machine-specific availability
# ------------------------------------------------------------

# Each row represents a different customer order.
# Each column represents one of the four production machines.
#
# Availability is generated independently across machines
# and customer orders.

availability_matrix = cell11_rng.beta(
    AVAILABILITY_BETA_ALPHA_CELL11,
    AVAILABILITY_BETA_BETA_CELL11,
    size=(
        N_ORDER_SCENARIOS,
        len(stage_ids)
    )
)

availability_matrix = np.clip(
    availability_matrix,
    MIN_MACHINE_AVAILABILITY_CELL11,
    MAX_MACHINE_AVAILABILITY_CELL11
)


# ------------------------------------------------------------
# 8. Calculate adjusted machine occupation and downtime
# ------------------------------------------------------------

# Availability-adjusted occupation:
#
# Adjusted occupation = Baseline machine busy time / Availability
#
# Additional downtime:
#
# Downtime = Adjusted occupation - Baseline busy time

adjusted_machine_occupation_h = (
    order_busy_matrix / availability_matrix
)

machine_downtime_matrix_h = (
    adjusted_machine_occupation_h
    - order_busy_matrix
)

total_production_downtime_h = (
    machine_downtime_matrix_h.sum(axis=1)
)


# ------------------------------------------------------------
# 9. Calculate stochastic production flow time
# ------------------------------------------------------------

# The jobs are processed serially.
# Therefore, all machine occupation and interstage intervals
# contribute to the order's total production flow time.

production_time_h = (
    adjusted_machine_occupation_h.sum(axis=1)
    + interstage_nonprocessing_time_h
)

production_time_increase_h = (
    production_time_h
    - baseline_production_flow_time_h
)


# ------------------------------------------------------------
# 10. Define the order release-time reference
# ------------------------------------------------------------

# All timestamps are measured relative to each order's release.
# The release time is therefore set to zero in each order's
# local simulation timeline.

order_release_time_h = np.zeros(
    N_ORDER_SCENARIOS,
    dtype=float
)

production_completion_time_h = (
    order_release_time_h
    + production_time_h
)


# ------------------------------------------------------------
# 11. Calculate machine availability indicators
# ------------------------------------------------------------

# Busy-time-weighted average availability across the four machines.

weighted_machine_availability = (
    (availability_matrix * order_busy_matrix).sum(axis=1)
    / total_machine_busy_h
)


# ------------------------------------------------------------
# 12. Calculate average machine utilization correctly
# ------------------------------------------------------------

# Machine utilization for each stage:
#
# Busy time at machine / Complete production flow time.
#
# Average system utilization is the arithmetic mean of
# the four machine-specific utilization ratios.
#
# This is not OEE or machine availability.

machine_utilization_matrix = (
    order_busy_matrix
    / production_time_h[:, np.newaxis]
)

average_machine_utilization = (
    machine_utilization_matrix.mean(axis=1)
)


# ------------------------------------------------------------
# 13. Add stochastic production variables
# ------------------------------------------------------------

multi_order_production_scenarios_df[
    "order_release_time_h"
] = order_release_time_h

multi_order_production_scenarios_df[
    "production_baseline_flow_time_h"
] = baseline_production_flow_time_h

multi_order_production_scenarios_df[
    "production_downtime_h"
] = total_production_downtime_h

multi_order_production_scenarios_df[
    "production_time_increase_h"
] = production_time_increase_h

multi_order_production_scenarios_df[
    "production_time_h"
] = production_time_h

multi_order_production_scenarios_df[
    "production_completion_time_h"
] = production_completion_time_h

multi_order_production_scenarios_df[
    "machine_availability"
] = weighted_machine_availability

multi_order_production_scenarios_df[
    "machine_availability_pct"
] = weighted_machine_availability * 100

multi_order_production_scenarios_df[
    "machine_utilization"
] = average_machine_utilization

multi_order_production_scenarios_df[
    "machine_utilization_pct"
] = average_machine_utilization * 100

# The number of pending jobs is retained as an order-structure
# indicator. It is not interpreted as measured machine congestion.

multi_order_production_scenarios_df[
    "pending_job_count"
] = multi_order_production_scenarios_df[
    "maximum_pending_job_count"
]


# ------------------------------------------------------------
# 14. Add individual machine-level indicators
# ------------------------------------------------------------

for machine_index, machine_id in enumerate(stage_ids):

    multi_order_production_scenarios_df[
        f"machine_availability_{machine_id}"
    ] = availability_matrix[:, machine_index]

    multi_order_production_scenarios_df[
        f"machine_downtime_{machine_id}_h"
    ] = machine_downtime_matrix_h[:, machine_index]

    multi_order_production_scenarios_df[
        f"machine_utilization_{machine_id}"
    ] = machine_utilization_matrix[:, machine_index]

    multi_order_production_scenarios_df[
        f"machine_utilization_{machine_id}_pct"
    ] = (
        machine_utilization_matrix[:, machine_index] * 100
    )


# ------------------------------------------------------------
# 15. Quality-control checks
# ------------------------------------------------------------

# A. Production times cannot be below the nominal baseline.

if not (
    multi_order_production_scenarios_df["production_time_h"]
    >= multi_order_production_scenarios_df[
        "production_baseline_flow_time_h"
    ]
).all():
    raise ValueError(
        "A simulated production time is below its nominal baseline."
    )

# B. Downtime must be non-negative.

if not (
    multi_order_production_scenarios_df["production_downtime_h"] >= 0
).all():
    raise ValueError(
        "Negative machine downtime was generated."
    )

# C. Availability bounds.

machine_availability_columns = [
    f"machine_availability_{machine_id}"
    for machine_id in stage_ids
]

if not multi_order_production_scenarios_df[
    machine_availability_columns
].apply(
    lambda col: col.between(
        MIN_MACHINE_AVAILABILITY_CELL11,
        MAX_MACHINE_AVAILABILITY_CELL11
    )
).all().all():
    raise ValueError(
        "Machine availability is outside the specified bounds."
    )

# D. Production-time increase equals additional downtime.

if not np.allclose(
    multi_order_production_scenarios_df[
        "production_time_increase_h"
    ],
    multi_order_production_scenarios_df[
        "production_downtime_h"
    ],
    atol=1e-8
):
    raise ValueError(
        "Production-time increase does not match additional downtime."
    )

# E. Verify completion-time calculation.

if not np.allclose(
    multi_order_production_scenarios_df[
        "production_completion_time_h"
    ],
    (
        multi_order_production_scenarios_df[
            "order_release_time_h"
        ]
        + multi_order_production_scenarios_df[
            "production_time_h"
        ]
    ),
    atol=1e-8
):
    raise ValueError(
        "Production completion times are inconsistent."
    )

# F. Verify average utilization.

if not np.allclose(
    multi_order_production_scenarios_df[
        "machine_utilization"
    ],
    multi_order_production_scenarios_df[
        [
            f"machine_utilization_{machine_id}"
            for machine_id in stage_ids
        ]
    ].mean(axis=1),
    atol=1e-8
):
    raise ValueError(
        "Average machine utilization was calculated incorrectly."
    )

# G. Verify one observation per distinct customer order.

if len(multi_order_production_scenarios_df) != len(
    multi_order_scenarios_df
):
    raise ValueError(
        "The production scenario count differs from the order portfolio."
    )

if multi_order_production_scenarios_df[
    "customer_order_id"
].duplicated().any():
    raise ValueError(
        "Duplicate customer orders were found after production modeling."
    )

# H. Check for non-finite numerical values.

production_output_columns = [
    "production_time_h",
    "production_downtime_h",
    "production_completion_time_h",
    "machine_availability",
    "machine_utilization"
]

if not np.isfinite(
    multi_order_production_scenarios_df[
        production_output_columns
    ].to_numpy()
).all():
    raise ValueError(
        "Non-finite values were generated in the production scenarios."
    )


# ------------------------------------------------------------
# 16. Verify the original reference order
# ------------------------------------------------------------

reference_order_output = (
    multi_order_production_scenarios_df[
        multi_order_production_scenarios_df[
            "customer_order_id"
        ] == "CO-001"
    ]
)

if len(reference_order_output) != 1:
    raise ValueError(
        "CO-001 must appear exactly once."
    )

reference_nominal_time_h = float(
    reference_order_output[
        "production_baseline_flow_time_h"
    ].iloc[0]
)

if not np.isclose(
    reference_nominal_time_h,
    215.51,
    atol=1e-6
):
    raise ValueError(
        "The original CO-001 nominal production time has changed."
    )


# ------------------------------------------------------------
# 17. Generate production summary statistics
# ------------------------------------------------------------

production_summary_columns = [
    "total_order_quantity",
    "number_of_jobs",
    "production_baseline_flow_time_h",
    "production_processing_time_h",
    "production_setup_time_h",
    "production_interstage_nonprocessing_time_h",
    "production_downtime_h",
    "production_time_h",
    "machine_availability_pct",
    "machine_utilization_pct"
]

multi_order_production_summary_df = (
    multi_order_production_scenarios_df[
        production_summary_columns
    ]
    .agg([
        "mean",
        "std",
        "min",
        "median",
        "max"
    ])
    .T
    .reset_index()
    .rename(columns={"index": "metric"})
)


# ------------------------------------------------------------
# 18. Display results
# ------------------------------------------------------------

print("CELL 11 COMPLETED SUCCESSFULLY")
print("=" * 70)

print(
    f"Number of distinct customer orders: "
    f"{N_ORDER_SCENARIOS}"
)

print(
    f"Number of production machines: "
    f"{len(stage_ids)}"
)

print(
    "Reference order nominal production time: "
    f"{reference_nominal_time_h:.2f} h"
)

print("-" * 70)

print(
    f"Mean nominal production time: "
    f"{baseline_production_flow_time_h.mean():.2f} h"
)

print(
    f"Mean additional production downtime: "
    f"{total_production_downtime_h.mean():.2f} h"
)

print(
    f"Mean simulated production time: "
    f"{production_time_h.mean():.2f} h"
)

print(
    f"Mean machine availability: "
    f"{weighted_machine_availability.mean() * 100:.2f}%"
)

print(
    f"Mean machine utilization: "
    f"{average_machine_utilization.mean() * 100:.2f}%"
)

print("-" * 70)

print("Observation unit: One distinct customer order")

print(
    "Availability assumption: "
    f"Beta({AVAILABILITY_BETA_ALPHA_CELL11}, "
    f"{AVAILABILITY_BETA_BETA_CELL11}), "
    f"bounded by {MIN_MACHINE_AVAILABILITY_CELL11:.2f} "
    f"and {MAX_MACHINE_AVAILABILITY_CELL11:.2f}"
)

print("=" * 70)

display(multi_order_production_summary_df)

display(
    multi_order_production_scenarios_df[
        [
            "scenario_id",
            "customer_order_id",
            "product_mix",
            "total_order_quantity",
            "number_of_jobs",
            "production_baseline_flow_time_h",
            "production_downtime_h",
            "production_time_h",
            "machine_availability_pct",
            "machine_utilization_pct"
        ]
    ].head(10)
)


CELL 11 COMPLETED SUCCESSFULLY
Number of distinct customer orders: 2000
Number of production machines: 4
Reference order nominal production time: 215.51 h
----------------------------------------------------------------------
Mean nominal production time: 227.04 h
Mean additional production downtime: 23.42 h
Mean simulated production time: 250.46 h
Mean machine availability: 90.41%
Mean machine utilization: 21.02%
----------------------------------------------------------------------
Observation unit: One distinct customer order
Availability assumption: Beta(18.0, 2.0), bounded by 0.80 and 1.00


,metric,mean,std,min,median,max
0,total_order_quantity,174.122500,54.155710,28.000000,178.000000,320.000000
1,number_of_jobs,2.561000,0.602877,1.000000,3.000000,3.000000
2,production_baseline_flow_time_h,227.044575,84.337616,24.794400,231.024300,433.034100
3,production_processing_time_h,198.254945,79.666143,14.484400,199.718600,399.334100
4,production_setup_time_h,13.757850,3.333100,3.800000,16.100000,16.100000
5,production_interstage_nonprocessing_time_h,15.031780,3.565122,5.010000,17.600000,17.600000
6,production_downtime_h,23.415674,12.980625,0.884686,21.671466,74.599902
7,production_time_h,250.460249,93.857170,26.978462,255.088754,485.174406
8,machine_availability_pct,90.408119,3.283131,80.531867,90.654062,97.874257
9,machine_utilization_pct,21.024328,0.904331,17.720481,21.080610,23.343976


,scenario_id,customer_order_id,product_mix,total_order_quantity,number_of_jobs,production_baseline_flow_time_h,production_downtime_h,production_time_h,machine_availability_pct,machine_utilization_pct
0,1,CO-001,"Ø900, Ø1000, Ø1200",190,3,215.5100,20.953667,236.463667,90.748959,20.923933
1,2,SIM-000002,"Ø1000, Ø1200",122,2,201.8991,10.553554,212.452654,94.848974,22.366054
2,3,SIM-000003,"Ø1000, Ø1200",125,2,197.1969,20.455479,217.652379,90.180717,21.291623
3,4,SIM-000004,"Ø900, Ø1000, Ø1200",239,3,290.9607,49.917016,340.877716,85.195007,20.048296
4,5,SIM-000005,"Ø1000, Ø1200",106,2,179.0294,22.230354,201.259754,88.864945,20.769105
5,6,SIM-000006,"Ø900, Ø1200",125,2,273.6880,33.511904,307.199904,88.719772,21.248216
6,7,SIM-000007,"Ø1000, Ø1200",118,2,193.5758,11.063014,204.638814,94.646493,22.203241
7,8,SIM-000008,"Ø900, Ø1000, Ø1200",190,3,246.9020,14.088277,260.990277,94.238356,21.964611
8,9,SIM-000009,"Ø900, Ø1000",119,2,81.5926,10.689945,92.282545,87.289775,19.183639
9,10,SIM-000010,"Ø900, Ø1000, Ø1200",209,3,267.1245,34.546921,301.671421,88.249018,20.678500


In [28]:

# ============================================================
# CELL 12 — WAREHOUSE SCENARIOS FOR DISTINCT CUSTOMER ORDERS
# ============================================================
# Purpose:
# 1. Model warehouse operations for 2,000 distinct orders.
# 2. Preserve the production-to-warehouse time connection.
# 3. Generate order-specific warehouse operating conditions.
# 4. Calculate inbound handling, storage, and loading times.
# 5. Calculate order-specific warehouse completion times.
#
# Observation unit:
# One distinct consolidated customer order.
#
# IMPORTANT:
# Warehouse parameters are provisional assumptions.
# Each order is simulated under an independent warehouse-state
# realization; the 2,000 orders are not modeled as a single
# chronological queue of actual factory arrivals.
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Validate production scenarios from Cell 11
# ------------------------------------------------------------

required_columns = [
    "scenario_id",
    "customer_order_id",
    "total_order_quantity",
    "number_of_jobs",
    "production_time_h",
    "production_completion_time_h",
    "machine_availability",
    "machine_utilization"
]

missing_columns = [
    col for col in required_columns
    if col not in multi_order_production_scenarios_df.columns
]

if missing_columns:
    raise ValueError(
        "Missing columns in multi_order_production_scenarios_df: "
        f"{missing_columns}"
    )

if multi_order_production_scenarios_df.empty:
    raise ValueError(
        "The production scenario dataset is empty."
    )

if multi_order_production_scenarios_df["scenario_id"].duplicated().any():
    raise ValueError(
        "Duplicate scenario IDs detected."
    )

if multi_order_production_scenarios_df["customer_order_id"].duplicated().any():
    raise ValueError(
        "Duplicate customer-order IDs detected."
    )

if "RANDOM_SEED" not in globals():
    raise RuntimeError(
        "RANDOM_SEED is missing. Check Cell 1."
    )


# ------------------------------------------------------------
# 2. Create an independent reproducible random generator
# ------------------------------------------------------------

cell12_rng = np.random.default_rng(
    RANDOM_SEED + 12
)


# ------------------------------------------------------------
# 3. Preserve the production scenarios
# ------------------------------------------------------------

multi_order_warehouse_scenarios_df = (
    multi_order_production_scenarios_df
    .copy()
    .sort_values("scenario_id")
    .reset_index(drop=True)
)

N_WAREHOUSE_ORDER_SCENARIOS = len(
    multi_order_warehouse_scenarios_df
)

if N_WAREHOUSE_ORDER_SCENARIOS == 0:
    raise ValueError(
        "No customer-order scenarios are available."
    )

scenario_ids = (
    multi_order_warehouse_scenarios_df["scenario_id"]
    .to_numpy()
)

customer_order_ids = (
    multi_order_warehouse_scenarios_df["customer_order_id"]
    .to_numpy()
)

order_quantity = (
    multi_order_warehouse_scenarios_df[
        "total_order_quantity"
    ].astype(float).to_numpy()
)

# Warehouse operations begin after each order completes production.

warehouse_start_time_h = (
    multi_order_warehouse_scenarios_df[
        "production_completion_time_h"
    ].astype(float).to_numpy()
)


# ------------------------------------------------------------
# 4. Define provisional warehouse parameters
# ------------------------------------------------------------

# A. Inbound handling capacity
# Unit: pipe-equivalent units per hour.

WAREHOUSE_INBOUND_RATE_MIN_CELL12 = 12.0
WAREHOUSE_INBOUND_RATE_MODE_CELL12 = 18.0
WAREHOUSE_INBOUND_RATE_MAX_CELL12 = 26.0

# B. Loading capacity
# Unit: pipe-equivalent units per hour.

WAREHOUSE_LOADING_RATE_MIN_CELL12 = 15.0
WAREHOUSE_LOADING_RATE_MODE_CELL12 = 22.0
WAREHOUSE_LOADING_RATE_MAX_CELL12 = 30.0

# C. Background warehouse workload.
#
# Gamma-distributed pipe-equivalent workload.
# Mean = shape * scale = 80 workload units.

WAREHOUSE_BACKLOG_GAMMA_SHAPE_CELL12 = 4.0
WAREHOUSE_BACKLOG_GAMMA_SCALE_CELL12 = 20.0

# D. Storage/staging duration.

WAREHOUSE_STORAGE_MIN_H_CELL12 = 0.5
WAREHOUSE_STORAGE_MODE_H_CELL12 = 1.5
WAREHOUSE_STORAGE_MAX_H_CELL12 = 4.0

WAREHOUSE_PARAMETER_STATUS_CELL12 = (
    "Provisional simulation assumptions requiring calibration "
    "with warehouse handling, staging, loading, and dispatch records."
)


# ------------------------------------------------------------
# 5. Generate warehouse operating conditions
# ------------------------------------------------------------

# Inbound handling rate.

warehouse_inbound_rate = cell12_rng.triangular(
    WAREHOUSE_INBOUND_RATE_MIN_CELL12,
    WAREHOUSE_INBOUND_RATE_MODE_CELL12,
    WAREHOUSE_INBOUND_RATE_MAX_CELL12,
    size=N_WAREHOUSE_ORDER_SCENARIOS
)

# Loading rate.

warehouse_loading_rate = cell12_rng.triangular(
    WAREHOUSE_LOADING_RATE_MIN_CELL12,
    WAREHOUSE_LOADING_RATE_MODE_CELL12,
    WAREHOUSE_LOADING_RATE_MAX_CELL12,
    size=N_WAREHOUSE_ORDER_SCENARIOS
)

# Background workload ahead of the arriving customer order.
#
# This represents pipe-equivalent workload, not an observed
# number of physical pipes waiting in the factory warehouse.

warehouse_background_queue_workload = (
    cell12_rng.gamma(
        shape=WAREHOUSE_BACKLOG_GAMMA_SHAPE_CELL12,
        scale=WAREHOUSE_BACKLOG_GAMMA_SCALE_CELL12,
        size=N_WAREHOUSE_ORDER_SCENARIOS
    )
)

# Storage/staging time.

warehouse_storage_dwell_time_h = cell12_rng.triangular(
    WAREHOUSE_STORAGE_MIN_H_CELL12,
    WAREHOUSE_STORAGE_MODE_H_CELL12,
    WAREHOUSE_STORAGE_MAX_H_CELL12,
    size=N_WAREHOUSE_ORDER_SCENARIOS
)


# ------------------------------------------------------------
# 6. Calculate warehouse queue waiting time
# ------------------------------------------------------------

# Assumption:
# Previously accumulated warehouse workload is processed before
# the incoming order under an effective FIFO policy.
#
# Queue waiting time =
# Background workload / Inbound handling rate

warehouse_queue_wait_time_h = (
    warehouse_background_queue_workload
    / warehouse_inbound_rate
)


# ------------------------------------------------------------
# 7. Calculate inbound handling time
# ------------------------------------------------------------

# The consolidated order is handled as one complete batch.
#
# Inbound handling time =
# Total order quantity / Inbound handling rate

warehouse_inbound_handling_time_h = (
    order_quantity / warehouse_inbound_rate
)


# ------------------------------------------------------------
# 8. Calculate loading time
# ------------------------------------------------------------

# Loading time =
# Total order quantity / Loading rate

warehouse_loading_time_h = (
    order_quantity / warehouse_loading_rate
)


# ------------------------------------------------------------
# 9. Calculate warehouse workload indicators
# ------------------------------------------------------------

warehouse_background_processing_load_h = (
    warehouse_background_queue_workload
    / warehouse_inbound_rate
)

warehouse_order_handling_load_h = (
    order_quantity / warehouse_inbound_rate
)

warehouse_total_inbound_workload_h = (
    warehouse_background_processing_load_h
    + warehouse_order_handling_load_h
)

# Relative background workload compared with the incoming order.
# This is not a physical warehouse occupancy ratio.

warehouse_backlog_ratio = (
    warehouse_background_queue_workload
    / order_quantity
)


# ------------------------------------------------------------
# 10. Calculate total warehouse elapsed time
# ------------------------------------------------------------

warehouse_processing_time_h = (
    warehouse_inbound_handling_time_h
    + warehouse_loading_time_h
)

warehouse_total_waiting_time_h = (
    warehouse_queue_wait_time_h
    + warehouse_storage_dwell_time_h
)

warehouse_total_elapsed_time_h = (
    warehouse_queue_wait_time_h
    + warehouse_inbound_handling_time_h
    + warehouse_storage_dwell_time_h
    + warehouse_loading_time_h
)


# ------------------------------------------------------------
# 11. Calculate warehouse completion time
# ------------------------------------------------------------

# Production and warehouse operations are sequential.
#
# Warehouse completion =
# Production completion + Warehouse elapsed time

warehouse_completion_time_h = (
    warehouse_start_time_h
    + warehouse_total_elapsed_time_h
)


# ------------------------------------------------------------
# 12. Add warehouse variables to the order scenarios
# ------------------------------------------------------------

multi_order_warehouse_scenarios_df[
    "warehouse_start_time_h"
] = warehouse_start_time_h

multi_order_warehouse_scenarios_df[
    "warehouse_background_queue_workload_equiv_pipes"
] = warehouse_background_queue_workload

multi_order_warehouse_scenarios_df[
    "warehouse_inbound_rate_equiv_pipes_per_h"
] = warehouse_inbound_rate

multi_order_warehouse_scenarios_df[
    "warehouse_loading_rate_equiv_pipes_per_h"
] = warehouse_loading_rate

multi_order_warehouse_scenarios_df[
    "warehouse_queue_wait_time_h"
] = warehouse_queue_wait_time_h

multi_order_warehouse_scenarios_df[
    "warehouse_inbound_handling_time_h"
] = warehouse_inbound_handling_time_h

multi_order_warehouse_scenarios_df[
    "warehouse_storage_dwell_time_h"
] = warehouse_storage_dwell_time_h

multi_order_warehouse_scenarios_df[
    "warehouse_loading_time_h"
] = warehouse_loading_time_h

multi_order_warehouse_scenarios_df[
    "warehouse_processing_time_h"
] = warehouse_processing_time_h

multi_order_warehouse_scenarios_df[
    "warehouse_total_waiting_time_h"
] = warehouse_total_waiting_time_h

multi_order_warehouse_scenarios_df[
    "warehouse_total_elapsed_time_h"
] = warehouse_total_elapsed_time_h

multi_order_warehouse_scenarios_df[
    "warehouse_total_inbound_workload_h"
] = warehouse_total_inbound_workload_h

multi_order_warehouse_scenarios_df[
    "warehouse_backlog_ratio"
] = warehouse_backlog_ratio

multi_order_warehouse_scenarios_df[
    "warehouse_completion_time_h"
] = warehouse_completion_time_h


# ------------------------------------------------------------
# 13. Quality-control checks
# ------------------------------------------------------------

# A. Warehouse duration variables must be non-negative.

warehouse_duration_columns = [
    "warehouse_queue_wait_time_h",
    "warehouse_inbound_handling_time_h",
    "warehouse_storage_dwell_time_h",
    "warehouse_loading_time_h",
    "warehouse_total_elapsed_time_h"
]

if (
    multi_order_warehouse_scenarios_df[
        warehouse_duration_columns
    ] < 0
).any().any():
    raise ValueError(
        "Negative warehouse durations were generated."
    )

# B. Check finite values.

warehouse_numeric_columns = [
    "warehouse_start_time_h",
    "warehouse_inbound_rate_equiv_pipes_per_h",
    "warehouse_loading_rate_equiv_pipes_per_h",
    "warehouse_queue_wait_time_h",
    "warehouse_inbound_handling_time_h",
    "warehouse_storage_dwell_time_h",
    "warehouse_loading_time_h",
    "warehouse_total_elapsed_time_h",
    "warehouse_completion_time_h"
]

if not np.isfinite(
    multi_order_warehouse_scenarios_df[
        warehouse_numeric_columns
    ].to_numpy()
).all():
    raise ValueError(
        "Non-finite warehouse values were generated."
    )

# C. Verify production-to-warehouse continuity.

if not np.allclose(
    multi_order_warehouse_scenarios_df[
        "warehouse_start_time_h"
    ],
    multi_order_warehouse_scenarios_df[
        "production_completion_time_h"
    ],
    atol=1e-8
):
    raise ValueError(
        "Warehouse start times do not match production completion times."
    )

# D. Verify warehouse time balance.

calculated_warehouse_elapsed_h = (
    multi_order_warehouse_scenarios_df[
        "warehouse_queue_wait_time_h"
    ]
    + multi_order_warehouse_scenarios_df[
        "warehouse_inbound_handling_time_h"
    ]
    + multi_order_warehouse_scenarios_df[
        "warehouse_storage_dwell_time_h"
    ]
    + multi_order_warehouse_scenarios_df[
        "warehouse_loading_time_h"
    ]
)

if not np.allclose(
    multi_order_warehouse_scenarios_df[
        "warehouse_total_elapsed_time_h"
    ],
    calculated_warehouse_elapsed_h,
    atol=1e-8
):
    raise ValueError(
        "Warehouse elapsed-time balance does not hold."
    )

# E. Verify warehouse completion time.

calculated_warehouse_completion_h = (
    multi_order_warehouse_scenarios_df[
        "production_completion_time_h"
    ]
    + multi_order_warehouse_scenarios_df[
        "warehouse_total_elapsed_time_h"
    ]
)

if not np.allclose(
    multi_order_warehouse_scenarios_df[
        "warehouse_completion_time_h"
    ],
    calculated_warehouse_completion_h,
    atol=1e-8
):
    raise ValueError(
        "Warehouse completion times are inconsistent."
    )

# F. Verify unique order-level observations.

if multi_order_warehouse_scenarios_df[
    "customer_order_id"
].duplicated().any():
    raise ValueError(
        "Duplicate customer orders were found."
    )

if len(multi_order_warehouse_scenarios_df) != len(
    multi_order_production_scenarios_df
):
    raise ValueError(
        "Warehouse scenario count differs from production scenario count."
    )

if not np.array_equal(
    multi_order_warehouse_scenarios_df["scenario_id"].to_numpy(),
    scenario_ids
):
    raise ValueError(
        "Scenario IDs changed during warehouse modeling."
    )


# ------------------------------------------------------------
# 14. Generate warehouse summary statistics
# ------------------------------------------------------------

warehouse_summary_columns = [
    "total_order_quantity",
    "warehouse_background_queue_workload_equiv_pipes",
    "warehouse_inbound_rate_equiv_pipes_per_h",
    "warehouse_loading_rate_equiv_pipes_per_h",
    "warehouse_queue_wait_time_h",
    "warehouse_inbound_handling_time_h",
    "warehouse_storage_dwell_time_h",
    "warehouse_loading_time_h",
    "warehouse_processing_time_h",
    "warehouse_total_waiting_time_h",
    "warehouse_total_elapsed_time_h",
    "warehouse_backlog_ratio",
    "warehouse_completion_time_h"
]

multi_order_warehouse_summary_df = (
    multi_order_warehouse_scenarios_df[
        warehouse_summary_columns
    ]
    .agg([
        "mean",
        "std",
        "min",
        "median",
        "max"
    ])
    .T
    .reset_index()
    .rename(columns={"index": "metric"})
)


# ------------------------------------------------------------
# 15. Analyze warehouse operations by order size
# ------------------------------------------------------------

# Descriptive analysis only.
# These groups are not used to impose different handling rates.

multi_order_warehouse_by_size_df = (
    multi_order_warehouse_scenarios_df
    .assign(
        order_size_group=pd.cut(
            multi_order_warehouse_scenarios_df[
                "total_order_quantity"
            ],
            bins=[0, 100, 200, 300, np.inf],
            labels=[
                "Small",
                "Medium",
                "Large",
                "Very large"
            ]
        )
    )
    .groupby(
        "order_size_group",
        observed=True
    )
    .agg(
        number_of_orders=("customer_order_id", "count"),
        mean_order_quantity=("total_order_quantity", "mean"),
        mean_inbound_time_h=("warehouse_inbound_handling_time_h", "mean"),
        mean_loading_time_h=("warehouse_loading_time_h", "mean"),
        mean_warehouse_elapsed_h=("warehouse_total_elapsed_time_h", "mean")
    )
    .reset_index()
)


# ------------------------------------------------------------
# 16. Display results
# ------------------------------------------------------------

print("CELL 12 COMPLETED SUCCESSFULLY")
print("=" * 70)

print(
    f"Number of distinct customer orders: "
    f"{N_WAREHOUSE_ORDER_SCENARIOS}"
)

print(
    f"Number of unique order IDs: "
    f"{multi_order_warehouse_scenarios_df['customer_order_id'].nunique()}"
)

print("-" * 70)

print(
    f"Mean production completion time: "
    f"{multi_order_warehouse_scenarios_df['production_completion_time_h'].mean():.2f} h"
)

print(
    f"Mean warehouse queue waiting time: "
    f"{multi_order_warehouse_scenarios_df['warehouse_queue_wait_time_h'].mean():.2f} h"
)

print(
    f"Mean inbound handling time: "
    f"{multi_order_warehouse_scenarios_df['warehouse_inbound_handling_time_h'].mean():.2f} h"
)

print(
    f"Mean storage/staging time: "
    f"{multi_order_warehouse_scenarios_df['warehouse_storage_dwell_time_h'].mean():.2f} h"
)

print(
    f"Mean loading time: "
    f"{multi_order_warehouse_scenarios_df['warehouse_loading_time_h'].mean():.2f} h"
)

print(
    f"Mean total warehouse elapsed time: "
    f"{multi_order_warehouse_scenarios_df['warehouse_total_elapsed_time_h'].mean():.2f} h"
)

print(
    f"Mean warehouse completion time: "
    f"{multi_order_warehouse_scenarios_df['warehouse_completion_time_h'].mean():.2f} h"
)

print("=" * 70)

display(multi_order_warehouse_summary_df)

display(multi_order_warehouse_by_size_df)

display(
    multi_order_warehouse_scenarios_df[
        [
            "scenario_id",
            "customer_order_id",
            "product_mix",
            "total_order_quantity",
            "production_completion_time_h",
            "warehouse_queue_wait_time_h",
            "warehouse_inbound_handling_time_h",
            "warehouse_storage_dwell_time_h",
            "warehouse_loading_time_h",
            "warehouse_total_elapsed_time_h",
            "warehouse_completion_time_h"
        ]
    ].head(10)
)


CELL 12 COMPLETED SUCCESSFULLY
Number of distinct customer orders: 2000
Number of unique order IDs: 2000
----------------------------------------------------------------------
Mean production completion time: 250.46 h
Mean warehouse queue waiting time: 4.37 h
Mean inbound handling time: 9.59 h
Mean storage/staging time: 2.04 h
Mean loading time: 7.97 h
Mean total warehouse elapsed time: 23.97 h
Mean warehouse completion time: 274.43 h


,metric,mean,std,min,median,max
0,total_order_quantity,174.122500,54.155710,28.000000,178.000000,320.000000
1,warehouse_background_queue_workload_equiv_pipes,79.278302,40.274202,5.872354,71.773275,320.038831
2,warehouse_inbound_rate_equiv_pipes_per_h,18.603773,2.831095,12.072033,18.456619,25.833898
3,warehouse_loading_rate_equiv_pipes_per_h,22.331127,3.116592,15.309116,22.291852,29.837437
4,warehouse_queue_wait_time_h,4.368616,2.329508,0.309452,3.888372,16.756478
5,warehouse_inbound_handling_time_h,9.594686,3.402708,1.696159,9.520876,22.883791
6,warehouse_storage_dwell_time_h,2.040426,0.733933,0.522138,1.928452,3.987726
7,warehouse_loading_time_h,7.969772,2.809177,1.344028,7.916163,18.923764
8,warehouse_processing_time_h,17.564458,5.888703,3.043449,17.606073,41.807556
9,warehouse_total_waiting_time_h,6.409042,2.427467,1.219474,6.030615,18.965864


,order_size_group,number_of_orders,mean_order_quantity,mean_inbound_time_h,mean_loading_time_h,mean_warehouse_elapsed_h
0,Small,214,75.672897,4.140965,3.440171,14.038879
1,Medium,1120,158.428571,8.696147,7.197170,22.286936
2,Large,663,231.776772,12.830151,10.700475,29.953516
3,Very large,3,314.333333,19.043653,16.033689,40.709772


,scenario_id,customer_order_id,product_mix,total_order_quantity,production_completion_time_h,warehouse_queue_wait_time_h,warehouse_inbound_handling_time_h,warehouse_storage_dwell_time_h,warehouse_loading_time_h,warehouse_total_elapsed_time_h,warehouse_completion_time_h
0,1,CO-001,"Ø900, Ø1000, Ø1200",190,236.463667,5.845538,9.587356,1.908276,8.385934,25.727105,262.190771
1,2,SIM-000002,"Ø1000, Ø1200",122,212.452654,2.972472,4.939171,2.600803,6.423510,16.935956,229.388610
2,3,SIM-000003,"Ø1000, Ø1200",125,217.652379,5.712994,7.635249,1.602441,5.280725,20.231409,237.883789
3,4,SIM-000004,"Ø900, Ø1000, Ø1200",239,340.877716,4.126150,18.002477,1.580462,12.154883,35.863973,376.741689
4,5,SIM-000005,"Ø1000, Ø1200",106,201.259754,2.406416,4.755249,2.608866,5.549918,15.320449,216.580203
5,6,SIM-000006,"Ø900, Ø1200",125,307.199904,0.687478,6.749805,1.462025,4.552911,13.452220,320.652124
6,7,SIM-000007,"Ø1000, Ø1200",118,204.638814,3.234581,6.467091,1.741417,6.068998,17.512087,222.150902
7,8,SIM-000008,"Ø900, Ø1000, Ø1200",190,260.990277,6.064594,11.409936,3.614728,7.110979,28.200237,289.190513
8,9,SIM-000009,"Ø900, Ø1000",119,92.282545,7.928089,7.757451,1.746859,4.367853,21.800251,114.082796
9,10,SIM-000010,"Ø900, Ø1000, Ø1200",209,301.671421,9.602286,16.078938,3.483039,10.888978,40.053241,341.724663


In [29]:

# ============================================================
# CELL 13R — FORECAST-BASED TRANSPORTATION MODEL
# ============================================================
# Prediction information is available before vehicle departure.
# Actual dispatch waiting and driving time are future outcomes.
#
# The distinction introduces operational uncertainty into
# the delivery-lateness prediction problem.
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Validate warehouse scenarios
# ------------------------------------------------------------

required = [
    "scenario_id",
    "customer_order_id",
    "total_order_quantity",
    "warehouse_completion_time_h",
    "warehouse_total_elapsed_time_h",
    "production_time_h",
    "delivery_distance_km"
]

missing = [c for c in required if c not in
           multi_order_warehouse_scenarios_df.columns]

if missing:
    raise ValueError(f"Missing warehouse variables: {missing}")

source = (
    multi_order_warehouse_scenarios_df
    .copy()
    .sort_values("scenario_id")
    .reset_index(drop=True)
)

if source["customer_order_id"].duplicated().any():
    raise ValueError("Duplicate customer orders detected.")

if not source["customer_order_id"].equals(
    source["customer_order_id"].drop_duplicates()
):
    raise ValueError("Customer-order observations are not unique.")

if "RANDOM_SEED" not in globals():
    raise RuntimeError("RANDOM_SEED is missing.")


# ------------------------------------------------------------
# 2. Independent reproducible random generator
# ------------------------------------------------------------

cell13_rng = np.random.default_rng(RANDOM_SEED + 13)

n = len(source)


# ------------------------------------------------------------
# 3. Define dispatch-time forecast assumptions
# ------------------------------------------------------------

# Forecast driving speed (km/h).

BASE_SPEED_MIN = 45.0
BASE_SPEED_MODE = 60.0
BASE_SPEED_MAX = 75.0

# Estimated vehicle availability.

VEHICLE_BETA_ALPHA = 16.0
VEHICLE_BETA_BETA = 4.0

MIN_VEHICLE_AVAILABILITY = 0.60
MAX_VEHICLE_AVAILABILITY = 1.00

# Forecast traffic:
# 1 = Low, 2 = Moderate, 3 = High, 4 = Severe.

TRAFFIC_LEVELS = [1, 2, 3, 4]
TRAFFIC_PROBABILITIES = [0.25, 0.40, 0.25, 0.10]

TRAFFIC_MULTIPLIERS = {
    1: 1.00,
    2: 1.15,
    3: 1.35,
    4: 1.60
}

# Forecast weather:
# 1 = Favorable, 2 = Adverse, 3 = Severe.

WEATHER_LEVELS = [1, 2, 3]
WEATHER_PROBABILITIES = [0.65, 0.25, 0.10]

WEATHER_MULTIPLIERS = {
    1: 1.00,
    2: 1.10,
    3: 1.25
}

# Fixed route characteristic because all orders currently
# have the same destination.
#
# Level 2 = Moderate route complexity.
# This is an illustrative assumption requiring verification.

ROUTE_COMPLEXITY_LEVEL = 2

ROUTE_COMPLEXITY_MULTIPLIER = 1.08


# ------------------------------------------------------------
# 4. Define future operating uncertainty
# ------------------------------------------------------------

# Coefficients of variation are provisional assumptions.
# They should be tested through sensitivity analysis.

TRAVEL_TIME_ERROR_CV = 0.15

DISPATCH_WAIT_ERROR_CV = 0.30

# The lognormal multiplier has an expected value of one.
# This preserves the forecast duration as the conditional
# mean while allowing positive, right-skewed deviations.

def generate_mean_one_lognormal(cv, size, rng):

    if cv <= 0:
        raise ValueError("Coefficient of variation must be positive.")

    sigma = np.sqrt(np.log(1.0 + cv**2))

    return np.exp(
        -0.5 * sigma**2
        + sigma * rng.standard_normal(size)
    )


# ------------------------------------------------------------
# 5. Generate information available at dispatch
# ------------------------------------------------------------

forecast_speed_kmh = cell13_rng.triangular(
    BASE_SPEED_MIN,
    BASE_SPEED_MODE,
    BASE_SPEED_MAX,
    size=n
)

forecast_vehicle_availability = cell13_rng.beta(
    VEHICLE_BETA_ALPHA,
    VEHICLE_BETA_BETA,
    size=n
)

forecast_vehicle_availability = np.clip(
    forecast_vehicle_availability,
    MIN_VEHICLE_AVAILABILITY,
    MAX_VEHICLE_AVAILABILITY
)

forecast_traffic_level = cell13_rng.choice(
    TRAFFIC_LEVELS,
    size=n,
    p=TRAFFIC_PROBABILITIES
)

forecast_weather_condition = cell13_rng.choice(
    WEATHER_LEVELS,
    size=n,
    p=WEATHER_PROBABILITIES
)

forecast_traffic_multiplier = np.array([
    TRAFFIC_MULTIPLIERS[int(x)]
    for x in forecast_traffic_level
])

forecast_weather_multiplier = np.array([
    WEATHER_MULTIPLIERS[int(x)]
    for x in forecast_weather_condition
])

# Physical route distance remains fixed for this destination.

delivery_distance_km = (
    source["delivery_distance_km"].astype(float).to_numpy()
)


# ------------------------------------------------------------
# 6. Calculate expected travel time at dispatch
# ------------------------------------------------------------

forecast_base_travel_time_h = (
    delivery_distance_km / forecast_speed_kmh
)

forecast_expected_travel_time_h = (
    forecast_base_travel_time_h
    * forecast_traffic_multiplier
    * forecast_weather_multiplier
    * ROUTE_COMPLEXITY_MULTIPLIER
)


# ------------------------------------------------------------
# 7. Estimate dispatch waiting from vehicle availability
# ------------------------------------------------------------

# This retains the previous availability-adjusted waiting
# approximation. The 8-hour reference coefficient is provisional,
# not a calibrated fleet-queue model.

DISPATCH_REFERENCE_WAIT_H = 8.0

dispatch_wait_reference_h = (
    DISPATCH_REFERENCE_WAIT_H
    * (1.0 - forecast_vehicle_availability)
    / forecast_vehicle_availability
)


# ------------------------------------------------------------
# 8. Generate future dispatch and travel-time uncertainty
# ------------------------------------------------------------

dispatch_wait_shock = generate_mean_one_lognormal(
    cv=DISPATCH_WAIT_ERROR_CV,
    size=n,
    rng=cell13_rng
)

travel_time_shock = generate_mean_one_lognormal(
    cv=TRAVEL_TIME_ERROR_CV,
    size=n,
    rng=cell13_rng
)

# Actual dispatch waiting time is not known before departure.

actual_dispatch_wait_time_h = (
    dispatch_wait_reference_h * dispatch_wait_shock
)

# Actual travel duration differs from the forecast because
# of unobserved route and traffic variability.

actual_driving_time_h = (
    forecast_expected_travel_time_h * travel_time_shock
)

travel_time_forecast_error_h = (
    actual_driving_time_h
    - forecast_expected_travel_time_h
)


# ------------------------------------------------------------
# 9. Calculate future transportation completion
# ------------------------------------------------------------

transportation_start_time_h = (
    source["warehouse_completion_time_h"]
    .astype(float)
    .to_numpy()
)

transportation_total_elapsed_time_h = (
    actual_dispatch_wait_time_h
    + actual_driving_time_h
)

transportation_completion_time_h = (
    transportation_start_time_h
    + transportation_total_elapsed_time_h
)

# Each order timeline uses release time zero.

actual_delivery_time_h = (
    transportation_completion_time_h
)


# ------------------------------------------------------------
# 10. Create revised integrated transportation dataset
# ------------------------------------------------------------

multi_order_transportation_scenarios_df = source.copy()

multi_order_transportation_scenarios_df[
    "transportation_start_time_h"
] = transportation_start_time_h

# Dispatch-time predictor variables.

multi_order_transportation_scenarios_df[
    "forecast_base_speed_kmh"
] = forecast_speed_kmh

multi_order_transportation_scenarios_df[
    "forecast_vehicle_availability"
] = forecast_vehicle_availability

multi_order_transportation_scenarios_df[
    "forecast_traffic_level"
] = forecast_traffic_level

multi_order_transportation_scenarios_df[
    "forecast_weather_condition"
] = forecast_weather_condition

multi_order_transportation_scenarios_df[
    "route_complexity_level"
] = ROUTE_COMPLEXITY_LEVEL

multi_order_transportation_scenarios_df[
    "forecast_expected_travel_time_h"
] = forecast_expected_travel_time_h

# Future outcomes. These must not be used as predictors.

multi_order_transportation_scenarios_df[
    "actual_dispatch_wait_time_h"
] = actual_dispatch_wait_time_h

multi_order_transportation_scenarios_df[
    "actual_driving_time_h"
] = actual_driving_time_h

multi_order_transportation_scenarios_df[
    "travel_time_forecast_error_h"
] = travel_time_forecast_error_h

multi_order_transportation_scenarios_df[
    "transportation_total_elapsed_time_h"
] = transportation_total_elapsed_time_h

multi_order_transportation_scenarios_df[
    "transportation_completion_time_h"
] = transportation_completion_time_h

multi_order_transportation_scenarios_df[
    "actual_delivery_time_h"
] = actual_delivery_time_h


# ------------------------------------------------------------
# 11. Verify chronological integration and time balance
# ------------------------------------------------------------

if not np.allclose(
    multi_order_transportation_scenarios_df[
        "transportation_start_time_h"
    ],
    multi_order_transportation_scenarios_df[
        "warehouse_completion_time_h"
    ],
    atol=1e-8
):
    raise ValueError("Warehouse-to-transportation continuity failed.")

if not np.allclose(
    multi_order_transportation_scenarios_df[
        "transportation_total_elapsed_time_h"
    ],
    (
        multi_order_transportation_scenarios_df[
            "actual_dispatch_wait_time_h"
        ]
        + multi_order_transportation_scenarios_df[
            "actual_driving_time_h"
        ]
    ),
    atol=1e-8
):
    raise ValueError("Transportation time balance failed.")

if not np.allclose(
    multi_order_transportation_scenarios_df[
        "actual_delivery_time_h"
    ],
    (
        multi_order_transportation_scenarios_df[
            "warehouse_completion_time_h"
        ]
        + multi_order_transportation_scenarios_df[
            "transportation_total_elapsed_time_h"
        ]
    ),
    atol=1e-8
):
    raise ValueError("Delivery completion time is inconsistent.")

if not (
    multi_order_transportation_scenarios_df[
        [
            "actual_dispatch_wait_time_h",
            "actual_driving_time_h",
            "transportation_total_elapsed_time_h"
        ]
    ] > 0
).all().all():
    raise ValueError("Non-positive transportation durations detected.")

if multi_order_transportation_scenarios_df[
    "customer_order_id"
].duplicated().any():
    raise ValueError("Duplicate customer orders detected.")

if not np.isfinite(
    multi_order_transportation_scenarios_df[
        [
            "forecast_expected_travel_time_h",
            "actual_dispatch_wait_time_h",
            "actual_driving_time_h",
            "actual_delivery_time_h"
        ]
    ].to_numpy()
).all():
    raise ValueError("Non-finite transportation values detected.")


# ------------------------------------------------------------
# 12. Generate concise transportation summary
# ------------------------------------------------------------

transportation_summary_df = (
    multi_order_transportation_scenarios_df[
        [
            "forecast_expected_travel_time_h",
            "actual_dispatch_wait_time_h",
            "actual_driving_time_h",
            "travel_time_forecast_error_h",
            "transportation_total_elapsed_time_h",
            "actual_delivery_time_h"
        ]
    ]
    .agg(["mean", "std", "min", "median", "max"])
    .T
    .reset_index()
    .rename(columns={"index": "metric"})
)


# ------------------------------------------------------------
# 13. Display results
# ------------------------------------------------------------

print("REVISED CELL 13 COMPLETED SUCCESSFULLY")
print("=" * 65)

print("Distinct customer orders:", n)

print("Destination diversity:",
      multi_order_transportation_scenarios_df[
          "destination_code"
      ].nunique())

print("Route complexity level:", ROUTE_COMPLEXITY_LEVEL)

print("Travel-time forecast error CV:",
      TRAVEL_TIME_ERROR_CV)

print("Dispatch-wait uncertainty CV:",
      DISPATCH_WAIT_ERROR_CV)

print("-" * 65)

print(
    "Mean forecast travel time:",
    round(forecast_expected_travel_time_h.mean(), 2),
    "h"
)

print(
    "Mean realized driving time:",
    round(actual_driving_time_h.mean(), 2),
    "h"
)

print(
    "Mean actual dispatch waiting:",
    round(actual_dispatch_wait_time_h.mean(), 2),
    "h"
)

print(
    "Mean actual delivery completion:",
    round(actual_delivery_time_h.mean(), 2),
    "h"
)

print("=" * 65)

display(transportation_summary_df)

display(
    multi_order_transportation_scenarios_df[
        [
            "customer_order_id",
            "delivery_distance_km",
            "forecast_base_speed_kmh",
            "forecast_vehicle_availability",
            "forecast_traffic_level",
            "forecast_weather_condition",
            "forecast_expected_travel_time_h",
            "actual_dispatch_wait_time_h",
            "actual_driving_time_h",
            "actual_delivery_time_h"
        ]
    ].head(10)
)


REVISED CELL 13 COMPLETED SUCCESSFULLY
Distinct customer orders: 2000
Destination diversity: 1
Route complexity level: 2
Travel-time forecast error CV: 0.15
Dispatch-wait uncertainty CV: 0.3
-----------------------------------------------------------------
Mean forecast travel time: 6.88 h
Mean realized driving time: 6.88 h
Mean actual dispatch waiting: 2.11 h
Mean actual delivery completion: 283.43 h


,metric,mean,std,min,median,max
0,forecast_expected_travel_time_h,6.883187,1.379250,4.349721,6.682246,13.117930
1,actual_dispatch_wait_time_h,2.114172,1.340365,0.104114,1.813359,10.349360
2,actual_driving_time_h,6.880553,1.744128,3.240753,6.620948,14.962408
3,travel_time_forecast_error_h,-0.002634,1.049389,-3.677297,-0.080147,4.232096
4,transportation_total_elapsed_time_h,8.994725,2.261015,3.868813,8.654217,20.130112
5,actual_delivery_time_h,283.428474,98.189186,46.863748,288.857353,532.390648


,customer_order_id,delivery_distance_km,forecast_base_speed_kmh,forecast_vehicle_availability,forecast_traffic_level,forecast_weather_condition,forecast_expected_travel_time_h,actual_dispatch_wait_time_h,actual_driving_time_h,actual_delivery_time_h
0,CO-001,300.0,61.889267,0.794306,2,1,6.020430,1.904090,6.673715,270.768577
1,SIM-000002,300.0,68.767707,0.720101,4,1,7.538422,4.723396,7.357958,241.469964
2,SIM-000003,300.0,66.407036,0.789039,2,2,6.171936,1.451485,6.560072,245.895346
3,SIM-000004,300.0,65.855325,0.843111,2,1,5.657857,0.873525,6.257292,383.872507
4,SIM-000005,300.0,56.826247,0.907947,3,1,7.697147,0.596059,7.092172,224.268434
5,SIM-000006,300.0,65.539515,0.782251,1,1,4.943583,1.642341,5.340156,327.634620
6,SIM-000007,300.0,47.800006,0.828821,4,1,10.845187,1.317840,7.846113,231.314855
7,SIM-000008,300.0,58.245915,0.848442,1,1,5.562622,1.538641,5.377396,296.106551
8,SIM-000009,300.0,62.958030,0.857794,3,1,6.947486,1.354652,6.973487,122.410935
9,SIM-000010,300.0,58.779319,0.749220,4,3,11.024286,1.969505,9.047732,352.741899


In [30]:

# ============================================================
# CELL 14 — INTEGRATED CUSTOMER-ORDER DATASET
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Validate the three operational datasets
# ------------------------------------------------------------

datasets = {
    "production": multi_order_production_scenarios_df,
    "warehouse": multi_order_warehouse_scenarios_df,
    "transportation": multi_order_transportation_scenarios_df
}

for name, df in datasets.items():

    required = [
        "scenario_id",
        "customer_order_id"
    ]

    missing = [c for c in required if c not in df.columns]

    if missing:
        raise ValueError(
            f"Missing columns in {name} dataset: {missing}"
        )

    if df["scenario_id"].duplicated().any():
        raise ValueError(
            f"Duplicate scenario IDs in {name} dataset."
        )

    if df["customer_order_id"].duplicated().any():
        raise ValueError(
            f"Duplicate customer orders in {name} dataset."
        )

# ------------------------------------------------------------
# 2. Check scenario and customer-order consistency
# ------------------------------------------------------------

production_df = multi_order_production_scenarios_df.copy()
warehouse_df = multi_order_warehouse_scenarios_df.copy()
transport_df = multi_order_transportation_scenarios_df.copy()

for df in [production_df, warehouse_df, transport_df]:
    df.sort_values("scenario_id", inplace=True)
    df.reset_index(drop=True, inplace=True)

if not (
    production_df["scenario_id"].equals(
        warehouse_df["scenario_id"]
    )
    and production_df["scenario_id"].equals(
        transport_df["scenario_id"]
    )
):
    raise ValueError("Scenario IDs do not match across stages.")

if not (
    production_df["customer_order_id"].equals(
        warehouse_df["customer_order_id"]
    )
    and production_df["customer_order_id"].equals(
        transport_df["customer_order_id"]
    )
):
    raise ValueError("Customer-order IDs do not match across stages.")

# ------------------------------------------------------------
# 3. Verify chronological connections between stages
# ------------------------------------------------------------

# Warehouse starts when production is completed.

if not np.allclose(
    warehouse_df["warehouse_start_time_h"],
    production_df["production_completion_time_h"],
    atol=1e-8
):
    raise ValueError(
        "Production completion and warehouse start are inconsistent."
    )

# Transportation starts when warehouse operations are completed.

if not np.allclose(
    transport_df["transportation_start_time_h"],
    warehouse_df["warehouse_completion_time_h"],
    atol=1e-8
):
    raise ValueError(
        "Warehouse completion and transportation start are inconsistent."
    )

# ------------------------------------------------------------
# 4. Construct the integrated order-level dataset
# ------------------------------------------------------------

integrated_order_dataset_df = transport_df.copy()

# ------------------------------------------------------------
# 5. Calculate actual delivery lead time
# ------------------------------------------------------------

# Each order's timeline starts at its own release time.
# The present model uses zero as the local release-time reference.

integrated_order_dataset_df["order_release_time_h"] = 0.0

integrated_order_dataset_df["actual_delivery_lead_time_h"] = (
    integrated_order_dataset_df["production_time_h"]
    + integrated_order_dataset_df["warehouse_total_elapsed_time_h"]
    + integrated_order_dataset_df[
        "transportation_total_elapsed_time_h"
    ]
)

# Retain the actual delivery completion time relative to release.

integrated_order_dataset_df["actual_delivery_time_h"] = (
    integrated_order_dataset_df["order_release_time_h"]
    + integrated_order_dataset_df["actual_delivery_lead_time_h"]
)

# ------------------------------------------------------------
# 6. Retrieve promised delivery periods
# ------------------------------------------------------------

# Promised delivery days must come from contractual order information.
# Missing values remain missing; no artificial deadlines are generated.

if "promised_delivery_days" not in integrated_order_dataset_df.columns:
    integrated_order_dataset_df["promised_delivery_days"] = np.nan

integrated_order_dataset_df["promised_delivery_days"] = (
    pd.to_numeric(
        integrated_order_dataset_df["promised_delivery_days"],
        errors="coerce"
    )
)

valid_deadlines = (
    integrated_order_dataset_df["promised_delivery_days"].notna()
)

if (
    integrated_order_dataset_df.loc[
        valid_deadlines,
        "promised_delivery_days"
    ] <= 0
).any():
    raise ValueError(
        "Promised delivery periods must be positive."
    )

# Convert promised delivery periods into hours.

integrated_order_dataset_df[
    "promised_delivery_lead_time_h"
] = (
    integrated_order_dataset_df["promised_delivery_days"] * 24.0
)

# ------------------------------------------------------------
# 7. Calculate delivery deviation and lateness
# ------------------------------------------------------------

integrated_order_dataset_df[
    "signed_delivery_deviation_h"
] = (
    integrated_order_dataset_df["actual_delivery_lead_time_h"]
    - integrated_order_dataset_df["promised_delivery_lead_time_h"]
)

# Primary prediction target:
# Non-negative customer-order delivery lateness.

integrated_order_dataset_df["delivery_lateness_h"] = (
    integrated_order_dataset_df[
        "signed_delivery_deviation_h"
    ].clip(lower=0)
)

# Early delivery is retained separately.

integrated_order_dataset_df["early_delivery_h"] = (
    -integrated_order_dataset_df[
        "signed_delivery_deviation_h"
    ].clip(upper=0)
)

integrated_order_dataset_df["delivery_status"] = np.select(
    [
        integrated_order_dataset_df[
            "promised_delivery_days"
        ].isna(),

        integrated_order_dataset_df[
            "signed_delivery_deviation_h"
        ] > 0
    ],
    [
        "Deadline unavailable",
        "Late"
    ],
    default="On time"
)

# ------------------------------------------------------------
# 8. Verify the integrated time balance
# ------------------------------------------------------------

calculated_delivery_time_h = (
    integrated_order_dataset_df["production_time_h"]
    + integrated_order_dataset_df["warehouse_total_elapsed_time_h"]
    + integrated_order_dataset_df[
        "transportation_total_elapsed_time_h"
    ]
)

if not np.allclose(
    integrated_order_dataset_df["actual_delivery_lead_time_h"],
    calculated_delivery_time_h,
    atol=1e-8
):
    raise ValueError(
        "Integrated delivery-time balance does not hold."
    )

# Verify the primary target wherever promised delivery is available.

valid_target = (
    integrated_order_dataset_df["promised_delivery_days"].notna()
)

if valid_target.any():

    expected_lateness = np.maximum(
        integrated_order_dataset_df.loc[
            valid_target, "actual_delivery_lead_time_h"
        ].to_numpy()
        -
        integrated_order_dataset_df.loc[
            valid_target, "promised_delivery_lead_time_h"
        ].to_numpy(),
        0
    )

    if not np.allclose(
        integrated_order_dataset_df.loc[
            valid_target, "delivery_lateness_h"
        ],
        expected_lateness,
        atol=1e-8
    ):
        raise ValueError("Delivery-lateness target is incorrectly calculated.")

# ------------------------------------------------------------
# 9. Final dataset validation
# ------------------------------------------------------------

if len(integrated_order_dataset_df) != 2000:
    raise ValueError(
        "Expected 2,000 distinct customer-order observations."
    )

if integrated_order_dataset_df["customer_order_id"].duplicated().any():
    raise ValueError(
        "Duplicate customer-order observations detected."
    )

if not np.isfinite(
    integrated_order_dataset_df[
        [
            "actual_delivery_lead_time_h",
            "actual_delivery_time_h",
            "production_time_h",
            "warehouse_total_elapsed_time_h",
            "transportation_total_elapsed_time_h"
        ]
    ].to_numpy()
).all():
    raise ValueError(
        "Invalid numerical values in the integrated dataset."
    )

# ------------------------------------------------------------
# 10. Report results
# ------------------------------------------------------------

TARGET_AVAILABLE_COUNT = int(valid_target.sum())
TARGET_MISSING_COUNT = int((~valid_target).sum())

print("CELL 14 COMPLETED SUCCESSFULLY")
print("=" * 60)

print(
    f"Distinct customer orders: "
    f"{len(integrated_order_dataset_df)}"
)

print(
    f"Mean production time: "
    f"{integrated_order_dataset_df['production_time_h'].mean():.2f} h"
)

print(
    f"Mean warehouse elapsed time: "
    f"{integrated_order_dataset_df['warehouse_total_elapsed_time_h'].mean():.2f} h"
)

print(
    f"Mean transportation elapsed time: "
    f"{integrated_order_dataset_df['transportation_total_elapsed_time_h'].mean():.2f} h"
)

print(
    f"Mean actual delivery lead time: "
    f"{integrated_order_dataset_df['actual_delivery_lead_time_h'].mean():.2f} h"
)

print("-" * 60)

print(
    f"Orders with contractual deadlines: "
    f"{TARGET_AVAILABLE_COUNT}"
)

print(
    f"Orders without contractual deadlines: "
    f"{TARGET_MISSING_COUNT}"
)

if TARGET_AVAILABLE_COUNT > 0:

    print(
        f"Mean delivery lateness: "
        f"{integrated_order_dataset_df['delivery_lateness_h'].mean():.2f} h"
    )

else:

    print(
        "Delivery-lateness target is pending contractual deadline data."
    )

print("=" * 60)

display(
    integrated_order_dataset_df[
        [
            "scenario_id",
            "customer_order_id",
            "product_mix",
            "total_order_quantity",
            "production_time_h",
            "warehouse_total_elapsed_time_h",
            "transportation_total_elapsed_time_h",
            "actual_delivery_lead_time_h",
            "promised_delivery_days",
            "delivery_lateness_h",
            "delivery_status"
        ]
    ].head(10)
)


CELL 14 COMPLETED SUCCESSFULLY
Distinct customer orders: 2000
Mean production time: 250.46 h
Mean warehouse elapsed time: 23.97 h
Mean transportation elapsed time: 8.99 h
Mean actual delivery lead time: 283.43 h
------------------------------------------------------------
Orders with contractual deadlines: 0
Orders without contractual deadlines: 2000
Delivery-lateness target is pending contractual deadline data.


,scenario_id,customer_order_id,product_mix,total_order_quantity,production_time_h,warehouse_total_elapsed_time_h,transportation_total_elapsed_time_h,actual_delivery_lead_time_h,promised_delivery_days,delivery_lateness_h,delivery_status
0,1,CO-001,"Ø900, Ø1000, Ø1200",190,236.463667,25.727105,8.577805,270.768577,NaN,NaN,Deadline unavailable
1,2,SIM-000002,"Ø1000, Ø1200",122,212.452654,16.935956,12.081354,241.469964,NaN,NaN,Deadline unavailable
2,3,SIM-000003,"Ø1000, Ø1200",125,217.652379,20.231409,8.011557,245.895346,NaN,NaN,Deadline unavailable
3,4,SIM-000004,"Ø900, Ø1000, Ø1200",239,340.877716,35.863973,7.130817,383.872507,NaN,NaN,Deadline unavailable
4,5,SIM-000005,"Ø1000, Ø1200",106,201.259754,15.320449,7.688231,224.268434,NaN,NaN,Deadline unavailable
5,6,SIM-000006,"Ø900, Ø1200",125,307.199904,13.452220,6.982497,327.634620,NaN,NaN,Deadline unavailable
6,7,SIM-000007,"Ø1000, Ø1200",118,204.638814,17.512087,9.163953,231.314855,NaN,NaN,Deadline unavailable
7,8,SIM-000008,"Ø900, Ø1000, Ø1200",190,260.990277,28.200237,6.916037,296.106551,NaN,NaN,Deadline unavailable
8,9,SIM-000009,"Ø900, Ø1000",119,92.282545,21.800251,8.328139,122.410935,NaN,NaN,Deadline unavailable
9,10,SIM-000010,"Ø900, Ø1000, Ø1200",209,301.671421,40.053241,11.017237,352.741899,NaN,NaN,Deadline unavailable


In [31]:

# ============================================================
# CELL 15 — SYNTHETIC DELIVERY COMMITMENT AND TARGET CONSTRUCTION
# ============================================================
# Purpose:
# 1. Establish a transparent standard delivery-planning rule.
# 2. Calculate order-specific promised delivery periods.
# 3. Calculate signed delivery deviation.
# 4. Construct the primary delivery-lateness target.
#
# Important:
# Promised delivery periods are synthetic planning commitments,
# NOT actual Farasan contractual delivery promises.
#
# The planning rule uses nominal operating times and standard
# capacities, not realized stochastic delays.
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Validate integrated dataset
# ------------------------------------------------------------

df = integrated_order_dataset_df.copy()

required_columns = [
    "customer_order_id",
    "total_order_quantity",
    "delivery_distance_km",
    "production_time_h",
    "warehouse_total_elapsed_time_h",
    "transportation_total_elapsed_time_h",
    "actual_delivery_lead_time_h"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing integrated dataset columns: {missing_columns}"
    )

if df["customer_order_id"].duplicated().any():
    raise ValueError("Duplicate customer orders detected.")

if df["customer_order_id"].nunique() != 2000:
    raise ValueError("Expected 2,000 distinct customer orders.")


# ------------------------------------------------------------
# 2. Retrieve nominal production time for each order
# ------------------------------------------------------------

planning_production_df = (
    multi_order_scenarios_df[
        [
            "customer_order_id",
            "nominal_production_flow_time_h"
        ]
    ].copy()
)

if planning_production_df["customer_order_id"].duplicated().any():
    raise ValueError(
        "Duplicate customer orders in nominal production data."
    )

df = df.drop(
    columns=["nominal_production_flow_time_h"],
    errors="ignore"
).merge(
    planning_production_df,
    on="customer_order_id",
    how="left",
    validate="one_to_one"
)

if df["nominal_production_flow_time_h"].isna().any():
    raise ValueError(
        "Nominal production times are missing for some orders."
    )


# ------------------------------------------------------------
# 3. Define standard planning assumptions
# ------------------------------------------------------------

# Standard warehouse inbound capacity:
# Pipe-equivalent units handled per hour.

STANDARD_INBOUND_RATE = 20.0

# Standard warehouse loading capacity:
# Pipe-equivalent units loaded per hour.

STANDARD_LOADING_RATE = 24.0

# Planned warehouse staging allowance.

STANDARD_STAGING_TIME_H = 1.5

# Standard transportation speed.

STANDARD_TRANSPORT_SPEED_KMH = 60.0

# Planning contingency allowance.

PROMISE_BUFFER_RATE = 0.10

# Delivery commitments are expressed in whole elapsed days.

HOURS_PER_DAY = 24.0

PROMISE_RULE_SOURCE = (
    "Synthetic planning rule based on nominal production time, "
    "standard warehouse capacities, standard transportation speed, "
    "and a 10% planning contingency. "
    "Not actual company contractual policy."
)

# Validate planning assumptions.

if (
    STANDARD_INBOUND_RATE <= 0
    or STANDARD_LOADING_RATE <= 0
    or STANDARD_TRANSPORT_SPEED_KMH <= 0
    or PROMISE_BUFFER_RATE < 0
):
    raise ValueError(
        "Invalid synthetic delivery-planning parameters."
    )

if (
    df["total_order_quantity"] <= 0
).any():
    raise ValueError("Order quantities must be positive.")

if (
    df["delivery_distance_km"] <= 0
).any():
    raise ValueError("Delivery distances must be positive.")


# ------------------------------------------------------------
# 4. Calculate planned warehouse time
# ------------------------------------------------------------

df["planned_inbound_handling_time_h"] = (
    df["total_order_quantity"]
    / STANDARD_INBOUND_RATE
)

df["planned_loading_time_h"] = (
    df["total_order_quantity"]
    / STANDARD_LOADING_RATE
)

df["planned_staging_time_h"] = (
    STANDARD_STAGING_TIME_H
)

df["planned_warehouse_time_h"] = (
    df["planned_inbound_handling_time_h"]
    + df["planned_loading_time_h"]
    + df["planned_staging_time_h"]
)


# ------------------------------------------------------------
# 5. Calculate planned transportation time
# ------------------------------------------------------------

df["planned_transportation_time_h"] = (
    df["delivery_distance_km"]
    / STANDARD_TRANSPORT_SPEED_KMH
)


# ------------------------------------------------------------
# 6. Calculate nominal integrated planning lead time
# ------------------------------------------------------------

df["planned_production_time_h"] = (
    df["nominal_production_flow_time_h"]
)

df["planned_delivery_lead_time_h"] = (
    df["planned_production_time_h"]
    + df["planned_warehouse_time_h"]
    + df["planned_transportation_time_h"]
)


# ------------------------------------------------------------
# 7. Apply planning contingency and calculate promised days
# ------------------------------------------------------------

df["planning_buffer_h"] = (
    df["planned_delivery_lead_time_h"]
    * PROMISE_BUFFER_RATE
)

df["buffered_planned_lead_time_h"] = (
    df["planned_delivery_lead_time_h"]
    + df["planning_buffer_h"]
)

# Round up to complete 24-hour days.

df["promised_delivery_days"] = np.ceil(
    df["buffered_planned_lead_time_h"]
    / HOURS_PER_DAY
).astype(int)

df["promised_delivery_lead_time_h"] = (
    df["promised_delivery_days"]
    * HOURS_PER_DAY
)

# All order timelines use release time zero.

df["promised_delivery_deadline_h"] = (
    df["order_release_time_h"]
    + df["promised_delivery_lead_time_h"]
)


# ------------------------------------------------------------
# 8. Calculate delivery deviation and lateness
# ------------------------------------------------------------

df["signed_delivery_deviation_h"] = (
    df["actual_delivery_lead_time_h"]
    - df["promised_delivery_lead_time_h"]
)

# Primary target: non-negative lateness.

df["delivery_lateness_h"] = (
    df["signed_delivery_deviation_h"].clip(lower=0)
)

# Early delivery is retained separately.

df["early_delivery_h"] = (
    -df["signed_delivery_deviation_h"].clip(upper=0)
)

df["on_time_delivery"] = (
    df["signed_delivery_deviation_h"] <= 0
)

df["delivery_status"] = np.where(
    df["signed_delivery_deviation_h"] > 0,
    "Late",
    "On time"
)

df["promised_delivery_rule_source"] = (
    PROMISE_RULE_SOURCE
)


# ------------------------------------------------------------
# 9. Validate target and time calculations
# ------------------------------------------------------------

# A. Integrated actual delivery time must equal
# the sum of the three operational-stage durations.

calculated_actual_lead_time = (
    df["production_time_h"]
    + df["warehouse_total_elapsed_time_h"]
    + df["transportation_total_elapsed_time_h"]
)

if not np.allclose(
    df["actual_delivery_lead_time_h"],
    calculated_actual_lead_time,
    atol=1e-6
):
    raise ValueError(
        "Actual delivery lead-time balance failed."
    )

# B. Promised time must never be shorter than its buffered plan.

if not (
    df["promised_delivery_lead_time_h"]
    >= df["buffered_planned_lead_time_h"] - 1e-8
).all():
    raise ValueError(
        "Promised delivery periods do not satisfy the planning rule."
    )

# C. Lateness must satisfy the mathematical target definition.

expected_lateness = np.maximum(
    df["actual_delivery_lead_time_h"]
    - df["promised_delivery_lead_time_h"],
    0
)

if not np.allclose(
    df["delivery_lateness_h"],
    expected_lateness,
    atol=1e-8
):
    raise ValueError(
        "Delivery-lateness calculation is inconsistent."
    )

# D. All 2,000 observations must have a valid target.

target_columns = [
    "promised_delivery_days",
    "promised_delivery_lead_time_h",
    "actual_delivery_lead_time_h",
    "delivery_lateness_h"
]

if df[target_columns].isna().any().any():
    raise ValueError(
        "Missing values detected in the target construction."
    )

if not np.isfinite(
    df[target_columns].to_numpy()
).all():
    raise ValueError(
        "Non-finite target values were generated."
    )

# E. Reference-order verification.

reference_order = df[
    df["customer_order_id"] == "CO-001"
]

if len(reference_order) != 1:
    raise ValueError(
        "CO-001 must appear exactly once."
    )


# ------------------------------------------------------------
# 10. Replace the integrated dataset
# ------------------------------------------------------------

integrated_order_dataset_df = df.copy()


# ------------------------------------------------------------
# 11. Generate concise target statistics
# ------------------------------------------------------------

target_summary_df = pd.DataFrame({

    "indicator": [
        "Actual delivery lead time",
        "Promised delivery lead time",
        "Signed delivery deviation",
        "Delivery lateness",
        "Early delivery"
    ],

    "mean_h": [
        df["actual_delivery_lead_time_h"].mean(),
        df["promised_delivery_lead_time_h"].mean(),
        df["signed_delivery_deviation_h"].mean(),
        df["delivery_lateness_h"].mean(),
        df["early_delivery_h"].mean()
    ],

    "median_h": [
        df["actual_delivery_lead_time_h"].median(),
        df["promised_delivery_lead_time_h"].median(),
        df["signed_delivery_deviation_h"].median(),
        df["delivery_lateness_h"].median(),
        df["early_delivery_h"].median()
    ]

})

# Delivery performance indicators.

late_order_percentage = (
    (~df["on_time_delivery"]).mean() * 100
)

on_time_order_percentage = (
    df["on_time_delivery"].mean() * 100
)

positive_lateness = df.loc[
    df["delivery_lateness_h"] > 0,
    "delivery_lateness_h"
]

mean_lateness_when_late_h = (
    positive_lateness.mean()
    if len(positive_lateness) > 0
    else 0.0
)


# ------------------------------------------------------------
# 12. Display results
# ------------------------------------------------------------

print("CELL 16 COMPLETED SUCCESSFULLY")
print("=" * 65)

print("Synthetic delivery-planning rule")
print(f"Standard inbound rate: {STANDARD_INBOUND_RATE:.1f} units/h")
print(f"Standard loading rate: {STANDARD_LOADING_RATE:.1f} units/h")
print(f"Standard staging time: {STANDARD_STAGING_TIME_H:.1f} h")
print(f"Standard transport speed: {STANDARD_TRANSPORT_SPEED_KMH:.1f} km/h")
print(f"Planning contingency: {PROMISE_BUFFER_RATE * 100:.1f}%")

print("-" * 65)

print(
    f"Distinct customer orders: {len(df)}"
)

print(
    f"Mean actual delivery lead time: "
    f"{df['actual_delivery_lead_time_h'].mean():.2f} h"
)

print(
    f"Mean promised delivery period: "
    f"{df['promised_delivery_days'].mean():.2f} days"
)

print(
    f"Orders delivered late: {late_order_percentage:.2f}%"
)

print(
    f"Orders delivered on time: {on_time_order_percentage:.2f}%"
)

print(
    f"Mean lateness among late orders: "
    f"{mean_lateness_when_late_h:.2f} h"
)

print("-" * 65)

print(
    "CO-001 promised delivery period:",
    int(reference_order["promised_delivery_days"].iloc[0]),
    "days"
)

print("=" * 65)

display(target_summary_df)

display(
    df[
        [
            "customer_order_id",
            "total_order_quantity",
            "nominal_production_flow_time_h",
            "planned_warehouse_time_h",
            "planned_transportation_time_h",
            "planned_delivery_lead_time_h",
            "planning_buffer_h",
            "promised_delivery_days",
            "actual_delivery_lead_time_h",
            "signed_delivery_deviation_h",
            "delivery_lateness_h",
            "delivery_status"
        ]
    ].head(10)
)


CELL 16 COMPLETED SUCCESSFULLY
Synthetic delivery-planning rule
Standard inbound rate: 20.0 units/h
Standard loading rate: 24.0 units/h
Standard staging time: 1.5 h
Standard transport speed: 60.0 km/h
Planning contingency: 10.0%
-----------------------------------------------------------------
Distinct customer orders: 2000
Mean actual delivery lead time: 283.43 h
Mean promised delivery period: 11.94 days
Orders delivered late: 37.55%
Orders delivered on time: 62.45%
Mean lateness among late orders: 9.25 h
-----------------------------------------------------------------
CO-001 promised delivery period: 11 days


,indicator,mean_h,median_h
0,Actual delivery lead time,283.428474,288.857353
1,Promised delivery lead time,286.536000,288.000000
2,Signed delivery deviation,-3.107526,-3.200693
3,Delivery lateness,3.473691,0.000000
4,Early delivery,6.581217,3.200693


,customer_order_id,total_order_quantity,nominal_production_flow_time_h,planned_warehouse_time_h,planned_transportation_time_h,planned_delivery_lead_time_h,planning_buffer_h,promised_delivery_days,actual_delivery_lead_time_h,signed_delivery_deviation_h,delivery_lateness_h,delivery_status
0,CO-001,190,215.5100,18.916667,5.0,239.426667,23.942667,11,270.768577,6.768577,6.768577,Late
1,SIM-000002,122,201.8991,12.683333,5.0,219.582433,21.958243,11,241.469964,-22.530036,0.000000,On time
2,SIM-000003,125,197.1969,12.958333,5.0,215.155233,21.515523,10,245.895346,5.895346,5.895346,Late
3,SIM-000004,239,290.9607,23.408333,5.0,319.369033,31.936903,15,383.872507,23.872507,23.872507,Late
4,SIM-000005,106,179.0294,11.216667,5.0,195.246067,19.524607,9,224.268434,8.268434,8.268434,Late
5,SIM-000006,125,273.6880,12.958333,5.0,291.646333,29.164633,14,327.634620,-8.365380,0.000000,On time
6,SIM-000007,118,193.5758,12.316667,5.0,210.892467,21.089247,10,231.314855,-8.685145,0.000000,On time
7,SIM-000008,190,246.9020,18.916667,5.0,270.818667,27.081867,13,296.106551,-15.893449,0.000000,On time
8,SIM-000009,119,81.5926,12.408333,5.0,99.000933,9.900093,5,122.410935,2.410935,2.410935,Late
9,SIM-000010,209,267.1245,20.658333,5.0,292.782833,29.278283,14,352.741899,16.741899,16.741899,Late


## 3. Feature engineering and training-only feature selection

In [32]:

# ============================================================
# CELL 16 — FINAL FEATURE ENGINEERING AND LEAKAGE CONTROL
# Revised for forecast-based Cell 13R
# ============================================================

import numpy as np
import pandas as pd

df = integrated_order_dataset_df.copy()

TARGET_VARIABLE = "delivery_lateness_h"

# ------------------------------------------------------------
# 1. Validate integrated dataset
# ------------------------------------------------------------

required_columns = [
    "scenario_id",
    "customer_order_id",
    "total_order_quantity",
    "number_of_jobs",
    "quantity_900mm",
    "quantity_1000mm",
    "production_time_h",
    "production_setup_time_h",
    "production_interstage_nonprocessing_time_h",
    "machine_downtime_M1_h",
    "machine_downtime_M2_h",
    "machine_downtime_M3_h",
    "machine_downtime_M4_h",
    "warehouse_backlog_ratio",
    "warehouse_queue_wait_time_h",
    "warehouse_inbound_handling_time_h",
    "warehouse_storage_dwell_time_h",
    "warehouse_loading_time_h",
    "delivery_distance_km",
    "forecast_base_speed_kmh",
    "forecast_vehicle_availability",
    "forecast_traffic_level",
    "forecast_weather_condition",
    "route_complexity_level",
    "forecast_expected_travel_time_h",
    "promised_delivery_days",
    TARGET_VARIABLE
]

missing = [
    col for col in required_columns
    if col not in df.columns
]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}. "
        "Check Cells 13R, 14 and 16."
    )

if df["customer_order_id"].duplicated().any():
    raise ValueError("Duplicate customer orders detected.")

if df[TARGET_VARIABLE].isna().any():
    raise ValueError("Delivery-lateness target contains missing values.")

if (df[TARGET_VARIABLE] < 0).any():
    raise ValueError("Delivery lateness cannot be negative.")


# ------------------------------------------------------------
# 2. Construct product composition features
# ------------------------------------------------------------

total_quantity = df["total_order_quantity"].astype(float)

df["quantity_share_900mm"] = (
    df["quantity_900mm"] / total_quantity
)

df["quantity_share_1000mm"] = (
    df["quantity_1000mm"] / total_quantity
)


# ------------------------------------------------------------
# 3. Construct cross-stage interaction features
# ------------------------------------------------------------

df["production_warehouse_interaction"] = (
    df["production_time_h"]
    * df["warehouse_backlog_ratio"]
)

df["production_transport_interaction"] = (
    df["production_time_h"]
    * df["forecast_expected_travel_time_h"]
)

df["warehouse_transport_interaction"] = (
    df["warehouse_total_elapsed_time_h"]
    * df["forecast_expected_travel_time_h"]
)


# ------------------------------------------------------------
# 4. Define candidate predictor groups
# ------------------------------------------------------------

feature_groups = {

    "order_characteristics": [
        "total_order_quantity",
        "number_of_jobs",
        "quantity_share_900mm",
        "quantity_share_1000mm"
    ],

    "contractual_commitment": [
        "promised_delivery_days"
    ],

    "production": [
        "production_time_h",
        "production_setup_time_h",
        "production_interstage_nonprocessing_time_h",
        "machine_downtime_M1_h",
        "machine_downtime_M2_h",
        "machine_downtime_M3_h",
        "machine_downtime_M4_h"
    ],

    "warehouse": [
        "warehouse_backlog_ratio",
        "warehouse_queue_wait_time_h",
        "warehouse_inbound_handling_time_h",
        "warehouse_storage_dwell_time_h",
        "warehouse_loading_time_h"
    ],

    "transportation": [
        "delivery_distance_km",
        "forecast_base_speed_kmh",
        "forecast_vehicle_availability",
        "forecast_traffic_level",
        "forecast_weather_condition",
        "route_complexity_level",
        "forecast_expected_travel_time_h"
    ],

    "cross_stage_interactions": [
        "production_warehouse_interaction",
        "production_transport_interaction",
        "warehouse_transport_interaction"
    ]
}

candidate_feature_columns = list(dict.fromkeys(
    feature
    for features in feature_groups.values()
    for feature in features
))


# ------------------------------------------------------------
# 5. Audit feature availability and variation
# ------------------------------------------------------------

feature_audit_records = []
selected_feature_columns = []

for feature in candidate_feature_columns:

    nonmissing = int(df[feature].notna().sum())

    missing_count = int(df[feature].isna().sum())

    unique_count = int(df[feature].nunique(dropna=True))

    if nonmissing == 0:
        status = "Excluded: no available values"

    elif missing_count > 0:
        status = "Requires missing-data treatment"

    elif unique_count <= 1:
        status = "Excluded: zero variance"

    else:
        status = "Eligible"
        selected_feature_columns.append(feature)

    feature_audit_records.append({

        "feature": feature,

        "missing_count": missing_count,

        "unique_values": unique_count,

        "status": status
    })

feature_audit_df = pd.DataFrame(
    feature_audit_records
)


# ------------------------------------------------------------
# 6. Define numerical and categorical predictors
# ------------------------------------------------------------

categorical_feature_columns = [
    feature
    for feature in [
        "forecast_traffic_level",
        "forecast_weather_condition",
        "route_complexity_level"
    ]
    if feature in selected_feature_columns
]

numeric_feature_columns = [
    feature
    for feature in selected_feature_columns
    if feature not in categorical_feature_columns
]


# ------------------------------------------------------------
# 7. Document variables excluded to prevent leakage
# ------------------------------------------------------------

# These variables describe realized events after vehicle departure.
# They are outcomes, not predictors available at the forecast time.

leakage_exclusion_columns = [
    "actual_dispatch_wait_time_h",
    "actual_driving_time_h",
    "travel_time_forecast_error_h",
    "transportation_total_elapsed_time_h",
    "transportation_completion_time_h",
    "actual_delivery_time_h",
    "actual_delivery_lead_time_h",
    "signed_delivery_deviation_h",
    "delivery_lateness_h",
    "early_delivery_h",
    "delivery_status"
]


# ------------------------------------------------------------
# 8. Construct feature matrix and target
# ------------------------------------------------------------

feature_matrix_df = df[
    [
        "scenario_id",
        "customer_order_id"
    ] + selected_feature_columns
].copy()

target_df = df[
    [
        "scenario_id",
        "customer_order_id",
        TARGET_VARIABLE
    ]
].copy()

modeling_dataset_df = feature_matrix_df.merge(
    target_df,
    on=[
        "scenario_id",
        "customer_order_id"
    ],
    how="inner",
    validate="one_to_one"
)

if modeling_dataset_df[
    selected_feature_columns + [TARGET_VARIABLE]
].isna().any().any():

    raise ValueError(
        "Missing predictor or target values detected."
    )


# ------------------------------------------------------------
# 9. Preserve feature groups for ablation analysis
# ------------------------------------------------------------

selected_feature_groups = {

    group: [
        feature
        for feature in features
        if feature in selected_feature_columns
    ]

    for group, features in feature_groups.items()
}

features_without_promise = [
    feature
    for feature in selected_feature_columns
    if feature != "promised_delivery_days"
]


# ------------------------------------------------------------
# 10. Create model-ready predictor and target objects
# ------------------------------------------------------------

feature_columns = selected_feature_columns.copy()

X = modeling_dataset_df[feature_columns].copy()

y = modeling_dataset_df[TARGET_VARIABLE].copy()

order_ids = modeling_dataset_df[
    "customer_order_id"
].copy()


# ------------------------------------------------------------
# 11. Final validation
# ------------------------------------------------------------

if len(modeling_dataset_df) != 2000:
    raise ValueError(
        "Expected 2,000 distinct customer orders."
    )

if modeling_dataset_df["customer_order_id"].duplicated().any():
    raise ValueError(
        "Duplicate observations detected."
    )

if y.isna().any():
    raise ValueError("Missing delivery-lateness targets.")

if not np.isfinite(y.to_numpy()).all():
    raise ValueError("Non-finite target values detected.")

numeric_values = (
    X.select_dtypes(include=[np.number]).to_numpy()
)

if not np.isfinite(numeric_values).all():
    raise ValueError(
        "Non-finite numerical predictor values detected."
    )

late_order_percentage = (
    (y > 0).mean() * 100
)

target_summary = y.describe(
    percentiles=[0.25, 0.50, 0.75, 0.90]
)


# ------------------------------------------------------------
# 12. Display essential results
# ------------------------------------------------------------

print("REVISED CELL 15 COMPLETED SUCCESSFULLY")
print("=" * 60)

print(f"Distinct orders: {len(modeling_dataset_df)}")

print(f"Candidate predictors: {len(candidate_feature_columns)}")

print(f"Selected predictors: {len(selected_feature_columns)}")

print(f"Numerical predictors: {len(numeric_feature_columns)}")

print(f"Categorical predictors: {len(categorical_feature_columns)}")

print("-" * 60)

print(f"Late-order percentage: {late_order_percentage:.2f}%")

print(f"Mean delivery lateness: {y.mean():.2f} h")

print(f"Median delivery lateness: {y.median():.2f} h")

print("-" * 60)

for group, features in selected_feature_groups.items():

    print(f"{group}: {len(features)} features")

print("=" * 60)

display(feature_audit_df)

display(target_summary)

display(X.head())


REVISED CELL 15 COMPLETED SUCCESSFULLY
Distinct orders: 2000
Candidate predictors: 27
Selected predictors: 25
Numerical predictors: 23
Categorical predictors: 2
------------------------------------------------------------
Late-order percentage: 37.55%
Mean delivery lateness: 3.47 h
Median delivery lateness: 0.00 h
------------------------------------------------------------
order_characteristics: 4 features
contractual_commitment: 1 features
production: 7 features
warehouse: 5 features
transportation: 5 features
cross_stage_interactions: 3 features


,feature,missing_count,unique_values,status
0,total_order_quantity,0,255,Eligible
1,number_of_jobs,0,3,Eligible
2,quantity_share_900mm,0,1421,Eligible
3,quantity_share_1000mm,0,1418,Eligible
4,promised_delivery_days,0,21,Eligible
5,production_time_h,0,2000,Eligible
6,production_setup_time_h,0,7,Eligible
7,production_interstage_nonprocessing_time_h,0,7,Eligible
8,machine_downtime_M1_h,0,1981,Eligible
9,machine_downtime_M2_h,0,1999,Eligible


count    2000.000000
mean        3.473691
std         6.624042
min         0.000000
25%         0.000000
50%         0.000000
75%         4.383595
90%        12.390282
max        48.510184
Name: delivery_lateness_h, dtype: float64

,total_order_quantity,number_of_jobs,quantity_share_900mm,quantity_share_1000mm,promised_delivery_days,production_time_h,production_setup_time_h,production_interstage_nonprocessing_time_h,machine_downtime_M1_h,machine_downtime_M2_h,...,warehouse_storage_dwell_time_h,warehouse_loading_time_h,forecast_base_speed_kmh,forecast_vehicle_availability,forecast_traffic_level,forecast_weather_condition,forecast_expected_travel_time_h,production_warehouse_interaction,production_transport_interaction,warehouse_transport_interaction
0,190,3,0.263158,0.526316,11,236.463667,16.1,17.60,5.077463,6.972925,...,1.908276,8.385934,61.889267,0.794306,2,1,6.020430,144.175038,1423.612960,154.888233
1,122,2,0.000000,0.549180,11,212.452654,12.3,11.83,2.025338,6.363616,...,2.600803,6.423510,68.767707,0.720101,4,1,7.538422,127.857409,1601.557786,127.670386
2,125,2,0.000000,0.584000,10,217.652379,12.3,11.83,12.120811,3.227698,...,1.602441,5.280725,66.407036,0.789039,2,2,6.171936,162.856084,1343.336637,124.866972
3,239,3,0.317992,0.414226,15,340.877716,16.1,17.60,32.437500,3.286241,...,1.580462,12.154883,65.855325,0.843111,2,1,5.657857,78.128843,1928.637310,202.913224
4,106,2,0.000000,0.547170,9,201.259754,12.3,11.83,18.027169,1.285790,...,2.608866,5.549918,56.826247,0.907947,3,1,7.697147,101.848434,1549.125981,117.923753


In [33]:

# ============================================================
# CELL 17 — TRAIN/TEST SPLIT AND PREPROCESSING
# ============================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder


# ------------------------------------------------------------
# 1. Rebuild X and y from the revised Cell 15 outputs
# ------------------------------------------------------------

if not isinstance(feature_matrix_df, pd.DataFrame):
    raise TypeError("feature_matrix_df must be a pandas DataFrame.")

if not isinstance(target_df, pd.DataFrame):
    raise TypeError("target_df must be a pandas DataFrame.")

if "delivery_lateness_h" not in target_df.columns:
    raise KeyError("delivery_lateness_h was not found in target_df.")

missing_features = [
    col for col in selected_feature_columns
    if col not in feature_matrix_df.columns
]

if missing_features:
    raise ValueError(
        f"Selected features are missing from feature_matrix_df: {missing_features}"
    )

X = feature_matrix_df.loc[:, selected_feature_columns].copy()
y = target_df["delivery_lateness_h"].copy().reset_index(drop=True)

X = X.reset_index(drop=True)
order_ids = np.asarray(order_ids).reshape(-1)


# ------------------------------------------------------------
# 2. Validate dataset consistency
# ------------------------------------------------------------

if not (len(X) == len(y) == len(order_ids)):
    raise ValueError(
        "X, y, and order_ids must contain the same number of observations."
    )

if X.empty:
    raise ValueError("The feature matrix is empty.")

if X.isna().all(axis=0).any():
    empty_columns = X.columns[X.isna().all(axis=0)].tolist()
    raise ValueError(f"Entirely missing feature columns detected: {empty_columns}")

if X.isna().any().any():
    raise ValueError("Missing values detected in X. Check revised Cell 15.")

if y.isna().any():
    raise ValueError("Missing target values detected in y.")

if not np.isfinite(y.to_numpy(dtype=float)).all():
    raise ValueError("The target contains infinite or non-finite values.")

if (y < 0).any():
    raise ValueError("Delivery lateness should not contain negative values.")

if pd.Series(order_ids).duplicated().any():
    raise ValueError("Duplicate order IDs detected.")

print("Dataset validation completed.")
print(f"Number of orders: {len(X):,}")
print(f"Number of selected features: {X.shape[1]}")


# ------------------------------------------------------------
# 3. Identify numerical and categorical features
# ------------------------------------------------------------

# These variables represent categorical operating conditions,
# even if their values are stored as integers.

known_categorical = {
    "forecast_traffic_level",
    "forecast_weather_condition",
    "route_complexity_level"
}

categorical_features = [
    col for col in X.columns
    if (
        col in known_categorical
        or pd.api.types.is_object_dtype(X[col])
        or isinstance(X[col].dtype, pd.CategoricalDtype)
    )
]

numeric_features = [
    col for col in X.columns
    if col not in categorical_features
]

print("\nNumerical features:", len(numeric_features))
print("Categorical features:", len(categorical_features))


# ------------------------------------------------------------
# 4. Define preprocessing pipelines
# ------------------------------------------------------------

transformers = []

if numeric_features:
    numeric_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    transformers.append(
        ("num", numeric_pipeline, numeric_features)
    )

if categorical_features:
    categorical_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    transformers.append(
        ("cat", categorical_pipeline, categorical_features)
    )

preprocessor = ColumnTransformer(
    transformers=transformers,
    remainder="drop"
)


# ------------------------------------------------------------
# 5. Create stratified 80/20 train/test split
# ------------------------------------------------------------

# Stratification preserves the proportion of:
#   0 = on-time orders
#   1 = late orders

stratify_labels = (y > 0).astype(int)

class_counts = stratify_labels.value_counts()

if len(class_counts) < 2 or class_counts.min() < 2:
    raise ValueError(
        "There are insufficient observations in the on-time/late "
        "categories for a stratified train/test split."
    )

indices = np.arange(len(X))

train_indices, test_indices = train_test_split(
    indices,
    test_size=0.20,
    random_state=2026,
    stratify=stratify_labels
)


# ------------------------------------------------------------
# 6. Construct training and test datasets
# ------------------------------------------------------------

X_train = X.iloc[train_indices].copy().reset_index(drop=True)
X_test = X.iloc[test_indices].copy().reset_index(drop=True)

y_train = y.iloc[train_indices].copy().reset_index(drop=True)
y_test = y.iloc[test_indices].copy().reset_index(drop=True)

train_order_ids = order_ids[train_indices]
test_order_ids = order_ids[test_indices]


# ------------------------------------------------------------
# 7. Verify split integrity
# ------------------------------------------------------------

if set(train_order_ids).intersection(set(test_order_ids)):
    raise ValueError("Order leakage detected between training and test sets.")

if len(X_train) + len(X_test) != len(X):
    raise ValueError("Train/test observations do not sum to the full dataset.")

split_summary = pd.DataFrame([
    {
        "Dataset": "Training",
        "Orders": len(y_train),
        "Late orders": int((y_train > 0).sum()),
        "Late orders (%)": 100 * (y_train > 0).mean(),
        "Mean lateness (h)": y_train.mean(),
        "Median lateness (h)": y_train.median()
    },
    {
        "Dataset": "Test",
        "Orders": len(y_test),
        "Late orders": int((y_test > 0).sum()),
        "Late orders (%)": 100 * (y_test > 0).mean(),
        "Mean lateness (h)": y_test.mean(),
        "Median lateness (h)": y_test.median()
    }
])


# ------------------------------------------------------------
# 8. Report results
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("TRAIN/TEST SPLIT COMPLETED")
print("=" * 60)

print(f"\nTraining set: {X_train.shape[0]} orders")
print(f"Test set:     {X_test.shape[0]} orders")

print("\nSplit summary:")
print(split_summary.round(3).to_string(index=False))

print("\nPreprocessing configuration:")
print(f"Numerical features:   {len(numeric_features)}")
print(f"Categorical features: {len(categorical_features)}")

print("\nPreprocessor defined but NOT fitted.")
print("Ready for Cell 19: model development and cross-validation.")


Dataset validation completed.
Number of orders: 2,000
Number of selected features: 25

Numerical features: 23
Categorical features: 2

TRAIN/TEST SPLIT COMPLETED

Training set: 1600 orders
Test set:     400 orders

Split summary:
 Dataset  Orders  Late orders  Late orders (%)  Mean lateness (h)  Median lateness (h)
Training    1600          601           37.562              3.427                  0.0
    Test     400          150           37.500              3.659                  0.0

Preprocessing configuration:
Numerical features:   23
Categorical features: 2

Preprocessor defined but NOT fitted.
Ready for Cell 19: model development and cross-validation.


In [35]:

# ============================================================
# CELL 18 — TRAINING-ONLY FEATURE ABLATION
# Primary model: Linear Regression
# Five-fold stratified cross-validation
# ============================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)


# ------------------------------------------------------------
# 1. Validate training data and feature definitions
# ------------------------------------------------------------

if len(X_train) != len(y_train):
    raise ValueError("Training feature and target lengths differ.")

if X_train.empty:
    raise ValueError("Training dataset is empty.")

if X_train.isna().any().any() or y_train.isna().any():
    raise ValueError("Missing values detected in the training data.")

base_features = list(X_train.columns)

print("=" * 70)
print("CELL 18 — TRAINING-ONLY FEATURE ABLATION")
print("=" * 70)

print("Training observations:", len(X_train))
print("Original predictors:", len(base_features))


# ------------------------------------------------------------
# 2. Define required variables for the alternative specification
# ------------------------------------------------------------

required_slack_features = [
    "promised_delivery_days",
    "production_time_h",
    "warehouse_queue_wait_time_h",
    "warehouse_inbound_handling_time_h",
    "warehouse_storage_dwell_time_h",
    "warehouse_loading_time_h"
]

missing_slack_features = [
    f for f in required_slack_features
    if f not in X_train.columns
]

if missing_slack_features:
    raise ValueError(
        f"Cannot calculate remaining slack. Missing features: "
        f"{missing_slack_features}"
    )


# ------------------------------------------------------------
# 3. Create five feature specifications
# ------------------------------------------------------------

ablation_datasets = {}

# A — Full model

ablation_datasets["A_Full_model"] = (
    X_train[base_features].copy()
)


# B — Remove promised delivery days

features_without_deadline = [
    f for f in base_features
    if f != "promised_delivery_days"
]

ablation_datasets["B_Without_deadline"] = (
    X_train[features_without_deadline].copy()
)


# C — Remove production time and its derived interactions

production_related_features = {
    "production_time_h",
    "production_transport_interaction",
    "production_warehouse_interaction"
}

features_without_production = [
    f for f in base_features
    if f not in production_related_features
]

ablation_datasets["C_Without_production_time"] = (
    X_train[features_without_production].copy()
)


# D — Remove deadline, production time,
# and interactions derived from production time

features_without_pair = [
    f for f in base_features
    if f not in (
        production_related_features
        | {"promised_delivery_days"}
    )
]

ablation_datasets["D_Without_deadline_and_production"] = (
    X_train[features_without_pair].copy()
)


# E — Replace deadline, production time and warehouse
# elapsed-time components with one remaining-slack variable.

warehouse_time_features = [
    "warehouse_queue_wait_time_h",
    "warehouse_inbound_handling_time_h",
    "warehouse_storage_dwell_time_h",
    "warehouse_loading_time_h"
]

interaction_features_to_remove = [
    "production_transport_interaction",
    "warehouse_transport_interaction",
    "production_warehouse_interaction"
]

replacement_features_to_remove = set(
    [
        "promised_delivery_days",
        "production_time_h"
    ]
    + warehouse_time_features
    + interaction_features_to_remove
)

features_for_slack_model = [
    f for f in base_features
    if f not in replacement_features_to_remove
]

X_slack = X_train[features_for_slack_model].copy()

X_slack["pre_dispatch_remaining_slack_h"] = (
    24 * X_train["promised_delivery_days"]
    - X_train["production_time_h"]
    - X_train[warehouse_time_features].sum(axis=1)
)

ablation_datasets["E_Remaining_slack_representation"] = (
    X_slack
)


# ------------------------------------------------------------
# 4. Validate ablation specifications
# ------------------------------------------------------------

for specification, X_spec in ablation_datasets.items():

    if X_spec.shape[1] == 0:
        raise ValueError(
            f"{specification} has no predictors."
        )

    if X_spec.isna().any().any():
        raise ValueError(
            f"Missing values detected in {specification}."
        )

    if X_spec.columns.duplicated().any():
        raise ValueError(
            f"Duplicate feature names detected in {specification}."
        )


specification_summary_df = pd.DataFrame([
    {
        "Specification": name,
        "Number_of_features": X_spec.shape[1]
    }
    for name, X_spec in ablation_datasets.items()
])

print("\n" + "=" * 70)
print("FEATURE SPECIFICATIONS")
print("=" * 70)

print(specification_summary_df.to_string(index=False))


# ------------------------------------------------------------
# 5. Define a fresh preprocessing pipeline for each specification
# ------------------------------------------------------------

known_categorical = {
    "forecast_traffic_level",
    "forecast_weather_condition",
    "route_complexity_level"
}


def build_ablation_pipeline(X_example):

    feature_columns = list(X_example.columns)

    categorical_features = [
        col for col in feature_columns
        if (
            col in known_categorical
            or pd.api.types.is_object_dtype(
                X_example[col].dtype
            )
            or isinstance(
                X_example[col].dtype,
                pd.CategoricalDtype
            )
        )
    ]

    numeric_features = [
        col for col in feature_columns
        if col not in categorical_features
    ]

    transformers = []

    if numeric_features:

        numeric_pipeline = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ])

        transformers.append(
            ("num", numeric_pipeline, numeric_features)
        )

    if categorical_features:

        categorical_pipeline = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ])

        transformers.append(
            ("cat", categorical_pipeline, categorical_features)
        )

    preprocessor = ColumnTransformer(
        transformers=transformers,
        remainder="drop"
    )

    return Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("regressor", LinearRegression())
    ])


# ------------------------------------------------------------
# 6. Run identical five-fold stratified CV
#    for all specifications
# ------------------------------------------------------------

stratify_labels = (
    y_train.to_numpy(dtype=float) > 0
).astype(int)

class_counts = pd.Series(
    stratify_labels
).value_counts()

if len(class_counts) < 2 or class_counts.min() < 5:
    raise ValueError(
        "Insufficient observations in one delivery-status category "
        "for five-fold stratified cross-validation."
    )

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=2026
)

y_train_array = y_train.to_numpy(dtype=float)

fold_records = []

print("\n" + "=" * 70)
print("FIVE-FOLD STRATIFIED CROSS-VALIDATION")
print("=" * 70)

for specification, X_spec in ablation_datasets.items():

    print(f"\nEvaluating: {specification}")

    for fold, (train_idx, valid_idx) in enumerate(
        cv.split(X_spec, stratify_labels),
        start=1
    ):

        X_fold_train = X_spec.iloc[train_idx]
        X_fold_valid = X_spec.iloc[valid_idx]

        y_fold_train = y_train_array[train_idx]
        y_fold_valid = y_train_array[valid_idx]

        model = build_ablation_pipeline(X_spec)

        model.fit(
            X_fold_train,
            y_fold_train
        )

        raw_predictions = model.predict(
            X_fold_valid
        )

        constrained_predictions = np.maximum(
            raw_predictions,
            0
        )

        # Raw metrics

        raw_rmse = np.sqrt(
            mean_squared_error(
                y_fold_valid,
                raw_predictions
            )
        )

        raw_mae = mean_absolute_error(
            y_fold_valid,
            raw_predictions
        )

        raw_r2 = r2_score(
            y_fold_valid,
            raw_predictions
        )

        # Nonnegative-constrained metrics

        constrained_rmse = np.sqrt(
            mean_squared_error(
                y_fold_valid,
                constrained_predictions
            )
        )

        constrained_mae = mean_absolute_error(
            y_fold_valid,
            constrained_predictions
        )

        constrained_r2 = r2_score(
            y_fold_valid,
            constrained_predictions
        )

        fold_records.append({

            "Specification": specification,
            "Fold": fold,

            "Raw_RMSE": raw_rmse,
            "Raw_MAE": raw_mae,
            "Raw_R2": raw_r2,

            "Constrained_RMSE": constrained_rmse,
            "Constrained_MAE": constrained_mae,
            "Constrained_R2": constrained_r2,

            "Number_of_features": X_spec.shape[1]
        })

        print(
            f"  Fold {fold}/5 | "
            f"Raw RMSE: {raw_rmse:.4f} | "
            f"Constrained RMSE: {constrained_rmse:.4f}"
        )


# ------------------------------------------------------------
# 7. Calculate mean and standard deviation across folds
# ------------------------------------------------------------

ablation_cv_fold_results_df = pd.DataFrame(
    fold_records
)

metric_columns = [
    "Raw_RMSE",
    "Raw_MAE",
    "Raw_R2",
    "Constrained_RMSE",
    "Constrained_MAE",
    "Constrained_R2"
]

summary_records = []

for specification, group in ablation_cv_fold_results_df.groupby(
    "Specification",
    sort=False
):

    record = {
        "Specification": specification,
        "Number_of_features": int(
            group["Number_of_features"].iloc[0]
        )
    }

    for metric in metric_columns:

        record[f"{metric}_Mean"] = group[metric].mean()

        record[f"{metric}_SD"] = group[metric].std(ddof=1)

    summary_records.append(record)


ablation_cv_summary_df = pd.DataFrame(
    summary_records
).sort_values(
    "Raw_RMSE_Mean"
).reset_index(drop=True)


# ------------------------------------------------------------
# 8. Calculate changes relative to the full model
# ------------------------------------------------------------

full_model_row = ablation_cv_summary_df[
    ablation_cv_summary_df["Specification"] == "A_Full_model"
]

if len(full_model_row) != 1:
    raise ValueError("Full-model CV results are missing.")

full_raw_rmse = float(
    full_model_row["Raw_RMSE_Mean"].iloc[0]
)

full_constrained_rmse = float(
    full_model_row["Constrained_RMSE_Mean"].iloc[0]
)

ablation_cv_summary_df["Raw_RMSE_Change_pct"] = (
    (
        ablation_cv_summary_df["Raw_RMSE_Mean"]
        / full_raw_rmse
    ) - 1
) * 100

ablation_cv_summary_df["Constrained_RMSE_Change_pct"] = (
    (
        ablation_cv_summary_df["Constrained_RMSE_Mean"]
        / full_constrained_rmse
    ) - 1
) * 100


# ------------------------------------------------------------
# 9. Display results
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FEATURE ABLATION RESULTS — FIVE-FOLD TRAINING CV")
print("=" * 70)

display_columns = [
    "Specification",
    "Number_of_features",
    "Raw_RMSE_Mean",
    "Raw_RMSE_SD",
    "Raw_MAE_Mean",
    "Raw_R2_Mean",
    "Constrained_RMSE_Mean",
    "Constrained_RMSE_SD",
    "Constrained_MAE_Mean",
    "Constrained_R2_Mean",
    "Raw_RMSE_Change_pct"
]

print(
    ablation_cv_summary_df[display_columns].to_string(
        index=False,
        formatters={
            col: "{:.4f}".format
            for col in display_columns
            if col not in [
                "Specification",
                "Number_of_features"
            ]
        }
    )
)


# ------------------------------------------------------------
# 10. Confirm the test set remains unused
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CELL 18 COMPLETED")
print("=" * 70)

print("Five feature specifications evaluated.")
print("Five-fold stratified CV performed on training data only.")
print("No test observations were used for model selection.")
print("Original models and test predictions were not modified.")

print("\nMain output:")
print("- ablation_cv_summary_df")
print("- ablation_cv_fold_results_df")
print("- specification_summary_df")


CELL 18 — TRAINING-ONLY FEATURE ABLATION
Training observations: 1600
Original predictors: 25

FEATURE SPECIFICATIONS
                    Specification  Number_of_features
                     A_Full_model                  25
               B_Without_deadline                  24
        C_Without_production_time                  22
D_Without_deadline_and_production                  21
 E_Remaining_slack_representation                  17

FIVE-FOLD STRATIFIED CROSS-VALIDATION

Evaluating: A_Full_model
  Fold 1/5 | Raw RMSE: 3.6419 | Constrained RMSE: 3.1251
  Fold 2/5 | Raw RMSE: 3.7096 | Constrained RMSE: 3.2538
  Fold 3/5 | Raw RMSE: 4.0351 | Constrained RMSE: 3.6283
  Fold 4/5 | Raw RMSE: 3.5388 | Constrained RMSE: 2.9359
  Fold 5/5 | Raw RMSE: 3.8778 | Constrained RMSE: 3.3284

Evaluating: B_Without_deadline
  Fold 1/5 | Raw RMSE: 4.5043 | Constrained RMSE: 4.3590
  Fold 2/5 | Raw RMSE: 4.8578 | Constrained RMSE: 4.6371
  Fold 3/5 | Raw RMSE: 4.7418 | Constrained RMSE: 4.5699
  Fold


# ============================================================
# CELL 18 — TRAINING-ONLY FEATURE ABLATION
# Primary model: Linear Regression
# Five-fold cross-validation
# ============================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)


# ------------------------------------------------------------
# 1. Validate training data and feature definitions
# ------------------------------------------------------------

if len(X_train) != len(y_train):
    raise ValueError("Training feature and target lengths differ.")

if X_train.empty:
    raise ValueError("Training dataset is empty.")

if X_train.isna().any().any() or y_train.isna().any():
    raise ValueError("Missing values detected in the training data.")

base_features = list(X_train.columns)

print("=" * 70)
print("CELL 23 — TRAINING-ONLY FEATURE ABLATION")
print("=" * 70)

print("Training observations:", len(X_train))
print("Original predictors:", len(base_features))


# ------------------------------------------------------------
# 2. Define required variables for the alternative specification
# ------------------------------------------------------------

required_slack_features = [
    "promised_delivery_days",
    "production_time_h",
    "warehouse_queue_wait_time_h",
    "warehouse_inbound_handling_time_h",
    "warehouse_storage_dwell_time_h",
    "warehouse_loading_time_h"
]

missing_slack_features = [
    f for f in required_slack_features
    if f not in X_train.columns
]

if missing_slack_features:
    raise ValueError(
        f"Cannot calculate remaining slack. Missing features: "
        f"{missing_slack_features}"
    )


# ------------------------------------------------------------
# 3. Create five feature specifications
# ------------------------------------------------------------

ablation_datasets = {}

# A — Full model
ablation_datasets["A_Full_model"] = X_train[base_features].copy()


# B — Remove promised delivery days
features_without_deadline = [
    f for f in base_features
    if f != "promised_delivery_days"
]

ablation_datasets["B_Without_deadline"] = (
    X_train[features_without_deadline].copy()
)


# C — Remove production time
features_without_production = [
    f for f in base_features
    if f != "production_time_h"
]

ablation_datasets["C_Without_production_time"] = (
    X_train[features_without_production].copy()
)


# D — Remove both highly correlated predictors
features_without_pair = [
    f for f in base_features
    if f not in {
        "promised_delivery_days",
        "production_time_h"
    }
]

ablation_datasets["D_Without_deadline_and_production"] = (
    X_train[features_without_pair].copy()
)


# E — Replace deadline, production time and warehouse
# elapsed-time components with one remaining-slack variable.

warehouse_time_features = [
    "warehouse_queue_wait_time_h",
    "warehouse_inbound_handling_time_h",
    "warehouse_storage_dwell_time_h",
    "warehouse_loading_time_h"
]

interaction_features_to_remove = [
    "production_transport_interaction",
    "warehouse_transport_interaction",
    "production_warehouse_interaction"
]

replacement_features_to_remove = set([
    "promised_delivery_days",
    "production_time_h"
] + warehouse_time_features + interaction_features_to_remove)

features_for_slack_model = [
    f for f in base_features
    if f not in replacement_features_to_remove
]

X_slack = X_train[features_for_slack_model].copy()

X_slack["pre_dispatch_remaining_slack_h"] = (

    24 * X_train["promised_delivery_days"]

    - X_train["production_time_h"]

    - X_train[warehouse_time_features].sum(axis=1)
)

ablation_datasets["E_Remaining_slack_representation"] = X_slack


# ------------------------------------------------------------
# 4. Validate ablation specifications
# ------------------------------------------------------------

for specification, X_spec in ablation_datasets.items():

    if X_spec.shape[1] == 0:
        raise ValueError(f"{specification} has no predictors.")

    if X_spec.isna().any().any():
        raise ValueError(
            f"Missing values detected in {specification}."
        )

    if X_spec.columns.duplicated().any():
        raise ValueError(
            f"Duplicate feature names detected in {specification}."
        )


specification_summary_df = pd.DataFrame([
    {
        "Specification": name,
        "Number_of_features": X_spec.shape[1]
    }
    for name, X_spec in ablation_datasets.items()
])

print("\n" + "=" * 70)
print("FEATURE SPECIFICATIONS")
print("=" * 70)

print(specification_summary_df.to_string(index=False))


# ------------------------------------------------------------
# 5. Define a fresh preprocessing pipeline for each specification
# ------------------------------------------------------------

known_categorical = {
    "forecast_traffic_level",
    "forecast_weather_condition",
    "route_complexity_level"
}


def build_ablation_pipeline(feature_columns):

    categorical_features = [
        col for col in feature_columns
        if (
            col in known_categorical
            or pd.api.types.is_object_dtype(
                ablation_datasets_feature_example[col].dtype
            )
            or isinstance(
                ablation_datasets_feature_example[col].dtype,
                pd.CategoricalDtype
            )
        )
    ]

    numeric_features = [
        col for col in feature_columns
        if col not in categorical_features
    ]

    transformers = []

    if numeric_features:

        numeric_pipeline = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ])

        transformers.append(
            ("num", numeric_pipeline, numeric_features)
        )

    if categorical_features:

        categorical_pipeline = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ])

        transformers.append(
            ("cat", categorical_pipeline, categorical_features)
        )

    preprocessor = ColumnTransformer(
        transformers=transformers,
        remainder="drop"
    )

    return Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("regressor", LinearRegression())
    ])


# ------------------------------------------------------------
# 6. Run identical five-fold CV for all specifications
# ------------------------------------------------------------

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=2026
)

y_train_array = y_train.to_numpy(dtype=float)

fold_records = []

print("\n" + "=" * 70)
print("FIVE-FOLD CROSS-VALIDATION")
print("=" * 70)

for specification, X_spec in ablation_datasets.items():

    print(f"\nEvaluating: {specification}")

    ablation_datasets_feature_example = X_spec

    for fold, (train_idx, valid_idx) in enumerate(
        cv.split(X_spec, y_train_array),
        start=1
    ):

        X_fold_train = X_spec.iloc[train_idx]
        X_fold_valid = X_spec.iloc[valid_idx]

        y_fold_train = y_train_array[train_idx]
        y_fold_valid = y_train_array[valid_idx]

        model = build_ablation_pipeline(
            list(X_spec.columns)
        )

        model.fit(
            X_fold_train,
            y_fold_train
        )

        raw_predictions = model.predict(
            X_fold_valid
        )

        constrained_predictions = np.maximum(
            raw_predictions,
            0
        )

        # Raw metrics: consistent with original Cell 19 CV results
        raw_rmse = np.sqrt(
            mean_squared_error(
                y_fold_valid,
                raw_predictions
            )
        )

        raw_mae = mean_absolute_error(
            y_fold_valid,
            raw_predictions
        )

        raw_r2 = r2_score(
            y_fold_valid,
            raw_predictions
        )

        # Nonnegative-constrained metrics
        constrained_rmse = np.sqrt(
            mean_squared_error(
                y_fold_valid,
                constrained_predictions
            )
        )

        constrained_mae = mean_absolute_error(
            y_fold_valid,
            constrained_predictions
        )

        constrained_r2 = r2_score(
            y_fold_valid,
            constrained_predictions
        )

        fold_records.append({

            "Specification": specification,
            "Fold": fold,

            "Raw_RMSE": raw_rmse,
            "Raw_MAE": raw_mae,
            "Raw_R2": raw_r2,

            "Constrained_RMSE": constrained_rmse,
            "Constrained_MAE": constrained_mae,
            "Constrained_R2": constrained_r2,

            "Number_of_features": X_spec.shape[1]
        })

        print(
            f"  Fold {fold}/5 | "
            f"Raw RMSE: {raw_rmse:.4f} | "
            f"Constrained RMSE: {constrained_rmse:.4f}"
        )


# ------------------------------------------------------------
# 7. Calculate mean and standard deviation across folds
# ------------------------------------------------------------

ablation_cv_fold_results_df = pd.DataFrame(
    fold_records
)

metric_columns = [
    "Raw_RMSE",
    "Raw_MAE",
    "Raw_R2",
    "Constrained_RMSE",
    "Constrained_MAE",
    "Constrained_R2"
]

summary_records = []

for specification, group in ablation_cv_fold_results_df.groupby(
    "Specification",
    sort=False
):

    record = {
        "Specification": specification,
        "Number_of_features": int(
            group["Number_of_features"].iloc[0]
        )
    }

    for metric in metric_columns:

        record[f"{metric}_Mean"] = group[metric].mean()

        record[f"{metric}_SD"] = group[metric].std(ddof=1)

    summary_records.append(record)


ablation_cv_summary_df = pd.DataFrame(
    summary_records
).sort_values(
    "Raw_RMSE_Mean"
).reset_index(drop=True)


# ------------------------------------------------------------
# 8. Calculate changes relative to the full model
# ------------------------------------------------------------

full_model_row = ablation_cv_summary_df[
    ablation_cv_summary_df["Specification"] == "A_Full_model"
]

if len(full_model_row) != 1:
    raise ValueError("Full-model CV results are missing.")

full_raw_rmse = float(
    full_model_row["Raw_RMSE_Mean"].iloc[0]
)

full_constrained_rmse = float(
    full_model_row["Constrained_RMSE_Mean"].iloc[0]
)

ablation_cv_summary_df["Raw_RMSE_Change_pct"] = (
    (
        ablation_cv_summary_df["Raw_RMSE_Mean"]
        / full_raw_rmse
    ) - 1
) * 100

ablation_cv_summary_df["Constrained_RMSE_Change_pct"] = (
    (
        ablation_cv_summary_df["Constrained_RMSE_Mean"]
        / full_constrained_rmse
    ) - 1
) * 100


# ------------------------------------------------------------
# 9. Display results
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FEATURE ABLATION RESULTS — FIVE-FOLD TRAINING CV")
print("=" * 70)

display_columns = [
    "Specification",
    "Number_of_features",
    "Raw_RMSE_Mean",
    "Raw_RMSE_SD",
    "Raw_MAE_Mean",
    "Raw_R2_Mean",
    "Constrained_RMSE_Mean",
    "Constrained_RMSE_SD",
    "Constrained_MAE_Mean",
    "Constrained_R2_Mean",
    "Raw_RMSE_Change_pct"
]

print(
    ablation_cv_summary_df[display_columns].to_string(
        index=False,
        formatters={
            col: "{:.4f}".format
            for col in display_columns
            if col not in [
                "Specification",
                "Number_of_features"
            ]
        }
    )
)


# ------------------------------------------------------------
# 10. Confirm the test set remains unused
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CELL 23 COMPLETED")
print("=" * 70)

print("Five feature specifications evaluated.")
print("Five-fold CV performed on training data only.")
print("No test observations were used for model selection.")
print("Original models and test predictions were not modified.")

print("\nMain output:")
print("- ablation_cv_summary_df")
print("- ablation_cv_fold_results_df")
print("- specification_summary_df")


### 3.1 Full-model multicollinearity diagnostic

In [37]:

# ============================================================
# VIF DIAGNOSTICS — TRAINING-ONLY NUMERICAL PREDICTORS
# ============================================================
# Purpose:
# Calculate multicollinearity diagnostics directly from the
# training-only candidate predictors.
#
# This avoids dependency on exploratory-model SHAP outputs.
# Categorical predictors are excluded from this numerical VIF
# analysis.
# ============================================================

import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression


# ------------------------------------------------------------
# 1. Identify numerical candidate predictors
# ------------------------------------------------------------

known_categorical = {
    "forecast_traffic_level",
    "forecast_weather_condition",
    "route_complexity_level"
}

feature_names = list(selected_feature_columns)

numeric_vif_features = [
    col for col in feature_names
    if (
        col not in known_categorical
        and pd.api.types.is_numeric_dtype(X_train[col])
    )
]


# ------------------------------------------------------------
# 2. Prepare numerical predictors
# ------------------------------------------------------------

X_vif = X_train[numeric_vif_features].astype(float).copy()

if not np.isfinite(X_vif.to_numpy()).all():
    raise ValueError(
        "Non-finite values detected in numerical VIF predictors."
    )

std = X_vif.std(ddof=0)

valid_vif_features = std[
    std > 1e-12
].index.tolist()

X_vif = X_vif[valid_vif_features]

if not valid_vif_features:
    raise ValueError(
        "No non-constant numerical predictors are available "
        "for VIF analysis."
    )


# Standardize predictors.

X_vif = (
    X_vif - X_vif.mean()
) / X_vif.std(ddof=0)


# ------------------------------------------------------------
# 3. Calculate VIF for each numerical predictor
# ------------------------------------------------------------

vif_records = []

for feature in valid_vif_features:

    others = [
        name for name in valid_vif_features
        if name != feature
    ]

    if not others:

        vif_value = 1.0

    else:

        model = LinearRegression().fit(
            X_vif[others],
            X_vif[feature]
        )

        r2 = float(
            np.clip(
                model.score(
                    X_vif[others],
                    X_vif[feature]
                ),
                0,
                1
            )
        )

        vif_value = (
            np.inf
            if r2 >= 1 - 1e-10
            else 1.0 / (1.0 - r2)
        )

    vif_records.append({
        "Feature": feature,
        "VIF": vif_value
    })


# ------------------------------------------------------------
# 4. Generate and display VIF diagnostics
# ------------------------------------------------------------

vif_diagnostics_df = (
    pd.DataFrame(vif_records)
    .sort_values(
        "VIF",
        ascending=False
    )
    .reset_index(drop=True)
)

print("Training-only full-model VIF diagnostics")

display(vif_diagnostics_df)


Training-only full-model VIF diagnostics


,Feature,VIF
0,production_interstage_nonprocessing_time_h,420.185560
1,production_time_h,307.711454
2,number_of_jobs,252.513888
3,promised_delivery_days,203.915017
4,production_setup_time_h,100.619645
5,warehouse_transport_interaction,68.719284
6,production_transport_interaction,55.761061
7,total_order_quantity,23.015481
8,warehouse_inbound_handling_time_h,19.141705
9,quantity_share_1000mm,17.079201


In [38]:

# ============================================================
# — VIF ANALYSIS FOR REMAINING-SLACK REPRESENTATION
# Training data only
# ============================================================

import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression


# ------------------------------------------------------------
# 1. Retrieve Specification E
# ------------------------------------------------------------

X_slack_vif = (
    ablation_datasets["E_Remaining_slack_representation"]
    .copy()
)

known_categorical = {
    "forecast_traffic_level",
    "forecast_weather_condition",
    "route_complexity_level"
}

numeric_features = [
    col for col in X_slack_vif.columns
    if (
        col not in known_categorical
        and pd.api.types.is_numeric_dtype(X_slack_vif[col])
    )
]

X_vif_slack = X_slack_vif[numeric_features].astype(float).copy()

# Remove constant variables
std_values = X_vif_slack.std(ddof=0)

valid_features = std_values[
    std_values > 1e-12
].index.tolist()

X_vif_slack = X_vif_slack[valid_features]

# Standardize numerical variables
X_vif_slack = (
    (X_vif_slack - X_vif_slack.mean())
    / X_vif_slack.std(ddof=0)
)


# ------------------------------------------------------------
# 2. Calculate full-model VIF
# ------------------------------------------------------------

vif_records = []

for feature in valid_features:

    other_features = [
        f for f in valid_features
        if f != feature
    ]

    if not other_features:
        vif_value = 1.0

    else:

        regression = LinearRegression()

        regression.fit(
            X_vif_slack[other_features],
            X_vif_slack[feature]
        )

        r_squared = np.clip(
            regression.score(
                X_vif_slack[other_features],
                X_vif_slack[feature]
            ),
            0,
            1
        )

        vif_value = (
            np.inf
            if r_squared >= 1 - 1e-10
            else 1 / (1 - r_squared)
        )

    vif_records.append({
        "Feature": feature,
        "VIF": vif_value
    })


vif_slack_df = pd.DataFrame(
    vif_records
).sort_values(
    "VIF",
    ascending=False
).reset_index(drop=True)


# ------------------------------------------------------------
# 3. Compare with original full-model VIF
# ------------------------------------------------------------

comparison_features = set(vif_slack_df["Feature"]).intersection(
    set(vif_diagnostics_df["Feature"])
)

vif_comparison_df = (

    vif_diagnostics_df[
        vif_diagnostics_df["Feature"].isin(comparison_features)
    ][["Feature", "VIF"]]

    .merge(
        vif_slack_df[
            ["Feature", "VIF"]
        ],
        on="Feature",
        suffixes=("_Full", "_Slack")
    )
)

vif_comparison_df["VIF_Reduction_pct"] = (
    (
        vif_comparison_df["VIF_Full"]
        - vif_comparison_df["VIF_Slack"]
    )
    / vif_comparison_df["VIF_Full"]
) * 100

vif_comparison_df = vif_comparison_df.sort_values(
    "VIF_Slack",
    ascending=False
).reset_index(drop=True)


# ------------------------------------------------------------
# 4. Report results
# ------------------------------------------------------------

print("=" * 70)
print("REMAINING-SLACK MODEL — FULL VIF DIAGNOSTICS")
print("=" * 70)

print(
    vif_slack_df.to_string(
        index=False,
        formatters={
            "VIF": lambda x: (
                "inf" if not np.isfinite(x) else f"{x:.3f}"
            )
        }
    )
)

print("\n" + "=" * 70)
print("COMPARISON WITH ORIGINAL FEATURE REPRESENTATION")
print("=" * 70)

print(
    vif_comparison_df.to_string(
        index=False,
        formatters={
            col: "{:.2f}".format
            for col in [
                "VIF_Full",
                "VIF_Slack",
                "VIF_Reduction_pct"
            ]
        }
    )
)

print("\n" + "=" * 70)
print("CELL 23A COMPLETED")
print("=" * 70)

print("Number of numerical predictors:", len(valid_features))
print("No models were retrained.")
print("Only training data were used.")


REMAINING-SLACK MODEL — FULL VIF DIAGNOSTICS
                                   Feature     VIF
production_interstage_nonprocessing_time_h 416.821
                            number_of_jobs 250.863
                   production_setup_time_h  99.605
                     quantity_share_1000mm   6.856
                      quantity_share_900mm   5.682
                      total_order_quantity   3.908
            pre_dispatch_remaining_slack_h   2.355
                     machine_downtime_M1_h   2.300
                     machine_downtime_M2_h   1.895
                   warehouse_backlog_ratio   1.679
                     machine_downtime_M3_h   1.419
           forecast_expected_travel_time_h   1.366
                   forecast_base_speed_kmh   1.364
                     machine_downtime_M4_h   1.283
             forecast_vehicle_availability   1.009

COMPARISON WITH ORIGINAL FEATURE REPRESENTATION
                                   Feature VIF_Full VIF_Slack VIF_Reduction_pct
production

In [39]:

# ============================================================
# CELL 23B — PRODUCTION FEATURE REDUCTION
# Training-only five-fold stratified cross-validation
# ============================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)


# ------------------------------------------------------------
# 1. Start from the remaining-slack representation (E)
# ------------------------------------------------------------

X_E = ablation_datasets[
    "E_Remaining_slack_representation"
].copy()

production_features = [
    "production_interstage_nonprocessing_time_h",
    "number_of_jobs",
    "production_setup_time_h"
]

missing = [
    f for f in production_features
    if f not in X_E.columns
]

if missing:
    raise ValueError(
        f"Missing production features: {missing}"
    )

if not isinstance(X_E, pd.DataFrame):
    raise TypeError(
        "X_E must be a pandas DataFrame."
    )

if not isinstance(y_train, (pd.Series, pd.DataFrame)):
    raise TypeError(
        "y_train must be a pandas Series or DataFrame."
    )

if isinstance(y_train, pd.DataFrame):
    if y_train.shape[1] != 1:
        raise ValueError(
            "y_train must contain exactly one target column."
        )
    y_array = y_train.iloc[:, 0].to_numpy(dtype=float)
else:
    y_array = y_train.to_numpy(dtype=float)

if len(X_E) != len(y_array):
    raise ValueError(
        "X_E and y_train have different numbers of observations."
    )

if not np.isfinite(y_array).all():
    raise ValueError(
        "The training target contains missing or non-finite values."
    )

if X_E.columns.duplicated().any():
    raise ValueError(
        "Duplicate feature names were detected in X_E."
    )


# ------------------------------------------------------------
# 2. Define alternative production representations
# ------------------------------------------------------------

feature_specs_23B = {

    "E_Remaining_slack_baseline": [],

    "F_Without_interstage_time": [
        "production_interstage_nonprocessing_time_h"
    ],

    "G_Without_number_of_jobs": [
        "number_of_jobs"
    ],

    "H_Without_setup_time": [
        "production_setup_time_h"
    ],

    "I_Without_setup_and_interstage": [
        "production_setup_time_h",
        "production_interstage_nonprocessing_time_h"
    ],

    "J_Without_jobs_and_interstage": [
        "number_of_jobs",
        "production_interstage_nonprocessing_time_h"
    ],

    "K_Without_all_three": production_features
}


# ------------------------------------------------------------
# 3. Construct production ablation datasets
# ------------------------------------------------------------

production_ablation_datasets = {}

for specification, features_to_remove in feature_specs_23B.items():

    remaining_features = [
        f for f in X_E.columns
        if f not in features_to_remove
    ]

    if not remaining_features:
        raise ValueError(
            f"{specification} contains no remaining predictors."
        )

    production_ablation_datasets[specification] = (
        X_E[remaining_features].copy()
    )


# ------------------------------------------------------------
# 4. Define preprocessing pipeline
# ------------------------------------------------------------

known_categorical = {
    "forecast_traffic_level",
    "forecast_weather_condition",
    "route_complexity_level"
}


def build_production_pipeline(X_spec):

    categorical_features = [
        col for col in X_spec.columns
        if (
            col in known_categorical
            or pd.api.types.is_object_dtype(X_spec[col])
            or isinstance(
                X_spec[col].dtype,
                pd.CategoricalDtype
            )
        )
    ]

    numeric_features = [
        col for col in X_spec.columns
        if col not in categorical_features
    ]

    transformers = []

    if numeric_features:

        numeric_pipeline = Pipeline(steps=[
            (
                "imputer",
                SimpleImputer(strategy="median")
            ),
            (
                "scaler",
                StandardScaler()
            )
        ])

        transformers.append(
            ("num", numeric_pipeline, numeric_features)
        )

    if categorical_features:

        categorical_pipeline = Pipeline(steps=[
            (
                "imputer",
                SimpleImputer(strategy="most_frequent")
            ),
            (
                "onehot",
                OneHotEncoder(handle_unknown="ignore")
            )
        ])

        transformers.append(
            ("cat", categorical_pipeline, categorical_features)
        )

    if not transformers:
        raise ValueError(
            "No valid numerical or categorical predictors were found."
        )

    return Pipeline(steps=[
        (
            "preprocessor",
            ColumnTransformer(
                transformers=transformers,
                remainder="drop"
            )
        ),
        (
            "regressor",
            LinearRegression()
        )
    ])


# ------------------------------------------------------------
# 5. Define numerical-only VIF calculation
# Training data only; descriptive diagnostic
# ------------------------------------------------------------

def calculate_numeric_vif(X_spec):

    numeric_features = [
        col for col in X_spec.columns
        if (
            col not in known_categorical
            and pd.api.types.is_numeric_dtype(X_spec[col])
        )
    ]

    if not numeric_features:
        return pd.DataFrame(
            columns=["Feature", "VIF"]
        )

    X_numeric = (
        X_spec[numeric_features]
        .astype(float)
        .copy()
    )

    # Replace infinite values before imputation
    X_numeric = X_numeric.replace(
        [np.inf, -np.inf],
        np.nan
    )

    # Median imputation for VIF diagnostics
    X_numeric = X_numeric.fillna(
        X_numeric.median()
    )

    if X_numeric.isna().any().any():
        raise ValueError(
            "Some numerical predictors could not be imputed "
            "for VIF calculation."
        )

    # Remove constant and near-constant variables
    std_values = X_numeric.std(ddof=0)

    valid_features = std_values[
        std_values > 1e-12
    ].index.tolist()

    X_numeric = X_numeric[valid_features]

    if len(valid_features) == 0:
        return pd.DataFrame(
            columns=["Feature", "VIF"]
        )

    if len(valid_features) == 1:
        return pd.DataFrame({
            "Feature": valid_features,
            "VIF": [1.0]
        })

    # Standardize numerical predictors
    X_numeric = (
        X_numeric - X_numeric.mean()
    ) / X_numeric.std(ddof=0)

    records = []

    for feature in valid_features:

        other_features = [
            f for f in valid_features
            if f != feature
        ]

        regression = LinearRegression()

        regression.fit(
            X_numeric[other_features],
            X_numeric[feature]
        )

        r_squared = np.clip(
            regression.score(
                X_numeric[other_features],
                X_numeric[feature]
            ),
            0,
            1
        )

        vif = (
            np.inf
            if r_squared >= 1 - 1e-10
            else 1 / (1 - r_squared)
        )

        records.append({
            "Feature": feature,
            "VIF": vif
        })

    return pd.DataFrame(records).sort_values(
        "VIF",
        ascending=False
    ).reset_index(drop=True)


# ------------------------------------------------------------
# 6. Run identical five-fold stratified cross-validation
# ------------------------------------------------------------

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=2026
)

# Late/on-time strata
y_strata = (y_array > 0).astype(int)

class_counts = pd.Series(y_strata).value_counts()

if class_counts.min() < 5:
    raise ValueError(
        "Five-fold stratified CV requires at least five "
        "observations in each late/on-time class."
    )

fold_records = []
vif_records = []

print("=" * 70)
print("PRODUCTION FEATURE REDUCTION — FIVE-FOLD STRATIFIED CV")
print("=" * 70)

for specification, X_spec in production_ablation_datasets.items():

    print(f"\nEvaluating: {specification}")

    # --------------------------------------------------------
    # Numerical VIF diagnostics
    # --------------------------------------------------------

    vif_df = calculate_numeric_vif(X_spec)

    max_vif = (
        vif_df["VIF"].max()
        if not vif_df.empty
        else np.nan
    )

    severe_vif_count = (
        int((vif_df["VIF"] >= 10).sum())
        if not vif_df.empty
        else 0
    )

    high_vif_count = (
        int((vif_df["VIF"] >= 5).sum())
        if not vif_df.empty
        else 0
    )

    for _, row in vif_df.iterrows():

        vif_records.append({
            "Specification": specification,
            "Feature": row["Feature"],
            "VIF": row["VIF"]
        })

    # --------------------------------------------------------
    # Cross-validation
    # --------------------------------------------------------

    for fold, (train_idx, valid_idx) in enumerate(
        cv.split(X_spec, y_strata),
        start=1
    ):

        X_fold_train = X_spec.iloc[train_idx]
        X_fold_valid = X_spec.iloc[valid_idx]

        y_fold_train = y_array[train_idx]
        y_fold_valid = y_array[valid_idx]

        # Fit preprocessing only on the current training fold
        model = build_production_pipeline(X_spec)

        model.fit(
            X_fold_train,
            y_fold_train
        )

        raw_pred = model.predict(
            X_fold_valid
        )

        # Nonnegative prediction constraint
        constrained_pred = np.maximum(
            raw_pred,
            0
        )

        fold_records.append({

            "Specification": specification,
            "Fold": fold,

            "Raw_RMSE": np.sqrt(
                mean_squared_error(
                    y_fold_valid,
                    raw_pred
                )
            ),

            "Raw_MAE": mean_absolute_error(
                y_fold_valid,
                raw_pred
            ),

            "Raw_R2": r2_score(
                y_fold_valid,
                raw_pred
            ),

            "Constrained_RMSE": np.sqrt(
                mean_squared_error(
                    y_fold_valid,
                    constrained_pred
                )
            ),

            "Constrained_MAE": mean_absolute_error(
                y_fold_valid,
                constrained_pred
            ),

            "Constrained_R2": r2_score(
                y_fold_valid,
                constrained_pred
            )
        })

    print(
        f"Features: {X_spec.shape[1]} | "
        f"Maximum numerical VIF: {max_vif:.2f} | "
        f"VIF >= 10: {severe_vif_count} | "
        f"VIF >= 5: {high_vif_count}"
    )


# ------------------------------------------------------------
# 7. Summarize cross-validation performance
# ------------------------------------------------------------

production_ablation_folds_df = pd.DataFrame(
    fold_records
)

metric_names = [
    "Raw_RMSE",
    "Raw_MAE",
    "Raw_R2",
    "Constrained_RMSE",
    "Constrained_MAE",
    "Constrained_R2"
]

summary_records = []

for specification, group in production_ablation_folds_df.groupby(
    "Specification",
    sort=False
):

    record = {
        "Specification": specification,

        "Number_of_features": production_ablation_datasets[
            specification
        ].shape[1]
    }

    for metric in metric_names:

        record[f"{metric}_Mean"] = group[metric].mean()

        record[f"{metric}_SD"] = group[metric].std(
            ddof=1
        )

    summary_records.append(record)


production_ablation_summary_df = pd.DataFrame(
    summary_records
)


# ------------------------------------------------------------
# 8. Add numerical VIF summaries
# ------------------------------------------------------------

production_vif_details_df = pd.DataFrame(
    vif_records,
    columns=[
        "Specification",
        "Feature",
        "VIF"
    ]
)

vif_summary_records = []

for specification in feature_specs_23B:

    group = production_vif_details_df[
        production_vif_details_df["Specification"]
        == specification
    ]

    vif_values = group["VIF"].to_numpy(dtype=float)

    vif_summary_records.append({

        "Specification": specification,

        "Max_Numeric_VIF": (
            np.max(vif_values)
            if len(vif_values) > 0
            else np.nan
        ),

        "Number_VIF_at_least_10": int(
            np.sum(vif_values >= 10)
        ),

        "Number_VIF_at_least_5": int(
            np.sum(vif_values >= 5)
        )
    })


production_vif_summary_df = pd.DataFrame(
    vif_summary_records
)

production_ablation_summary_df = (
    production_ablation_summary_df.merge(
        production_vif_summary_df,
        on="Specification",
        how="left"
    )
)


# ------------------------------------------------------------
# 9. Calculate performance changes against E
# ------------------------------------------------------------

baseline = production_ablation_summary_df[
    production_ablation_summary_df["Specification"]
    == "E_Remaining_slack_baseline"
]

if len(baseline) != 1:
    raise ValueError(
        "The E remaining-slack baseline must appear exactly once."
    )

baseline_rmse = float(
    baseline["Raw_RMSE_Mean"].iloc[0]
)

if not np.isfinite(baseline_rmse) or baseline_rmse <= 0:
    raise ValueError(
        "The baseline RMSE must be positive and finite."
    )

production_ablation_summary_df["Raw_RMSE_Change_pct"] = (
    (
        production_ablation_summary_df["Raw_RMSE_Mean"]
        / baseline_rmse
    ) - 1
) * 100


# ------------------------------------------------------------
# 10. Display results
# ------------------------------------------------------------

production_ablation_summary_df = (
    production_ablation_summary_df.sort_values(
        "Raw_RMSE_Mean"
    ).reset_index(drop=True)
)

print("\n" + "=" * 70)
print("PRODUCTION ABLATION — PERFORMANCE AND MULTICOLLINEARITY")
print("=" * 70)

display_columns = [
    "Specification",
    "Number_of_features",
    "Raw_RMSE_Mean",
    "Raw_RMSE_SD",
    "Raw_MAE_Mean",
    "Raw_R2_Mean",
    "Max_Numeric_VIF",
    "Number_VIF_at_least_10",
    "Number_VIF_at_least_5",
    "Raw_RMSE_Change_pct"
]

print(
    production_ablation_summary_df[
        display_columns
    ].to_string(
        index=False,
        formatters={
            col: "{:.4f}".format
            for col in display_columns
            if col not in [
                "Specification",
                "Number_of_features",
                "Number_VIF_at_least_10",
                "Number_VIF_at_least_5"
            ]
        }
    )
)


# ------------------------------------------------------------
# 11. Display remaining severe VIF values
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FEATURES WITH VIF >= 10")
print("=" * 70)

severe_vif_features_df = (
    production_vif_details_df[
        production_vif_details_df["VIF"] >= 10
    ]
    .sort_values(
        ["Specification", "VIF"],
        ascending=[True, False]
    )
)

if severe_vif_features_df.empty:
    print("No numerical predictors have VIF >= 10.")
else:
    print(
        severe_vif_features_df.to_string(
            index=False,
            formatters={
                "VIF": lambda x: (
                    "inf" if not np.isfinite(x)
                    else f"{x:.3f}"
                )
            }
        )
    )


# ------------------------------------------------------------
# 12. Completion summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CELL 23B COMPLETED")
print("=" * 70)

print("Generated outputs:")
print("- production_ablation_summary_df")
print("- production_ablation_folds_df")
print("- production_vif_summary_df")
print("- production_vif_details_df")

print("\nAll performance comparisons used training data only.")
print("VIF diagnostics used the complete training partition.")
print("No model was evaluated on the test set.")


PRODUCTION FEATURE REDUCTION — FIVE-FOLD STRATIFIED CV

Evaluating: E_Remaining_slack_baseline
Features: 17 | Maximum numerical VIF: 416.82 | VIF >= 10: 3 | VIF >= 5: 5

Evaluating: F_Without_interstage_time
Features: 16 | Maximum numerical VIF: 55.11 | VIF >= 10: 2 | VIF >= 5: 3

Evaluating: G_Without_number_of_jobs
Features: 16 | Maximum numerical VIF: 93.17 | VIF >= 10: 2 | VIF >= 5: 3

Evaluating: H_Without_setup_time
Features: 16 | Maximum numerical VIF: 234.67 | VIF >= 10: 2 | VIF >= 5: 3

Evaluating: I_Without_setup_and_interstage
Features: 15 | Maximum numerical VIF: 3.85 | VIF >= 10: 0 | VIF >= 5: 0

Evaluating: J_Without_jobs_and_interstage
Features: 15 | Maximum numerical VIF: 3.71 | VIF >= 10: 0 | VIF >= 5: 0

Evaluating: K_Without_all_three
Features: 14 | Maximum numerical VIF: 2.63 | VIF >= 10: 0 | VIF >= 5: 0

PRODUCTION ABLATION — PERFORMANCE AND MULTICOLLINEARITY
                 Specification  Number_of_features Raw_RMSE_Mean Raw_RMSE_SD Raw_MAE_Mean Raw_R2_Mean Max_N

In [41]:

# ============================================================
# CELL 23C — TESTING THE VALUE OF CROSS-STAGE INTERACTIONS
# Training-only five-fold stratified cross-validation
# ============================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)


# ------------------------------------------------------------
# 1. Define the two candidate specifications
# ------------------------------------------------------------

X_K = production_ablation_datasets[
    "K_Without_all_three"
].copy()

interaction_features = [
    "production_transport_interaction",
    "warehouse_transport_interaction",
    "production_warehouse_interaction"
]

# Check that all interaction features are available
missing = [
    f for f in interaction_features
    if f not in X_train.columns
]

if missing:
    raise ValueError(
        f"Missing interaction features: {missing}"
    )

# Verify row alignment before combining predictors
if not X_K.index.equals(X_train.index):
    raise ValueError(
        "X_K and X_train do not have identical row indices. "
        "Interaction features cannot be safely concatenated."
    )

# Retrieve interaction features from the original training matrix
X_interactions = X_train[
    interaction_features
].copy()

# Construct Specification L
X_L = pd.concat(
    [
        X_K,
        X_interactions
    ],
    axis=1
)

# Check duplicate predictors
if X_L.columns.duplicated().any():
    raise ValueError(
        "Duplicate predictors detected in Specification L."
    )

# Validate target alignment
if len(X_K) != len(y_train):
    raise ValueError(
        "X_K and y_train have different numbers of observations."
    )

if len(X_L) != len(y_train):
    raise ValueError(
        "X_L and y_train have different numbers of observations."
    )

# Define candidate models
candidate_models = {
    "K_Remaining_slack": X_K,
    "L_Remaining_slack_plus_interactions": X_L
}

print("=" * 70)
print("CELL 23C — CROSS-STAGE INTERACTION ABLATION")
print("=" * 70)

print(
    "Specification K:",
    X_K.shape[1],
    "features"
)

print(
    "Specification L:",
    X_L.shape[1],
    "features"
)


# ------------------------------------------------------------
# 2. Prepare target and stratification labels
# ------------------------------------------------------------

if isinstance(y_train, pd.DataFrame):

    if y_train.shape[1] != 1:
        raise ValueError(
            "y_train must contain exactly one target column."
        )

    y_array = y_train.iloc[:, 0].to_numpy(dtype=float)

else:

    y_array = y_train.to_numpy(dtype=float)

if not np.isfinite(y_array).all():
    raise ValueError(
        "The training target contains missing or non-finite values."
    )

# Late/on-time classification for stratified fold construction
y_strata = (y_array > 0).astype(int)

class_counts = pd.Series(y_strata).value_counts()

if len(class_counts) != 2 or class_counts.min() < 5:
    raise ValueError(
        "Five-fold stratified CV requires at least five "
        "observations in each late/on-time class."
    )


# ------------------------------------------------------------
# 3. Define preprocessing
# ------------------------------------------------------------

known_categorical = {
    "forecast_traffic_level",
    "forecast_weather_condition",
    "route_complexity_level"
}


def make_pipeline_for_candidate(X):

    categorical_features = [
        col for col in X.columns
        if (
            col in known_categorical
            or pd.api.types.is_object_dtype(X[col])
            or isinstance(
                X[col].dtype,
                pd.CategoricalDtype
            )
        )
    ]

    numeric_features = [
        col for col in X.columns
        if col not in categorical_features
    ]

    transformers = []

    if numeric_features:

        transformers.append((
            "num",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(strategy="median")
                ),
                (
                    "scaler",
                    StandardScaler()
                )
            ]),
            numeric_features
        ))

    if categorical_features:

        transformers.append((
            "cat",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(strategy="most_frequent")
                ),
                (
                    "onehot",
                    OneHotEncoder(handle_unknown="ignore")
                )
            ]),
            categorical_features
        ))

    if not transformers:
        raise ValueError(
            "No valid numerical or categorical predictors were found."
        )

    return Pipeline([
        (
            "preprocessor",
            ColumnTransformer(
                transformers=transformers,
                remainder="drop"
            )
        ),
        (
            "regressor",
            LinearRegression()
        )
    ])


# ------------------------------------------------------------
# 4. Calculate numerical VIF
# Training data only; descriptive diagnostic
# ------------------------------------------------------------

def calculate_candidate_vif(X):

    numeric_features = [
        col for col in X.columns
        if (
            col not in known_categorical
            and pd.api.types.is_numeric_dtype(X[col])
        )
    ]

    if not numeric_features:
        return pd.DataFrame(
            columns=["Feature", "VIF"]
        )

    X_num = (
        X[numeric_features]
        .astype(float)
        .copy()
    )

    # Replace infinite values
    X_num = X_num.replace(
        [np.inf, -np.inf],
        np.nan
    )

    # Median imputation for VIF diagnostics
    X_num = X_num.fillna(
        X_num.median()
    )

    if X_num.isna().any().any():
        raise ValueError(
            "Some numerical predictors could not be imputed "
            "for candidate VIF calculation."
        )

    # Remove constant and near-constant predictors
    std_values = X_num.std(ddof=0)

    valid_features = std_values[
        std_values > 1e-12
    ].index.tolist()

    X_num = X_num[valid_features]

    if len(valid_features) == 0:
        return pd.DataFrame(
            columns=["Feature", "VIF"]
        )

    if len(valid_features) == 1:
        return pd.DataFrame({
            "Feature": valid_features,
            "VIF": [1.0]
        })

    # Standardize numerical predictors
    X_num = (
        X_num - X_num.mean()
    ) / X_num.std(ddof=0)

    records = []

    for feature in valid_features:

        others = [
            col for col in valid_features
            if col != feature
        ]

        reg = LinearRegression()

        reg.fit(
            X_num[others],
            X_num[feature]
        )

        r2 = np.clip(
            reg.score(
                X_num[others],
                X_num[feature]
            ),
            0,
            1
        )

        vif = (
            np.inf
            if r2 >= 1 - 1e-10
            else 1 / (1 - r2)
        )

        records.append({
            "Feature": feature,
            "VIF": vif
        })

    return pd.DataFrame(records).sort_values(
        "VIF",
        ascending=False
    ).reset_index(drop=True)


# ------------------------------------------------------------
# 5. Run identical five-fold stratified cross-validation
# ------------------------------------------------------------

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=2026
)

fold_records = []
vif_records = []

for specification, X_candidate in candidate_models.items():

    print(f"\nEvaluating {specification}")

    # --------------------------------------------------------
    # Numerical VIF diagnostics
    # --------------------------------------------------------

    candidate_vif = calculate_candidate_vif(
        X_candidate
    )

    max_vif = (
        candidate_vif["VIF"].max()
        if not candidate_vif.empty
        else np.nan
    )

    vif_at_least_10 = (
        int((candidate_vif["VIF"] >= 10).sum())
        if not candidate_vif.empty
        else 0
    )

    vif_at_least_5 = (
        int((candidate_vif["VIF"] >= 5).sum())
        if not candidate_vif.empty
        else 0
    )

    vif_records.append({

        "Specification": specification,

        "Maximum_VIF": max_vif,

        "VIF_at_least_10": vif_at_least_10,

        "VIF_at_least_5": vif_at_least_5
    })

    # --------------------------------------------------------
    # Cross-validation
    # --------------------------------------------------------

    for fold, (train_idx, valid_idx) in enumerate(
        cv.split(X_candidate, y_strata),
        start=1
    ):

        X_fold_train = X_candidate.iloc[
            train_idx
        ]

        X_fold_valid = X_candidate.iloc[
            valid_idx
        ]

        y_fold_train = y_array[
            train_idx
        ]

        y_fold_valid = y_array[
            valid_idx
        ]

        # Fit preprocessing and regression within each fold
        model = make_pipeline_for_candidate(
            X_candidate
        )

        model.fit(
            X_fold_train,
            y_fold_train
        )

        raw_pred = model.predict(
            X_fold_valid
        )

        # Apply nonnegative prediction constraint
        constrained_pred = np.maximum(
            raw_pred,
            0
        )

        fold_records.append({

            "Specification": specification,

            "Fold": fold,

            "Raw_RMSE": np.sqrt(
                mean_squared_error(
                    y_fold_valid,
                    raw_pred
                )
            ),

            "Raw_MAE": mean_absolute_error(
                y_fold_valid,
                raw_pred
            ),

            "Raw_R2": r2_score(
                y_fold_valid,
                raw_pred
            ),

            "Constrained_RMSE": np.sqrt(
                mean_squared_error(
                    y_fold_valid,
                    constrained_pred
                )
            ),

            "Constrained_MAE": mean_absolute_error(
                y_fold_valid,
                constrained_pred
            ),

            "Constrained_R2": r2_score(
                y_fold_valid,
                constrained_pred
            )
        })


# ------------------------------------------------------------
# 6. Summarize cross-validation results
# ------------------------------------------------------------

fold_results_23C_df = pd.DataFrame(
    fold_records
)

summary_23C_df = (
    fold_results_23C_df
    .groupby(
        "Specification",
        sort=False
    )
    .agg(
        Raw_RMSE_Mean=("Raw_RMSE", "mean"),
        Raw_RMSE_SD=("Raw_RMSE", "std"),

        Raw_MAE_Mean=("Raw_MAE", "mean"),

        Raw_R2_Mean=("Raw_R2", "mean"),

        Constrained_RMSE_Mean=(
            "Constrained_RMSE",
            "mean"
        ),

        Constrained_MAE_Mean=(
            "Constrained_MAE",
            "mean"
        ),

        Constrained_R2_Mean=(
            "Constrained_R2",
            "mean"
        )
    )
    .reset_index()
)


# ------------------------------------------------------------
# 7. Add VIF summaries
# ------------------------------------------------------------

vif_summary_23C_df = pd.DataFrame(
    vif_records
)

summary_23C_df = summary_23C_df.merge(
    vif_summary_23C_df,
    on="Specification",
    how="left"
)


# ------------------------------------------------------------
# 8. Calculate performance change relative to K
# ------------------------------------------------------------

baseline = summary_23C_df[
    summary_23C_df["Specification"]
    == "K_Remaining_slack"
]

if len(baseline) != 1:
    raise ValueError(
        "Specification K must appear exactly once."
    )

baseline_rmse = float(
    baseline["Raw_RMSE_Mean"].iloc[0]
)

if not np.isfinite(baseline_rmse) or baseline_rmse <= 0:
    raise ValueError(
        "The baseline RMSE must be positive and finite."
    )

summary_23C_df["Raw_RMSE_Change_pct"] = (
    (
        summary_23C_df["Raw_RMSE_Mean"]
        / baseline_rmse
    ) - 1
) * 100


# ------------------------------------------------------------
# 9. Display results
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CROSS-STAGE INTERACTION COMPARISON")
print("=" * 70)

display_columns = [
    "Specification",
    "Raw_RMSE_Mean",
    "Raw_RMSE_SD",
    "Raw_MAE_Mean",
    "Raw_R2_Mean",
    "Constrained_RMSE_Mean",
    "Constrained_MAE_Mean",
    "Constrained_R2_Mean",
    "Maximum_VIF",
    "VIF_at_least_10",
    "VIF_at_least_5",
    "Raw_RMSE_Change_pct"
]

print(
    summary_23C_df[display_columns].to_string(
        index=False,
        formatters={
            col: (
                lambda x: (
                    "inf" if not np.isfinite(x)
                    else f"{x:.4f}"
                )
            )
            for col in display_columns
            if col not in [
                "Specification",
                "VIF_at_least_10",
                "VIF_at_least_5"
            ]
        }
    )
)


# ------------------------------------------------------------
# 10. Display detailed VIF results
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("NUMERICAL VIF SUMMARY")
print("=" * 70)

print(
    vif_summary_23C_df.to_string(
        index=False,
        formatters={
            "Maximum_VIF": lambda x: (
                "inf" if not np.isfinite(x)
                else f"{x:.3f}"
            )
        }
    )
)


# ------------------------------------------------------------
# 11. Completion summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CELL 23C COMPLETED")
print("=" * 70)

print("Generated outputs:")
print("- summary_23C_df")
print("- fold_results_23C_df")
print("- vif_summary_23C_df")

print("\nBoth specifications used identical stratified folds.")
print("All performance comparisons used training data only.")
print("VIF diagnostics used the complete training partition.")
print("No model was evaluated on the test set.")


CELL 23C — CROSS-STAGE INTERACTION ABLATION
Specification K: 14 features
Specification L: 17 features

Evaluating K_Remaining_slack

Evaluating L_Remaining_slack_plus_interactions

CROSS-STAGE INTERACTION COMPARISON
                      Specification Raw_RMSE_Mean Raw_RMSE_SD Raw_MAE_Mean Raw_R2_Mean Constrained_RMSE_Mean Constrained_MAE_Mean Constrained_R2_Mean Maximum_VIF  VIF_at_least_10  VIF_at_least_5 Raw_RMSE_Change_pct
                  K_Remaining_slack        3.7401      0.1588       2.9148      0.6766                3.2331               2.1382              0.7597      2.6348                0               0              0.0000
L_Remaining_slack_plus_interactions        3.7413      0.1696       2.9238      0.6763                3.2328               2.1470              0.7597     16.9443                2               6              0.0322

NUMERICAL VIF SUMMARY
                      Specification Maximum_VIF  VIF_at_least_10  VIF_at_least_5
                  K_Remaining_slack

In [42]:

# ============================================================
# CELL 24 — FINAL 14-FEATURE DATASET (SPECIFICATION K)
# Corrected: construct remaining slack for all 2,000 orders
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Define the final 14 predictors
# ------------------------------------------------------------

final_feature_columns = [

    # Order characteristics
    "total_order_quantity",
    "quantity_share_900mm",
    "quantity_share_1000mm",

    # Production reliability
    "machine_downtime_M1_h",
    "machine_downtime_M2_h",
    "machine_downtime_M3_h",
    "machine_downtime_M4_h",

    # Warehouse
    "warehouse_backlog_ratio",

    # Transportation forecasts
    "forecast_vehicle_availability",
    "forecast_base_speed_kmh",
    "forecast_traffic_level",
    "forecast_weather_condition",
    "forecast_expected_travel_time_h",

    # Remaining contractual time
    "pre_dispatch_remaining_slack_h"
]

base_final_features = [
    f for f in final_feature_columns
    if f != "pre_dispatch_remaining_slack_h"
]


# ------------------------------------------------------------
# 2. Validate source datasets
# ------------------------------------------------------------

required_objects = [
    "feature_matrix_df",
    "target_df",
    "order_ids",
    "integrated_order_dataset_df",
    "X_train",
    "X_test",
    "y_train",
    "y_test",
    "train_order_ids",
    "test_order_ids"
]

for obj in required_objects:

    if obj not in globals():
        raise ValueError(
            f"Required object '{obj}' is missing. "
            "Check the execution of the previous cells."
        )


# ------------------------------------------------------------
# 3. Reconstruct remaining slack from integrated data
# ------------------------------------------------------------

required_integrated_columns = [
    "customer_order_id",
    "promised_delivery_days",
    "production_time_h",
    "warehouse_total_elapsed_time_h"
]

missing = [
    f for f in required_integrated_columns
    if f not in integrated_order_dataset_df.columns
]

if missing:
    raise ValueError(
        f"Missing integrated dataset columns: {missing}. "
        "Check Cell 14 and Cell 16."
    )

integrated_lookup = integrated_order_dataset_df.copy()

if integrated_lookup["customer_order_id"].duplicated().any():
    raise ValueError(
        "Duplicate customer order IDs found in integrated dataset."
    )

# Standardize ID representation for matching.
integrated_lookup["_order_key"] = (
    integrated_lookup["customer_order_id"].astype(str)
)

if integrated_lookup["_order_key"].duplicated().any():
    raise ValueError("Duplicate normalized order IDs detected.")

integrated_lookup = integrated_lookup.set_index("_order_key")

# Calculate the remaining contractual time for every order.
integrated_lookup["pre_dispatch_remaining_slack_h"] = (

    24 * integrated_lookup["promised_delivery_days"]

    - integrated_lookup["production_time_h"]

    - integrated_lookup["warehouse_total_elapsed_time_h"]
)


# ------------------------------------------------------------
# 4. Create an ID-based lookup function
# ------------------------------------------------------------

def get_remaining_slack(order_id_array):

    keys = pd.Index(
        pd.Series(order_id_array).astype(str)
    )

    missing_ids = keys.difference(
        integrated_lookup.index
    )

    if len(missing_ids) > 0:

        raise ValueError(
            f"{len(missing_ids)} order IDs were not found "
            "in the integrated dataset."
        )

    values = integrated_lookup.loc[
        keys,
        "pre_dispatch_remaining_slack_h"
    ].to_numpy(dtype=float)

    if not np.isfinite(values).all():
        raise ValueError(
            "Non-finite remaining-slack values detected."
        )

    return values


# ------------------------------------------------------------
# 5. Construct the complete 2,000-order feature matrix
# ------------------------------------------------------------

missing_features = [
    f for f in base_final_features
    if f not in feature_matrix_df.columns
]

if missing_features:
    raise ValueError(
        f"Missing base predictors: {missing_features}"
    )

final_feature_matrix_df = (
    feature_matrix_df[base_final_features]
    .copy()
    .reset_index(drop=True)
)

final_order_ids = np.asarray(order_ids).reshape(-1)

if len(final_feature_matrix_df) != len(final_order_ids):
    raise ValueError(
        "Feature matrix and order ID lengths do not match."
    )

final_feature_matrix_df["pre_dispatch_remaining_slack_h"] = (
    get_remaining_slack(final_order_ids)
)

final_feature_matrix_df = (
    final_feature_matrix_df[final_feature_columns]
)


# ------------------------------------------------------------
# 6. Construct the complete modeling dataset
# ------------------------------------------------------------

final_target = (
    target_df["delivery_lateness_h"]
    .reset_index(drop=True)
)

if not (
    len(final_feature_matrix_df)
    == len(final_target)
    == len(final_order_ids)
):
    raise ValueError("Final dataset dimensions are inconsistent.")

final_modeling_dataset_df = (
    final_feature_matrix_df.copy()
)

final_modeling_dataset_df.insert(
    0,
    "order_id",
    final_order_ids
)

final_modeling_dataset_df["actual_lateness_h"] = (
    final_target.to_numpy()
)


# ------------------------------------------------------------
# 7. Construct final training and test datasets by order ID
# ------------------------------------------------------------

X_final_train = X_train[
    base_final_features
].copy().reset_index(drop=True)

X_final_test = X_test[
    base_final_features
].copy().reset_index(drop=True)

X_final_train["pre_dispatch_remaining_slack_h"] = (
    get_remaining_slack(train_order_ids)
)

X_final_test["pre_dispatch_remaining_slack_h"] = (
    get_remaining_slack(test_order_ids)
)

X_final_train = X_final_train[
    final_feature_columns
]

X_final_test = X_final_test[
    final_feature_columns
]

y_final_train = y_train.copy().reset_index(drop=True)
y_final_test = y_test.copy().reset_index(drop=True)

train_order_ids_final = np.asarray(train_order_ids).copy()
test_order_ids_final = np.asarray(test_order_ids).copy()


# ------------------------------------------------------------
# 8. Validate final feature specifications
# ------------------------------------------------------------

if X_final_train.shape[1] != 14:
    raise ValueError("Training dataset must contain 14 predictors.")

if X_final_test.shape[1] != 14:
    raise ValueError("Test dataset must contain 14 predictors.")

if X_final_train.isna().any().any():
    raise ValueError("Missing values in final training dataset.")

if X_final_test.isna().any().any():
    raise ValueError("Missing values in final test dataset.")

if not np.isfinite(y_final_train.to_numpy()).all():
    raise ValueError("Invalid training target values.")

if not np.isfinite(y_final_test.to_numpy()).all():
    raise ValueError("Invalid test target values.")

if (
    set(train_order_ids_final)
    & set(test_order_ids_final)
):
    raise ValueError("Training/test order overlap detected.")


# Verify that final data preserve the original target values.
original_target = (
    target_df["delivery_lateness_h"].to_numpy(dtype=float)
)

if not np.allclose(
    final_modeling_dataset_df["actual_lateness_h"],
    original_target
):
    raise ValueError("The final target differs from the original target.")


# ------------------------------------------------------------
# 9. Define final operational groups
# ------------------------------------------------------------

final_feature_groups = {

    "Order characteristics": [
        "total_order_quantity",
        "quantity_share_900mm",
        "quantity_share_1000mm"
    ],

    "Production reliability": [
        "machine_downtime_M1_h",
        "machine_downtime_M2_h",
        "machine_downtime_M3_h",
        "machine_downtime_M4_h"
    ],

    "Warehouse": [
        "warehouse_backlog_ratio"
    ],

    "Transportation forecasts": [
        "forecast_vehicle_availability",
        "forecast_base_speed_kmh",
        "forecast_traffic_level",
        "forecast_weather_condition",
        "forecast_expected_travel_time_h"
    ],

    "Pre-dispatch remaining slack": [
        "pre_dispatch_remaining_slack_h"
    ]
}


# ------------------------------------------------------------
# 10. Report final dataset
# ------------------------------------------------------------

print("=" * 70)
print("CELL 24 — FINAL DATASET CREATED SUCCESSFULLY")
print("=" * 70)

print("\nComplete dataset:", final_modeling_dataset_df.shape)
print("Training dataset:", X_final_train.shape)
print("Test dataset:", X_final_test.shape)

print("\nFinal predictors:")

for i, feature in enumerate(final_feature_columns, 1):
    print(f"{i:02d}. {feature}")

print("\nRemaining slack summary:")
print(
    final_feature_matrix_df[
        "pre_dispatch_remaining_slack_h"
    ].describe().round(3)
)

print("\nTarget summary:")
print(final_target.describe().round(3))

print("\nTraining orders:", len(X_final_train))
print("Test orders:", len(X_final_test))

print("\nCELL 24 COMPLETED.")


CELL 24 — FINAL DATASET CREATED SUCCESSFULLY

Complete dataset: (2000, 16)
Training dataset: (1600, 14)
Test dataset: (400, 14)

Final predictors:
01. total_order_quantity
02. quantity_share_900mm
03. quantity_share_1000mm
04. machine_downtime_M1_h
05. machine_downtime_M2_h
06. machine_downtime_M3_h
07. machine_downtime_M4_h
08. warehouse_backlog_ratio
09. forecast_vehicle_availability
10. forecast_base_speed_kmh
11. forecast_traffic_level
12. forecast_weather_condition
13. forecast_expected_travel_time_h
14. pre_dispatch_remaining_slack_h

Remaining slack summary:
count    2000.000
mean       12.102
std        12.095
min       -31.303
25%         4.617
50%        12.504
75%        20.382
max        43.391
Name: pre_dispatch_remaining_slack_h, dtype: float64

Target summary:
count    2000.000
mean        3.474
std         6.624
min         0.000
25%         0.000
50%         0.000
75%         4.384
max        48.510
Name: delivery_lateness_h, dtype: float64

Training orders: 1600
Test 

In [43]:

# ============================================================
# CELL 25 — FINAL LINEAR REGRESSION MODEL
# Specification K — 14 selected predictors
# ============================================================

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)


# ------------------------------------------------------------
# 1. Validate final training dataset
# ------------------------------------------------------------

if X_final_train.shape[1] != 14:
    raise ValueError("Final model must contain exactly 14 predictors.")

if len(X_final_train) != len(y_final_train):
    raise ValueError("Training feature and target lengths differ.")

if X_final_train.isna().any().any():
    raise ValueError("Missing values detected in training predictors.")

if not np.isfinite(y_final_train.to_numpy(dtype=float)).all():
    raise ValueError("Invalid training target values.")

print("=" * 70)
print("CELL 25 — FINAL LINEAR REGRESSION MODEL")
print("=" * 70)

print("Training observations:", len(X_final_train))
print("Selected predictors:", X_final_train.shape[1])


# ------------------------------------------------------------
# 2. Identify numerical and categorical predictors
# ------------------------------------------------------------

categorical_features = [
    "forecast_traffic_level",
    "forecast_weather_condition"
]

categorical_features = [
    f for f in categorical_features
    if f in X_final_train.columns
]

numeric_features = [
    f for f in X_final_train.columns
    if f not in categorical_features
]

print("\nNumerical predictors:", len(numeric_features))
print("Categorical predictors:", len(categorical_features))


# ------------------------------------------------------------
# 3. Define preprocessing pipeline
# ------------------------------------------------------------

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_final = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features)
    ],
    remainder="drop"
)


# ------------------------------------------------------------
# 4. Define the final Linear Regression pipeline
# ------------------------------------------------------------

final_lr_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor_final),
    ("regressor", LinearRegression())
])


# ------------------------------------------------------------
# 5. Five-fold cross-validation and OOF predictions
# ------------------------------------------------------------

cv_final = KFold(
    n_splits=5,
    shuffle=True,
    random_state=2026
)

y_train_array = y_final_train.to_numpy(dtype=float)

oof_predictions_final = np.full(
    len(X_final_train),
    np.nan
)

fold_records = []

print("\n" + "=" * 70)
print("FIVE-FOLD CROSS-VALIDATION")
print("=" * 70)

for fold, (train_idx, valid_idx) in enumerate(
    cv_final.split(X_final_train),
    start=1
):

    X_fold_train = X_final_train.iloc[train_idx]
    X_fold_valid = X_final_train.iloc[valid_idx]

    y_fold_train = y_train_array[train_idx]
    y_fold_valid = y_train_array[valid_idx]

    fold_model = clone(final_lr_pipeline)

    fold_model.fit(
        X_fold_train,
        y_fold_train
    )

    raw_predictions = fold_model.predict(
        X_fold_valid
    )

    constrained_predictions_fold = np.maximum(
        raw_predictions,
        0
    )

    # Save out-of-fold predictions
    oof_predictions_final[valid_idx] = raw_predictions

    # Raw performance
    raw_rmse = np.sqrt(
        mean_squared_error(
            y_fold_valid,
            raw_predictions
        )
    )

    raw_mae = mean_absolute_error(
        y_fold_valid,
        raw_predictions
    )

    raw_r2 = r2_score(
        y_fold_valid,
        raw_predictions
    )

    # Nonnegative-constrained performance
    constrained_rmse = np.sqrt(
        mean_squared_error(
            y_fold_valid,
            constrained_predictions_fold
        )
    )

    constrained_mae = mean_absolute_error(
        y_fold_valid,
        constrained_predictions_fold
    )

    constrained_r2 = r2_score(
        y_fold_valid,
        constrained_predictions_fold
    )

    fold_records.append({

        "Fold": fold,

        "Raw_RMSE": raw_rmse,
        "Raw_MAE": raw_mae,
        "Raw_R2": raw_r2,

        "Constrained_RMSE": constrained_rmse,
        "Constrained_MAE": constrained_mae,
        "Constrained_R2": constrained_r2
    })

    print(
        f"Fold {fold}/5 | "
        f"RMSE: {raw_rmse:.4f} | "
        f"MAE: {raw_mae:.4f} | "
        f"R²: {raw_r2:.4f}"
    )


# ------------------------------------------------------------
# 6. Validate OOF predictions
# ------------------------------------------------------------

if np.isnan(oof_predictions_final).any():
    raise ValueError(
        "Some training observations do not have OOF predictions."
    )

oof_predictions_constrained_final = np.maximum(
    oof_predictions_final,
    0
)


# ------------------------------------------------------------
# 7. Summarize cross-validation results
# ------------------------------------------------------------

final_lr_cv_results_df = pd.DataFrame(fold_records)

metric_columns = [
    "Raw_RMSE",
    "Raw_MAE",
    "Raw_R2",
    "Constrained_RMSE",
    "Constrained_MAE",
    "Constrained_R2"
]

summary_records = []

for metric in metric_columns:

    summary_records.append({

        "Metric": metric,

        "Mean": final_lr_cv_results_df[metric].mean(),

        "SD": final_lr_cv_results_df[metric].std(ddof=1)
    })


final_lr_cv_summary_df = pd.DataFrame(summary_records)

print("\n" + "=" * 70)
print("FINAL LINEAR REGRESSION — CROSS-VALIDATION SUMMARY")
print("=" * 70)

print(
    final_lr_cv_summary_df.to_string(
        index=False,
        formatters={
            "Mean": "{:.4f}".format,
            "SD": "{:.4f}".format
        }
    )
)


# ------------------------------------------------------------
# 8. Calculate pooled OOF performance
# ------------------------------------------------------------

pooled_oof_metrics = {

    "Raw_OOF_RMSE": np.sqrt(
        mean_squared_error(
            y_train_array,
            oof_predictions_final
        )
    ),

    "Raw_OOF_MAE": mean_absolute_error(
        y_train_array,
        oof_predictions_final
    ),

    "Raw_OOF_R2": r2_score(
        y_train_array,
        oof_predictions_final
    ),

    "Constrained_OOF_RMSE": np.sqrt(
        mean_squared_error(
            y_train_array,
            oof_predictions_constrained_final
        )
    ),

    "Constrained_OOF_MAE": mean_absolute_error(
        y_train_array,
        oof_predictions_constrained_final
    ),

    "Constrained_OOF_R2": r2_score(
        y_train_array,
        oof_predictions_constrained_final
    )
}

final_lr_oof_summary_df = pd.DataFrame([
    {
        "Metric": metric,
        "Value": value
    }
    for metric, value in pooled_oof_metrics.items()
])

print("\n" + "=" * 70)
print("POOLED OUT-OF-FOLD PERFORMANCE")
print("=" * 70)

print(
    final_lr_oof_summary_df.to_string(
        index=False,
        formatters={"Value": "{:.4f}".format}
    )
)


# ------------------------------------------------------------
# 9. Fit final model on all training observations
# ------------------------------------------------------------

final_lr_model = clone(final_lr_pipeline)

final_lr_model.fit(
    X_final_train,
    y_final_train
)

# Store the fitted preprocessing and regression components
final_lr_preprocessor = (
    final_lr_model.named_steps["preprocessor"]
)

final_lr_regressor = (
    final_lr_model.named_steps["regressor"]
)

final_lr_transformed_feature_names = (
    final_lr_preprocessor.get_feature_names_out()
)

final_lr_coefficients = (
    final_lr_regressor.coef_
)


# ------------------------------------------------------------
# 10. Create coefficient audit table
# ------------------------------------------------------------

final_lr_coefficients_df = pd.DataFrame({

    "Transformed_Feature":
        final_lr_transformed_feature_names,

    "Coefficient":
        final_lr_coefficients
})

print("\n" + "=" * 70)
print("FINAL LINEAR REGRESSION MODEL")
print("=" * 70)

print("Intercept:", round(final_lr_regressor.intercept_, 6))

print(
    "Transformed predictors:",
    len(final_lr_transformed_feature_names)
)

print("\nModel successfully fitted using all 1,600 training orders.")


# ------------------------------------------------------------
# 11. Save OOF prediction audit
# ------------------------------------------------------------

final_lr_oof_predictions_df = pd.DataFrame({

    "order_id": train_order_ids_final,

    "actual_lateness_h": y_train_array,

    "oof_prediction_raw_h": oof_predictions_final,

    "oof_prediction_constrained_h":
        oof_predictions_constrained_final

})

final_lr_oof_predictions_df["absolute_error_h"] = (
    np.abs(
        final_lr_oof_predictions_df[
            "oof_prediction_constrained_h"
        ]
        - final_lr_oof_predictions_df["actual_lateness_h"]
    )
)


# ------------------------------------------------------------
# 12. Completion summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CELL 25 COMPLETED")
print("=" * 70)

print("Final model: Linear Regression")
print("Feature specification: K")
print("Number of selected predictors:", X_final_train.shape[1])

print("\nGenerated outputs:")
print("- final_lr_model")
print("- final_lr_cv_results_df")
print("- final_lr_cv_summary_df")
print("- final_lr_oof_summary_df")
print("- final_lr_coefficients_df")
print("- final_lr_oof_predictions_df")

print("\nThe 400 test observations were NOT used.")


CELL 25 — FINAL LINEAR REGRESSION MODEL
Training observations: 1600
Selected predictors: 14

Numerical predictors: 12
Categorical predictors: 2

FIVE-FOLD CROSS-VALIDATION
Fold 1/5 | RMSE: 3.6990 | MAE: 2.8524 | R²: 0.6908
Fold 2/5 | RMSE: 3.6920 | MAE: 2.9290 | R²: 0.6752
Fold 3/5 | RMSE: 3.5725 | MAE: 2.9102 | R²: 0.6049
Fold 4/5 | RMSE: 3.9282 | MAE: 2.8903 | R²: 0.7265
Fold 5/5 | RMSE: 3.7994 | MAE: 2.9841 | R²: 0.6755

FINAL LINEAR REGRESSION — CROSS-VALIDATION SUMMARY
          Metric   Mean     SD
        Raw_RMSE 3.7382 0.1332
         Raw_MAE 2.9132 0.0487
          Raw_R2 0.6746 0.0442
Constrained_RMSE 3.2338 0.2040
 Constrained_MAE 2.1372 0.0704
  Constrained_R2 0.7582 0.0186

POOLED OUT-OF-FOLD PERFORMANCE
              Metric  Value
        Raw_OOF_RMSE 3.7401
         Raw_OOF_MAE 2.9132
          Raw_OOF_R2 0.6822
Constrained_OOF_RMSE 3.2389
 Constrained_OOF_MAE 2.1372
  Constrained_OOF_R2 0.7616

FINAL LINEAR REGRESSION MODEL
Intercept: 3.304568
Transformed predictors: 1

In [44]:

# ============================================================
# CELL 27 — FINAL XGBOOST NONLINEAR BENCHMARK
# Specification K — Same 14 predictors as Linear Regression
# ============================================================

import numpy as np
import pandas as pd

from scipy.stats import randint, uniform, loguniform

from sklearn.base import clone
from sklearn.model_selection import KFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline

from xgboost import XGBRegressor


# ------------------------------------------------------------
# 1. Validate final datasets
# ------------------------------------------------------------

if X_final_train.shape[1] != 14:
    raise ValueError("XGBoost must use the same 14 predictors.")

if list(X_final_train.columns) != list(X_final_test.columns):
    raise ValueError("Training and test predictors are inconsistent.")

if X_final_train.isna().any().any():
    raise ValueError("Missing values detected in training data.")

if y_final_train.isna().any():
    raise ValueError("Missing target values detected.")


print("=" * 70)
print("CELL 27 — FINAL XGBOOST BENCHMARK")
print("=" * 70)

print("Training observations:", len(X_final_train))
print("Selected predictors:", X_final_train.shape[1])


# ------------------------------------------------------------
# 2. Define XGBoost pipeline
# ------------------------------------------------------------

final_xgb_pipeline = Pipeline(steps=[

    ("preprocessor", clone(preprocessor_final)),

    ("regressor", XGBRegressor(
        objective="reg:squarederror",
        eval_metric="rmse",
        tree_method="hist",
        random_state=2026,
        n_jobs=1,
        verbosity=0
    ))
])


# ------------------------------------------------------------
# 3. Define hyperparameter search space
# ------------------------------------------------------------

final_xgb_param_distributions = {

    "regressor__n_estimators": randint(250, 701),

    "regressor__max_depth": randint(2, 6),

    "regressor__learning_rate": loguniform(0.025, 0.12),

    "regressor__subsample": uniform(0.70, 0.30),

    "regressor__colsample_bytree": uniform(0.70, 0.30),

    "regressor__min_child_weight": [1, 3, 5],

    "regressor__gamma": [0.0, 0.05, 0.1, 0.2],

    "regressor__reg_alpha": loguniform(1e-4, 0.1),

    "regressor__reg_lambda": loguniform(0.5, 8.0)
}


# ------------------------------------------------------------
# 4. Define five-fold cross-validation
# ------------------------------------------------------------

final_xgb_cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=2026
)

final_xgb_scoring = {

    "RMSE": "neg_root_mean_squared_error",

    "MAE": "neg_mean_absolute_error",

    "R2": "r2"
}


# ------------------------------------------------------------
# 5. Randomized hyperparameter optimization
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("XGBOOST HYPERPARAMETER OPTIMIZATION")
print("=" * 70)

final_xgb_search = RandomizedSearchCV(

    estimator=final_xgb_pipeline,

    param_distributions=final_xgb_param_distributions,

    n_iter=15,

    scoring=final_xgb_scoring,

    refit="RMSE",

    cv=final_xgb_cv,

    random_state=2026,

    n_jobs=1,

    verbose=1,

    return_train_score=False,

    error_score="raise"
)

final_xgb_search.fit(
    X_final_train,
    y_final_train
)


# ------------------------------------------------------------
# 6. Retrieve best XGBoost model
# ------------------------------------------------------------

final_xgb_model = final_xgb_search.best_estimator_

best_index = final_xgb_search.best_index_

print("\nBest XGBoost parameters:")

for parameter, value in final_xgb_search.best_params_.items():
    print(f"{parameter}: {value}")


# ------------------------------------------------------------
# 7. Extract best cross-validation performance
# ------------------------------------------------------------

final_xgb_cv_summary_df = pd.DataFrame([{

    "Model": "Final XGBoost",

    "Number_of_features": X_final_train.shape[1],

    "CV_RMSE": -final_xgb_search.cv_results_[
        "mean_test_RMSE"
    ][best_index],

    "CV_RMSE_SD": final_xgb_search.cv_results_[
        "std_test_RMSE"
    ][best_index],

    "CV_MAE": -final_xgb_search.cv_results_[
        "mean_test_MAE"
    ][best_index],

    "CV_MAE_SD": final_xgb_search.cv_results_[
        "std_test_MAE"
    ][best_index],

    "CV_R2": final_xgb_search.cv_results_[
        "mean_test_R2"
    ][best_index],

    "CV_R2_SD": final_xgb_search.cv_results_[
        "std_test_R2"
    ][best_index]

}])


# ------------------------------------------------------------
# 8. Compare final LR and XGBoost cross-validation
# ------------------------------------------------------------

lr_cv_row = final_lr_cv_results_df

comparison_records = [{

    "Model": "Final Linear Regression",

    "Number_of_features": 14,

    "CV_RMSE": lr_cv_row["Raw_RMSE"].mean(),

    "CV_RMSE_SD": lr_cv_row["Raw_RMSE"].std(ddof=1),

    "CV_MAE": lr_cv_row["Raw_MAE"].mean(),

    "CV_MAE_SD": lr_cv_row["Raw_MAE"].std(ddof=1),

    "CV_R2": lr_cv_row["Raw_R2"].mean(),

    "CV_R2_SD": lr_cv_row["Raw_R2"].std(ddof=1)

},

{

    "Model": "Final XGBoost",

    "Number_of_features": 14,

    "CV_RMSE": final_xgb_cv_summary_df[
        "CV_RMSE"
    ].iloc[0],

    "CV_RMSE_SD": final_xgb_cv_summary_df[
        "CV_RMSE_SD"
    ].iloc[0],

    "CV_MAE": final_xgb_cv_summary_df[
        "CV_MAE"
    ].iloc[0],

    "CV_MAE_SD": final_xgb_cv_summary_df[
        "CV_MAE_SD"
    ].iloc[0],

    "CV_R2": final_xgb_cv_summary_df[
        "CV_R2"
    ].iloc[0],

    "CV_R2_SD": final_xgb_cv_summary_df[
        "CV_R2_SD"
    ].iloc[0]
}]


final_model_cv_comparison_df = pd.DataFrame(
    comparison_records
).sort_values(
    "CV_RMSE"
).reset_index(drop=True)


# ------------------------------------------------------------
# 9. Report final comparison
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL LR vs XGBOOST — FIVE-FOLD CV")
print("=" * 70)

print(
    final_model_cv_comparison_df.to_string(
        index=False,
        formatters={
            col: "{:.4f}".format
            for col in [
                "CV_RMSE",
                "CV_RMSE_SD",
                "CV_MAE",
                "CV_MAE_SD",
                "CV_R2",
                "CV_R2_SD"
            ]
        }
    )
)


# ------------------------------------------------------------
# 10. Save the hyperparameter search results
# ------------------------------------------------------------

final_xgb_search_results_df = pd.DataFrame(
    final_xgb_search.cv_results_
)

print("\n" + "=" * 70)
print("CELL 27 COMPLETED")
print("=" * 70)

print("Final XGBoost model fitted on all training observations.")

print("\nGenerated outputs:")
print("- final_xgb_model")
print("- final_xgb_search")
print("- final_xgb_cv_summary_df")
print("- final_xgb_search_results_df")
print("- final_model_cv_comparison_df")

print("\nThe 400 test observations were NOT used.")


CELL 27 — FINAL XGBOOST BENCHMARK
Training observations: 1600
Selected predictors: 14

XGBOOST HYPERPARAMETER OPTIMIZATION
Fitting 5 folds for each of 15 candidates, totalling 75 fits

Best XGBoost parameters:
regressor__colsample_bytree: 0.9496857161337158
regressor__gamma: 0.0
regressor__learning_rate: 0.06035667363696704
regressor__max_depth: 4
regressor__min_child_weight: 3
regressor__n_estimators: 366
regressor__reg_alpha: 0.0008082529129356661
regressor__reg_lambda: 0.6234943644699736
regressor__subsample: 0.7054200897844257

FINAL LR vs XGBOOST — FIVE-FOLD CV
                  Model  Number_of_features CV_RMSE CV_RMSE_SD CV_MAE CV_MAE_SD  CV_R2 CV_R2_SD
          Final XGBoost                  14  0.9863     0.0958 0.5234    0.0305 0.9777   0.0012
Final Linear Regression                  14  3.7382     0.1332 2.9132    0.0487 0.6746   0.0442

CELL 27 COMPLETED
Final XGBoost model fitted on all training observations.

Generated outputs:
- final_xgb_model
- final_xgb_search
- fina

In [ ]:

# ============================================================
# CELL 27A — NESTED CROSS-VALIDATION
# Final Linear Regression vs. Final XGBoost
# ============================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, RandomizedSearchCV
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

# ------------------------------------------------------------
# 1. Validate the required objects
# ------------------------------------------------------------

required_objects = [
    "final_xgb_pipeline",
    "final_xgb_param_distributions",
    "X_final_train",
    "y_final_train",
    "oof_predictions_final"
]

for obj in required_objects:
    if obj not in globals():
        raise ValueError(f"Required object missing: {obj}")

if len(oof_predictions_final) != len(X_final_train):
    raise ValueError("LR OOF predictions are inconsistent with training data.")

y_nested = y_final_train.to_numpy(dtype=float)

if not np.isfinite(y_nested).all():
    raise ValueError("Invalid training target values.")

if (y_nested < 0).any():
    raise ValueError("Negative lateness values detected.")

print("=" * 70)
print("CELL 27A — NESTED CROSS-VALIDATION")
print("=" * 70)


# ------------------------------------------------------------
# 2. Define outer and inner cross-validation
# ------------------------------------------------------------

outer_cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=2026
)

NESTED_SEARCH_ITERATIONS = 10

nested_xgb_oof_predictions = np.full(
    len(X_final_train),
    np.nan
)

nested_fold_records = []
nested_parameters_records = []


# ------------------------------------------------------------
# 3. Nested XGBoost training
# ------------------------------------------------------------

print("\nGenerating nested XGBoost predictions...")

for fold, (outer_train_idx, outer_valid_idx) in enumerate(
    outer_cv.split(X_final_train),
    start=1
):

    print(f"\nOuter fold {fold}/5")

    X_outer_train = X_final_train.iloc[outer_train_idx]
    X_outer_valid = X_final_train.iloc[outer_valid_idx]

    y_outer_train = y_nested[outer_train_idx]
    y_outer_valid = y_nested[outer_valid_idx]

    # Hyperparameter tuning occurs only within this
    # outer training subset.

    inner_cv = KFold(
        n_splits=3,
        shuffle=True,
        random_state=2026 + fold
    )

    inner_search = RandomizedSearchCV(

        estimator=final_xgb_pipeline,

        param_distributions=final_xgb_param_distributions,

        n_iter=NESTED_SEARCH_ITERATIONS,

        scoring="neg_root_mean_squared_error",

        cv=inner_cv,

        random_state=2026 + fold,

        n_jobs=1,

        verbose=0,

        refit=True,

        error_score="raise"
    )

    inner_search.fit(
        X_outer_train,
        y_outer_train
    )

    # Predict only the outer validation observations.
    fold_predictions = inner_search.predict(
        X_outer_valid
    )

    nested_xgb_oof_predictions[outer_valid_idx] = (
        fold_predictions
    )

    # Fold-level performance
    fold_rmse = np.sqrt(
        mean_squared_error(
            y_outer_valid,
            fold_predictions
        )
    )

    fold_mae = mean_absolute_error(
        y_outer_valid,
        fold_predictions
    )

    fold_r2 = r2_score(
        y_outer_valid,
        fold_predictions
    )

    nested_fold_records.append({

        "Fold": fold,

        "Training_observations": len(outer_train_idx),

        "Validation_observations": len(outer_valid_idx),

        "XGBoost_RMSE": fold_rmse,

        "XGBoost_MAE": fold_mae,

        "XGBoost_R2": fold_r2,

        "Best_inner_CV_RMSE": -inner_search.best_score_
    })

    nested_parameters_records.append({

        "Fold": fold,

        "Best_parameters": inner_search.best_params_

    })

    print(
        f"Nested RMSE: {fold_rmse:.4f} | "
        f"MAE: {fold_mae:.4f} | "
        f"R²: {fold_r2:.4f}"
    )


# ------------------------------------------------------------
# 4. Validate all nested predictions
# ------------------------------------------------------------

if np.isnan(nested_xgb_oof_predictions).any():
    raise ValueError(
        "Some observations do not have nested XGBoost predictions."
    )

if not np.isfinite(nested_xgb_oof_predictions).all():
    raise ValueError("Non-finite nested XGBoost predictions detected.")


# ------------------------------------------------------------
# 5. Calculate pooled raw and constrained metrics
# ------------------------------------------------------------

def evaluate_prediction_set(actual, predicted):

    constrained = np.maximum(predicted, 0)

    return {

        "Raw_RMSE": np.sqrt(
            mean_squared_error(actual, predicted)
        ),

        "Raw_MAE": mean_absolute_error(
            actual,
            predicted
        ),

        "Raw_R2": r2_score(
            actual,
            predicted
        ),

        "Constrained_RMSE": np.sqrt(
            mean_squared_error(actual, constrained)
        ),

        "Constrained_MAE": mean_absolute_error(
            actual,
            constrained
        ),

        "Constrained_R2": r2_score(
            actual,
            constrained
        ),

        "Negative_Raw_Predictions": int(
            (predicted < 0).sum()
        )
    }


lr_nested_metrics = evaluate_prediction_set(
    y_nested,
    np.asarray(oof_predictions_final)
)

xgb_nested_metrics = evaluate_prediction_set(
    y_nested,
    nested_xgb_oof_predictions
)


nested_comparison_df = pd.DataFrame([

    {
        "Model": "Linear Regression",
        **lr_nested_metrics
    },

    {
        "Model": "XGBoost",
        **xgb_nested_metrics
    }

])


# ------------------------------------------------------------
# 6. Display pooled nested performance
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("POOLED NESTED OOF PERFORMANCE")
print("=" * 70)

print(
    nested_comparison_df.to_string(
        index=False,
        formatters={
            col: "{:.4f}".format
            for col in [
                "Raw_RMSE",
                "Raw_MAE",
                "Raw_R2",
                "Constrained_RMSE",
                "Constrained_MAE",
                "Constrained_R2"
            ]
        }
    )
)


# ------------------------------------------------------------
# 7. Compare fold-level performance
# ------------------------------------------------------------

nested_xgb_fold_df = pd.DataFrame(
    nested_fold_records
)

lr_fold_metrics = final_lr_cv_results_df[
    [
        "Fold",
        "Raw_RMSE",
        "Raw_MAE",
        "Raw_R2"
    ]
].copy()

lr_fold_metrics.columns = [
    "Fold",
    "LR_RMSE",
    "LR_MAE",
    "LR_R2"
]

nested_fold_comparison_df = nested_xgb_fold_df.merge(
    lr_fold_metrics,
    on="Fold",
    how="inner"
)

nested_fold_comparison_df["RMSE_difference_XGB_minus_LR"] = (
    nested_fold_comparison_df["XGBoost_RMSE"]
    - nested_fold_comparison_df["LR_RMSE"]
)

print("\n" + "=" * 70)
print("FOLD-LEVEL COMPARISON")
print("=" * 70)

print(
    nested_fold_comparison_df[
        [
            "Fold",
            "LR_RMSE",
            "XGBoost_RMSE",
            "LR_MAE",
            "XGBoost_MAE",
            "RMSE_difference_XGB_minus_LR"
        ]
    ].to_string(
        index=False,
        formatters={
            col: "{:.4f}".format
            for col in [
                "LR_RMSE",
                "XGBoost_RMSE",
                "LR_MAE",
                "XGBoost_MAE",
                "RMSE_difference_XGB_minus_LR"
            ]
        }
    )
)


# ------------------------------------------------------------
# 8. Calculate relative RMSE improvement
# ------------------------------------------------------------

lr_rmse = lr_nested_metrics["Raw_RMSE"]
xgb_rmse = xgb_nested_metrics["Raw_RMSE"]

rmse_improvement_pct = (
    (lr_rmse - xgb_rmse) / lr_rmse
) * 100

lr_mae = lr_nested_metrics["Raw_MAE"]
xgb_mae = xgb_nested_metrics["Raw_MAE"]

mae_improvement_pct = (
    (lr_mae - xgb_mae) / lr_mae
) * 100

print("\n" + "=" * 70)
print("NESTED XGBOOST IMPROVEMENT")
print("=" * 70)

print(f"Relative RMSE improvement: {rmse_improvement_pct:.2f}%")
print(f"Relative MAE improvement:  {mae_improvement_pct:.2f}%")


# ------------------------------------------------------------
# 9. Save order-level nested predictions
# ------------------------------------------------------------

nested_model_predictions_df = pd.DataFrame({

    "order_id": train_order_ids_final,

    "actual_lateness_h": y_nested,

    "LR_OOF_raw_h": oof_predictions_final,

    "XGBoost_nested_OOF_raw_h": nested_xgb_oof_predictions,

    "LR_OOF_constrained_h": np.maximum(
        oof_predictions_final,
        0
    ),

    "XGBoost_nested_OOF_constrained_h": np.maximum(
        nested_xgb_oof_predictions,
        0
    )

})


# ------------------------------------------------------------
# 10. Completion summary
# ------------------------------------------------------------

nested_parameters_df = pd.DataFrame(
    nested_parameters_records
)

print("\n" + "=" * 70)
print("CELL 27A COMPLETED")
print("=" * 70)

print("Generated outputs:")
print("- nested_xgb_oof_predictions")
print("- nested_comparison_df")
print("- nested_xgb_fold_df")
print("- nested_fold_comparison_df")
print("- nested_parameters_df")
print("- nested_model_predictions_df")

print("\nAll XGBoost tuning occurred within outer training folds.")
print("The 400 test observations were NOT used.")


In [ ]:

# ============================================================
# CELL 27B — FINAL HOLDOUT EVALUATION
# Specification K: Linear Regression vs XGBoost
# ============================================================

import numpy as np
import pandas as pd

from scipy.stats import wilcoxon

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score
)


# ------------------------------------------------------------
# 1. Validate models and holdout data
# ------------------------------------------------------------

if len(X_final_test) != len(y_final_test):
    raise ValueError("Test feature and target lengths differ.")

if X_final_test.shape[1] != 14:
    raise ValueError("Final evaluation requires 14 predictors.")

if X_final_test.isna().any().any():
    raise ValueError("Missing values detected in test predictors.")

y_actual = y_final_test.to_numpy(dtype=float)

if not np.isfinite(y_actual).all() or (y_actual < 0).any():
    raise ValueError("Invalid test target values.")

print("=" * 70)
print("CELL 27B — FINAL HOLDOUT EVALUATION")
print("=" * 70)

print("Test observations:", len(X_final_test))
print("Selected predictors:", X_final_test.shape[1])


# ------------------------------------------------------------
# 2. Generate predictions
# ------------------------------------------------------------

lr_raw = final_lr_model.predict(X_final_test)

xgb_raw = final_xgb_model.predict(X_final_test)

if not (
    np.isfinite(lr_raw).all()
    and np.isfinite(xgb_raw).all()
):
    raise ValueError("Non-finite predictions detected.")

lr_constrained = np.maximum(lr_raw, 0)

xgb_constrained = np.maximum(xgb_raw, 0)


# ------------------------------------------------------------
# 3. Calculate regression metrics
# ------------------------------------------------------------

def regression_metrics(actual, predicted):

    return {

        "RMSE_h": np.sqrt(
            mean_squared_error(actual, predicted)
        ),

        "MAE_h": mean_absolute_error(
            actual, predicted
        ),

        "R2": r2_score(
            actual, predicted
        ),

        "Mean_Error_h": np.mean(
            predicted - actual
        )
    }


performance_records = []

for name, raw, constrained in [
    ("Linear Regression", lr_raw, lr_constrained),
    ("XGBoost", xgb_raw, xgb_constrained)
]:

    raw_metrics = regression_metrics(
        y_actual,
        raw
    )

    constrained_metrics = regression_metrics(
        y_actual,
        constrained
    )

    performance_records.append({

        "Model": name,

        "Raw_RMSE_h": raw_metrics["RMSE_h"],
        "Raw_MAE_h": raw_metrics["MAE_h"],
        "Raw_R2": raw_metrics["R2"],

        "Constrained_RMSE_h": constrained_metrics["RMSE_h"],
        "Constrained_MAE_h": constrained_metrics["MAE_h"],
        "Constrained_R2": constrained_metrics["R2"],

        "Constrained_Mean_Error_h": (
            constrained_metrics["Mean_Error_h"]
        ),

        "Negative_Raw_Predictions": int(
            (raw < 0).sum()
        )
    })


final_holdout_performance_df = pd.DataFrame(
    performance_records
)


# ------------------------------------------------------------
# 4. Calculate performance on actually late orders
# ------------------------------------------------------------

late_mask = y_actual > 0

late_performance_records = []

for name, predictions in [
    ("Linear Regression", lr_constrained),
    ("XGBoost", xgb_constrained)
]:

    late_performance_records.append({

        "Model": name,

        "Late_Orders": int(late_mask.sum()),

        "RMSE_h": np.sqrt(
            mean_squared_error(
                y_actual[late_mask],
                predictions[late_mask]
            )
        ),

        "MAE_h": mean_absolute_error(
            y_actual[late_mask],
            predictions[late_mask]
        ),

        "Mean_Actual_Lateness_h": y_actual[late_mask].mean(),

        "Mean_Predicted_Lateness_h": (
            predictions[late_mask].mean()
        ),

        "Mean_Error_h": (
            predictions[late_mask] - y_actual[late_mask]
        ).mean()
    })


final_late_performance_df = pd.DataFrame(
    late_performance_records
)


# ------------------------------------------------------------
# 5. Classification diagnostics
# ------------------------------------------------------------

y_actual_class = (y_actual > 0).astype(int)

classification_records = []

for name, predictions in [
    ("Linear Regression", lr_constrained),
    ("XGBoost", xgb_constrained)
]:

    predicted_class = (predictions > 0).astype(int)

    classification_records.append({

        "Model": name,

        "Accuracy": accuracy_score(
            y_actual_class,
            predicted_class
        ),

        "Balanced_Accuracy": balanced_accuracy_score(
            y_actual_class,
            predicted_class
        ),

        "Precision": precision_score(
            y_actual_class,
            predicted_class,
            zero_division=0
        ),

        "Recall": recall_score(
            y_actual_class,
            predicted_class,
            zero_division=0
        ),

        "F1": f1_score(
            y_actual_class,
            predicted_class,
            zero_division=0
        )
    })


final_classification_performance_df = pd.DataFrame(
    classification_records
)


# ------------------------------------------------------------
# 6. Paired Wilcoxon comparison of absolute errors
# ------------------------------------------------------------

lr_absolute_errors = np.abs(
    lr_constrained - y_actual
)

xgb_absolute_errors = np.abs(
    xgb_constrained - y_actual
)

wilcoxon_result = wilcoxon(
    lr_absolute_errors,
    xgb_absolute_errors,
    alternative="two-sided",
    zero_method="wilcox",
    method="auto"
)

holdout_wilcoxon_df = pd.DataFrame([{

    "Comparison": "Linear Regression vs XGBoost",

    "Wilcoxon_statistic": wilcoxon_result.statistic,

    "P_value": wilcoxon_result.pvalue,

    "Mean_absolute_error_LR": lr_absolute_errors.mean(),

    "Mean_absolute_error_XGBoost": xgb_absolute_errors.mean()
}])


# ------------------------------------------------------------
# 7. Save order-level predictions
# ------------------------------------------------------------

final_holdout_predictions_df = pd.DataFrame({

    "order_id": test_order_ids_final,

    "actual_lateness_h": y_actual,

    "LR_raw_prediction_h": lr_raw,

    "LR_constrained_prediction_h": lr_constrained,

    "XGBoost_raw_prediction_h": xgb_raw,

    "XGBoost_constrained_prediction_h": xgb_constrained

})

final_holdout_predictions_df["XGBoost_absolute_error_h"] = (
    np.abs(xgb_constrained - y_actual)
)

final_holdout_predictions_df["LR_absolute_error_h"] = (
    np.abs(lr_constrained - y_actual)
)


# ------------------------------------------------------------
# 8. Display results
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL HOLDOUT REGRESSION PERFORMANCE")
print("=" * 70)

print(
    final_holdout_performance_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


print("\n" + "=" * 70)
print("PERFORMANCE ON ACTUALLY LATE ORDERS")
print("=" * 70)

print(
    final_late_performance_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


print("\n" + "=" * 70)
print("ON-TIME / LATE CLASSIFICATION PERFORMANCE")
print("=" * 70)

print(
    final_classification_performance_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


print("\n" + "=" * 70)
print("PAIRED WILCOXON COMPARISON")
print("=" * 70)

print(
    holdout_wilcoxon_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# ------------------------------------------------------------
# 9. Completion summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CELL 27B COMPLETED")
print("=" * 70)

print("Generated outputs:")
print("- final_holdout_performance_df")
print("- final_late_performance_df")
print("- final_classification_performance_df")
print("- holdout_wilcoxon_df")
print("- final_holdout_predictions_df")

print("\nBoth models use the same 14 predictors.")
print("No test observations were used for model fitting.")


## 5. SHAP-based model explainability

SHAP values are calculated for the 400-observation holdout set using a 100-observation background sample drawn from the training set. Transformed one-hot contributions are aggregated back to the 14 original predictors. Additivity is checked against raw XGBoost predictions. Because remaining slack is algebraically linked to the target, its dominant contribution must not be interpreted as a causal effect or as evidence of independent information.

In [ ]:

# ============================================================
# CELL 28 — FINAL XGBOOST SHAP EXPLAINABILITY
# Primary predictive model — Specification K
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

from pathlib import Path
from scipy import sparse


# ------------------------------------------------------------
# 1. Validate model and datasets
# ------------------------------------------------------------

if "final_xgb_model" not in globals():
    raise ValueError("Run Cell 27 before executing Cell 28.")

if X_final_train.shape[1] != 14:
    raise ValueError("Expected 14 original predictors.")

if X_final_test.shape[1] != 14:
    raise ValueError("Expected 14 original predictors in test data.")

if list(X_final_train.columns) != list(final_feature_columns):
    raise ValueError("Training feature order is inconsistent.")

print("=" * 70)
print("CELL 28 — FINAL XGBOOST SHAP ANALYSIS")
print("=" * 70)


# ------------------------------------------------------------
# 2. Transform training and test observations
# ------------------------------------------------------------

def make_dense_xgb(matrix):

    if sparse.issparse(matrix):
        return matrix.toarray()

    return np.asarray(matrix)


final_xgb_preprocessor = (
    final_xgb_model.named_steps["preprocessor"]
)

final_xgb_regressor = (
    final_xgb_model.named_steps["regressor"]
)

X_train_xgb_shap = make_dense_xgb(
    final_xgb_preprocessor.transform(X_final_train)
)

X_test_xgb_shap = make_dense_xgb(
    final_xgb_preprocessor.transform(X_final_test)
)

xgb_transformed_feature_names = np.asarray(
    final_xgb_preprocessor.get_feature_names_out(),
    dtype=str
)

if X_train_xgb_shap.shape[1] != len(
    xgb_transformed_feature_names
):
    raise ValueError("Training transformed feature dimensions are inconsistent.")

if X_test_xgb_shap.shape[1] != len(
    xgb_transformed_feature_names
):
    raise ValueError("Test transformed feature dimensions are inconsistent.")

print("Original predictors:", len(final_feature_columns))
print("Transformed predictors:", len(xgb_transformed_feature_names))
print("Training observations:", len(X_final_train))
print("Explanation observations:", len(X_final_test))


# ------------------------------------------------------------
# 3. Select SHAP background observations from training data
# ------------------------------------------------------------

SHAP_BACKGROUND_SIZE = min(
    100,
    len(X_train_xgb_shap)
)

rng = np.random.default_rng(2026)

background_indices = rng.choice(
    len(X_train_xgb_shap),
    size=SHAP_BACKGROUND_SIZE,
    replace=False
)

xgb_shap_background = (
    X_train_xgb_shap[background_indices]
)

print("\nSHAP background observations:", SHAP_BACKGROUND_SIZE)


# ------------------------------------------------------------
# 4. Create XGBoost TreeExplainer
# ------------------------------------------------------------

print("\nCalculating XGBoost SHAP values...")

final_xgb_shap_explainer = shap.TreeExplainer(

    final_xgb_regressor,

    data=xgb_shap_background,

    feature_perturbation="interventional",

    model_output="raw"
)

xgb_shap_output = final_xgb_shap_explainer.shap_values(
    X_test_xgb_shap
)

if isinstance(xgb_shap_output, list):
    xgb_shap_output = xgb_shap_output[0]

if hasattr(xgb_shap_output, "values"):
    final_xgb_shap_values = np.asarray(
        xgb_shap_output.values
    )
else:
    final_xgb_shap_values = np.asarray(
        xgb_shap_output
    )

if (
    final_xgb_shap_values.ndim == 3
    and final_xgb_shap_values.shape[-1] == 1
):
    final_xgb_shap_values = (
        final_xgb_shap_values[:, :, 0]
    )

if final_xgb_shap_values.shape != X_test_xgb_shap.shape:
    raise ValueError(
        "Unexpected XGBoost SHAP dimensions: "
        f"{final_xgb_shap_values.shape}"
    )


# ------------------------------------------------------------
# 5. Verify SHAP additivity
# ------------------------------------------------------------

final_xgb_expected_value = float(
    np.asarray(
        final_xgb_shap_explainer.expected_value
    ).reshape(-1)[0]
)

final_xgb_raw_predictions = final_xgb_model.predict(
    X_final_test
)

xgb_reconstructed_predictions = (
    final_xgb_expected_value
    + final_xgb_shap_values.sum(axis=1)
)

xgb_additivity_errors = np.abs(
    final_xgb_raw_predictions
    - xgb_reconstructed_predictions
)

print("\n" + "=" * 70)
print("XGBOOST SHAP ADDITIVITY VERIFICATION")
print("=" * 70)

print(
    f"Expected value: {final_xgb_expected_value:.6f} hours"
)

print(
    "Mean absolute difference:",
    f"{xgb_additivity_errors.mean():.10f}"
)

print(
    "Maximum absolute difference:",
    f"{xgb_additivity_errors.max():.10f}"
)

if xgb_additivity_errors.max() > 1e-3:

    print(
        "WARNING: Additivity difference exceeds the "
        "1e-3 numerical tolerance. Check SHAP/model compatibility."
    )


# ------------------------------------------------------------
# 6. Map transformed predictors to original predictors
# ------------------------------------------------------------

xgb_feature_indices = {}

for original_feature in final_feature_columns:

    matching_indices = []

    numerical_name = f"num__{original_feature}"

    categorical_name = f"cat__{original_feature}"

    categorical_prefix = f"cat__{original_feature}_"

    for idx, transformed_name in enumerate(
        xgb_transformed_feature_names
    ):

        if (
            transformed_name == numerical_name
            or transformed_name == categorical_name
            or transformed_name.startswith(categorical_prefix)
        ):
            matching_indices.append(idx)

    if not matching_indices:
        raise ValueError(
            f"Unable to map transformed feature: {original_feature}"
        )

    xgb_feature_indices[original_feature] = matching_indices


# Confirm every transformed column is mapped exactly once.
mapped_indices = [
    idx
    for indices in xgb_feature_indices.values()
    for idx in indices
]

if sorted(mapped_indices) != list(
    range(len(xgb_transformed_feature_names))
):
    raise ValueError(
        "Transformed features were not mapped exactly once."
    )


# ------------------------------------------------------------
# 7. Aggregate transformed SHAP values to 14 original predictors
# ------------------------------------------------------------

final_xgb_source_shap_df = pd.DataFrame(
    index=np.arange(len(X_final_test))
)

for original_feature in final_feature_columns:

    indices = xgb_feature_indices[original_feature]

    final_xgb_source_shap_df[original_feature] = (
        final_xgb_shap_values[:, indices].sum(axis=1)
    )


# Check aggregated SHAP additivity
aggregated_xgb_shap_sum = (
    final_xgb_source_shap_df.sum(axis=1).to_numpy()
)

aggregated_xgb_predictions = (
    final_xgb_expected_value
    + aggregated_xgb_shap_sum
)

aggregated_xgb_additivity_error = np.abs(
    final_xgb_raw_predictions
    - aggregated_xgb_predictions
)

print("\nOriginal-feature aggregation check:")
print(
    "Maximum difference:",
    f"{aggregated_xgb_additivity_error.max():.12f}"
)


# ------------------------------------------------------------
# 8. Calculate global importance for original predictors
# ------------------------------------------------------------

final_xgb_feature_importance_df = pd.DataFrame({

    "Feature": final_feature_columns,

    "Operational_Group": [

        next(
            group
            for group, features in final_feature_groups.items()
            if feature in features
        )

        for feature in final_feature_columns
    ],

    "Mean_Absolute_SHAP_h": [

        final_xgb_source_shap_df[feature].abs().mean()

        for feature in final_feature_columns
    ],

    "Mean_SHAP_h": [

        final_xgb_source_shap_df[feature].mean()

        for feature in final_feature_columns
    ]

}).sort_values(
    "Mean_Absolute_SHAP_h",
    ascending=False
).reset_index(drop=True)


print("\n" + "=" * 70)
print("FINAL XGBOOST — GLOBAL FEATURE IMPORTANCE")
print("=" * 70)

print(
    final_xgb_feature_importance_df.to_string(
        index=False,
        formatters={
            "Mean_Absolute_SHAP_h": "{:.4f}".format,
            "Mean_SHAP_h": "{:.4f}".format
        }
    )
)


# ------------------------------------------------------------
# 9. Aggregate SHAP contributions by operational group
# ------------------------------------------------------------

final_xgb_grouped_shap_df = pd.DataFrame(
    index=np.arange(len(X_final_test))
)

for group, features in final_feature_groups.items():

    final_xgb_grouped_shap_df[group] = (
        final_xgb_source_shap_df[features].sum(axis=1)
    )


# Calculate group-level contribution
group_records = []

for group, features in final_feature_groups.items():

    group_values = (
        final_xgb_grouped_shap_df[group].to_numpy()
    )

    group_records.append({

        "Operational_Group": group,

        "Number_of_Features": len(features),

        "Mean_Absolute_Group_SHAP_h": np.mean(
            np.abs(group_values)
        ),

        "Mean_Group_SHAP_h": np.mean(
            group_values
        )
    })


final_xgb_group_importance_df = pd.DataFrame(
    group_records
).sort_values(
    "Mean_Absolute_Group_SHAP_h",
    ascending=False
).reset_index(drop=True)


# Relative contribution share
group_total = (
    final_xgb_group_importance_df[
        "Mean_Absolute_Group_SHAP_h"
    ].sum()
)

final_xgb_group_importance_df[
    "Grouped_Contribution_Share_pct"
] = (

    100

    * final_xgb_group_importance_df[
        "Mean_Absolute_Group_SHAP_h"
    ]

    / group_total
)


print("\n" + "=" * 70)
print("FINAL XGBOOST — OPERATIONAL GROUP IMPORTANCE")
print("=" * 70)

print(
    final_xgb_group_importance_df.to_string(
        index=False,
        formatters={
            "Mean_Absolute_Group_SHAP_h": "{:.4f}".format,
            "Mean_Group_SHAP_h": "{:.4f}".format,
            "Grouped_Contribution_Share_pct": "{:.2f}".format
        }
    )
)


# ------------------------------------------------------------
# 10. Verify grouped SHAP additivity
# ------------------------------------------------------------

grouped_xgb_shap_sum = (
    final_xgb_grouped_shap_df.sum(axis=1).to_numpy()
)

grouped_xgb_predictions = (
    final_xgb_expected_value
    + grouped_xgb_shap_sum
)

grouped_xgb_additivity_error = np.abs(
    final_xgb_raw_predictions
    - grouped_xgb_predictions
)

print("\n" + "=" * 70)
print("GROUPED SHAP ADDITIVITY CHECK")
print("=" * 70)

print(
    "Maximum difference:",
    f"{grouped_xgb_additivity_error.max():.12f}"
)


# ------------------------------------------------------------
# 14. Save order-level SHAP audit
# ------------------------------------------------------------

final_xgb_test_shap_audit_df = pd.DataFrame({

    "order_id": test_order_ids_final,

    "actual_lateness_h": y_final_test.to_numpy(),

    "raw_predicted_lateness_h": final_xgb_raw_predictions

})

for feature in final_feature_columns:

    final_xgb_test_shap_audit_df[
        f"SHAP_{feature}"
    ] = final_xgb_source_shap_df[feature].to_numpy()


# ------------------------------------------------------------
# 15. Completion summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CELL 28 COMPLETED")
print("=" * 70)

print("Original predictors explained:", len(final_feature_columns))
print("Test observations explained:", len(X_final_test))
print("SHAP background observations:", SHAP_BACKGROUND_SIZE)

print("\nGenerated outputs:")
print("- final_xgb_shap_explainer")
print("- final_xgb_shap_values")
print("- final_xgb_source_shap_df")
print("- final_xgb_feature_importance_df")
print("- final_xgb_grouped_shap_df")
print("- final_xgb_group_importance_df")
print("- final_xgb_test_shap_audit_df")


print("\nSHAP values explain raw XGBoost predictions.")
print("No model was retrained in this cell.")

## 6. Process-informed operational benchmarks

The deterministic benchmark uses the specified vehicle-availability-to-dispatch-wait relationship and forecast driving time. The distribution-aware benchmark additionally uses the known mean-one lognormal shock distributions and their coefficients of variation (0.30 for dispatch waiting and 0.15 for driving time). This benchmark has privileged access to the assumed data-generating distributions; it is a process-informed, near-oracle comparator under the simulation assumptions, not a learned model or an empirical performance ceiling.

In [ ]:

# ============================================================
# CELL 30 — RULE-BASED DELIVERY-LATENESS BENCHMARK
# Compare rule-based prediction, LR and XGBoost
# ============================================================

import numpy as np
import pandas as pd

from scipy.stats import wilcoxon

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)


# ------------------------------------------------------------
# 1. Validate the required predictors
# ------------------------------------------------------------

required_features = [
    "pre_dispatch_remaining_slack_h",
    "forecast_vehicle_availability",
    "forecast_expected_travel_time_h"
]

for feature in required_features:

    if feature not in X_final_train.columns:
        raise ValueError(f"Missing training predictor: {feature}")

    if feature not in X_final_test.columns:
        raise ValueError(f"Missing test predictor: {feature}")


# ------------------------------------------------------------
# 2. Define the rule-based prediction function
# ------------------------------------------------------------

def rule_based_prediction(X):

    availability = X[
        "forecast_vehicle_availability"
    ].to_numpy(dtype=float)

    expected_driving = X[
        "forecast_expected_travel_time_h"
    ].to_numpy(dtype=float)

    remaining_slack = X[
        "pre_dispatch_remaining_slack_h"
    ].to_numpy(dtype=float)

    if not np.isfinite(availability).all():
        raise ValueError("Invalid vehicle availability values.")

    if not np.isfinite(expected_driving).all():
        raise ValueError("Invalid forecast driving-time values.")

    if not np.isfinite(remaining_slack).all():
        raise ValueError("Invalid remaining-slack values.")

    if ((availability <= 0) | (availability > 1)).any():
        raise ValueError(
            "Vehicle availability must be greater than zero "
            "and no greater than one."
        )

    if (expected_driving < 0).any():
        raise ValueError("Negative expected driving time detected.")

    # Expected dispatch wait under Cell 13R
    estimated_dispatch_wait = (
        8 * (1 - availability) / availability
    )

    # Expected total transportation duration
    estimated_transportation_time = (
        estimated_dispatch_wait + expected_driving
    )

    # Deterministic rule-based lateness prediction
    predicted_lateness = np.maximum(
        0,
        estimated_transportation_time - remaining_slack
    )

    return pd.DataFrame({

        "estimated_dispatch_wait_h": estimated_dispatch_wait,

        "forecast_driving_time_h": expected_driving,

        "estimated_transportation_time_h":
            estimated_transportation_time,

        "remaining_slack_h": remaining_slack,

        "rule_predicted_lateness_h": predicted_lateness
    })


# ------------------------------------------------------------
# 3. Generate rule-based predictions
# ------------------------------------------------------------

rule_train_results_df = rule_based_prediction(
    X_final_train
)

rule_test_results_df = rule_based_prediction(
    X_final_test
)

rule_train_predictions = (
    rule_train_results_df[
        "rule_predicted_lateness_h"
    ].to_numpy()
)

rule_test_predictions = (
    rule_test_results_df[
        "rule_predicted_lateness_h"
    ].to_numpy()
)


# ------------------------------------------------------------
# 4. Prepare nested OOF model predictions
# ------------------------------------------------------------

y_train_rule = y_final_train.to_numpy(dtype=float)

y_test_rule = y_final_test.to_numpy(dtype=float)

prediction_sets = {

    "Training OOF": {

        "Linear Regression": np.maximum(
            np.asarray(oof_predictions_final),
            0
        ),

        "XGBoost": np.maximum(
            np.asarray(nested_xgb_oof_predictions),
            0
        ),

        "Rule-based": rule_train_predictions
    },

    "Holdout": {

        "Linear Regression": np.maximum(
            final_lr_model.predict(X_final_test),
            0
        ),

        "XGBoost": np.maximum(
            final_xgb_model.predict(X_final_test),
            0
        ),

        "Rule-based": rule_test_predictions
    }
}

actual_sets = {
    "Training OOF": y_train_rule,
    "Holdout": y_test_rule
}


# Validate prediction lengths
for dataset_name, predictions in prediction_sets.items():

    actual = actual_sets[dataset_name]

    for model_name, values in predictions.items():

        if len(values) != len(actual):
            raise ValueError(
                f"Prediction length mismatch: {dataset_name}, {model_name}"
            )

        if not np.isfinite(values).all():
            raise ValueError(
                f"Invalid predictions: {dataset_name}, {model_name}"
            )


# ------------------------------------------------------------
# 5. Define evaluation function
# ------------------------------------------------------------

def evaluate_rule_benchmark(actual, predictions):

    actual_class = (actual > 0).astype(int)

    records = []

    for model_name, predicted in predictions.items():

        predicted_class = (predicted > 0).astype(int)

        tn, fp, fn, tp = confusion_matrix(
            actual_class,
            predicted_class,
            labels=[0, 1]
        ).ravel()

        late_mask = actual > 0

        records.append({

            "Model": model_name,

            "RMSE_h": np.sqrt(
                mean_squared_error(actual, predicted)
            ),

            "MAE_h": mean_absolute_error(
                actual,
                predicted
            ),

            "R2": r2_score(
                actual,
                predicted
            ),

            "Late_Order_RMSE_h": np.sqrt(
                mean_squared_error(
                    actual[late_mask],
                    predicted[late_mask]
                )
            ),

            "Late_Order_MAE_h": mean_absolute_error(
                actual[late_mask],
                predicted[late_mask]
            ),

            "Accuracy": accuracy_score(
                actual_class,
                predicted_class
            ),

            "Balanced_Accuracy": balanced_accuracy_score(
                actual_class,
                predicted_class
            ),

            "Precision": precision_score(
                actual_class,
                predicted_class,
                zero_division=0
            ),

            "Recall": recall_score(
                actual_class,
                predicted_class,
                zero_division=0
            ),

            "F1": f1_score(
                actual_class,
                predicted_class,
                zero_division=0
            ),

            "True_Negatives": tn,
            "False_Positives": fp,
            "False_Negatives": fn,
            "True_Positives": tp
        })

    return pd.DataFrame(records)


# ------------------------------------------------------------
# 6. Evaluate training OOF and holdout predictions
# ------------------------------------------------------------

rule_oof_comparison_df = evaluate_rule_benchmark(
    actual_sets["Training OOF"],
    prediction_sets["Training OOF"]
)

rule_holdout_comparison_df = evaluate_rule_benchmark(
    actual_sets["Holdout"],
    prediction_sets["Holdout"]
)


print("\n" + "=" * 70)
print("TRAINING — NESTED OOF COMPARISON")
print("=" * 70)

print(
    rule_oof_comparison_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


print("\n" + "=" * 70)
print("HOLDOUT — RULE-BASED AND ML COMPARISON")
print("=" * 70)

print(
    rule_holdout_comparison_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ------------------------------------------------------------
# 7. Paired Wilcoxon tests on absolute errors
# ------------------------------------------------------------

def calculate_paired_tests(actual, predictions):

    comparisons = [

        ("Linear Regression", "Rule-based"),

        ("XGBoost", "Rule-based"),

        ("XGBoost", "Linear Regression")
    ]

    records = []

    for model_a, model_b in comparisons:

        errors_a = np.abs(
            predictions[model_a] - actual
        )

        errors_b = np.abs(
            predictions[model_b] - actual
        )

        result = wilcoxon(
            errors_a,
            errors_b,
            alternative="two-sided",
            zero_method="wilcox",
            method="auto"
        )

        records.append({

            "Model_A": model_a,

            "Model_B": model_b,

            "Mean_Absolute_Error_A": errors_a.mean(),

            "Mean_Absolute_Error_B": errors_b.mean(),

            "Wilcoxon_statistic": result.statistic,

            "P_value": result.pvalue
        })

    result_df = pd.DataFrame(records)

    # Holm correction
    p_values = result_df["P_value"].to_numpy()

    m = len(p_values)

    order = np.argsort(p_values)

    sorted_p = p_values[order]

    adjusted_sorted = np.maximum.accumulate(
        (m - np.arange(m)) * sorted_p
    )

    adjusted_sorted = np.minimum(
        adjusted_sorted,
        1.0
    )

    adjusted = np.empty(m)

    adjusted[order] = adjusted_sorted

    result_df["Holm_adjusted_p"] = adjusted

    result_df["Significant_at_0.05"] = (
        result_df["Holm_adjusted_p"] < 0.05
    )

    return result_df


rule_oof_wilcoxon_df = calculate_paired_tests(
    actual_sets["Training OOF"],
    prediction_sets["Training OOF"]
)

rule_holdout_wilcoxon_df = calculate_paired_tests(
    actual_sets["Holdout"],
    prediction_sets["Holdout"]
)


print("\n" + "=" * 70)
print("TRAINING OOF — PAIRED WILCOXON TESTS WITH HOLM CORRECTION")
print("=" * 70)

print(
    rule_oof_wilcoxon_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


print("\n" + "=" * 70)
print("HOLDOUT — PAIRED WILCOXON TESTS WITH HOLM CORRECTION")
print("=" * 70)

print(
    rule_holdout_wilcoxon_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# ------------------------------------------------------------
# 8. Calculate rule-based benchmark summary
# ------------------------------------------------------------

rule_summary_df = pd.DataFrame([

    {
        "Dataset": "Training OOF",

        "Mean_Dispatch_Wait_h":
            rule_train_results_df[
                "estimated_dispatch_wait_h"
            ].mean(),

        "Mean_Forecast_Driving_h":
            rule_train_results_df[
                "forecast_driving_time_h"
            ].mean(),

        "Mean_Remaining_Slack_h":
            rule_train_results_df[
                "remaining_slack_h"
            ].mean(),

        "Mean_Predicted_Lateness_h":
            rule_train_predictions.mean(),

        "Predicted_Late_Orders":
            int((rule_train_predictions > 0).sum())
    },

    {
        "Dataset": "Holdout",

        "Mean_Dispatch_Wait_h":
            rule_test_results_df[
                "estimated_dispatch_wait_h"
            ].mean(),

        "Mean_Forecast_Driving_h":
            rule_test_results_df[
                "forecast_driving_time_h"
            ].mean(),

        "Mean_Remaining_Slack_h":
            rule_test_results_df[
                "remaining_slack_h"
            ].mean(),

        "Mean_Predicted_Lateness_h":
            rule_test_predictions.mean(),

        "Predicted_Late_Orders":
            int((rule_test_predictions > 0).sum())
    }

])


print("\n" + "=" * 70)
print("RULE-BASED BENCHMARK CHARACTERISTICS")
print("=" * 70)

print(
    rule_summary_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ------------------------------------------------------------
# 9. Calculate relative XGBoost improvement over the rule
# ------------------------------------------------------------

improvement_records = []

for dataset_name, comparison_df in [
    ("Training OOF", rule_oof_comparison_df),
    ("Holdout", rule_holdout_comparison_df)
]:

    rule_rmse = comparison_df.loc[
        comparison_df["Model"] == "Rule-based",
        "RMSE_h"
    ].iloc[0]

    xgb_rmse = comparison_df.loc[
        comparison_df["Model"] == "XGBoost",
        "RMSE_h"
    ].iloc[0]

    rule_mae = comparison_df.loc[
        comparison_df["Model"] == "Rule-based",
        "MAE_h"
    ].iloc[0]

    xgb_mae = comparison_df.loc[
        comparison_df["Model"] == "XGBoost",
        "MAE_h"
    ].iloc[0]

    improvement_records.append({

        "Dataset": dataset_name,

        "Rule_RMSE_h": rule_rmse,

        "XGBoost_RMSE_h": xgb_rmse,

        "XGBoost_RMSE_Improvement_pct": (
            (rule_rmse - xgb_rmse) / rule_rmse
        ) * 100,

        "Rule_MAE_h": rule_mae,

        "XGBoost_MAE_h": xgb_mae,

        "XGBoost_MAE_Improvement_pct": (
            (rule_mae - xgb_mae) / rule_mae
        ) * 100
    })


rule_improvement_df = pd.DataFrame(
    improvement_records
)

print("\n" + "=" * 70)
print("XGBOOST IMPROVEMENT OVER RULE-BASED BENCHMARK")
print("=" * 70)

print(
    rule_improvement_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.2f}"
    )
)


# ------------------------------------------------------------
# 10. Save order-level benchmark predictions
# ------------------------------------------------------------

rule_oof_predictions_df = pd.DataFrame({

    "order_id": train_order_ids_final,

    "actual_lateness_h": y_train_rule,

    "rule_predicted_lateness_h": rule_train_predictions,

    "LR_nested_OOF_h": prediction_sets[
        "Training OOF"
    ]["Linear Regression"],

    "XGBoost_nested_OOF_h": prediction_sets[
        "Training OOF"
    ]["XGBoost"]
})


rule_holdout_predictions_df = pd.DataFrame({

    "order_id": test_order_ids_final,

    "actual_lateness_h": y_test_rule,

    "rule_predicted_lateness_h": rule_test_predictions,

    "LR_predicted_lateness_h": prediction_sets[
        "Holdout"
    ]["Linear Regression"],

    "XGBoost_predicted_lateness_h": prediction_sets[
        "Holdout"
    ]["XGBoost"]
})


# ------------------------------------------------------------
# 11. Completion summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CELL 30 COMPLETED")
print("=" * 70)

print("Generated outputs:")
print("- rule_train_results_df")
print("- rule_test_results_df")
print("- rule_oof_comparison_df")
print("- rule_holdout_comparison_df")
print("- rule_oof_wilcoxon_df")
print("- rule_holdout_wilcoxon_df")
print("- rule_summary_df")
print("- rule_improvement_df")
print("- rule_oof_predictions_df")
print("- rule_holdout_predictions_df")

print("\nNo model was retrained.")
print("No realized transportation outcomes were used as predictors.")


In [ ]:

# ============================================================
# CELL 30A — DISTRIBUTION-AWARE RULE-BASED BENCHMARK
# Uses known stochastic distributions from Cell 13R
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

from scipy.stats import wilcoxon


# ------------------------------------------------------------
# 1. Define simulation uncertainty parameters
# ------------------------------------------------------------

DISPATCH_WAIT_CV = 0.30
DRIVING_TIME_CV = 0.15

N_MONTE_CARLO = 50000
MONTE_CARLO_SEED = 2026
CHUNK_SIZE = 32

rng = np.random.default_rng(MONTE_CARLO_SEED)


# ------------------------------------------------------------
# 2. Generate mean-one lognormal shocks
# ------------------------------------------------------------

def generate_mean_one_lognormal(cv, size, rng):

    sigma_squared = np.log(1 + cv**2)

    sigma = np.sqrt(sigma_squared)

    mu = -0.5 * sigma_squared

    return rng.lognormal(
        mean=mu,
        sigma=sigma,
        size=size
    )


dispatch_shocks = generate_mean_one_lognormal(
    DISPATCH_WAIT_CV,
    N_MONTE_CARLO,
    rng
)

driving_shocks = generate_mean_one_lognormal(
    DRIVING_TIME_CV,
    N_MONTE_CARLO,
    rng
)


# ------------------------------------------------------------
# 3. Define distribution-aware lateness estimator
# ------------------------------------------------------------

def distribution_aware_prediction(X):

    availability = X[
        "forecast_vehicle_availability"
    ].to_numpy(dtype=float)

    expected_driving = X[
        "forecast_expected_travel_time_h"
    ].to_numpy(dtype=float)

    remaining_slack = X[
        "pre_dispatch_remaining_slack_h"
    ].to_numpy(dtype=float)

    if ((availability <= 0) | (availability > 1)).any():
        raise ValueError("Invalid vehicle availability.")

    n_orders = len(X)

    conditional_mean = np.empty(n_orders)
    conditional_median = np.empty(n_orders)
    probability_late = np.empty(n_orders)

    # Expected dispatch waiting time before random deviation
    reference_dispatch_wait = (
        8 * (1 - availability) / availability
    )

    # Use common random numbers across orders.
    # This improves consistency when comparing scenarios.

    for start in range(0, n_orders, CHUNK_SIZE):

        end = min(start + CHUNK_SIZE, n_orders)

        dispatch_base = reference_dispatch_wait[start:end]

        driving_base = expected_driving[start:end]

        slack = remaining_slack[start:end]

        # Simulate transportation outcomes conditional
        # on the information available before dispatch.
        actual_dispatch = (
            dispatch_base[:, None]
            * dispatch_shocks[None, :]
        )

        actual_driving = (
            driving_base[:, None]
            * driving_shocks[None, :]
        )

        simulated_transportation = (
            actual_dispatch + actual_driving
        )

        simulated_lateness = np.maximum(
            0,
            simulated_transportation - slack[:, None]
        )

        conditional_mean[start:end] = (
            simulated_lateness.mean(axis=1)
        )

        conditional_median[start:end] = (
            np.median(simulated_lateness, axis=1)
        )

        probability_late[start:end] = (
            simulated_lateness > 0
        ).mean(axis=1)

    return pd.DataFrame({

        "Conditional_Mean_Lateness_h": conditional_mean,

        "Conditional_Median_Lateness_h": conditional_median,

        "Probability_Late": probability_late,

        "Reference_Dispatch_Wait_h": reference_dispatch_wait,

        "Forecast_Driving_Time_h": expected_driving,

        "Remaining_Slack_h": remaining_slack

    })


# ------------------------------------------------------------
# 4. Estimate training and holdout conditional distributions
# ------------------------------------------------------------

print("=" * 70)
print("CELL 30A — DISTRIBUTION-AWARE BENCHMARK")
print("=" * 70)

print("Monte Carlo simulations per order:", N_MONTE_CARLO)

distribution_train_df = distribution_aware_prediction(
    X_final_train
)

distribution_test_df = distribution_aware_prediction(
    X_final_test
)


# ------------------------------------------------------------
# 5. Define prediction sets for comparison
# ------------------------------------------------------------

training_predictions = {

    "Plug-in rule": rule_train_predictions,

    "Distribution-aware conditional mean":
        distribution_train_df[
            "Conditional_Mean_Lateness_h"
        ].to_numpy(),

    "Distribution-aware conditional median":
        distribution_train_df[
            "Conditional_Median_Lateness_h"
        ].to_numpy(),

    "Linear Regression": np.maximum(
        oof_predictions_final, 0
    ),

    "XGBoost": np.maximum(
        nested_xgb_oof_predictions, 0
    )
}

holdout_predictions = {

    "Plug-in rule": rule_test_predictions,

    "Distribution-aware conditional mean":
        distribution_test_df[
            "Conditional_Mean_Lateness_h"
        ].to_numpy(),

    "Distribution-aware conditional median":
        distribution_test_df[
            "Conditional_Median_Lateness_h"
        ].to_numpy(),

    "Linear Regression": np.maximum(
        final_lr_model.predict(X_final_test), 0
    ),

    "XGBoost": np.maximum(
        final_xgb_model.predict(X_final_test), 0
    )
}


# ------------------------------------------------------------
# 6. Evaluate regression performance
# ------------------------------------------------------------

def evaluate_distribution_benchmark(actual, predictions):

    records = []

    late_mask = actual > 0

    for model_name, predicted in predictions.items():

        records.append({

            "Model": model_name,

            "RMSE_h": np.sqrt(
                mean_squared_error(actual, predicted)
            ),

            "MAE_h": mean_absolute_error(
                actual, predicted
            ),

            "R2": r2_score(
                actual, predicted
            ),

            "Late_Order_RMSE_h": np.sqrt(
                mean_squared_error(
                    actual[late_mask],
                    predicted[late_mask]
                )
            ),

            "Late_Order_MAE_h": mean_absolute_error(
                actual[late_mask],
                predicted[late_mask]
            )
        })

    return pd.DataFrame(records).sort_values(
        "RMSE_h"
    ).reset_index(drop=True)


distribution_train_comparison_df = (
    evaluate_distribution_benchmark(
        y_final_train.to_numpy(dtype=float),
        training_predictions
    )
)

distribution_holdout_comparison_df = (
    evaluate_distribution_benchmark(
        y_final_test.to_numpy(dtype=float),
        holdout_predictions
    )
)


# ------------------------------------------------------------
# 7. Evaluate classification using probability-based threshold
# ------------------------------------------------------------

def evaluate_probability_classification(
    actual,
    prediction_dict,
    probability_late
):

    actual_class = (actual > 0).astype(int)

    classification_predictions = {
        "Plug-in rule": (
            prediction_dict["Plug-in rule"] > 0
        ).astype(int),

        "Distribution-aware probability rule": (
            probability_late > 0.50
        ).astype(int),

        "Linear Regression": (
            prediction_dict["Linear Regression"] > 0
        ).astype(int),

        "XGBoost": (
            prediction_dict["XGBoost"] > 0
        ).astype(int)
    }

    records = []

    for name, predicted_class in classification_predictions.items():

        tn, fp, fn, tp = confusion_matrix(
            actual_class,
            predicted_class,
            labels=[0, 1]
        ).ravel()

        records.append({

            "Model": name,

            "Accuracy": accuracy_score(
                actual_class,
                predicted_class
            ),

            "Balanced_Accuracy": balanced_accuracy_score(
                actual_class,
                predicted_class
            ),

            "Precision": precision_score(
                actual_class,
                predicted_class,
                zero_division=0
            ),

            "Recall": recall_score(
                actual_class,
                predicted_class,
                zero_division=0
            ),

            "F1": f1_score(
                actual_class,
                predicted_class,
                zero_division=0
            ),

            "TN": tn,
            "FP": fp,
            "FN": fn,
            "TP": tp
        })

    return pd.DataFrame(records)


distribution_train_classification_df = (
    evaluate_probability_classification(

        y_final_train.to_numpy(dtype=float),

        training_predictions,

        distribution_train_df[
            "Probability_Late"
        ].to_numpy()
    )
)

distribution_holdout_classification_df = (
    evaluate_probability_classification(

        y_final_test.to_numpy(dtype=float),

        holdout_predictions,

        distribution_test_df[
            "Probability_Late"
        ].to_numpy()
    )
)


# ------------------------------------------------------------
# 8. Paired comparisons using absolute errors
# ------------------------------------------------------------

def paired_distribution_tests(actual, predictions):

    comparisons = [

        (
            "XGBoost",
            "Plug-in rule"
        ),

        (
            "XGBoost",
            "Distribution-aware conditional mean"
        ),

        (
            "Plug-in rule",
            "Distribution-aware conditional mean"
        )
    ]

    records = []

    for model_a, model_b in comparisons:

        error_a = np.abs(
            predictions[model_a] - actual
        )

        error_b = np.abs(
            predictions[model_b] - actual
        )

        result = wilcoxon(
            error_a,
            error_b,
            alternative="two-sided",
            zero_method="wilcox",
            method="auto"
        )

        records.append({

            "Model_A": model_a,

            "Model_B": model_b,

            "MAE_A": error_a.mean(),

            "MAE_B": error_b.mean(),

            "Wilcoxon_statistic": result.statistic,

            "P_value": result.pvalue
        })

    result_df = pd.DataFrame(records)

    p_values = result_df["P_value"].to_numpy()

    m = len(p_values)

    order = np.argsort(p_values)

    sorted_p = p_values[order]

    adjusted_sorted = np.maximum.accumulate(
        (m - np.arange(m)) * sorted_p
    )

    adjusted_sorted = np.minimum(
        adjusted_sorted, 1.0
    )

    adjusted = np.empty(m)

    adjusted[order] = adjusted_sorted

    result_df["Holm_adjusted_p"] = adjusted

    result_df["Significant_at_0.05"] = (
        adjusted < 0.05
    )

    return result_df


distribution_train_wilcoxon_df = (
    paired_distribution_tests(
        y_final_train.to_numpy(dtype=float),
        training_predictions
    )
)

distribution_holdout_wilcoxon_df = (
    paired_distribution_tests(
        y_final_test.to_numpy(dtype=float),
        holdout_predictions
    )
)


# ------------------------------------------------------------
# 9. Display results
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("TRAINING — DISTRIBUTION-AWARE BENCHMARK")
print("=" * 70)

print(
    distribution_train_comparison_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


print("\n" + "=" * 70)
print("HOLDOUT — DISTRIBUTION-AWARE BENCHMARK")
print("=" * 70)

print(
    distribution_holdout_comparison_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


print("\n" + "=" * 70)
print("HOLDOUT — PROBABILITY-BASED CLASSIFICATION")
print("=" * 70)

print(
    distribution_holdout_classification_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


print("\n" + "=" * 70)
print("HOLDOUT — PAIRED WILCOXON TESTS")
print("=" * 70)

print(
    distribution_holdout_wilcoxon_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6g}"
    )
)


# ------------------------------------------------------------
# 10. Save benchmark prediction audit
# ------------------------------------------------------------

distribution_benchmark_holdout_df = pd.DataFrame({

    "order_id": test_order_ids_final,

    "actual_lateness_h": y_final_test.to_numpy(),

    "plug_in_rule_h": rule_test_predictions,

    "conditional_mean_lateness_h":
        distribution_test_df[
            "Conditional_Mean_Lateness_h"
        ].to_numpy(),

    "conditional_median_lateness_h":
        distribution_test_df[
            "Conditional_Median_Lateness_h"
        ].to_numpy(),

    "probability_late":
        distribution_test_df[
            "Probability_Late"
        ].to_numpy(),

    "xgboost_prediction_h":
        holdout_predictions["XGBoost"],

    "linear_regression_prediction_h":
        holdout_predictions["Linear Regression"]

})


# ------------------------------------------------------------
# 11. Completion summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CELL 30A COMPLETED")
print("=" * 70)

print("Generated outputs:")
print("- distribution_train_df")
print("- distribution_test_df")
print("- distribution_train_comparison_df")
print("- distribution_holdout_comparison_df")
print("- distribution_holdout_classification_df")
print("- distribution_holdout_wilcoxon_df")
print("- distribution_benchmark_holdout_df")

print("\nKnown simulation distributions were used.")
print("No realized future transportation values were used as predictors.")


## 7. Publication tables and six figures

In [ ]:

# ============================================================
# CELL 29 — FINAL RESULTS TABLES AND PUBLICATION FIGURES
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
import json


# ------------------------------------------------------------
# 1. Validate required outputs
# ------------------------------------------------------------

required_objects = [
    "nested_comparison_df",
    "final_holdout_performance_df",
    "final_late_performance_df",
    "final_classification_performance_df",
    "holdout_wilcoxon_df",
    "final_xgb_feature_importance_df",
    "final_xgb_group_importance_df",
    "final_holdout_predictions_df",
    "nested_xgb_fold_df",
    "final_lr_cv_results_df",
    "final_xgb_search",
    "final_xgb_source_shap_df",
    "X_final_test"
]

for obj in required_objects:

    if obj not in globals():
        raise ValueError(
            f"Required results object is missing: {obj}"
        )


# ------------------------------------------------------------
# 2. Create output directories
# ------------------------------------------------------------

output_directory = Path("Final_Publication_Results")

figure_directory = output_directory / "Figures"

table_directory = output_directory / "Tables"

figure_directory.mkdir(
    parents=True,
    exist_ok=True
)

table_directory.mkdir(
    parents=True,
    exist_ok=True
)

print("=" * 70)
print("CELL 29 — FINAL RESULTS AND FIGURES")
print("=" * 70)


# ------------------------------------------------------------
# 3. Table 1 — Nested cross-validation results
# ------------------------------------------------------------

table_nested_cv_df = nested_comparison_df.copy()

table_nested_cv_df = table_nested_cv_df.rename(columns={

    "Model": "Model",

    "Raw_RMSE": "RMSE_h",

    "Raw_MAE": "MAE_h",

    "Raw_R2": "R2",

    "Constrained_RMSE": "Constrained_RMSE_h",

    "Constrained_MAE": "Constrained_MAE_h",

    "Constrained_R2": "Constrained_R2"
})


# Add fold-level RMSE standard deviations
cv_sd_records = [

    {
        "Model": "Linear Regression",

        "RMSE_SD": final_lr_cv_results_df[
            "Raw_RMSE"
        ].std(ddof=1),

        "MAE_SD": final_lr_cv_results_df[
            "Raw_MAE"
        ].std(ddof=1)
    },

    {
        "Model": "XGBoost",

        "RMSE_SD": nested_xgb_fold_df[
            "XGBoost_RMSE"
        ].std(ddof=1),

        "MAE_SD": nested_xgb_fold_df[
            "XGBoost_MAE"
        ].std(ddof=1)
    }
]

table_cv_sd_df = pd.DataFrame(cv_sd_records)

table_nested_cv_df = table_nested_cv_df.merge(
    table_cv_sd_df,
    on="Model",
    how="left"
)

table_nested_cv_df = table_nested_cv_df[
    [
        "Model",
        "Raw_RMSE" if "Raw_RMSE" in table_nested_cv_df.columns else "RMSE_h",
        "RMSE_SD",
        "Raw_MAE" if "Raw_MAE" in table_nested_cv_df.columns else "MAE_h",
        "MAE_SD",
        "Raw_R2" if "Raw_R2" in table_nested_cv_df.columns else "R2"
    ]
].copy()

# Standardize final column labels
table_nested_cv_df.columns = [
    "Model",
    "Nested_CV_RMSE_h",
    "RMSE_Fold_SD",
    "Nested_CV_MAE_h",
    "MAE_Fold_SD",
    "Nested_CV_R2"
]


# ------------------------------------------------------------
# 4. Table 2 — Holdout regression performance
# ------------------------------------------------------------

table_holdout_df = final_holdout_performance_df[
    [
        "Model",
        "Raw_RMSE_h",
        "Raw_MAE_h",
        "Raw_R2",
        "Constrained_RMSE_h",
        "Constrained_MAE_h",
        "Constrained_R2",
        "Constrained_Mean_Error_h",
        "Negative_Raw_Predictions"
    ]
].copy()


# ------------------------------------------------------------
# 5. Table 3 — Performance on genuinely late orders
# ------------------------------------------------------------

table_late_orders_df = final_late_performance_df.copy()


# ------------------------------------------------------------
# 6. Table 4 — On-time / late classification performance
# ------------------------------------------------------------

table_classification_df = (
    final_classification_performance_df.copy()
)


# ------------------------------------------------------------
# 7. Table 5 — Paired Wilcoxon comparison
# ------------------------------------------------------------

table_wilcoxon_df = holdout_wilcoxon_df.copy()

# Add an appropriate reporting interpretation.
table_wilcoxon_df["Statistical_Conclusion"] = (
    table_wilcoxon_df["P_value"].apply(
        lambda p: (
            "p < 0.001"
            if p < 0.001
            else f"p = {p:.4f}"
        )
    )
)


# ------------------------------------------------------------
# 8. Table 6 — XGBoost global feature importance
# ------------------------------------------------------------

table_shap_features_df = (
    final_xgb_feature_importance_df.copy()
)

table_shap_features_df.insert(
    0,
    "Rank",
    np.arange(1, len(table_shap_features_df) + 1)
)


# ------------------------------------------------------------
# 9. Table 7 — XGBoost grouped operational contributions
# ------------------------------------------------------------

table_shap_groups_df = (
    final_xgb_group_importance_df.copy()
)


# ------------------------------------------------------------
# 10. Table 8 — Final XGBoost hyperparameters
# ------------------------------------------------------------

table_xgb_parameters_df = pd.DataFrame([

    {
        "Parameter": parameter,

        "Selected_Value": json.dumps(
            value.item() if isinstance(value, np.generic) else value
        )

    }

    for parameter, value
    in final_xgb_search.best_params_.items()

])


# ------------------------------------------------------------
# 11. Table 9 — Nested cross-validation fold results
# ------------------------------------------------------------

table_fold_results_df = nested_fold_comparison_df.copy()


# ------------------------------------------------------------
# 12. Create combined manuscript comparison table
# ------------------------------------------------------------

table_combined_performance_df = (

    table_nested_cv_df[
        [
            "Model",
            "Nested_CV_RMSE_h",
            "Nested_CV_MAE_h",
            "Nested_CV_R2"
        ]
    ]

    .merge(
        table_holdout_df[
            [
                "Model",
                "Raw_RMSE_h",
                "Raw_MAE_h",
                "Raw_R2"
            ]
        ],
        on="Model",
        how="inner"
    )

    .merge(
        table_late_orders_df[
            [
                "Model",
                "RMSE_h",
                "MAE_h"
            ]
        ].rename(columns={
            "RMSE_h": "Late_Order_RMSE_h",
            "MAE_h": "Late_Order_MAE_h"
        }),
        on="Model",
        how="inner"
    )
)


# ------------------------------------------------------------
# 13. Display final tables
# ------------------------------------------------------------

tables_to_display = {

    "NESTED CROSS-VALIDATION": table_nested_cv_df,

    "HOLDOUT REGRESSION PERFORMANCE": table_holdout_df,

    "LATE-ORDER PERFORMANCE": table_late_orders_df,

    "CLASSIFICATION PERFORMANCE": table_classification_df,

    "WILCOXON COMPARISON": table_wilcoxon_df,

    "GLOBAL SHAP IMPORTANCE": table_shap_features_df,

    "GROUPED SHAP CONTRIBUTIONS": table_shap_groups_df,

    "FINAL XGBOOST PARAMETERS": table_xgb_parameters_df,

    "COMBINED MANUSCRIPT COMPARISON": table_combined_performance_df
}

for title, dataframe in tables_to_display.items():

    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)

    print(
        dataframe.round(4).to_string(index=False)
        if all(
            pd.api.types.is_numeric_dtype(dtype)
            for dtype in dataframe.dtypes
        )
        else dataframe.to_string(index=False)
    )


# ------------------------------------------------------------
# 14. Export tables to Excel
# ------------------------------------------------------------

excel_path = (
    table_directory / "Final_Model_Results_Tables.xlsx"
)

with pd.ExcelWriter(
    excel_path,
    engine="openpyxl"
) as writer:

    table_nested_cv_df.to_excel(
        writer,
        sheet_name="Nested_CV",
        index=False
    )

    table_holdout_df.to_excel(
        writer,
        sheet_name="Holdout_Performance",
        index=False
    )

    table_late_orders_df.to_excel(
        writer,
        sheet_name="Late_Orders",
        index=False
    )

    table_classification_df.to_excel(
        writer,
        sheet_name="Classification",
        index=False
    )

    table_wilcoxon_df.to_excel(
        writer,
        sheet_name="Wilcoxon",
        index=False
    )

    table_shap_features_df.to_excel(
        writer,
        sheet_name="SHAP_Features",
        index=False
    )

    table_shap_groups_df.to_excel(
        writer,
        sheet_name="SHAP_Groups",
        index=False
    )

    table_xgb_parameters_df.to_excel(
        writer,
        sheet_name="XGBoost_Parameters",
        index=False
    )

    table_fold_results_df.to_excel(
        writer,
        sheet_name="Nested_Folds",
        index=False
    )

    table_combined_performance_df.to_excel(
        writer,
        sheet_name="Combined_Results",
        index=False
    )


# ------------------------------------------------------------
# 15. Figure 1 — Nested CV model comparison
# ------------------------------------------------------------

fig, axes = plt.subplots(
    1,
    2,
    figsize=(12, 5)
)

model_order = [
    "Linear Regression",
    "XGBoost"
]

cv_plot_data = (
    table_nested_cv_df.set_index("Model")
    .loc[model_order]
)

axes[0].bar(
    model_order,
    cv_plot_data["Nested_CV_RMSE_h"]
)

axes[0].set_title("Nested CV RMSE")
axes[0].set_ylabel("RMSE (hours)")
axes[0].tick_params(axis="x", rotation=15)

axes[1].bar(
    model_order,
    cv_plot_data["Nested_CV_MAE_h"]
)

axes[1].set_title("Nested CV MAE")
axes[1].set_ylabel("MAE (hours)")
axes[1].tick_params(axis="x", rotation=15)

fig.suptitle(
    "Predictive Performance under Nested Cross-Validation"
)

fig.tight_layout()

fig.savefig(
    figure_directory / "Figure_1_Nested_CV_Comparison.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)


# ------------------------------------------------------------
# 16. Figure 2 — Observed versus predicted holdout lateness
# ------------------------------------------------------------

actual = final_holdout_predictions_df[
    "actual_lateness_h"
].to_numpy()

lr_predictions = final_holdout_predictions_df[
    "LR_constrained_prediction_h"
].to_numpy()

xgb_predictions = final_holdout_predictions_df[
    "XGBoost_constrained_prediction_h"
].to_numpy()

maximum_value = max(
    actual.max(),
    lr_predictions.max(),
    xgb_predictions.max()
) * 1.05

fig, axes = plt.subplots(
    1,
    2,
    figsize=(12, 5)
)

axes[0].scatter(
    actual,
    lr_predictions,
    alpha=0.60,
    s=24
)

axes[0].plot(
    [0, maximum_value],
    [0, maximum_value],
    linestyle="--"
)

axes[0].set_title("Linear Regression")
axes[0].set_xlabel("Actual lateness (h)")
axes[0].set_ylabel("Predicted lateness (h)")
axes[0].set_xlim(0, maximum_value)
axes[0].set_ylim(0, maximum_value)


axes[1].scatter(
    actual,
    xgb_predictions,
    alpha=0.60,
    s=24
)

axes[1].plot(
    [0, maximum_value],
    [0, maximum_value],
    linestyle="--"
)

axes[1].set_title("XGBoost")
axes[1].set_xlabel("Actual lateness (h)")
axes[1].set_ylabel("Predicted lateness (h)")
axes[1].set_xlim(0, maximum_value)
axes[1].set_ylim(0, maximum_value)

fig.suptitle(
    "Observed versus Predicted Delivery Lateness"
)

fig.tight_layout()

fig.savefig(
    figure_directory / "Figure_2_Observed_vs_Predicted.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)


# ------------------------------------------------------------
# 17. Figure 3 — XGBoost holdout residual diagnostics
# ------------------------------------------------------------

xgb_residuals = (
    xgb_predictions - actual
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(12, 5)
)

axes[0].scatter(
    xgb_predictions,
    xgb_residuals,
    alpha=0.60,
    s=24
)

axes[0].axhline(
    y=0,
    linestyle="--"
)

axes[0].set_xlabel("Predicted lateness (h)")
axes[0].set_ylabel("Residual (predicted - actual, h)")
axes[0].set_title("Residuals versus predictions")


axes[1].hist(
    xgb_residuals,
    bins=25
)

axes[1].axvline(
    xgb_residuals.mean(),
    linestyle="--",
    label="Mean residual"
)

axes[1].set_xlabel("Residual (h)")
axes[1].set_ylabel("Frequency")
axes[1].set_title("Residual distribution")
axes[1].legend()

fig.suptitle(
    "XGBoost Holdout Residual Diagnostics"
)

fig.tight_layout()

fig.savefig(
    figure_directory / "Figure_3_XGBoost_Residuals.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)


# ------------------------------------------------------------
# 18. Figure 4 — XGBoost global SHAP importance
# ------------------------------------------------------------

plot_features = (
    final_xgb_feature_importance_df
    .sort_values("Mean_Absolute_SHAP_h")
)

fig, ax = plt.subplots(figsize=(10, 7))

ax.barh(
    plot_features["Feature"],
    plot_features["Mean_Absolute_SHAP_h"]
)

ax.set_xlabel(
    "Mean absolute SHAP contribution (hours)"
)

ax.set_ylabel("Predictor")

ax.set_title(
    "Global Importance of Operational Predictors — XGBoost"
)

fig.tight_layout()

fig.savefig(
    figure_directory / "Figure_4_XGBoost_SHAP_Importance.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)


# ------------------------------------------------------------
# 19. Figure 5 — Grouped SHAP contributions
# ------------------------------------------------------------

plot_groups = (
    final_xgb_group_importance_df
    .sort_values("Mean_Absolute_Group_SHAP_h")
)

fig, ax = plt.subplots(figsize=(10, 6))

ax.barh(
    plot_groups["Operational_Group"],
    plot_groups["Mean_Absolute_Group_SHAP_h"]
)

ax.set_xlabel(
    "Mean absolute grouped SHAP contribution (hours)"
)

ax.set_ylabel("Operational group")

ax.set_title(
    "Operational Group Contributions — XGBoost"
)

fig.tight_layout()

fig.savefig(
    figure_directory / "Figure_5_XGBoost_Grouped_SHAP.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)


# ------------------------------------------------------------
# 20. Figure 6 — SHAP dependence for three leading predictors
# ------------------------------------------------------------

dependence_features = [

    "pre_dispatch_remaining_slack_h",

    "forecast_vehicle_availability",

    "forecast_expected_travel_time_h"
]

fig, axes = plt.subplots(
    1,
    3,
    figsize=(16, 5)
)

for ax, feature in zip(
    axes,
    dependence_features
):

    x_values = pd.to_numeric(
        X_final_test[feature],
        errors="coerce"
    ).to_numpy()

    y_values = (
        final_xgb_source_shap_df[feature].to_numpy()
    )

    valid = (
        np.isfinite(x_values)
        & np.isfinite(y_values)
    )

    ax.scatter(
        x_values[valid],
        y_values[valid],
        alpha=0.60,
        s=22
    )

    ax.axhline(
        y=0,
        linestyle="--"
    )

    ax.set_xlabel(feature)

    ax.set_ylabel("SHAP contribution (h)")

    ax.set_title(feature.replace("_", " ").title())


fig.suptitle(
    "SHAP Dependence of the Leading Predictors"
)

fig.tight_layout()

fig.savefig(
    figure_directory / "Figure_6_XGBoost_SHAP_Dependence.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)


# ------------------------------------------------------------
# 21. Save SHAP tables separately as CSV
# ------------------------------------------------------------

table_shap_features_df.to_csv(
    table_directory / "XGBoost_SHAP_Feature_Importance.csv",
    index=False
)

table_shap_groups_df.to_csv(
    table_directory / "XGBoost_SHAP_Group_Importance.csv",
    index=False
)

table_combined_performance_df.to_csv(
    table_directory / "Final_Model_Comparison.csv",
    index=False
)


# ------------------------------------------------------------
# 22. Final report
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CELL 29 COMPLETED")
print("=" * 70)

print("\nExcel workbook:")
print(excel_path.resolve())

print("\nFigures saved in:")
print(figure_directory.resolve())

print("\nTables saved in:")
print(table_directory.resolve())

print("\nPublication figures:")
for file in sorted(figure_directory.glob("*.png")):
    print("-", file.name)

print("\nFinal consolidated results generated successfully.")


### 7.1 Export process-informed benchmark results

In [ ]:
# Save benchmark results in a separate workbook so all comparisons are preserved.
from pathlib import Path
import pandas as pd

benchmark_workbook = table_directory / "Process_Informed_Benchmark_Results.xlsx"
with pd.ExcelWriter(benchmark_workbook, engine="openpyxl") as writer:
    rule_oof_comparison_df.to_excel(writer, sheet_name="Rule_Training_OOF", index=False)
    rule_holdout_comparison_df.to_excel(writer, sheet_name="Rule_Holdout", index=False)
    distribution_train_comparison_df.to_excel(writer, sheet_name="Distribution_Training", index=False)
    distribution_holdout_comparison_df.to_excel(writer, sheet_name="Distribution_Holdout", index=False)
    distribution_holdout_classification_df.to_excel(writer, sheet_name="Probability_Classification", index=False)
    distribution_holdout_wilcoxon_df.to_excel(writer, sheet_name="Benchmark_Wilcoxon", index=False)
print("Benchmark workbook saved:", benchmark_workbook.resolve())

## 8. Reproducibility record

The final execution should be performed from a fresh kernel. Record the Python and package versions, execution date, random seed, observation counts, feature names, fitted hyperparameters, validation metrics, benchmark metrics, SHAP additivity diagnostics, and output-file inventory. A successful run confirms computational reproducibility under the stated simulation assumptions; it does not establish empirical validity against real delivery records. Holdout-based paired significance tests should be treated as exploratory because the holdout set has been inspected during model development.